# Modelamiento de Grafos aplicado a la detección de fraude

### Del dato tabular al Graph Machine Learning — 24 módulos sobre el *Elliptic Bitcoin Dataset*

---

Este notebook recorre el ciclo completo del análisis de grafos: qué es un grafo, cómo se
modela a partir de tablas, cómo se construye, cómo se mide, cómo se aprende de él y cómo
se lleva a producción. Cada módulo combina **teoría**, **funciones reutilizables** y
**resultados sobre datos reales**.

## Los dos grafos de trabajo

Trabajamos con dos grafos deliberadamente distintos, porque ninguno de los dos basta por sí solo:

| | **Grafo A — Elliptic** | **Grafo B — Banca sintética** |
|---|---|---|
| Origen | Kaggle `ellipticco/elliptic-data-set` (real) | Generado en el Módulo 1 (sintético, semilla fija) |
| Escala | 203.769 nodos · 234.355 aristas | ~6.000 nodos · ~12.000 aristas |
| Tipo | Homogéneo, dirigido, acíclico, temporal | **Heterogéneo**, dirigido, multigrafo, con atributos |
| Nodos | Transacciones de Bitcoin | Cliente, Cuenta, Tarjeta, Comercio, IP, Dispositivo, ATM… |
| Etiquetas | 46.564 transacciones marcadas lícito/ilícito | Ground truth de 6 patrones de fraude inyectados |
| Para qué sirve | Escala real, métricas, ML, GNN, análisis temporal | Modelado de entidades, tipología de fraude, Cypher, detectores |

Elliptic aporta **realismo y escala**; el grafo bancario aporta **riqueza semántica**. Los
Módulos 1, 18 y 19 no se pueden demostrar con Elliptic porque su grafo es homogéneo:
solo existe el tipo de nodo «transacción» y la relación «flujo de fondos». Ahí entra el Grafo B.

## Mapa de módulos

| | Módulo | Grafo | Entregable principal |
|---|---|---|---|
| 0 | Introducción a grafos | juguete | Tipología completa de grafos, grafo vs. relacional |
| 1 | Modelado de datos | A + B | `tabular_a_grafo`, `build_banking_graph`, Property Graph / RDF / KG |
| 2 | Construcción del grafo | A + B | `construir_grafo_elliptic`, subgrafos, muestreo |
| 3 | Estadísticas descriptivas | A | Densidad, grados, ley de potencias, hubs |
| 4 | Componentes | A + B | WCC / SCC / componente gigante |
| 5 | Caminos | A | Dijkstra, Bellman-Ford, Floyd-Warshall, A\*, diámetro |
| 6 | Centralidad | A | 8 centralidades en una sola tabla comparada |
| 7 | Comunidades | A | Louvain, Leiden, Girvan-Newman, Label Propagation, Infomap |
| 8 | Motifs | A + B | Triángulos, estrellas, cadenas, ciclos, feed-forward loop |
| 9 | Clustering | A | Coeficiente local/global, cierre triádico |
| 10 | Redes temporales | A | 49 snapshots, evolución y ciclo de vida de comunidades |
| 11 | Ingeniería de features | A | Tabla ML con features topológicas y de vecindad |
| 12 | Link prediction | A | Jaccard, Adamic-Adar, RA, Preferential Attachment |
| 13 | Embeddings | A | SVD/HOPE, DeepWalk, Node2Vec, LINE + PCA/t-SNE |
| 14 | Detección de anomalías | A | OddBall, Isolation Forest, autoencoder, DOMINANT |
| 15 | Graph Machine Learning | A | Clasificación de nodos, aristas y grafos |
| 16 | Graph Neural Networks | A + B | GCN, GraphSAGE, GAT, GIN, RGCN en PyTorch puro |
| 17 | Visualización | A + B | 5 layouts, Plotly, PyVis, export a Gephi/Cytoscape |
| 18 | Fraude bancario | B | Modelado de 12 entidades y 6 relaciones |
| 19 | Casos de uso | B | 6 detectores validados contra ground truth |
| 20 | Bases de datos de grafos | B | Comparativa + export a Neo4j |
| 21 | Lenguajes de consulta | B | La misma consulta en Cypher, Gremlin y SPARQL |
| 22 | Escalabilidad | A | Benchmark NetworkX vs igraph, patrones dispersos |
| 23 | Investigación (SOTA) | B | TGN, Graph Transformers, KG embeddings, XAI |

## Cómo ejecutar

1. **Google Colab** (recomendado): `Entorno de ejecución → Ejecutar todo`. La celda de
   dependencias instala solo lo que falte y los datos se descargan con `kagglehub`.
2. **Local**: `pip install -r requirements.txt` y luego ejecutar de arriba a abajo. Si ya
   tienes los CSV, colócalos en `./data/` y se usarán sin descargar nada.

> **Orden de ejecución**: el notebook está diseñado para correr **linealmente**. Cada módulo
> reutiliza objetos definidos en los anteriores (`G`, `df_nodos`, `snapshots`…). Ejecutar
> celdas sueltas fuera de orden producirá `NameError`.

> **Coste computacional**: las celdas marcadas con ⏱️ tardan más de un minuto sobre el grafo
> completo. Cada una documenta por qué y qué aproximación se usa en su lugar.

---

## Configuración del entorno

El notebook es **portable**: funciona igual en Colab que en local. La estrategia es no instalar
nada que ya esté disponible y **degradar con elegancia** cuando falta una librería opcional.

Cada dependencia opcional levanta una bandera (`TORCH_OK`, `GENSIM_OK`…) y las celdas que la
necesitan comprueban la bandera antes de ejecutarse. Así, un entorno mínimo con
`networkx + pandas + numpy + scipy + matplotlib + scikit-learn` recorre el notebook completo
sin lanzar una sola excepción; simplemente se salta las demostraciones que no puede hacer.

In [ ]:
# =============================================================================
# Instalación portable de dependencias
# =============================================================================
import importlib.util
import subprocess
import sys

# Paquetes imprescindibles: si falta alguno, se instala.
BASE = {
    "networkx": "networkx>=3.1",
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "kagglehub": "kagglehub",
}

# Paquetes opcionales: NO se instalan automáticamente (torch pesa ~2 GB y en Colab ya viene).
# Cambia INSTALAR_OPCIONALES a True si quieres cobertura total en un entorno local.
OPCIONALES = {
    "torch": "torch",             # Módulos 14 y 16 (GNN, autoencoder)
    "gensim": "gensim",           # Módulo 13 (Word2Vec para DeepWalk/Node2Vec)
    "plotly": "plotly",           # Módulo 17 (visualización interactiva)
    "pyvis": "pyvis",             # Módulo 17 (grafo interactivo en HTML)
    "igraph": "python-igraph",    # Módulo 22 (benchmark de escalabilidad)
    "leidenalg": "leidenalg",     # Módulo 7 (algoritmo de Leiden)
}
INSTALAR_OPCIONALES = False


def _disponible(modulo: str) -> bool:
    """True si el módulo está instalado.

    `find_spec` lanza ModuleNotFoundError cuando el paquete *padre* de un nombre
    con puntos no existe (p. ej. "google.colab" fuera de Colab), así que la
    excepción se captura y se traduce a False.
    """
    try:
        return importlib.util.find_spec(modulo) is not None
    except (ImportError, ValueError):
        return False


def _instalar(spec: str) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", spec],
        stdout=subprocess.DEVNULL,
    )


EN_COLAB = _disponible("google.colab")
print(f"Entorno detectado : {'Google Colab' if EN_COLAB else 'Local / Jupyter'}")
print(f"Python            : {sys.version.split()[0]}\n")

faltantes = [spec for mod, spec in BASE.items() if not _disponible(mod)]
if faltantes:
    print(f"Instalando {len(faltantes)} paquete(s) base: {', '.join(faltantes)}")
    for spec in faltantes:
        _instalar(spec)
    print("Instalación completada.\n")
else:
    print("Todas las dependencias base ya están disponibles.\n")

if INSTALAR_OPCIONALES:
    for mod, spec in OPCIONALES.items():
        if not _disponible(mod):
            print(f"Instalando opcional: {spec}")
            _instalar(spec)

print("Dependencias opcionales:")
for mod in OPCIONALES:
    print(f"  {'[OK]     ' if _disponible(mod) else '[ausente]'} {mod}")

In [ ]:
# =============================================================================
# Imports globales, banderas de capacidad y configuración de gráficos
# =============================================================================
import itertools
import json
import math
import os
import random
import time
import warnings
from collections import Counter, defaultdict, deque
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# `display` existe como builtin solo dentro de IPython/Jupyter. Lo importamos de forma
# explícita para que el notebook también funcione ejecutado como script (defecto del
# notebook original: usaba display() sin importarlo).
try:
    from IPython.display import display
except ImportError:  # pragma: no cover - ejecución fuera de IPython
    display = print

# Banderas de capacidad: cada celda opcional las consulta antes de ejecutarse.
TORCH_OK = _disponible("torch")
GENSIM_OK = _disponible("gensim")
PLOTLY_OK = _disponible("plotly")
PYVIS_OK = _disponible("pyvis")
IGRAPH_OK = _disponible("igraph")
LEIDEN_OK = _disponible("leidenalg") and IGRAPH_OK

# Reproducibilidad. Todo algoritmo estocástico del notebook recibe esta semilla.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Estilo visual único para las ~40 figuras del notebook.
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})

# Paleta consistente: un color por rol semántico, no por capricho.
COLORES = {
    "licito": "#2E86AB",
    "ilicito": "#D7263D",
    "desconocido": "#B0B7BE",
    "acento": "#F18F01",
    "neutro": "#4A5568",
    "ok": "#2A9D8F",
}

print(f"NetworkX {nx.__version__} | pandas {pd.__version__} | NumPy {np.__version__}")
print(f"Capacidades -> torch={TORCH_OK} gensim={GENSIM_OK} plotly={PLOTLY_OK} "
      f"pyvis={PYVIS_OK} igraph={IGRAPH_OK} leiden={LEIDEN_OK}")

In [ ]:
# =============================================================================
# Utilidades transversales usadas por todos los módulos
# =============================================================================

def titulo(texto: str, nivel: int = 1) -> None:
    """Imprime un encabezado legible en la salida de la celda."""
    ancho = 78
    if nivel == 1:
        print("=" * ancho)
        print(texto.upper())
        print("=" * ancho)
    else:
        print(f"\n{texto}")
        print("-" * min(len(texto), ancho))


def cronometrar(func, *args, etiqueta: str | None = None, **kwargs):
    """Ejecuta `func` midiendo el tiempo de pared.

    Devuelve `(resultado, segundos)`. Se usa en todo el notebook para hacer
    explícito el coste de cada algoritmo sobre un grafo de 200 mil nodos.
    """
    nombre = etiqueta or getattr(func, "__name__", "función")
    t0 = time.perf_counter()
    resultado = func(*args, **kwargs)
    dt = time.perf_counter() - t0
    print(f"  · {nombre}: {dt:.2f} s")
    return resultado, dt


def tabla_resumen(d: dict, titulo_col: str = "Métrica") -> pd.DataFrame:
    """Convierte un diccionario de métricas en un DataFrame de una columna."""
    return pd.DataFrame({titulo_col: list(d.keys()), "Valor": list(d.values())}).set_index(titulo_col)


def formatear(x) -> str:
    """Formato numérico legible: miles con separador, notación científica si es diminuto."""
    if isinstance(x, (int, np.integer)):
        return f"{x:,}"
    if isinstance(x, (float, np.floating)):
        if x != 0 and abs(x) < 1e-3:
            return f"{x:.3e}"
        return f"{x:,.4f}"
    return str(x)


def top_n(df: pd.DataFrame, columna: str, n: int = 10, ascendente: bool = False) -> pd.DataFrame:
    """Top-N filas de un DataFrame ordenadas por una columna."""
    return df.sort_values(columna, ascending=ascendente).head(n)


print("Utilidades transversales cargadas: titulo, cronometrar, tabla_resumen, formatear, top_n")

---

## Carga y perfilado del dataset

El **Elliptic Bitcoin Dataset** es el mayor conjunto público de transacciones de Bitcoin
etiquetadas para detección de actividad ilícita. Se distribuye como tres CSV que forman un
**esquema en estrella** alrededor de la clave `txId`:

```
elliptic_txs_features.csv[0]  ──┐
                                ├── txId  (203.769 identificadores únicos)
elliptic_txs_classes.csv[txId] ─┤
                                │
elliptic_txs_edgelist.csv[txId1] ─┤  FK → txId  (origen)
elliptic_txs_edgelist.csv[txId2] ─┘  FK → txId  (destino)
```

| Archivo | Cabecera | Contenido | Filas |
|---|---|---|---|
| `elliptic_txs_classes.csv` | **sí** | `txId`, `class` → `'1'` ilícito, `'2'` lícito, `'unknown'` sin etiquetar | 203.769 |
| `elliptic_txs_features.csv` | **no** | col `0` = txId, col `1` = timestep (1–49), cols `2–94` locales, cols `95–166` agregadas | 203.769 |
| `elliptic_txs_edgelist.csv` | **sí** | `txId1` → `txId2` (flujo de fondos) | 234.355 |

Tres detalles que condicionan todo el análisis posterior:

1. **`class` es texto, no número.** Contiene el literal `'unknown'`, así que un `astype(int)`
   directo revienta. Solo el 22,9 % de los nodos está etiquetado.
2. **`features` no tiene cabecera.** Hay que leerlo con `header=None`; sus columnas son
   enteros `0..166`. Las renombramos a nombres con significado para poder unirlas después.
3. **Las 93 features locales ya vienen estandarizadas** (media ≈ 0, desviación ≈ 1) y
   anonimizadas: Elliptic no revela qué mide cada una. Las 72 agregadas son estadísticos
   (máx., mín., desviación, correlación) de las features locales de los vecinos a un salto —
   es decir, el dataset ya trae una capa de *feature engineering de grafo* incorporada.

In [ ]:
# =============================================================================
# Carga portable del dataset
# =============================================================================

NOMBRES_CSV = {
    "classes": "elliptic_txs_classes.csv",
    "features": "elliptic_txs_features.csv",
    "edgelist": "elliptic_txs_edgelist.csv",
}


def localizar_csv(directorio: str | Path) -> dict[str, str] | None:
    """Busca recursivamente los tres CSV de Elliptic bajo `directorio`.

    Devuelve un dict {clave: ruta} solo si encuentra los **tres**; en otro caso None.
    Se recorre el árbol porque kagglehub anida los ficheros dentro de
    `.../versions/1/elliptic_bitcoin_dataset/`.
    """
    directorio = Path(directorio)
    if not directorio.exists():
        return None
    encontrados: dict[str, str] = {}
    for raiz, _dirs, ficheros in os.walk(directorio):
        for clave, nombre in NOMBRES_CSV.items():
            if nombre in ficheros and clave not in encontrados:
                encontrados[clave] = os.path.join(raiz, nombre)
    return encontrados if len(encontrados) == 3 else None


def normalizar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Renombra las 167 columnas anónimas de `features` a nombres con significado.

    `0 → txId`, `1 → timestep`, `2..94 → local_1..local_93`, `95..166 → agg_1..agg_72`.
    Sin esto, unir features con el resto de tablas obliga a arrastrar índices numéricos
    por todo el notebook.
    """
    columnas = ["txId", "timestep"]
    columnas += [f"local_{i}" for i in range(1, 94)]   # cols 2..94  -> 93 features
    columnas += [f"agg_{i}" for i in range(1, 73)]     # cols 95..166 -> 72 features
    if len(columnas) != df.shape[1]:
        raise ValueError(f"Se esperaban {len(columnas)} columnas y llegaron {df.shape[1]}")
    df = df.copy()
    df.columns = columnas
    return df


def cargar_datos(dir_local: str = "data", usar_kagglehub: bool = True) -> dict:
    """Carga los tres CSV de Elliptic, vengan de donde vengan.

    Orden de preferencia:
      1. `./data/` local — instantáneo, sin red, ideal para reejecutar.
      2. `kagglehub.dataset_download(...)` — descarga 146 MB y cachea.

    Devuelve un dict con `classes`, `features`, `edgelist` y `ruta`.
    Lanza `FileNotFoundError` con instrucciones accionables si ninguna vía funciona;
    el notebook original asignaba `df = None` en el caso de fallo, lo que producía un
    `NameError` confuso varias celdas más abajo en lugar de un error claro aquí.
    """
    rutas = localizar_csv(dir_local)
    origen = f"directorio local '{dir_local}'"

    if rutas is None and usar_kagglehub:
        try:
            import kagglehub
            print("No hay datos en local; descargando desde Kaggle (146 MB)…")
            destino = kagglehub.dataset_download("ellipticco/elliptic-data-set")
            rutas = localizar_csv(destino)
            origen = f"caché de kagglehub ({destino})"
        except Exception as exc:
            print(f"kagglehub no pudo descargar el dataset: {exc}")

    if rutas is None:
        raise FileNotFoundError(
            "No se encontraron los CSV de Elliptic.\n"
            "Opción A: autentícate en Kaggle y reejecuta esta celda.\n"
            f"Opción B: descarga manualmente 'ellipticco/elliptic-data-set' y deja los "
            f"tres ficheros en '{dir_local}/':\n  - " + "\n  - ".join(NOMBRES_CSV.values())
        )

    print(f"Origen de los datos: {origen}")
    datos = {
        "classes": pd.read_csv(rutas["classes"]),                 # sí tiene cabecera
        "features": normalizar_features(pd.read_csv(rutas["features"], header=None)),
        "edgelist": pd.read_csv(rutas["edgelist"]),               # sí tiene cabecera
        "ruta": rutas,
    }
    for clave in ("classes", "features", "edgelist"):
        print(f"  {clave:9s}: {datos[clave].shape[0]:>7,} filas x {datos[clave].shape[1]:>3} columnas")
    return datos


DATOS = cargar_datos()
df_classes = DATOS["classes"]      # el notebook original lo llamaba `df_clasess` (typo)
df_features = DATOS["features"]
df_edgelist = DATOS["edgelist"]

In [ ]:
# =============================================================================
# Perfilado: entender los datos ANTES de convertirlos en grafo
# =============================================================================

def perfilar_datos(classes: pd.DataFrame, features: pd.DataFrame, edgelist: pd.DataFrame) -> dict:
    """Audita integridad, cardinalidad y balance de clases del dataset tabular.

    Comprueba explícitamente las tres precondiciones del modelado de grafo:
    integridad referencial (¿todo txId de la edgelist existe como nodo?),
    unicidad de la clave, y existencia de nodos aislados.
    """
    ids_classes = set(classes["txId"])
    ids_features = set(features["txId"])
    ids_aristas = set(edgelist["txId1"]) | set(edgelist["txId2"])

    dist_clase = classes["class"].value_counts()
    etiquetados = int(dist_clase.get("1", 0) + dist_clase.get("2", 0))

    resumen = {
        "Nodos en classes": len(ids_classes),
        "Nodos en features": len(ids_features),
        "txId únicos en aristas": len(ids_aristas),
        "classes.txId duplicados": int(classes["txId"].duplicated().sum()),
        "features.txId duplicados": int(features["txId"].duplicated().sum()),
        "classes == features (misma clave)": ids_classes == ids_features,
        "Aristas huérfanas (txId sin nodo)": len(ids_aristas - ids_classes),
        "Nodos aislados (sin arista)": len(ids_classes - ids_aristas),
        "Aristas totales": len(edgelist),
        "Aristas duplicadas": int(edgelist.duplicated().sum()),
        "Self-loops en la tabla": int((edgelist["txId1"] == edgelist["txId2"]).sum()),
        "Ilícitos (class=1)": int(dist_clase.get("1", 0)),
        "Lícitos (class=2)": int(dist_clase.get("2", 0)),
        "Sin etiquetar (unknown)": int(dist_clase.get("unknown", 0)),
        "% etiquetado": round(100 * etiquetados / len(classes), 2),
        "% ilícito sobre etiquetados": round(100 * dist_clase.get("1", 0) / etiquetados, 2),
        "Timesteps": int(features["timestep"].nunique()),
        "Nulos en features": int(features.isna().sum().sum()),
    }
    return resumen


titulo("Perfilado del dataset Elliptic")
perfil = perfilar_datos(df_classes, df_features, df_edgelist)
for k, v in perfil.items():
    print(f"  {k:38s} {formatear(v):>14s}")

print("\nPrimeras filas de cada tabla:")
display(df_classes.head(3))
display(df_features.iloc[:3, :6])
display(df_edgelist.head(3))

### Lectura del perfilado

Cuatro conclusiones que van a condicionar todos los módulos siguientes:

- **Integridad referencial perfecta**: cero aristas huérfanas y cero nodos aislados. Todo
  `txId` de la lista de aristas existe como nodo y todo nodo participa en al menos una arista.
  Esto significa que `nx.from_pandas_edgelist` producirá exactamente los 203.769 nodos, sin
  necesidad de añadirlos por separado.
- **Sin duplicados ni self-loops**: las 234.355 filas de la edgelist son 234.355 aristas
  distintas. Un `MultiDiGraph` sería redundante para Elliptic (sí lo usaremos en el grafo bancario,
  donde un cliente puede transferir a la misma cuenta muchas veces).
- **Desbalance severo**: solo el ~23 % está etiquetado y, dentro de lo etiquetado, apenas el
  ~9,8 % es ilícito. Es el escenario clásico de fraude: la clase de interés es rara. Esto obliga
  a evaluar con precision/recall de la clase minoritaria, **nunca con accuracy** (un modelo que
  prediga «todo lícito» acertaría el 90 % y sería inútil).
- **49 timesteps** separados por unas dos semanas cada uno. La dimensión temporal no es
  decorativa: define cómo hay que partir los datos en entrenamiento y prueba (Módulo 15).

In [ ]:
# Distribución de clases y de transacciones en el tiempo
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))

conteo = df_classes["class"].value_counts()
etiquetas = {"1": "Ilícito", "2": "Lícito", "unknown": "Desconocido"}
colores = {"1": COLORES["ilicito"], "2": COLORES["licito"], "unknown": COLORES["desconocido"]}
ejes[0].bar(
    [etiquetas[c] for c in conteo.index],
    conteo.values,
    color=[colores[c] for c in conteo.index],
)
ejes[0].set_title("Distribución de clases")
ejes[0].set_ylabel("Nº de transacciones")
for i, v in enumerate(conteo.values):
    ejes[0].text(i, v, f"{v:,}\n({100*v/len(df_classes):.1f}%)", ha="center", va="bottom", fontsize=9)
ejes[0].set_ylim(0, conteo.max() * 1.18)

# Volumen por timestep, separando ilícito de lícito
tmp = df_features[["txId", "timestep"]].merge(df_classes, on="txId")
pivote = tmp.pivot_table(index="timestep", columns="class", aggfunc="size", fill_value=0)
pivote.plot(
    kind="bar", stacked=True, ax=ejes[1], width=0.85, legend=False,
    color=[colores.get(c, COLORES["neutro"]) for c in pivote.columns],
)
ejes[1].set_title("Volumen de transacciones por timestep")
ejes[1].set_xlabel("Timestep (1–49)")
ejes[1].set_ylabel("Nº de transacciones")
ejes[1].set_xticks(range(0, 49, 4))
ejes[1].legend([etiquetas[c] for c in pivote.columns], fontsize=8)
plt.tight_layout()
plt.show()

# Proporción de ilícitos en el tiempo: aquí aparece el hallazgo más interesante del dataset
ratio = (
    tmp[tmp["class"] != "unknown"]
    .assign(ilicito=lambda d: (d["class"] == "1").astype(int))
    .groupby("timestep")["ilicito"].mean() * 100
)
plt.figure(figsize=(12, 3.8))
plt.plot(ratio.index, ratio.values, marker="o", ms=4, color=COLORES["ilicito"], lw=1.6)
plt.axvline(43, ls="--", c=COLORES["neutro"], lw=1.2)
plt.annotate("cierre del mercado\nnegro (t≈43)", xy=(43, ratio.max() * 0.8),
             xytext=(35, ratio.max() * 0.92), fontsize=9,
             arrowprops=dict(arrowstyle="->", color=COLORES["neutro"]))
plt.title("Porcentaje de transacciones ilícitas por timestep (solo etiquetadas)")
plt.xlabel("Timestep")
plt.ylabel("% ilícito")
plt.tight_layout()
plt.show()

print(f"Ratio de ilícitos: media {ratio.mean():.1f}% | máximo {ratio.max():.1f}% "
      f"(t={ratio.idxmax()}) | mínimo {ratio.min():.1f}% (t={ratio.idxmin()})")

---
---

# Módulo 0 — Introducción a Grafos

## ¿Qué es un grafo?

Un grafo es un par $G = (V, E)$ donde $V$ es un conjunto de **vértices** (o nodos) y
$E \subseteq V \times V$ un conjunto de **aristas** (o enlaces) que conectan pares de vértices.

Esa definición austera esconde el cambio de mentalidad importante: en un grafo, **la relación
es un objeto de primera clase**. En una tabla, la relación entre dos filas es algo que se
*calcula* cuando haces un JOIN. En un grafo, la relación *está almacenada* y recorrerla cuesta
lo mismo que seguir un puntero. Toda la ventaja del modelo de grafos deriva de ese único hecho.

## Grafo vs. base de datos relacional

| | **Relacional** | **Grafo** |
|---|---|---|
| Unidad de dato | Fila en una tabla | Nodo con propiedades |
| Relación | Clave foránea, materializada al hacer JOIN | Arista almacenada físicamente |
| Coste de "amigos de amigos" | JOIN por cada salto: recorre la relación entera | Recorrido local: solo toca los vecinos visitados |
| Coste de $k$ saltos | Crece con el **tamaño de la tabla** × $k$ | Crece con el **tamaño del vecindario** explorado |
| Esquema | Rígido, definido por adelantado | Flexible, cada nodo puede tener propiedades distintas |
| Pregunta natural | "¿Cuánto sumaron las compras de marzo?" | "¿Qué camino conecta a este cliente con esta cuenta señalada?" |
| Punto débil | Consultas de conectividad profunda | Agregaciones masivas sobre todos los datos |

La propiedad que hace posible la columna derecha se llama **adyacencia sin índice**
(*index-free adjacency*): cada nodo guarda referencias directas a sus vecinos, así que
expandir un salto no requiere consultar ningún índice global. La celda siguiente lo mide.

## Vértices y aristas

- **Vértice (nodo)**: la entidad. En fraude: un cliente, una cuenta, una tarjeta, una IP.
- **Arista (enlace)**: la relación. En fraude: *transfiere a*, *posee*, *se conecta desde*.
- **Grado** de un vértice: número de aristas que lo tocan. En grafos dirigidos se desdobla en
  **grado de entrada** (in-degree) y **grado de salida** (out-degree).
- **Adyacencia**: dos vértices son adyacentes si comparten una arista.
- **Incidencia**: una arista es incidente a los vértices que conecta.

## Tipología de grafos

| Familia | Definición | Ejemplo en fraude |
|---|---|---|
| **No dirigido** | La arista no tiene sentido: $(u,v) \equiv (v,u)$ | Dos clientes *comparten* un dispositivo |
| **Dirigido** (digrafo) | La arista tiene sentido: $(u,v) \neq (v,u)$ | La cuenta A *transfiere a* la cuenta B |
| **Ponderado** | Cada arista lleva un peso $w(u,v)$ | Importe transferido, número de operaciones |
| **Bipartito** | $V = V_1 \cup V_2$ y toda arista va de $V_1$ a $V_2$ | Clientes ↔ Comercios; Usuarios ↔ Dispositivos |
| **Multigrafo** | Se admiten varias aristas entre el mismo par | Tres transferencias distintas de A a B |
| **Hipergrafo** | Una arista puede conectar más de dos nodos | Una transacción con 5 entradas y 3 salidas |
| **Heterogéneo** | Varios tipos de nodo y varios tipos de arista | Cliente–Cuenta–Tarjeta–Comercio–IP |

Dos observaciones prácticas:

- **El hipergrafo casi siempre se reifica**. En vez de una arista que toque 8 nodos, se crea un
  nodo «Transacción» que conecta con sus 8 participantes. Se pierde algo de pureza formal y se
  gana poder usar todo el instrumental de grafos ordinarios. Es exactamente lo que hace Elliptic:
  la transacción de Bitcoin —que es intrínsecamente una hiperarista— aparece como un nodo.
- **El grafo heterogéneo es la norma en banca**, no la excepción. Un grafo homogéneo es una
  simplificación que se toma cuando el algoritmo la exige (Louvain, PageRank), no porque el
  dominio lo sea.

## Propiedades de un grafo

| Propiedad | Qué mide | Rango |
|---|---|---|
| **Orden** $n$ | Número de vértices | $\geq 0$ |
| **Tamaño** $m$ | Número de aristas | $\geq 0$ |
| **Densidad** | $m$ sobre el máximo posible de aristas | $[0, 1]$ |
| **Conexidad** | ¿Existe camino entre cualquier par? | booleano |
| **Aciclicidad** | ¿Hay ciclos? Un digrafo sin ciclos es un **DAG** | booleano |
| **Regularidad** | ¿Todos los vértices tienen el mismo grado? | booleano |
| **Reciprocidad** | Fracción de aristas $(u,v)$ con recíproca $(v,u)$ | $[0, 1]$ |
| **Asortatividad** | ¿Los nodos de alto grado se conectan entre sí? | $[-1, 1]$ |

Para un grafo dirigido simple con $n$ vértices, el máximo de aristas es $n(n-1)$; para uno no
dirigido, $n(n-1)/2$. La densidad es el cociente entre las aristas reales y ese máximo. En
redes del mundo real la densidad es diminuta: veremos que Elliptic tiene densidad $5{,}6\times10^{-6}$,
o sea que existe **una de cada 180.000** aristas posibles. Esa escasez no es un defecto del dato;
es la razón por la que los algoritmos de grafos son viables a gran escala.

In [ ]:
# =============================================================================
# M0.1 — Un ejemplar mínimo de cada familia de grafos
# =============================================================================

def grafos_ejemplo() -> dict:
    """Construye el ejemplar más pequeño posible de cada familia de grafos.

    Cada uno es un juguete de 4–8 nodos, elegido para que la propiedad que
    caracteriza a la familia sea visible de un vistazo al dibujarlo.
    """
    ejemplos: dict[str, nx.Graph] = {}

    aristas = [("A", "B"), ("B", "C"), ("C", "A"), ("C", "D")]

    # 1. No dirigido: la relación es simétrica.
    ejemplos["No dirigido"] = nx.Graph(aristas)

    # 2. Dirigido: la relación tiene sentido. A→B no implica B→A.
    ejemplos["Dirigido"] = nx.DiGraph(aristas)

    # 3. Ponderado: cada arista lleva un número asociado.
    g_pond = nx.Graph()
    g_pond.add_weighted_edges_from([("A", "B", 5.0), ("B", "C", 1.5), ("C", "A", 9.0), ("C", "D", 3.0)])
    ejemplos["Ponderado"] = g_pond

    # 4. Bipartito: dos conjuntos disjuntos, aristas solo entre ellos (nunca dentro).
    g_bip = nx.Graph()
    g_bip.add_nodes_from(["U1", "U2", "U3"], bipartite=0)
    g_bip.add_nodes_from(["D1", "D2"], bipartite=1)
    g_bip.add_edges_from([("U1", "D1"), ("U2", "D1"), ("U3", "D1"), ("U3", "D2")])
    ejemplos["Bipartito"] = g_bip

    # 5. Multigrafo: varias aristas paralelas entre el mismo par de nodos.
    g_multi = nx.MultiDiGraph()
    g_multi.add_edges_from([("A", "B"), ("A", "B"), ("A", "B"), ("B", "C"), ("C", "A")])
    ejemplos["Multigrafo"] = g_multi

    # 6. Hipergrafo reificado: la hiperarista H1 = {A,B,C} pasa a ser un NODO
    #    que conecta con sus tres miembros. Es la representación bipartita de incidencia.
    g_hiper = nx.Graph()
    g_hiper.add_nodes_from(["A", "B", "C", "D"], tipo="elemento")
    g_hiper.add_nodes_from(["H1", "H2"], tipo="hiperarista")
    g_hiper.add_edges_from([("H1", "A"), ("H1", "B"), ("H1", "C"), ("H2", "C"), ("H2", "D")])
    ejemplos["Hipergrafo (reificado)"] = g_hiper

    # 7. Heterogéneo: varios tipos de nodo y de arista conviviendo.
    g_het = nx.DiGraph()
    g_het.add_node("Cliente1", tipo="cliente")
    g_het.add_node("Cuenta1", tipo="cuenta")
    g_het.add_node("Tarjeta1", tipo="tarjeta")
    g_het.add_node("Comercio1", tipo="comercio")
    g_het.add_node("IP1", tipo="ip")
    g_het.add_edges_from([
        ("Cliente1", "Cuenta1", {"rel": "posee"}),
        ("Cliente1", "Tarjeta1", {"rel": "posee"}),
        ("Tarjeta1", "Comercio1", {"rel": "compra"}),
        ("Cliente1", "IP1", {"rel": "utiliza"}),
    ])
    ejemplos["Heterogéneo"] = g_het

    # 8. DAG: dirigido y sin ciclos. Es la forma de Elliptic (el dinero no vuelve atrás).
    ejemplos["DAG"] = nx.DiGraph([("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E")])

    return ejemplos


def propiedades_grafo(G: nx.Graph, nombre: str = "") -> dict:
    """Calcula el juego completo de propiedades estructurales de un grafo.

    Funciona igual sobre grafos dirigidos y no dirigidos: las métricas que solo
    aplican a uno de los dos casos se devuelven como None en lugar de fallar.
    """
    dirigido = G.is_directed()
    multi = G.is_multigraph()
    n, m = G.number_of_nodes(), G.number_of_edges()
    grados = [d for _, d in G.degree()]

    props = {
        "nombre": nombre,
        "orden (n)": n,
        "tamaño (m)": m,
        "dirigido": dirigido,
        "multigrafo": multi,
        "ponderado": any("weight" in d for _, _, d in G.edges(data=True)),
        "densidad": round(nx.density(G), 4),
        "grado_medio": round(sum(grados) / n, 2) if n else 0.0,
        "grado_max": max(grados) if grados else 0,
        "self_loops": nx.number_of_selfloops(G),
        "regular": len(set(grados)) == 1 if grados else True,
    }

    # Bipartición y aciclicidad se comprueban de forma distinta según el tipo.
    G_simple = nx.DiGraph(G) if (multi and dirigido) else (nx.Graph(G) if multi else G)
    props["bipartito"] = nx.is_bipartite(G_simple)

    if dirigido:
        props["conexo (débil)"] = nx.is_weakly_connected(G) if n else False
        props["conexo (fuerte)"] = nx.is_strongly_connected(G) if n else False
        props["acíclico (DAG)"] = nx.is_directed_acyclic_graph(G_simple)
        props["reciprocidad"] = round(nx.reciprocity(G_simple), 3) if m else None
    else:
        props["conexo"] = nx.is_connected(G) if n else False
        props["acíclico (bosque)"] = nx.is_forest(G_simple)

    return props


EJEMPLOS = grafos_ejemplo()
tabla_props = pd.DataFrame([propiedades_grafo(g, nom) for nom, g in EJEMPLOS.items()]).set_index("nombre")
titulo("Propiedades estructurales de cada familia de grafos")
display(tabla_props.fillna("—"))

In [ ]:
# =============================================================================
# M0.2 — Visualización comparada de las 8 familias
# =============================================================================

def dibujar_familias(ejemplos: dict, columnas: int = 4) -> None:
    """Dibuja cada grafo de ejemplo con la codificación visual propia de su familia.

    Bipartito usa layout de dos columnas, el heterogéneo colorea por tipo de nodo,
    el ponderado escribe los pesos y el multigrafo curva las aristas paralelas.
    """
    filas = math.ceil(len(ejemplos) / columnas)
    fig, ejes = plt.subplots(filas, columnas, figsize=(4.2 * columnas, 3.6 * filas))
    ejes = np.atleast_1d(ejes).ravel()

    for eje, (nombre, g) in zip(ejes, ejemplos.items()):
        if nombre == "Bipartito":
            pos = nx.bipartite_layout(g, [n for n, d in g.nodes(data=True) if d.get("bipartite") == 0])
        elif nombre.startswith("Hipergrafo"):
            pos = nx.bipartite_layout(g, [n for n, d in g.nodes(data=True) if d.get("tipo") == "hiperarista"])
        elif nombre == "DAG":
            pos = nx.multipartite_layout(nx.DiGraph(g), subset_key=dict(enumerate(nx.topological_generations(g))))
        else:
            pos = nx.spring_layout(g, seed=SEED)

        # Color por tipo cuando el grafo es heterogéneo o reificado.
        tipos = nx.get_node_attributes(g, "tipo")
        if tipos:
            paleta = {t: c for t, c in zip(
                sorted(set(tipos.values())),
                [COLORES["licito"], COLORES["acento"], COLORES["ok"], COLORES["ilicito"], COLORES["neutro"]],
            )}
            colores_nodo = [paleta[tipos[n]] for n in g.nodes()]
        else:
            colores_nodo = COLORES["licito"]

        nx.draw_networkx_nodes(g, pos, ax=eje, node_color=colores_nodo, node_size=520,
                               edgecolors="white", linewidths=1.5)
        nx.draw_networkx_labels(g, pos, ax=eje, font_size=7.5, font_color="white", font_weight="bold")
        estilo = "arc3,rad=0.18" if g.is_multigraph() else "arc3,rad=0.0"
        nx.draw_networkx_edges(g, pos, ax=eje, edge_color="#7A8492", width=1.6,
                               arrows=g.is_directed(), arrowsize=13, connectionstyle=estilo)
        if nombre == "Ponderado":
            nx.draw_networkx_edge_labels(g, pos, ax=eje, font_size=8,
                                         edge_labels=nx.get_edge_attributes(g, "weight"))

        eje.set_title(f"{nombre}\nn={g.number_of_nodes()}, m={g.number_of_edges()}", fontsize=10)
        eje.axis("off")

    for eje in ejes[len(ejemplos):]:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


dibujar_familias(EJEMPLOS)

In [ ]:
# =============================================================================
# M0.3 — Grafo vs. relacional: midiendo el coste de k saltos
# =============================================================================

def comparar_grafo_vs_relacional(n_nodos: int = 60_000, n_aristas: int = 240_000,
                                 saltos=(1, 2, 3, 4, 5), seed: int = SEED) -> pd.DataFrame:
    """Compara empíricamente el coste de una consulta de k saltos en ambos modelos.

    Genera la MISMA relación en dos representaciones y ejecuta la misma pregunta
    ("¿qué nodos alcanzo desde X en como mucho k saltos?"):

      · Relacional: k self-JOINs encadenados. Cada JOIN debe recorrer la tabla
        entera, aunque la frontera de búsqueda tenga tres filas.
      · Grafo: un BFS que solo toca los nodos que va descubriendo.

    El resultado no es "pandas es lento": es que el coste relacional depende del
    tamaño de la RELACIÓN y el coste del grafo depende del tamaño del VECINDARIO.
    """
    rng = np.random.default_rng(seed)
    tabla = pd.DataFrame({
        "origen": rng.integers(0, n_nodos, n_aristas),
        "destino": rng.integers(0, n_nodos, n_aristas),
    }).drop_duplicates()
    G = nx.from_pandas_edgelist(tabla, "origen", "destino", create_using=nx.DiGraph())

    # Semilla: un nodo con grado de salida alto, para que la consulta no sea trivial.
    semilla = int(tabla["origen"].value_counts().index[0])
    filas = []

    for k in saltos:
        # --- Modelo relacional: k self-JOINs ---
        t0 = time.perf_counter()
        visitados_rel = {semilla}
        frontera = pd.DataFrame({"id": [semilla]})
        for _ in range(k):
            frontera = (
                frontera.merge(tabla, left_on="id", right_on="origen")[["destino"]]
                .rename(columns={"destino": "id"})
                .drop_duplicates()
            )
            visitados_rel |= set(frontera["id"])
            if frontera.empty:
                break
        t_rel = time.perf_counter() - t0

        # --- Modelo de grafo: un BFS con corte a profundidad k ---
        t0 = time.perf_counter()
        visitados_grafo = set(nx.single_source_shortest_path_length(G, semilla, cutoff=k))
        t_grafo = time.perf_counter() - t0

        filas.append({
            "saltos": k,
            "nodos alcanzados": len(visitados_grafo),
            "relacional (ms)": round(t_rel * 1000, 2),
            "grafo (ms)": round(t_grafo * 1000, 2),
            "aceleración": round(t_rel / max(t_grafo, 1e-9), 1),
            "mismo resultado": visitados_rel == visitados_grafo,
        })

    print(f"Relación de prueba: {len(tabla):,} aristas sobre {n_nodos:,} nodos")
    print(f"Nodo semilla: {semilla} (grado de salida = {G.out_degree(semilla)})\n")
    return pd.DataFrame(filas).set_index("saltos")


titulo("Coste de una consulta de k saltos: relacional vs. grafo")
comparativa = comparar_grafo_vs_relacional()
display(comparativa)

fig, eje = plt.subplots(figsize=(9, 4))
eje.plot(comparativa.index, comparativa["relacional (ms)"], marker="o",
         label="Relacional (k self-JOINs)", color=COLORES["ilicito"], lw=2)
eje.plot(comparativa.index, comparativa["grafo (ms)"], marker="s",
         label="Grafo (BFS con corte)", color=COLORES["ok"], lw=2)
eje.set_yscale("log")
eje.set_xlabel("Profundidad de la consulta (saltos)")
eje.set_ylabel("Tiempo (ms, escala log)")
eje.set_title("El coste relacional crece con cada JOIN; el del grafo, con el vecindario explorado")
eje.set_xticks(list(comparativa.index))
eje.legend()
plt.tight_layout()
plt.show()

### Lectura del experimento

La columna `mismo resultado` confirma que ambos modelos responden **exactamente la misma
pregunta**; lo único que cambia es cómo la ejecutan.

El patrón que aparece es el esperado: a un salto la diferencia es modesta, y crece con la
profundidad. La razón es estructural, no de implementación —un motor SQL real con índices
sería más rápido que estos `merge`, pero el crecimiento sería el mismo:

- Cada JOIN adicional obliga a **volver a recorrer la relación completa**, incluso cuando la
  frontera de búsqueda tiene un puñado de filas. El coste depende de $|E|$.
- El BFS solo toca **los nodos que descubre**. El coste depende del vecindario a $k$ saltos,
  que en una red dispersa es una fracción minúscula del grafo.

Ese es todo el argumento a favor del modelo de grafos, y también su límite: en una consulta
de un solo salto o en una agregación sobre todos los datos, el modelo relacional gana. La
pregunta correcta nunca es "¿grafo o tabla?" sino "**¿mi pregunta es de conectividad o de
agregación?**". Las preguntas de fraude —"¿quién está a tres saltos de esta cuenta señalada?",
"¿existe un ciclo de transferencias que vuelva al origen?"— son de conectividad.

---
---

# Módulo 1 — Modelado de datos

> Aquí empieza el trabajo de verdad. Los algoritmos de los módulos 3 a 16 son código ajeno que
> se invoca; **el modelado es la parte que nadie puede hacer por ti**, y es la que determina si
> el análisis encontrará algo o no. Un grafo mal modelado produce métricas impecables sobre
> preguntas equivocadas.

## Cómo transformar datos tabulares en un grafo

El proceso tiene cuatro decisiones, y conviene tomarlas en este orden:

### 1. Definir los nodos — ¿qué es una entidad?

La regla operativa: **es un nodo aquello sobre lo que quieres preguntar "¿con qué se conecta?"**.
Si de una columna solo vas a querer filtrar o agregar, es un atributo. Si vas a querer recorrerla,
es un nodo.

Ejemplo concreto: `pais_del_cliente` normalmente es un *atributo* (filtras por él). Pero si tu
pregunta es "¿qué clientes comparten país con cuentas señaladas y además comparten dispositivo?",
entonces el país se convierte en un *nodo* y esa consulta pasa a ser un recorrido de dos saltos.
La misma columna, dos modelados distintos, según la pregunta.

### 2. Definir las relaciones — ¿qué es una arista?

Cada clave foránea es una arista candidata. Tres cuestiones a resolver por cada una:

- **¿Dirigida?** *transfiere_a* sí (el dinero va en un sentido); *comparte_dispositivo_con* no.
- **¿Múltiple?** Si dos entidades pueden relacionarse varias veces y cada ocurrencia importa,
  necesitas un multigrafo o **reificar** la relación como nodo.
- **¿Reificar?** Si la relación tiene atributos propios ricos (importe, fecha, canal, resultado)
  y quieres consultarla como entidad, conviértela en nodo. Este es el paso más importante del
  modelado de fraude.

### 3. Reificación: cuándo una relación se convierte en nodo

```
Modelo A (arista con propiedades)        Modelo B (relación reificada)

  Cuenta ──transfiere{monto,fecha}──►     Cuenta ──emite──► Transferencia ──recibe──► Cuenta
  Cuenta                                                    {monto, fecha, canal}
```

El modelo A es más compacto. El modelo B permite preguntar cosas sobre la transferencia
*en sí misma*: ¿qué transferencias comparten dispositivo de origen?, ¿qué ocurre si una
transferencia se divide entre tres destinos? Elliptic usa implícitamente el modelo B: sus
nodos **son** transacciones, no cuentas.

### 4. Definir atributos

| | Atributos de **nodo** | Atributos de **arista** |
|---|---|---|
| Qué guardan | Propiedades intrínsecas de la entidad | Propiedades de la interacción |
| Ejemplos en banca | Antigüedad, segmento, país, scoring | Importe, timestamp, canal, divisa |
| En Elliptic | `timestep`, `class`, 165 features | (ninguno: las aristas son solo topología) |
| Uso en ML | Features de nodo para GNN | Pesos para caminos mínimos, filtros temporales |

## Diseño de un esquema de grafo

Un **esquema** declara qué tipos de nodo existen, qué tipos de arista los conectan y en qué
dirección. Es el equivalente al diagrama entidad-relación, y conviene escribirlo antes de tocar
código. El esquema de nuestro grafo bancario:

```
                 ┌────────────┐
       ┌─posee──►│   Cuenta   │◄──recibe──┐
       │         └─────┬──────┘           │
       │               │ emite            │
       │               ▼                  │
       │        ┌──────────────┐          │
       │        │Transferencia │──────────┘
       │        └──────────────┘
       │
  ┌────┴─────┐  posee   ┌──────────┐ realiza  ┌────────┐  en   ┌──────────┐
  │ Cliente  ├─────────►│ Tarjeta  ├─────────►│ Compra ├──────►│ Comercio │
  └────┬─────┘          └──────────┘          └────────┘       └──────────┘
       │ utiliza
       ├──────────► Dispositivo      ┌──────────────────────────────────┐
       ├──────────► IP               │ Los nodos compartidos (Dispositivo,│
       ├──────────► Teléfono         │ IP, Teléfono, Dirección) son los  │
       └─reside_en► Dirección        │ que revelan anillos de fraude.    │
                                     └──────────────────────────────────┘
  Cuenta ──opera_en──► ATM ──pertenece_a──► Agencia
```

Los nodos compartidos merecen un comentario. Un dispositivo o una IP no son interesantes por
sí mismos: son interesantes porque **crean caminos entre clientes que se declaran independientes**.
Sin esos nodos, dos titulares distintos no tienen ninguna conexión en el grafo; con ellos,
están a dos saltos. Casi toda la detección de anillos de fraude vive en esa distinción.

## Property Graph vs. RDF vs. Knowledge Graph

Tres formalismos que se confunden a menudo:

| | **Property Graph** | **RDF** | **Knowledge Graph** |
|---|---|---|---|
| Unidad | Nodo/arista con dict de propiedades | Tripleta `(sujeto, predicado, objeto)` | Grafo + ontología + inferencia |
| Atributos en aristas | Sí, nativamente | No: hay que reificar la tripleta | Depende de la base |
| Estándar | De facto (cada motor el suyo) | W3C (RDF, RDFS, OWL) | — |
| Consulta | Cypher, Gremlin | SPARQL | SPARQL / Cypher |
| Motores | Neo4j, Memgraph, TigerGraph | GraphDB, Blazegraph, Neptune | Wikidata, Neptune |
| Fuerte en | Analítica y recorridos | Interoperabilidad, datos enlazados | Razonamiento, integración semántica |
| Débil en | Semántica formal | Ergonomía y atributos de arista | Rendimiento a gran escala |

En detección de fraude domina el **Property Graph**: las aristas llevan importe y timestamp de
forma natural, y las consultas son recorridos, no razonamiento lógico. RDF aparece cuando hay
que integrar fuentes externas (listas de sanciones, registros mercantiles, datos de terceros)
donde el vocabulario compartido importa más que la velocidad.

Un **Knowledge Graph** es un grafo (de cualquiera de los dos formalismos) al que se le añade una
capa de ontología: jerarquías de tipos, restricciones y reglas que permiten **inferir aristas que
no están almacenadas**. Ejemplo: si la ontología declara que `comparte_dispositivo` es simétrica
y transitiva, el motor deduce el anillo completo sin que nadie lo escriba.

In [ ]:
# =============================================================================
# M1.1 — Transformador genérico de tablas a grafo
# =============================================================================

def tabular_a_grafo(df: pd.DataFrame, especificaciones: list, crear_con=nx.MultiDiGraph) -> nx.Graph:
    """Convierte un DataFrame en un grafo siguiendo una lista de especificaciones.

    Cada especificación describe una relación a extraer de la tabla:

        {
          "origen":       nombre de la columna con el nodo origen,
          "destino":      nombre de la columna con el nodo destino,
          "rel":          nombre del tipo de arista,
          "tipo_origen":  tipo semántico del nodo origen,
          "tipo_destino": tipo semántico del nodo destino,
          "attrs":        [columnas que pasan a ser atributos de la arista],
          "prefijo_origen"/"prefijo_destino": prefijo opcional para evitar colisiones
                          de identificador entre tipos distintos (p. ej. cliente 7 y cuenta 7).
        }

    Es el puente reutilizable entre el mundo tabular y el mundo de grafos: la misma
    función sirve para Elliptic (una sola especificación) y para un modelo bancario
    heterogéneo (una especificación por clave foránea).
    """
    G = crear_con()

    for esp in especificaciones:
        col_o, col_d = esp["origen"], esp["destino"]
        pre_o = esp.get("prefijo_origen", "")
        pre_d = esp.get("prefijo_destino", "")
        attrs = esp.get("attrs", [])
        sub = df[[col_o, col_d] + attrs].dropna(subset=[col_o, col_d])

        for fila in sub.itertuples(index=False):
            valores = dict(zip([col_o, col_d] + attrs, fila))
            u = f"{pre_o}{valores[col_o]}"
            v = f"{pre_d}{valores[col_d]}"
            G.add_node(u, tipo=esp.get("tipo_origen", "nodo"))
            G.add_node(v, tipo=esp.get("tipo_destino", "nodo"))
            G.add_edge(u, v, rel=esp["rel"], **{a: valores[a] for a in attrs})

    return G


# Demostración sobre una tabla de operaciones bancarias en formato "una fila = un evento",
# que es como suelen llegar los datos de un core bancario.
operaciones = pd.DataFrame({
    "cliente":   ["C1", "C1", "C2", "C3", "C3"],
    "cuenta":    ["A1", "A1", "A2", "A3", "A3"],
    "destino":   ["A2", "A3", "A3", "A1", "A2"],
    "dispositivo": ["D1", "D1", "D1", "D2", "D2"],
    "monto":     [1500.0, 230.5, 9800.0, 45.0, 12000.0],
    "ts":        [1, 2, 3, 4, 5],
})

ESPEC_DEMO = [
    {"origen": "cliente", "destino": "cuenta", "rel": "posee",
     "tipo_origen": "cliente", "tipo_destino": "cuenta"},
    {"origen": "cuenta", "destino": "destino", "rel": "transfiere",
     "tipo_origen": "cuenta", "tipo_destino": "cuenta", "attrs": ["monto", "ts"]},
    {"origen": "cliente", "destino": "dispositivo", "rel": "utiliza",
     "tipo_origen": "cliente", "tipo_destino": "dispositivo"},
]

G_demo = tabular_a_grafo(operaciones, ESPEC_DEMO)
titulo("De una tabla de eventos a un grafo heterogéneo")
print(f"Tabla de entrada : {operaciones.shape[0]} filas x {operaciones.shape[1]} columnas")
print(f"Grafo resultante : {G_demo.number_of_nodes()} nodos, {G_demo.number_of_edges()} aristas")
print(f"Tipos de nodo    : {dict(Counter(nx.get_node_attributes(G_demo, 'tipo').values()))}")
print(f"Tipos de arista  : {dict(Counter(d['rel'] for _, _, d in G_demo.edges(data=True)))}")
print("\nObservación: C1 y C2 no comparten ninguna columna en la tabla, pero en el grafo")
print("están a dos saltos a través del dispositivo D1. Esa conexión es invisible en la tabla.")

In [ ]:
# =============================================================================
# M1.2 — El esquema de Elliptic expresado con la misma función
# =============================================================================

ESPEC_ELLIPTIC = [
    {
        "origen": "txId1", "destino": "txId2", "rel": "flujo_de_fondos",
        "tipo_origen": "transaccion", "tipo_destino": "transaccion",
    }
]

titulo("Esquema de Elliptic")
print("""
ESQUEMA (grafo homogéneo, dirigido, acíclico y temporal)

    ┌──────────────────────────────────────┐
    │            Transacción               │      flujo_de_fondos
    │  txId       (identidad)              │  ────────────────────────►
    │  timestep   1..49  (partición temp.) │      Transacción
    │  clase      ilícito|lícito|desconoc. │
    │  local_1..93   features propias      │
    │  agg_1..72     features del vecindario│
    └──────────────────────────────────────┘

DECISIONES DE MODELADO
  · El nodo es la TRANSACCIÓN, no la cuenta. Es una reificación: la transacción de
    Bitcoin es realmente una hiperarista (n entradas → m salidas) y Elliptic la
    convierte en nodo para poder usar algoritmos de grafos ordinarios.
  · Las aristas NO tienen atributos: son pura topología (el importe está anonimizado
    dentro de las features del nodo).
  · `timestep` es atributo de NODO, no de arista. Veremos en el Módulo 4 que ninguna
    arista cruza timesteps: eso convierte al timestep en clave natural de partición.
  · No se necesita `tabular_a_grafo` aquí: con una sola relación, el atajo de NetworkX
    `from_pandas_edgelist` es más directo y mucho más rápido (Módulo 2).
""")

### El grafo bancario sintético

Elliptic no puede demostrar los Módulos 18 y 19: su grafo tiene **un solo tipo de nodo**, así
que preguntas como "¿qué clientes comparten dispositivo?" no tienen dónde apoyarse. Generamos
por tanto un segundo grafo, **heterogéneo y con ground truth conocido**.

Que sea sintético es una ventaja metodológica, no una concesión: **sabemos exactamente qué
fraude hay dentro**, así que podemos medir el recall real de cada detector del Módulo 19. Con
datos reales de fraude nunca se conoce el denominador (el fraude no detectado sigue sin
detectarse) y toda evaluación es optimista por construcción.

Se inyectan seis tipologías, cada una con una firma topológica distinta:

| Patrón | Firma en el grafo | Módulo que lo detecta |
|---|---|---|
| **Money mule** | Fan-in de muchas cuentas → concentración → fan-out inmediato a un recolector | 8, 19 |
| **Device sharing** | Muchos clientes «independientes» colgando de un mismo dispositivo/IP | 8, 19 |
| **Card testing** | Una tarjeta → decenas de compras diminutas en muchos comercios en pocas horas | 19 |
| **Account takeover** | Dispositivo e IP nuevos en una cuenta antigua + transferencia grande inmediata | 19 |
| **Identidad sintética** | Clientes distintos que comparten teléfono/dirección y nada más | 19 |
| **Layering (AML)** | Ciclo dirigido de transferencias que vuelve al origen con importes similares | 8, 19 |

In [ ]:
# =============================================================================
# M1.3 — Generador del grafo bancario heterogéneo con fraude inyectado
# =============================================================================

CIUDADES = ["Lima", "Arequipa", "Trujillo", "Cusco", "Piura", "Chiclayo", "Iquitos"]
CATEGORIAS = ["retail", "electrónica", "viajes", "alimentación", "juego online",
              "criptomoneda", "farmacia", "combustible"]
SEGMENTOS = ["retail", "premium", "pyme"]

# Tipos de nodo y de relación que componen el esquema (Módulo 18 los usa como catálogo).
TIPOS_NODO = ["cliente", "cuenta", "tarjeta", "comercio", "ip", "dispositivo",
              "telefono", "direccion", "atm", "agencia", "transferencia", "compra"]
TIPOS_RELACION = ["posee", "compra", "transfiere", "utiliza", "comparte", "recibe"]


def build_banking_graph(seed: int = SEED, n_clientes: int = 320, n_comercios: int = 60,
                        n_agencias: int = 8, n_atm: int = 25, n_transferencias: int = 1400,
                        n_compras: int = 2400, dias: int = 90) -> tuple:
    """Genera un grafo bancario heterogéneo sintético con seis tipologías de fraude.

    Devuelve `(G, ground_truth)` donde:
      · `G` es un `MultiDiGraph`; cada nodo lleva `tipo` y cada arista lleva `rel`.
        Las transferencias y compras están REIFICADAS como nodos, de modo que
        arrastran sus propios atributos (`monto`, `ts`).
      · `ground_truth` es un dict {patrón: set de nodos implicados}, la verdad de
        campo contra la que se evalúan los detectores del Módulo 19.

    Todo es determinista dada la semilla: dos ejecuciones producen el mismo grafo.
    """
    rng = random.Random(seed)
    G = nx.MultiDiGraph()
    gt: dict[str, set] = defaultdict(set)
    cont: Counter = Counter()

    def nuevo(prefijo: str, tipo: str, **attrs) -> str:
        cont[prefijo] += 1
        nid = f"{prefijo}_{cont[prefijo]:04d}"
        G.add_node(nid, tipo=tipo, patron_fraude=None, **attrs)
        return nid

    def marcar(nodos, patron: str) -> None:
        for n in nodos:
            gt[patron].add(n)
            G.nodes[n]["patron_fraude"] = patron

    SEG = dias * 86400  # ventana temporal total en segundos

    # ---------------------------------------------------------------- infraestructura
    agencias = [nuevo("AGE", "agencia", ciudad=rng.choice(CIUDADES)) for _ in range(n_agencias)]
    atms = []
    for _ in range(n_atm):
        a = nuevo("ATM", "atm", ciudad=rng.choice(CIUDADES))
        G.add_edge(a, rng.choice(agencias), rel="pertenece_a")
        atms.append(a)
    comercios = [
        nuevo("COM", "comercio", categoria=rng.choice(CATEGORIAS), riesgo=round(rng.uniform(0, 1), 2))
        for _ in range(n_comercios)
    ]

    # ---------------------------------------------------------------- clientes y activos
    clientes, cuentas, tarjetas, dispositivos, ips = [], [], [], [], []
    cuentas_de: dict[str, list] = {}
    tarjetas_de: dict[str, list] = {}

    for _ in range(n_clientes):
        cli = nuevo("CLI", "cliente",
                    antiguedad_dias=rng.randint(30, 3600),
                    segmento=rng.choices(SEGMENTOS, weights=[7, 2, 1])[0],
                    ciudad=rng.choice(CIUDADES))
        clientes.append(cli)

        # Identificadores personales: por defecto uno por cliente (los patrones los compartirán).
        dirc = nuevo("DIR", "direccion", ciudad=G.nodes[cli]["ciudad"])
        tel = nuevo("TEL", "telefono", operador=rng.choice(["OpA", "OpB", "OpC"]))
        G.add_edge(cli, dirc, rel="reside_en")
        G.add_edge(cli, tel, rel="utiliza")

        cuentas_de[cli] = []
        for _ in range(rng.choices([1, 2, 3], weights=[6, 3, 1])[0]):
            cta = nuevo("CTA", "cuenta", saldo=round(rng.lognormvariate(7.5, 1.1), 2),
                        moneda=rng.choice(["PEN", "USD"]))
            G.add_edge(cli, cta, rel="posee")
            G.add_edge(cta, rng.choice(atms), rel="opera_en")
            cuentas.append(cta)
            cuentas_de[cli].append(cta)

        tarjetas_de[cli] = []
        for _ in range(rng.choices([0, 1, 2], weights=[2, 6, 2])[0]):
            tar = nuevo("TAR", "tarjeta", tipo_tarjeta=rng.choice(["débito", "crédito"]),
                        limite=rng.choice([1000, 3000, 5000, 15000]))
            G.add_edge(cli, tar, rel="posee")
            tarjetas.append(tar)
            tarjetas_de[cli].append(tar)

        for _ in range(rng.choices([1, 2], weights=[7, 3])[0]):
            dev = nuevo("DEV", "dispositivo", so=rng.choice(["Android", "iOS", "Windows"]))
            G.add_edge(cli, dev, rel="utiliza", ts=rng.randint(0, SEG))
            dispositivos.append(dev)

        for _ in range(rng.choices([1, 2, 3], weights=[5, 3, 2])[0]):
            ip = nuevo("IP", "ip", pais=rng.choices(["PE", "US", "RU", "NG"], weights=[85, 8, 4, 3])[0])
            G.add_edge(cli, ip, rel="utiliza", ts=rng.randint(0, SEG))
            ips.append(ip)

    # ---------------------------------------------------------------- actividad legítima
    def transferencia(origen: str, destino: str, monto: float, ts: int, patron=None) -> str:
        trf = nuevo("TRF", "transferencia", monto=round(monto, 2), ts=ts,
                    canal=rng.choice(["app", "web", "oficina"]))
        G.add_edge(origen, trf, rel="emite", ts=ts, monto=round(monto, 2))
        G.add_edge(trf, destino, rel="recibe", ts=ts, monto=round(monto, 2))
        if patron:
            marcar([trf], patron)
        return trf

    def compra(tarjeta: str, comercio: str, monto: float, ts: int, patron=None) -> str:
        cmp_ = nuevo("CMP", "compra", monto=round(monto, 2), ts=ts)
        G.add_edge(tarjeta, cmp_, rel="realiza", ts=ts, monto=round(monto, 2))
        G.add_edge(cmp_, comercio, rel="en", ts=ts)
        if patron:
            marcar([cmp_], patron)
        return cmp_

    for _ in range(n_transferencias):
        o, d = rng.sample(cuentas, 2)
        transferencia(o, d, rng.lognormvariate(6.0, 1.2), rng.randint(0, SEG))

    tarjetas_activas = [t for t in tarjetas]
    for _ in range(n_compras):
        compra(rng.choice(tarjetas_activas), rng.choice(comercios),
               rng.lognormvariate(4.0, 1.0), rng.randint(0, SEG))

    # ---------------------------------------------------------------- patrón 1: money mule
    # Firma: muchas cuentas envían importes pequeños a una mula en una ventana corta,
    # y la mula reenvía casi todo a un recolector poco después.
    mulas = rng.sample(cuentas, 6)
    for mula in mulas:
        recolector = rng.choice([c for c in cuentas if c != mula])
        t0 = rng.randint(0, SEG - 5 * 86400)
        origenes = rng.sample([c for c in cuentas if c not in (mula, recolector)], rng.randint(9, 16))
        total = 0.0
        for i, org in enumerate(origenes):
            monto = rng.uniform(800, 2600)
            total += monto
            transferencia(org, mula, monto, t0 + i * rng.randint(600, 5400), patron="money_mule")
        transferencia(mula, recolector, total * rng.uniform(0.90, 0.97),
                      t0 + 5 * 86400, patron="money_mule")
        marcar([mula, recolector], "money_mule")

    # ---------------------------------------------------------------- patrón 2: device sharing
    # Firma: N clientes sin ninguna otra relación cuelgan del mismo dispositivo e IP.
    for _ in range(3):
        anillo = rng.sample(clientes, rng.randint(6, 11))
        dev = nuevo("DEV", "dispositivo", so=rng.choice(["Android", "Windows"]))
        ip = nuevo("IP", "ip", pais=rng.choice(["RU", "NG", "US"]))
        dispositivos.append(dev)
        ips.append(ip)
        for cli in anillo:
            ts = rng.randint(0, SEG)
            G.add_edge(cli, dev, rel="utiliza", ts=ts)
            G.add_edge(cli, ip, rel="utiliza", ts=ts)
        marcar(anillo + [dev, ip], "device_sharing")

    # ---------------------------------------------------------------- patrón 3: card testing
    # Firma: una tarjeta prueba importes minúsculos en muchos comercios en pocas horas.
    for tar in rng.sample(tarjetas, 4):
        t0 = rng.randint(0, SEG - 86400)
        for i in range(rng.randint(22, 40)):
            compra(tar, rng.choice(comercios), rng.uniform(0.5, 4.0),
                   t0 + i * rng.randint(60, 420), patron="card_testing")
        marcar([tar], "card_testing")

    # ---------------------------------------------------------------- patrón 4: account takeover
    # Firma: cuenta antigua que estrena dispositivo e IP y vacía saldo acto seguido.
    for cli in rng.sample([c for c in clientes if cuentas_de[c]], 5):
        t_ato = rng.randint(int(SEG * 0.7), SEG - 3600)
        dev = nuevo("DEV", "dispositivo", so="Android")
        ip = nuevo("IP", "ip", pais=rng.choice(["RU", "NG"]))
        dispositivos.append(dev)
        ips.append(ip)
        G.add_edge(cli, dev, rel="utiliza", ts=t_ato)
        G.add_edge(cli, ip, rel="utiliza", ts=t_ato)
        origen = cuentas_de[cli][0]
        destino = rng.choice([c for c in cuentas if c not in cuentas_de[cli]])
        transferencia(origen, destino, rng.uniform(9000, 30000), t_ato + rng.randint(300, 2400),
                      patron="account_takeover")
        marcar([cli, dev, ip, origen], "account_takeover")

    # ---------------------------------------------------------------- patrón 5: identidad sintética
    # Firma: clientes "distintos" que comparten teléfono y dirección y poco más.
    for _ in range(4):
        familia = rng.sample(clientes, rng.randint(3, 6))
        tel = nuevo("TEL", "telefono", operador="OpA")
        dirc = nuevo("DIR", "direccion", ciudad=rng.choice(CIUDADES))
        for cli in familia:
            G.add_edge(cli, tel, rel="utiliza")
            G.add_edge(cli, dirc, rel="reside_en")
        marcar(familia + [tel, dirc], "identidad_sintetica")

    # ---------------------------------------------------------------- patrón 6: layering AML
    # Firma: ciclo dirigido de transferencias con importes decrecientes que vuelve al origen.
    for _ in range(3):
        ciclo = rng.sample(cuentas, rng.randint(4, 7))
        monto = rng.uniform(15000, 60000)
        t0 = rng.randint(0, SEG - 10 * 86400)
        for i in range(len(ciclo)):
            transferencia(ciclo[i], ciclo[(i + 1) % len(ciclo)],
                          monto * (0.97 ** i), t0 + i * rng.randint(3600, 86400),
                          patron="aml_layering")
        marcar(ciclo, "aml_layering")

    return G, dict(gt)


G_banco, GT_BANCO = build_banking_graph()

titulo("Grafo bancario sintético generado")
conteo_tipos = Counter(nx.get_node_attributes(G_banco, "tipo").values())
conteo_rel = Counter(d["rel"] for _, _, d in G_banco.edges(data=True))

print(f"Nodos  : {G_banco.number_of_nodes():,}")
print(f"Aristas: {G_banco.number_of_edges():,}")
print(f"Tipo   : MultiDiGraph heterogéneo\n")

resumen_banco = pd.DataFrame({
    "Tipo de nodo": list(conteo_tipos.keys()),
    "Cantidad": list(conteo_tipos.values()),
}).sort_values("Cantidad", ascending=False).reset_index(drop=True)
resumen_rel = pd.DataFrame({
    "Tipo de relación": list(conteo_rel.keys()),
    "Cantidad": list(conteo_rel.values()),
}).sort_values("Cantidad", ascending=False).reset_index(drop=True)
display(resumen_banco.T)
display(resumen_rel.T)

print("\nGround truth inyectado (nodos implicados por patrón):")
for patron, nodos in sorted(GT_BANCO.items()):
    tipos_impl = Counter(G_banco.nodes[n]["tipo"] for n in nodos)
    print(f"  {patron:22s} {len(nodos):>4} nodos  →  {dict(tipos_impl)}")

In [ ]:
# =============================================================================
# M1.4 — El mismo grafo expresado como Property Graph, RDF y Knowledge Graph
# =============================================================================

def a_property_graph(G: nx.MultiDiGraph, nodo: str) -> dict:
    """Vista Property Graph de un nodo: etiqueta, propiedades y relaciones salientes."""
    return {
        "id": nodo,
        "etiqueta": G.nodes[nodo]["tipo"],
        "propiedades": {k: v for k, v in G.nodes[nodo].items() if k != "tipo" and v is not None},
        "relaciones_salientes": [
            {"rel": d["rel"], "destino": v, "propiedades": {k: x for k, x in d.items() if k != "rel"}}
            for _, v, d in G.out_edges(nodo, data=True)
        ],
    }


def a_tripletas_rdf(G: nx.MultiDiGraph, nodos=None, base: str = "http://banca.ejemplo/") -> list:
    """Traduce un fragmento del Property Graph a tripletas RDF (sujeto, predicado, objeto).

    Dos observaciones sobre la traducción, que ilustran el precio de RDF:
      · Las propiedades de NODO se vuelven tripletas con objeto literal. Directo.
      · Las propiedades de ARISTA no tienen sitio: una tripleta no admite atributos.
        Hay que reificar la arista como un recurso propio, que es justamente lo que
        RDF-star vino a resolver. Aquí se marca dónde ocurriría esa pérdida.
    """
    nodos = list(G.nodes()) if nodos is None else list(nodos)
    conjunto = set(nodos)
    tripletas = []

    for n in nodos:
        datos = G.nodes[n]
        tripletas.append((f"{base}{n}", "rdf:type", f"{base}clase/{datos['tipo'].capitalize()}"))
        for clave, valor in datos.items():
            if clave in ("tipo", "patron_fraude") or valor is None:
                continue
            tripletas.append((f"{base}{n}", f"{base}prop/{clave}", f'"{valor}"'))

    for u, v, d in G.edges(data=True):
        if u in conjunto and v in conjunto:
            tripletas.append((f"{base}{u}", f"{base}rel/{d['rel']}", f"{base}{v}"))

    return tripletas


def tripletas_a_ntriples(tripletas: list) -> str:
    """Serializa tripletas al formato N-Triples, legible por cualquier motor RDF."""
    lineas = []
    for s, p, o in tripletas:
        s_ = f"<{s}>"
        p_ = p if p.startswith("rdf:") else f"<{p}>"
        o_ = o if o.startswith('"') else f"<{o}>"
        lineas.append(f"{s_} {p_} {o_} .")
    return "\n".join(lineas)


# Tomamos un cliente con actividad y lo mostramos en los tres formalismos.
cliente_ejemplo = next(n for n, d in G_banco.nodes(data=True)
                       if d["tipo"] == "cliente" and G_banco.out_degree(n) >= 4)
vecindario = [cliente_ejemplo] + list(G_banco.successors(cliente_ejemplo))[:4]

titulo("El mismo nodo en tres formalismos")

print(">>> PROPERTY GRAPH (dict de propiedades sobre nodo y arista)\n")
print(json.dumps(a_property_graph(G_banco, cliente_ejemplo), indent=2, ensure_ascii=False)[:1100])

print("\n\n>>> RDF / N-TRIPLES (todo son tripletas sujeto-predicado-objeto)\n")
print(tripletas_a_ntriples(a_tripletas_rdf(G_banco, vecindario)[:12]))

print("\n\n>>> KNOWLEDGE GRAPH (grafo + ontología + reglas de inferencia)\n")
print("""Ontología declarada (OWL/RDFS):
    :Cuenta        rdfs:subClassOf  :ProductoFinanciero .
    :Tarjeta       rdfs:subClassOf  :ProductoFinanciero .
    :comparte_dispositivo  rdf:type  owl:SymmetricProperty ,
                                     owl:TransitiveProperty .
    :emite  owl:propertyChainAxiom  ( :posee :emite ) .

Aristas INFERIDAS, que no están almacenadas en ninguna tabla:
    CLI_0007  :comparte_dispositivo  CLI_0142     (ambos usan DEV_0311; simetría)
    CLI_0142  :comparte_dispositivo  CLI_0089     (transitividad → anillo completo)
    CLI_0007  :emite                 TRF_0455     (cadena posee∘emite)

Ese es el valor del Knowledge Graph: la consulta "dame el anillo de clientes que
comparten dispositivo" no necesita un JOIN recursivo ni código; el razonador la
deriva de la ontología.""")

---
---

# Módulo 2 — Construcción del grafo con NetworkX

NetworkX ofrece cuatro clases y elegir mal cuesta caro más adelante:

| Clase | Dirigido | Aristas paralelas | Cuándo usarla |
|---|---|---|---|
| `Graph` | No | No | Relaciones simétricas: *comparte dispositivo* |
| `DiGraph` | Sí | No | Flujos: *transfiere a*, *paga a* |
| `MultiGraph` | No | Sí | Varias interacciones simétricas entre el mismo par |
| `MultiDiGraph` | Sí | Sí | **Banca**: A transfiere a B muchas veces y cada vez cuenta |

Dos avisos que ahorran depuración:

- **El multigrafo no es gratis.** Muchos algoritmos (`triangles`, `clustering`, los de link
  prediction, Louvain) no aceptan multigrafos y hay que colapsarlos antes. La estrategia
  habitual: mantener el `MultiDiGraph` como fuente de verdad y derivar proyecciones simples
  cuando un algoritmo lo exija, guardando el número de aristas colapsadas como peso.
- **Los atributos de nodo consumen memoria.** Adjuntar las 165 features de Elliptic como
  atributos de nodo significa 203.769 diccionarios de 165 claves: del orden de 1–2 GB en dicts
  de Python frente a los 135 MB del mismo dato en un array NumPy. Por eso mantenemos las
  features en un DataFrame indexado por `txId` y solo colgamos del grafo lo que la topología
  necesita (`timestep`, `clase`).

In [ ]:
# =============================================================================
# M2.1 — La API básica: nodos, aristas y atributos
# =============================================================================

titulo("API básica de NetworkX")

# --- Creación y adición explícita ---
g = nx.DiGraph()
g.add_node("CTA_A", tipo="cuenta", saldo=15000, pais="PE")          # nodo con atributos
g.add_nodes_from(["CTA_B", "CTA_C"], tipo="cuenta")                 # varios de golpe
g.add_edge("CTA_A", "CTA_B", monto=2500.0, ts=1700000000)           # arista con atributos
g.add_edges_from([("CTA_B", "CTA_C", {"monto": 900.0}),
                  ("CTA_C", "CTA_A", {"monto": 120.0})])

# --- Modificación posterior de atributos ---
g.nodes["CTA_B"]["saldo"] = 300                                     # sobre un nodo concreto
nx.set_node_attributes(g, {"CTA_C": "alto"}, name="riesgo")         # de forma masiva
nx.set_edge_attributes(g, {("CTA_A", "CTA_B"): "app"}, name="canal")

print(f"Nodos     : {list(g.nodes())}")
print(f"Aristas   : {list(g.edges())}")
print(f"Nodo CTA_A: {g.nodes['CTA_A']}")
print(f"Arista A→B: {g.edges['CTA_A', 'CTA_B']}")
print(f"Sucesores de CTA_A: {list(g.successors('CTA_A'))} | Predecesores: {list(g.predecessors('CTA_A'))}")
print(f"Grados    : entrada={dict(g.in_degree())} salida={dict(g.out_degree())}")

# --- DiGraph vs MultiDiGraph: qué se pierde al colapsar ---
simple = nx.DiGraph()
multi = nx.MultiDiGraph()
tres_transferencias = [("CTA_A", "CTA_B", {"monto": 100}),
                       ("CTA_A", "CTA_B", {"monto": 250}),
                       ("CTA_A", "CTA_B", {"monto": 900})]
simple.add_edges_from(tres_transferencias)
multi.add_edges_from(tres_transferencias)

print(f"\nTres transferencias A→B de 100, 250 y 900:")
print(f"  DiGraph      → {simple.number_of_edges()} arista.  monto conservado: "
      f"{simple.edges['CTA_A','CTA_B']['monto']}  (¡se pierden las dos primeras!)")
print(f"  MultiDiGraph → {multi.number_of_edges()} aristas. montos conservados: "
      f"{[d['monto'] for _, _, d in multi.edges(data=True)]}")
print("  Moraleja: si cada interacción individual importa (y en fraude importa), MultiDiGraph.")

In [ ]:
# =============================================================================
# M2.2 — Construcción del grafo de Elliptic desde DataFrames
# =============================================================================

MAPA_CLASE = {"1": "ilicito", "2": "licito", "unknown": "desconocido"}
MAPA_Y = {"1": 1, "2": 0, "unknown": -1}   # codificación numérica para ML: -1 = sin etiqueta


def construir_grafo_elliptic(edgelist: pd.DataFrame, features: pd.DataFrame,
                             classes: pd.DataFrame, dirigido: bool = True,
                             con_atributos: bool = True) -> nx.Graph:
    """Construye el grafo de transacciones de Elliptic a partir de los tres DataFrames.

    Solo se adjuntan al grafo los atributos que la topología necesita (`timestep`,
    `clase`, `y`). Las 165 features numéricas se quedan en `df_features`, indexadas
    por `txId`, y se alinean por posición cuando hacen falta (Módulos 15 y 16).
    Adjuntarlas como atributos de nodo multiplicaría por diez el consumo de memoria
    sin ninguna ventaja algorítmica.
    """
    crear_con = nx.DiGraph() if dirigido else nx.Graph()
    G = nx.from_pandas_edgelist(edgelist, source="txId1", target="txId2", create_using=crear_con)

    if con_atributos:
        nx.set_node_attributes(G, features.set_index("txId")["timestep"].to_dict(), "timestep")
        clase = classes.set_index("txId")["class"]
        nx.set_node_attributes(G, clase.map(MAPA_CLASE).to_dict(), "clase")
        nx.set_node_attributes(G, clase.map(MAPA_Y).to_dict(), "y")

    return G


def tabla_nodos(G: nx.Graph, features: pd.DataFrame, classes: pd.DataFrame) -> pd.DataFrame:
    """Tabla maestra de nodos: una fila por nodo del grafo, en el MISMO orden que `G.nodes()`.

    Ese orden es crítico: las matrices dispersas y los tensores de los Módulos 13 y 16
    se construyen con `nx.to_scipy_sparse_array(G, nodelist=...)`, y si la tabla y la
    matriz no comparten orden, las features quedan asignadas al nodo equivocado —un
    error silencioso que produce modelos que "funcionan" pero no significan nada.
    """
    nodos = list(G.nodes())
    df = pd.DataFrame({"txId": nodos})
    df = df.merge(features[["txId", "timestep"]], on="txId", how="left")
    df = df.merge(classes, on="txId", how="left")
    df["clase"] = df["class"].map(MAPA_CLASE).fillna("desconocido")
    df["y"] = df["class"].map(MAPA_Y).fillna(-1).astype(int)
    return df.drop(columns=["class"])


titulo("Construcción del grafo de Elliptic")
G, t_construccion = cronometrar(
    construir_grafo_elliptic, df_edgelist, df_features, df_classes,
    etiqueta="construir_grafo_elliptic",
)
df_nodos = tabla_nodos(G, df_features, df_classes)

print(f"\n{G}")
print(f"Atributos del primer nodo: {G.nodes[list(G.nodes())[0]]}")
print(f"\nTabla maestra de nodos ({len(df_nodos):,} filas), alineada con G.nodes():")
display(df_nodos.head())
print(f"Orden idéntico a G.nodes(): {list(df_nodos['txId']) == list(G.nodes())}")
print(f"\nDistribución de clases en el grafo:\n{df_nodos['clase'].value_counts().to_string()}")

# Versión no dirigida: la necesitan Louvain, clustering, link prediction y los embeddings.
G_und = G.to_undirected()
print(f"\nProyección no dirigida: {G_und}")
print(f"Aristas colapsadas por reciprocidad: {G.number_of_edges() - G_und.number_of_edges()}")

In [ ]:
# =============================================================================
# M2.3 — Subgrafos: las tres formas de reducir un grafo grande
# =============================================================================

def subgrafo_timestep(G: nx.Graph, t: int, df_nodos: pd.DataFrame) -> nx.Graph:
    """Subgrafo inducido por todos los nodos de un timestep concreto.

    En Elliptic esta partición es especial: ninguna arista cruza timesteps, así que
    el resultado no pierde ni una sola arista de los nodos seleccionados (Módulo 4).
    """
    nodos = df_nodos.loc[df_nodos["timestep"] == t, "txId"]
    return G.subgraph(nodos).copy()


def subgrafo_ego(G: nx.Graph, nodo, radio: int = 2, dirigido: bool = False) -> nx.Graph:
    """Red ego: el nodo, sus vecinos hasta `radio` saltos y las aristas entre ellos.

    Es la unidad de análisis de la investigación de fraude: cuando un analista abre
    una alerta, lo que mira es exactamente esto.
    """
    return nx.ego_graph(G, nodo, radius=radio, undirected=not dirigido)


def muestrear_subgrafo(G: nx.Graph, n_objetivo: int = 2000, metodo: str = "bfs",
                       nodo_inicial=None, seed: int = SEED) -> nx.Graph:
    """Extrae un subgrafo manejable de un grafo grande. Cada método sesga distinto.

    · "bfs"       — bola de nieve desde un nodo. Conserva la estructura local y
                    produce subgrafos conexos, pero sobre-representa la zona densa
                    donde arranca.
    · "aleatorio" — nodos al azar y aristas inducidas. Insesgado en nodos, pero
                    DESTRUYE la conectividad: en un grafo disperso casi todas las
                    aristas se pierden porque rara vez se muestrean ambos extremos.
    · "aristas"   — aristas al azar. Conserva mejor la densidad, pero sobre-representa
                    los nodos de grado alto (aparecen en más aristas).
    · "grado"     — los N nodos de mayor grado. Útil para inspeccionar hubs; no vale
                    como muestra representativa porque es el sesgo llevado al extremo.

    El notebook original usaba `list(G.nodes())[:80]`, que es un cuarto método no
    listado aquí —"los primeros del diccionario"— y produce un subgrafo casi sin
    aristas: por eso su cálculo de camino mínimo promedio siempre fallaba.
    """
    rng = random.Random(seed)

    if metodo == "bfs":
        inicio = nodo_inicial if nodo_inicial is not None else max(G.degree, key=lambda x: x[1])[0]
        vista = G.to_undirected(as_view=True) if G.is_directed() else G
        visitados = {inicio}
        cola = deque([inicio])
        while cola and len(visitados) < n_objetivo:
            actual = cola.popleft()
            for vecino in vista.neighbors(actual):
                if vecino not in visitados:
                    visitados.add(vecino)
                    cola.append(vecino)
                    if len(visitados) >= n_objetivo:
                        break
        seleccion = visitados

    elif metodo == "aleatorio":
        seleccion = set(rng.sample(list(G.nodes()), min(n_objetivo, G.number_of_nodes())))

    elif metodo == "aristas":
        aristas = list(G.edges())
        seleccion = set()
        for u, v in rng.sample(aristas, len(aristas)):
            seleccion.update([u, v])
            if len(seleccion) >= n_objetivo:
                break

    elif metodo == "grado":
        seleccion = {n for n, _ in sorted(G.degree, key=lambda x: x[1], reverse=True)[:n_objetivo]}

    else:
        raise ValueError(f"Método desconocido: {metodo!r}")

    return G.subgraph(seleccion).copy()


titulo("Comparación de estrategias de muestreo (objetivo: 2.000 nodos)")
filas = []
for metodo in ["bfs", "aleatorio", "aristas", "grado"]:
    sub = muestrear_subgrafo(G, 2000, metodo=metodo)
    sub_und = sub.to_undirected()
    filas.append({
        "método": metodo,
        "nodos": sub.number_of_nodes(),
        "aristas": sub.number_of_edges(),
        "densidad": round(nx.density(sub), 6),
        "grado medio": round(2 * sub.number_of_edges() / max(sub.number_of_nodes(), 1), 2),
        "componentes": nx.number_connected_components(sub_und),
        "% en la mayor comp.": round(
            100 * len(max(nx.connected_components(sub_und), key=len)) / max(sub.number_of_nodes(), 1), 1
        ) if sub.number_of_nodes() else 0,
    })
display(pd.DataFrame(filas).set_index("método"))

print("\nEl muestreo aleatorio conserva 2.000 nodos pero casi ninguna arista: en un grafo con")
print("densidad 5,6e-06, la probabilidad de capturar ambos extremos de una arista es ínfima.")
print("Para analizar ESTRUCTURA hay que muestrear con BFS; para estimar propiedades de NODO,")
print("el muestreo aleatorio es el único insesgado. El método correcto depende de la pregunta.")

In [ ]:
# =============================================================================
# M2.4 — Subgrafos de trabajo reutilizados por el resto del notebook
# =============================================================================

# H: subgrafo conexo mediano. Aquí van los algoritmos O(n²)/O(n³) que no admiten
#    el grafo completo (Floyd-Warshall, Girvan-Newman, betweenness exacta).
H = muestrear_subgrafo(G, 1200, metodo="bfs")
H_und = H.to_undirected()

# T35: un timestep completo. Es la unidad natural de análisis en Elliptic porque
#      ninguna arista cruza timesteps: el snapshot está completo, no truncado.
TIMESTEP_FOCO = 35
G_t = subgrafo_timestep(G, TIMESTEP_FOCO, df_nodos)

titulo("Subgrafos de trabajo")
for nombre, sub in [("H (BFS, 1.200 nodos)", H), (f"G_t (timestep {TIMESTEP_FOCO})", G_t)]:
    und = sub.to_undirected()
    mayor = max(nx.connected_components(und), key=len) if sub.number_of_nodes() else set()
    print(f"{nombre:28s} {sub.number_of_nodes():>6,} nodos  {sub.number_of_edges():>6,} aristas  "
          f"{nx.number_connected_components(und):>4} comp.  mayor={len(mayor):>5,}")

# Red ego de un hub: la vista que tendría un analista al abrir una alerta.
hub = max(G.in_degree, key=lambda x: x[1])[0]
ego = subgrafo_ego(G, hub, radio=2)
print(f"\nRed ego de radio 2 del mayor receptor (txId {hub}, in-degree "
      f"{G.in_degree(hub)}): {ego.number_of_nodes()} nodos, {ego.number_of_edges()} aristas")

fig, ejes = plt.subplots(1, 2, figsize=(14, 5.5))
for eje, (sub, titulo_ax) in zip(ejes, [(H, "H — muestreo BFS (1.200 nodos)"),
                                        (ego, f"Red ego radio 2 de txId {hub}")]):
    pos = nx.spring_layout(sub, seed=SEED, k=0.35, iterations=45)
    colores_nodo = [
        {"ilicito": COLORES["ilicito"], "licito": COLORES["licito"]}.get(
            sub.nodes[n].get("clase"), COLORES["desconocido"])
        for n in sub.nodes()
    ]
    tam = [90 if n == hub else 14 for n in sub.nodes()]
    nx.draw_networkx_edges(sub, pos, ax=eje, edge_color="#C9CFD6", width=0.5,
                           arrows=False, alpha=0.7)
    nx.draw_networkx_nodes(sub, pos, ax=eje, node_color=colores_nodo, node_size=tam,
                           linewidths=0)
    eje.set_title(titulo_ax)
    eje.axis("off")

manijas = [plt.Line2D([], [], marker="o", ls="", color=c, label=l, markersize=7)
           for l, c in [("Ilícito", COLORES["ilicito"]), ("Lícito", COLORES["licito"]),
                        ("Desconocido", COLORES["desconocido"])]]
ejes[1].legend(handles=manijas, loc="upper right", fontsize=8, frameon=True)
plt.tight_layout()
plt.show()

---
---

# Módulo 3 — Estadísticas descriptivas

Las primeras métricas. Su función no es lucirse, sino **decidir qué algoritmos son viables**
en los módulos siguientes: la densidad determina si conviene una matriz dispersa o densa, la
distribución de grados anticipa si habrá hubs que dominen el PageRank, y el número de nodos
descarta de entrada cualquier algoritmo cuadrático.

## Definiciones

**Densidad** — cuántas de las aristas posibles existen realmente:

$$\rho_{\text{dirigido}} = \frac{m}{n(n-1)} \qquad \rho_{\text{no dirigido}} = \frac{2m}{n(n-1)}$$

**Grado** — número de aristas incidentes a un vértice. En un digrafo:

$$k_i^{\text{in}} = \sum_j A_{ji} \qquad k_i^{\text{out}} = \sum_j A_{ij} \qquad k_i = k_i^{\text{in}} + k_i^{\text{out}}$$

**Distribución de grados** $P(k)$ — probabilidad de que un vértice al azar tenga grado $k$.
En redes reales suele seguir una **ley de potencias**:

$$P(k) \sim k^{-\gamma}, \quad \gamma \in [2, 3]$$

Esto tiene una consecuencia que conviene interiorizar: **la media del grado no describe la
red**. Con $\gamma < 3$ la varianza diverge, así que "el grado medio es 1,15" es un dato casi
vacío cuando existe un nodo con grado 472. La distribución es de cola pesada: la mayoría de
nodos tiene grado 1 o 2 y unos pocos concentran todo. Cualquier intuición basada en la media
—o en una distribución normal— será errónea.

**Hub** — nodo cuyo grado excede en órdenes de magnitud al típico. En un grafo de transacciones,
un hub de entrada suele ser un exchange o un servicio de custodia (mucha gente le envía); un
hub de salida suele ser un mezclador o un pagador masivo (envía a mucha gente).

**Self-loop** — arista de un nodo a sí mismo. En Elliptic no debería haber ninguno (una
transacción no se financia a sí misma); si aparecieran, indicarían un problema en el dato.

In [ ]:
# =============================================================================
# M3.1 — Estadísticas estructurales básicas
# =============================================================================

def estadisticas_basicas(G: nx.Graph, nombre: str = "", calcular_asortatividad: bool = True) -> dict:
    """Panel completo de métricas descriptivas de un grafo.

    Reúne en una sola pasada todo lo que se necesita para decidir qué algoritmos son
    viables después. Las métricas que solo aplican a digrafos (in/out degree,
    reciprocidad, aciclicidad) se omiten limpiamente en grafos no dirigidos.
    """
    n, m = G.number_of_nodes(), G.number_of_edges()
    dirigido = G.is_directed()
    grados = np.array([d for _, d in G.degree()])

    est = {
        "grafo": nombre,
        "nodos (n)": n,
        "aristas (m)": m,
        "densidad": nx.density(G),
        "self-loops": nx.number_of_selfloops(G),
        "nodos aislados": int(np.sum(grados == 0)),
        "grado medio": float(grados.mean()) if n else 0.0,
        "grado mediano": float(np.median(grados)) if n else 0.0,
        "grado máximo": int(grados.max()) if n else 0,
        "desv. típica del grado": float(grados.std()) if n else 0.0,
    }

    if dirigido:
        gin = np.array([d for _, d in G.in_degree()])
        gout = np.array([d for _, d in G.out_degree()])
        est.update({
            "in-degree medio": float(gin.mean()),
            "in-degree máximo": int(gin.max()),
            "out-degree máximo": int(gout.max()),
            "fuentes (in=0)": int(np.sum(gin == 0)),
            "sumideros (out=0)": int(np.sum(gout == 0)),
            "reciprocidad": nx.reciprocity(G) if m else 0.0,
            "es DAG": nx.is_directed_acyclic_graph(G),
        })

    if calcular_asortatividad and m > 0:
        try:
            est["asortatividad de grado"] = nx.degree_assortativity_coefficient(G)
        except Exception:
            est["asortatividad de grado"] = float("nan")

    return est


titulo("Estadísticas descriptivas de Elliptic")
est_G, _ = cronometrar(estadisticas_basicas, G, etiqueta="estadisticas_basicas(G)", nombre="Elliptic completo")
print()
for k, v in est_G.items():
    if k != "grafo":
        print(f"  {k:26s} {formatear(v):>16s}")

# Comparativa entre los grafos de trabajo del notebook.
comparativa_grafos = pd.DataFrame([
    estadisticas_basicas(G, "Elliptic completo"),
    estadisticas_basicas(G_t, f"Timestep {TIMESTEP_FOCO}"),
    estadisticas_basicas(H, "H (BFS 1.200)"),
    estadisticas_basicas(nx.DiGraph(G_banco), "Banco (proyectado)"),
]).set_index("grafo").T
titulo("Comparativa entre grafos de trabajo", nivel=2)
display(comparativa_grafos.fillna("—").apply(lambda col: col.map(formatear)))

### Lo que dicen estos números

- **Densidad $5{,}6\times10^{-6}$**: existe una de cada ~177.000 aristas posibles. Es un grafo
  extremadamente disperso, lo cual es una buena noticia: la matriz de adyacencia densa ocuparía
  $203.769^2 \times 8$ bytes ≈ **332 GB**, mientras que en formato disperso son unos 3 MB. Todos
  los módulos posteriores usan `scipy.sparse` por esta razón.
- **Cero self-loops y cero nodos aislados**: el dato está limpio y todo nodo participa en el flujo.
- **Grado medio ≈ 2,3** (contando entrada y salida) con **grado máximo 472**: una diferencia de
  dos órdenes de magnitud que confirma la cola pesada.
- **Es un DAG**: no hay ni un solo ciclo dirigido. Tiene sentido —el output de una transacción
  de Bitcoin solo puede gastarse en transacciones posteriores, así que el tiempo impone un orden
  topológico. Esto tiene consecuencias fuertes: **no habrá componentes fuertemente conexas**
  no triviales (Módulo 4), **no habrá ciclos de layering que detectar** (Módulo 8) y la matriz de
  adyacencia es nilpotente, lo que hace que la centralidad de Katz converja para cualquier $\alpha$
  (Módulo 6).
- **Reciprocidad 0**: consecuencia directa de ser un DAG. Si A financia a B, B nunca financia a A.

In [ ]:
# =============================================================================
# M3.2 — Tabla de grados y detección de hubs
# =============================================================================

def tabla_grados(G: nx.Graph, df_base: pd.DataFrame | None = None) -> pd.DataFrame:
    """DataFrame con el grado de cada nodo, listo para enriquecerse con más métricas.

    Sustituye al `degree_df` del notebook original y corrige dos cosas: construye
    los tres grados de una vez a partir de las vistas de NetworkX (sin asumir que
    `in_degree` y `out_degree` iteran en el mismo orden) y se une opcionalmente a
    la tabla maestra de nodos para arrastrar `timestep` y `clase`.
    """
    if G.is_directed():
        gin, gout = dict(G.in_degree()), dict(G.out_degree())
        nodos = list(G.nodes())
        df = pd.DataFrame({
            "txId": nodos,
            "in_degree": [gin[n] for n in nodos],
            "out_degree": [gout[n] for n in nodos],
        })
        df["degree"] = df["in_degree"] + df["out_degree"]
    else:
        grados = dict(G.degree())
        df = pd.DataFrame({"txId": list(grados.keys()), "degree": list(grados.values())})

    if df_base is not None:
        df = df.merge(df_base, on="txId", how="left")
    return df


def detectar_hubs(df: pd.DataFrame, columna: str = "degree", n: int = 10,
                  umbral_sigma: float = 3.0) -> tuple:
    """Identifica hubs por dos criterios complementarios.

    · Top-N absoluto: los N nodos con mayor valor. Siempre devuelve algo.
    · Criterio estadístico: nodos por encima de media + `umbral_sigma`·desviación.
      En una distribución de cola pesada este criterio marca muchísimos nodos, y esa
      es precisamente la señal de que la distribución NO es normal: en una gaussiana
      esperaríamos un 0,13 % de nodos más allá de 3σ.
    """
    serie = df[columna]
    umbral = serie.mean() + umbral_sigma * serie.std()
    return top_n(df, columna, n), int((serie > umbral).sum()), float(umbral)


df_grados = tabla_grados(G, df_nodos)

titulo("Hubs de Elliptic")
for col, descripcion in [("in_degree", "RECEPCIÓN (posibles exchanges / servicios de custodia)"),
                         ("out_degree", "DISPERSIÓN (posibles mezcladores / pagadores masivos)")]:
    top, n_sobre, umbral = detectar_hubs(df_grados, col, n=5)
    esperado_normal = 0.0013 * len(df_grados)
    print(f"\nHubs de {descripcion}")
    print(f"  Umbral 3σ = {umbral:.1f} · nodos por encima: {n_sobre:,} "
          f"({100*n_sobre/len(df_grados):.2f}%, en una normal se esperaría {esperado_normal:.0f} = 0.13%)")
    display(top[["txId", "in_degree", "out_degree", "timestep", "clase"]])

In [ ]:
# =============================================================================
# M3.3 — Distribución de grados y ajuste de ley de potencias
# =============================================================================

def ajustar_ley_potencias(grados, k_min: int = 1) -> dict:
    """Estima el exponente γ de P(k) ~ k^(-γ) por dos vías.

    · Regresión lineal sobre log P(k) vs log k: es la que se ve en todos los papers
      antiguos y la que produce el gráfico intuitivo, pero está sesgada porque los
      bins de la cola tienen muy pocas observaciones y pesan lo mismo que los del
      inicio, donde hay decenas de miles.
    · Estimador de máxima verosimilitud (Clauset, Shalizi & Newman 2009), que es el
      correcto: γ = 1 + n / Σ ln(k_i / (k_min − ½)).

    Se devuelven los dos para que se vea la diferencia; si divergen mucho, la
    distribución probablemente no es una ley de potencias pura.
    """
    valores = np.asarray([g for g in grados if g >= k_min], dtype=float)
    conteo = Counter(valores)
    k = np.array(sorted(conteo), dtype=float)
    pk = np.array([conteo[x] for x in k], dtype=float)
    pk = pk / pk.sum()

    coef = np.polyfit(np.log10(k), np.log10(pk), 1)
    gamma_ols = -coef[0]
    gamma_mle = 1.0 + len(valores) / np.sum(np.log(valores / (k_min - 0.5)))

    # R² del ajuste log-log, para saber cuánto fiarse de la recta.
    pred = np.polyval(coef, np.log10(k))
    ss_res = np.sum((np.log10(pk) - pred) ** 2)
    ss_tot = np.sum((np.log10(pk) - np.log10(pk).mean()) ** 2)

    return {
        "k": k, "pk": pk, "coef": coef,
        "gamma_ols": gamma_ols, "gamma_mle": gamma_mle,
        "r2": 1 - ss_res / ss_tot, "k_min": k_min, "n": len(valores),
    }


def distribucion_grados(G: nx.Graph, df_grados: pd.DataFrame, titulo_fig: str = "") -> dict:
    """Cuatro vistas de la distribución de grados, cada una con un propósito distinto.

    1. Histograma lineal: el que produce el notebook original. Muestra que casi todo
       está en el primer bin y poco más — es informativo justamente por lo que NO deja ver.
    2. Histograma log-log: revela la cola que el lineal aplasta.
    3. CCDF: P(K ≥ k). Es la representación correcta de una cola pesada porque no
       depende de cómo se elijan los bins.
    4. Dispersión in vs out: separa receptores puros de dispersores puros.
    """
    grados = df_grados["degree"].values
    ajuste = ajustar_ley_potencias(grados)

    fig, ejes = plt.subplots(2, 2, figsize=(13, 8.5))
    if titulo_fig:
        fig.suptitle(titulo_fig, fontsize=13, fontweight="bold")

    # 1. Histograma lineal
    ejes[0, 0].hist(grados, bins=60, color=COLORES["licito"], edgecolor="white", linewidth=0.4)
    ejes[0, 0].set_title("1. Histograma lineal — engañoso")
    ejes[0, 0].set_xlabel("Grado")
    ejes[0, 0].set_ylabel("Nº de nodos")
    ejes[0, 0].text(0.45, 0.75, f"El {100*np.mean(grados <= 3):.1f}% de los nodos\ntiene grado ≤ 3",
                    transform=ejes[0, 0].transAxes, fontsize=9,
                    bbox=dict(boxstyle="round", fc="#FFF3CD", ec="#E0C97F"))

    # 2. Log-log con la recta ajustada
    ejes[0, 1].scatter(ajuste["k"], ajuste["pk"], s=16, color=COLORES["licito"], alpha=0.65)
    k_line = np.logspace(0, np.log10(ajuste["k"].max()), 60)
    ejes[0, 1].plot(k_line, 10 ** np.polyval(ajuste["coef"], np.log10(k_line)),
                    "--", color=COLORES["ilicito"], lw=1.8,
                    label=f"γ(OLS) = {ajuste['gamma_ols']:.2f}  (R²={ajuste['r2']:.2f})")
    ejes[0, 1].set_xscale("log")
    ejes[0, 1].set_yscale("log")
    ejes[0, 1].set_title("2. P(k) en log-log — aparece la ley de potencias")
    ejes[0, 1].set_xlabel("Grado k")
    ejes[0, 1].set_ylabel("P(k)")
    ejes[0, 1].legend(fontsize=8)

    # 3. CCDF
    orden = np.sort(grados)
    ccdf = 1.0 - np.arange(len(orden)) / len(orden)
    ejes[1, 0].plot(orden, ccdf, color=COLORES["acento"], lw=1.8)
    ejes[1, 0].set_xscale("log")
    ejes[1, 0].set_yscale("log")
    ejes[1, 0].set_title("3. CCDF P(K ≥ k) — sin dependencia de los bins")
    ejes[1, 0].set_xlabel("Grado k")
    ejes[1, 0].set_ylabel("P(K ≥ k)")

    # 4. In vs out
    if "in_degree" in df_grados:
        ejes[1, 1].scatter(df_grados["in_degree"] + 1, df_grados["out_degree"] + 1,
                           s=7, alpha=0.25, color=COLORES["neutro"])
        ejes[1, 1].set_xscale("log")
        ejes[1, 1].set_yscale("log")
        ejes[1, 1].set_title("4. In-degree vs out-degree (+1 para la escala log)")
        ejes[1, 1].set_xlabel("In-degree + 1")
        ejes[1, 1].set_ylabel("Out-degree + 1")
        ejes[1, 1].text(0.03, 0.92, "Receptores puros ↓", transform=ejes[1, 1].transAxes, fontsize=8)
        ejes[1, 1].text(0.62, 0.06, "← Dispersores puros", transform=ejes[1, 1].transAxes, fontsize=8)

    plt.tight_layout()
    plt.show()
    return ajuste


ajuste = distribucion_grados(G, df_grados, "Distribución de grados — Elliptic (203.769 nodos)")

titulo("Ajuste de la ley de potencias", nivel=2)
print(f"  γ por regresión log-log (OLS) : {ajuste['gamma_ols']:.3f}   (R² = {ajuste['r2']:.3f})")
print(f"  γ por máxima verosimilitud    : {ajuste['gamma_mle']:.3f}   ← el estimador correcto")
print(f"  Grados distintos observados   : {len(ajuste['k'])}")
print(f"\n  Media del grado    : {df_grados['degree'].mean():.2f}")
print(f"  Mediana del grado  : {df_grados['degree'].median():.0f}")
print(f"  Máximo             : {df_grados['degree'].max()}  ({df_grados['degree'].max() / df_grados['degree'].mean():.0f}x la media)")
print(f"  Percentil 99       : {df_grados['degree'].quantile(0.99):.0f}")
print("\n  Con γ < 3 la varianza teórica diverge: la media NO es un resumen válido de esta")
print("  distribución. Los módulos siguientes trabajan con percentiles y rangos, no con medias.")

---
---

# Módulo 4 — Componentes

Una **componente conexa** es un subconjunto máximo de vértices en el que existe camino entre
cualquier par. En grafos dirigidos la noción se desdobla, y la diferencia importa:

| Tipo | Definición | Interpretación en fraude |
|---|---|---|
| **Débilmente conexa (WCC)** | Conexa ignorando la dirección de las aristas | "Este grupo de entidades está relacionado de alguna forma" |
| **Fuertemente conexa (SCC)** | Existe camino dirigido $u \to v$ **y** $v \to u$ | "El dinero puede circular en círculo dentro de este grupo" — señal de *layering* |

- **Largest Connected Component (LCC)**: la componente de mayor tamaño.
- **Giant Component**: cuando la LCC contiene una fracción constante del grafo (típicamente
  >50 %) y las demás son minúsculas. Es un fenómeno de transición de fase: en un grafo aleatorio
  aparece de golpe cuando el grado medio supera 1.

## Por qué importa en detección de fraude

- **Redes criminales**: un anillo de fraude organizado suele aparecer como una componente
  densa y **separada** del resto. Que un grupo de 40 cuentas forme su propia isla, sin ninguna
  conexión con la componente principal, es en sí mismo una anomalía.
- **Redes aisladas**: componentes pequeñas de entidades recién creadas que solo interactúan
  entre sí son la firma clásica de identidades sintéticas.
- **SCC no triviales**: en un grafo de flujo de dinero, un ciclo dirigido significa que los
  fondos vuelven al punto de partida. Casi nunca es legítimo.

In [ ]:
# =============================================================================
# M4.1 — Análisis de componentes
# =============================================================================

def analizar_componentes(G: nx.Graph, nombre: str = "") -> dict:
    """Descompone el grafo en componentes y resume su estructura.

    Devuelve tanto el resumen numérico como las listas de componentes, para que
    quien llame pueda seguir trabajando con ellas sin recalcularlas (el cálculo es
    O(n+m) pero sobre 200 mil nodos se nota si se repite en cada celda).
    """
    dirigido = G.is_directed()
    n = G.number_of_nodes()

    if dirigido:
        debiles = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
        fuertes = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
    else:
        debiles = sorted(nx.connected_components(G), key=len, reverse=True)
        fuertes = []

    tam_debiles = [len(c) for c in debiles]
    lcc = debiles[0] if debiles else set()

    resumen = {
        "grafo": nombre,
        "componentes débiles": len(debiles),
        "tamaño LCC": len(lcc),
        "% del grafo en la LCC": round(100 * len(lcc) / n, 2) if n else 0,
        "¿componente gigante?": (len(lcc) / n > 0.5) if n else False,
        "tamaño mediano de componente": int(np.median(tam_debiles)) if tam_debiles else 0,
        "componentes de 1 nodo": sum(1 for t in tam_debiles if t == 1),
    }
    if dirigido:
        tam_fuertes = [len(c) for c in fuertes]
        resumen.update({
            "componentes fuertes": len(fuertes),
            "mayor SCC": max(tam_fuertes) if tam_fuertes else 0,
            "SCC no triviales (>1 nodo)": sum(1 for t in tam_fuertes if t > 1),
        })

    return {"resumen": resumen, "debiles": debiles, "fuertes": fuertes, "lcc": lcc}


titulo("Componentes de Elliptic")
comp, _ = cronometrar(analizar_componentes, G, etiqueta="analizar_componentes(G)", nombre="Elliptic")
for k, v in comp["resumen"].items():
    if k != "grafo":
        print(f"  {k:32s} {formatear(v):>12s}")

tam_componentes = np.array([len(c) for c in comp["debiles"]])

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2))
ejes[0].hist(tam_componentes, bins=30, color=COLORES["licito"], edgecolor="white", linewidth=0.5)
ejes[0].set_title("Distribución del tamaño de las componentes débiles")
ejes[0].set_xlabel("Nodos por componente")
ejes[0].set_ylabel("Nº de componentes")

ejes[1].bar(range(len(tam_componentes)), np.sort(tam_componentes)[::-1],
            color=COLORES["acento"], width=0.85)
ejes[1].set_title("Componentes ordenadas por tamaño")
ejes[1].set_xlabel("Ranking de la componente")
ejes[1].set_ylabel("Nodos")
plt.tight_layout()
plt.show()

print(f"\nTamaños: mínimo {tam_componentes.min():,} · mediana {int(np.median(tam_componentes)):,} · "
      f"máximo {tam_componentes.max():,}")
print("Ninguna componente domina: NO hay componente gigante. Es un perfil atípico para una red")
print("real, y la explicación no es estructural sino temporal. La celda siguiente lo demuestra.")

In [ ]:
# =============================================================================
# M4.2 — El hallazgo: las componentes SON los timesteps
# =============================================================================

def verificar_componentes_vs_timesteps(G: nx.Graph, df_nodos: pd.DataFrame,
                                       componentes=None) -> dict:
    """Comprueba si la partición en componentes coincide con la partición temporal.

    Contrasta tres afirmaciones independientes:
      1. Ninguna arista conecta nodos de timesteps distintos.
      2. Cada componente débil contiene nodos de un único timestep (componente "pura").
      3. Cada timestep se corresponde con exactamente una componente.

    Si las tres se cumplen, el grafo no es una red única con dimensión temporal: es
    una colección de 49 redes independientes, y eso cambia por completo qué análisis
    tienen sentido.
    """
    ts = df_nodos.set_index("txId")["timestep"].to_dict()
    if componentes is None:
        componentes = list(nx.weakly_connected_components(G))

    aristas_cruzadas = sum(1 for u, v in G.edges() if ts.get(u) != ts.get(v))

    puras, timesteps_por_comp = 0, []
    for c in componentes:
        distintos = {ts.get(n) for n in c}
        timesteps_por_comp.append(distintos)
        if len(distintos) == 1:
            puras += 1

    timesteps_cubiertos = set().union(*timesteps_por_comp) if timesteps_por_comp else set()
    biyeccion = (len(componentes) == len(timesteps_cubiertos) == puras)

    return {
        "aristas que cruzan timesteps": aristas_cruzadas,
        "componentes": len(componentes),
        "componentes puras (un solo timestep)": puras,
        "timesteps distintos": len(timesteps_cubiertos),
        "biyección componente ↔ timestep": biyeccion,
    }


titulo("¿Coinciden las componentes con los timesteps?")
verificacion = verificar_componentes_vs_timesteps(G, df_nodos, comp["debiles"])
for k, v in verificacion.items():
    print(f"  {k:42s} {formatear(v):>10s}")

if verificacion["biyección componente ↔ timestep"]:
    print("""
CONFIRMADO. Las 49 componentes débiles son exactamente los 49 timesteps.

Consecuencias para todo lo que viene después:

  · El grafo NO es una red temporal continua. Es una colección de 49 grafos
    independientes. No existe ningún camino que conecte una transacción del
    timestep 3 con una del 40, así que preguntas del tipo "rastrea este dinero
    a lo largo de seis meses" no tienen respuesta en este dataset.
  · Los snapshots del Módulo 10 no son cortes arbitrarios de una red continua:
    son las componentes reales del grafo, completas y sin truncar.
  · Diámetro, radio y excentricidad (Módulo 5) solo tienen sentido DENTRO de un
    timestep. Calcularlos sobre el grafo completo devolvería infinito.
  · El análisis de "evolución de comunidades" del Módulo 10 no puede seguir a un
    nodo en el tiempo: cada timestep tiene nodos distintos. Lo que se sigue son
    propiedades agregadas, no identidades.""")

# Tabla comparativa: tamaño de cada componente frente al volumen de su timestep.
TS_POR_NODO = df_nodos.set_index("txId")["timestep"].to_dict()   # se reutiliza en módulos posteriores
tam_por_ts = df_nodos.groupby("timestep").size()
comp_por_ts = pd.DataFrame({
    "timestep": [TS_POR_NODO[next(iter(c))] for c in comp["debiles"][:8]],
    "nodos en la componente": [len(c) for c in comp["debiles"][:8]],
})
comp_por_ts["nodos en el timestep"] = comp_por_ts["timestep"].map(tam_por_ts)
comp_por_ts["coinciden"] = comp_por_ts["nodos en la componente"] == comp_por_ts["nodos en el timestep"]
titulo("Las 8 mayores componentes frente a su timestep", nivel=2)
display(comp_por_ts)

In [ ]:
# =============================================================================
# M4.3 — Componentes fuertemente conexas: por qué aquí no hay ninguna
# =============================================================================

titulo("Componentes fuertemente conexas")
scc_no_triviales = [c for c in comp["fuertes"] if len(c) > 1]
print(f"  SCC totales                : {len(comp['fuertes']):,}")
print(f"  SCC con más de un nodo     : {len(scc_no_triviales)}")
print(f"  ¿El grafo es un DAG?       : {nx.is_directed_acyclic_graph(G)}")
print("""
Las 203.769 SCC son todas de un solo nodo, que es la definición operativa de DAG.
No es una casualidad del dato: una transacción de Bitcoin solo puede gastar salidas
de transacciones ANTERIORES, así que el propio protocolo impone un orden topológico.

Lo que esto implica para la detección de fraude:

  · En Elliptic NO se pueden buscar ciclos de layering. La técnica clásica de AML
    "detecta dinero que vuelve al origen" es inaplicable a nivel de transacción.
    Solo funciona a nivel de ENTIDAD (cuenta, wallet, cliente), donde sí hay ciclos.
  · Por eso el grafo bancario del Módulo 1 modela CUENTAS: ahí el layering sí existe
    y se detecta (Módulo 8).
  · La profundidad topológica del DAG sí es informativa: mide cuántos saltos de
    blanqueo separan a una transacción de su origen.
""")

profundidad = {}
for capa, generacion in enumerate(nx.topological_generations(G_t)):
    for n in generacion:
        profundidad[n] = capa
if profundidad:
    prof = pd.Series(profundidad)
    print(f"Profundidad topológica dentro del timestep {TIMESTEP_FOCO}: "
          f"máxima {prof.max()}, media {prof.mean():.2f}")

In [ ]:
# =============================================================================
# M4.4 — Aplicación: anillos criminales en el grafo bancario
# =============================================================================

def proyectar_clientes_por_recurso(G: nx.MultiDiGraph,
                                   tipos_recurso=("dispositivo", "ip", "telefono", "direccion")
                                   ) -> nx.Graph:
    """Proyecta el grafo heterogéneo a una red cliente–cliente de recursos compartidos.

    Dos clientes quedan conectados si comparten al menos un dispositivo, IP, teléfono
    o dirección. El peso de la arista es el número de recursos compartidos.

    Esta proyección es el corazón de la detección de anillos: en los datos originales
    dos clientes no tienen ninguna relación (son titulares distintos, con documentos
    distintos), pero al proyectar por recursos compartidos el anillo se materializa
    como una componente conexa.
    """
    P = nx.Graph()
    P.add_nodes_from(n for n, d in G.nodes(data=True) if d["tipo"] == "cliente")

    for recurso, datos in G.nodes(data=True):
        if datos["tipo"] not in tipos_recurso:
            continue
        usuarios = sorted({u for u in G.predecessors(recurso)
                           if G.nodes[u]["tipo"] == "cliente"})
        for a, b in itertools.combinations(usuarios, 2):
            if P.has_edge(a, b):
                P[a][b]["peso"] += 1
                P[a][b]["recursos"].append(recurso)
            else:
                P.add_edge(a, b, peso=1, recursos=[recurso])
    return P


def anillos_por_componente(P: nx.Graph, tam_min: int = 3) -> list:
    """Componentes conexas de la proyección con al menos `tam_min` clientes = anillos candidatos."""
    return sorted((c for c in nx.connected_components(P) if len(c) >= tam_min),
                  key=len, reverse=True)


titulo("Anillos de clientes en el grafo bancario")
P_clientes = proyectar_clientes_por_recurso(G_banco)
anillos = anillos_por_componente(P_clientes, tam_min=3)

print(f"Clientes en la proyección : {P_clientes.number_of_nodes():,}")
print(f"Aristas (recurso compartido): {P_clientes.number_of_edges():,}")
print(f"Componentes con ≥3 clientes : {len(anillos)}\n")

# Contraste con el ground truth: ¿los anillos detectados son los que inyectamos?
sospechosos_reales = GT_BANCO.get("device_sharing", set()) | GT_BANCO.get("identidad_sintetica", set())
clientes_gt = {n for n in sospechosos_reales if G_banco.nodes[n]["tipo"] == "cliente"}

filas = []
for i, anillo in enumerate(anillos[:10], 1):
    coincide = len(anillo & clientes_gt)
    recursos = Counter()
    for a, b in itertools.combinations(sorted(anillo), 2):
        if P_clientes.has_edge(a, b):
            for r in P_clientes[a][b]["recursos"]:
                recursos[G_banco.nodes[r]["tipo"]] += 1
    filas.append({
        "anillo": i,
        "clientes": len(anillo),
        "en ground truth": coincide,
        "precisión": f"{100*coincide/len(anillo):.0f}%",
        "recursos compartidos": dict(recursos),
    })
display(pd.DataFrame(filas).set_index("anillo"))

cubiertos = set().union(*anillos) if anillos else set()
print(f"Recall sobre clientes marcados como fraude: "
      f"{100*len(cubiertos & clientes_gt)/max(len(clientes_gt),1):.1f}% "
      f"({len(cubiertos & clientes_gt)}/{len(clientes_gt)})")
print("\nUn simple análisis de componentes conexas sobre la proyección correcta recupera casi")
print("todos los anillos inyectados, sin machine learning de por medio. El trabajo estaba en")
print("el modelado (decidir que dispositivo e IP son NODOS), no en el algoritmo.")

---
---

# Módulo 5 — Caminos

Un **camino** es una secuencia de vértices $v_0, v_1, \dots, v_k$ donde cada par consecutivo está
unido por una arista. Su **longitud** es el número de aristas ($k$) si el grafo no está ponderado,
o la suma de los pesos si lo está. El **camino mínimo** es el de longitud menor entre dos vértices.

## Los cuatro algoritmos y cuándo usar cada uno

| Algoritmo | Complejidad | Pesos negativos | Resuelve | Úsalo cuando |
|---|---|---|---|---|
| **BFS** | $O(n + m)$ | — (sin pesos) | 1 origen → todos | El grafo no está ponderado. Es lo más rápido. |
| **Dijkstra** | $O(m + n\log n)$ | ❌ No | 1 origen → todos | Pesos positivos. El caso habitual. |
| **Bellman-Ford** | $O(n \cdot m)$ | ✅ Sí | 1 origen → todos | Hay pesos negativos, o hay que detectar ciclos negativos |
| **Floyd-Warshall** | $O(n^3)$ | ✅ Sí | **todos → todos** | El grafo es pequeño (< ~1.000 nodos) y necesitas la matriz completa |
| **A\*** | $O(m)$ en el mejor caso | ❌ No | 1 origen → 1 destino | Existe una heurística que estime la distancia al objetivo |

Un par de precisiones que suelen confundirse:

- **Dijkstra falla con pesos negativos, no se ralentiza**: da una respuesta incorrecta. Su
  invariante es que una vez un nodo se cierra, su distancia es definitiva — algo que un peso
  negativo posterior puede invalidar. NetworkX lo detecta y lanza excepción en vez de mentir.
- **A\* es Dijkstra con una pista**. Si la heurística $h(u)$ nunca sobreestima la distancia real
  al destino (es *admisible*), A\* garantiza el óptimo explorando muchos menos nodos. Si la
  sobreestima, va más rápido pero puede devolver un camino subóptimo. En grafos sin geometría
  natural —como los de transacciones— construir una heurística admisible es el problema difícil.

### ¿Pesos negativos en un grafo de fraude?

Suena artificial pero no lo es. Aparecen al modelar **coste de oportunidad** o **evidencia
acumulada**: si se define el peso de una arista como $-\log P(\text{legítima})$, el camino
mínimo es el de máxima probabilidad conjunta, y ciertas transformaciones de esa escala producen
valores negativos. También aparecen en modelos de arbitraje, donde un ciclo de peso negativo
significa beneficio sin riesgo — detectar esos ciclos es una aplicación clásica de Bellman-Ford.

## Métricas de distancia global

- **Excentricidad** $\epsilon(v)$: la distancia al nodo más lejano desde $v$.
- **Diámetro**: $\max_v \epsilon(v)$. La mayor distancia entre dos nodos cualesquiera.
- **Radio**: $\min_v \epsilon(v)$. La excentricidad del nodo más "central".
- **Centro**: los nodos cuya excentricidad iguala al radio.
- **Periferia**: los nodos cuya excentricidad iguala al diámetro.

Las tres exigen que el grafo sea conexo: si no, la distancia entre componentes es infinita y
las métricas no están definidas. En Elliptic esto obliga a trabajar **dentro de un timestep**,
porque el Módulo 4 demostró que las componentes son exactamente los timesteps.

In [ ]:
# =============================================================================
# M5.1 — API unificada de caminos mínimos
# =============================================================================

def camino_minimo(G: nx.Graph, origen, destino, algoritmo: str = "auto",
                  peso: str | None = None, heuristica=None) -> dict:
    """Calcula el camino mínimo con el algoritmo elegido y reporta su coste real.

    `algoritmo` acepta "bfs", "dijkstra", "bellman-ford", "astar" o "auto"
    (que elige BFS si no hay pesos y Dijkstra si los hay).

    Devuelve un dict con el camino, su longitud, el tiempo empleado y —cuando el
    algoritmo falla— el motivo. Nunca lanza excepción: un camino inexistente es un
    resultado legítimo, no un error.
    """
    t0 = time.perf_counter()
    if algoritmo == "auto":
        algoritmo = "dijkstra" if peso else "bfs"

    try:
        if algoritmo == "bfs":
            camino = nx.shortest_path(G, origen, destino)
        elif algoritmo == "dijkstra":
            camino = nx.dijkstra_path(G, origen, destino, weight=peso or "weight")
        elif algoritmo == "bellman-ford":
            camino = nx.bellman_ford_path(G, origen, destino, weight=peso or "weight")
        elif algoritmo == "astar":
            if heuristica is None:
                raise ValueError("A* necesita una heurística")
            camino = nx.astar_path(G, origen, destino, heuristic=heuristica, weight=peso)
        else:
            raise ValueError(f"Algoritmo desconocido: {algoritmo!r}")

        if peso:
            longitud = sum(G[u][v][peso] for u, v in zip(camino, camino[1:]))
        else:
            longitud = len(camino) - 1

        return {"algoritmo": algoritmo, "camino": camino, "saltos": len(camino) - 1,
                "longitud": longitud, "ms": round((time.perf_counter() - t0) * 1000, 3),
                "error": None}

    except (nx.NetworkXNoPath, nx.NodeNotFound, ValueError, nx.NetworkXUnbounded) as exc:
        return {"algoritmo": algoritmo, "camino": None, "saltos": None, "longitud": None,
                "ms": round((time.perf_counter() - t0) * 1000, 3),
                "error": f"{type(exc).__name__}: {exc}"}


# --- Demostración 1: pesos positivos, los cuatro algoritmos coinciden ---
G_pesos = nx.DiGraph()
G_pesos.add_weighted_edges_from([
    ("CTA_A", "CTA_B", 4), ("CTA_A", "CTA_C", 2), ("CTA_C", "CTA_B", 1),
    ("CTA_B", "CTA_D", 5), ("CTA_C", "CTA_D", 8), ("CTA_D", "CTA_E", 3),
    ("CTA_C", "CTA_E", 12),
])

titulo("Caminos mínimos con pesos positivos (A → E)")
filas = []
for alg in ["bfs", "dijkstra", "bellman-ford"]:
    r = camino_minimo(G_pesos, "CTA_A", "CTA_E", algoritmo=alg, peso=None if alg == "bfs" else "weight")
    filas.append({"algoritmo": alg, "camino": " → ".join(r["camino"]) if r["camino"] else "—",
                  "saltos": r["saltos"], "coste": r["longitud"], "ms": r["ms"]})
display(pd.DataFrame(filas).set_index("algoritmo"))
print("BFS minimiza SALTOS (A→C→E, 2 aristas, coste 14).")
print("Dijkstra y Bellman-Ford minimizan COSTE (A→C→B→D→E, 4 aristas, coste 11).")
print("No es que uno acierte y otro falle: optimizan objetivos distintos.\n")

# --- Demostración 2: pesos negativos, aquí Dijkstra ya no vale ---
G_neg = nx.DiGraph()
G_neg.add_weighted_edges_from([
    ("A", "B", 4), ("A", "C", 5), ("C", "B", -3), ("B", "D", 2), ("C", "D", 6),
])

titulo("Con un peso negativo (arista C→B = −3)", nivel=2)
for alg in ["dijkstra", "bellman-ford"]:
    r = camino_minimo(G_neg, "A", "D", algoritmo=alg, peso="weight")
    if r["error"]:
        print(f"  {alg:14s} → FALLA. {r['error'][:95]}")
    else:
        print(f"  {alg:14s} → {' → '.join(r['camino'])}  (coste {r['longitud']})")
print("\nDijkstra no es 'lento' con pesos negativos: su invariante deja de ser válido.")
print("NetworkX lo detecta y lanza excepción en lugar de devolver una respuesta incorrecta.")

In [ ]:
# =============================================================================
# M5.2 — Floyd-Warshall: todos contra todos, y por qué no escala
# =============================================================================

def floyd_warshall_demo(G: nx.Graph, peso: str | None = None) -> tuple:
    """Matriz completa de distancias por Floyd-Warshall, con su coste medido.

    Floyd-Warshall responde de una vez a "distancia entre CUALQUIER par", con
    complejidad O(n³) en tiempo y O(n²) en memoria. Ese O(n²) de memoria es lo que
    lo mata antes que el tiempo: para los 203.769 nodos de Elliptic la matriz de
    distancias ocuparía 203.769² × 8 bytes ≈ 332 GB.
    """
    t0 = time.perf_counter()
    D = nx.floyd_warshall_numpy(G, weight=peso)
    return D, time.perf_counter() - t0


sub_fw = muestrear_subgrafo(G, 500, metodo="bfs")
titulo("Floyd-Warshall sobre un subgrafo de 500 nodos")
D_fw, t_fw = floyd_warshall_demo(sub_fw)
finitas = D_fw[np.isfinite(D_fw) & (D_fw > 0)]

print(f"  Matriz calculada    : {D_fw.shape[0]}x{D_fw.shape[1]} = {D_fw.size:,} distancias")
print(f"  Tiempo              : {t_fw:.2f} s")
print(f"  Memoria de la matriz: {D_fw.nbytes / 1024**2:.1f} MB")
print(f"  Pares alcanzables   : {len(finitas):,} ({100*len(finitas)/D_fw.size:.2f}% del total)")
print(f"  Distancia media     : {finitas.mean():.2f} saltos · máxima: {finitas.max():.0f}")

n_total = G.number_of_nodes()
factor = (n_total / D_fw.shape[0]) ** 3
print(f"\n  Extrapolación al grafo completo ({n_total:,} nodos):")
print(f"    Tiempo estimado : {t_fw * factor / 3600:,.0f} horas  (factor {factor:,.0f}x)")
print(f"    Memoria estimada: {n_total**2 * 8 / 1024**3:,.0f} GB")
print("    Conclusión: Floyd-Warshall es una herramienta didáctica y para grafos pequeños.")
print("    Para 'todos contra todos' a escala se usa BFS desde una MUESTRA de orígenes.")

In [ ]:
# =============================================================================
# M5.3 — A*: Dijkstra con una heurística geométrica
# =============================================================================

def heuristica_por_layout(pos: dict, escala: float = 1.0):
    """Construye una heurística euclídea para A* a partir de posiciones de layout.

    Los grafos de transacciones no tienen geometría natural (no hay "coordenadas" de
    una transacción), así que se las fabricamos con un layout de fuerzas: nodos
    topológicamente cercanos quedan cerca en el plano. La distancia euclídea en ese
    plano es entonces una estimación razonable de la distancia en saltos.

    Advertencia honesta: esta heurística NO es admisible en general —puede
    sobreestimar—, así que A* deja de garantizar el óptimo. La celda comprueba
    empíricamente si el camino encontrado coincide con el de BFS.
    """
    def h(u, v):
        (x1, y1), (x2, y2) = pos[u], pos[v]
        return escala * math.hypot(x2 - x1, y2 - y1)
    return h


sub_astar = muestrear_subgrafo(G, 900, metodo="bfs").to_undirected()
sub_astar = sub_astar.subgraph(max(nx.connected_components(sub_astar), key=len)).copy()
pos_astar = nx.spring_layout(sub_astar, seed=SEED, iterations=60)

# Calibración: escala = 1 / (longitud euclídea media de una arista), para que la
# heurística devuelva algo comparable a un número de saltos.
long_aristas = [math.dist(pos_astar[u], pos_astar[v]) for u, v in sub_astar.edges()]
escala = 1.0 / np.mean(long_aristas)
h = heuristica_por_layout(pos_astar, escala)

titulo("A* frente a Dijkstra y BFS sobre 30 pares aleatorios")
rng = random.Random(SEED)
nodos_sub = list(sub_astar.nodes())
filas, optimos = [], 0
for _ in range(30):
    o, d = rng.sample(nodos_sub, 2)
    r_bfs = camino_minimo(sub_astar, o, d, algoritmo="bfs")
    r_ast = camino_minimo(sub_astar, o, d, algoritmo="astar", heuristica=h)
    if r_bfs["camino"] and r_ast["camino"]:
        es_optimo = r_ast["saltos"] == r_bfs["saltos"]
        optimos += es_optimo
        filas.append({"saltos BFS": r_bfs["saltos"], "saltos A*": r_ast["saltos"],
                      "ms BFS": r_bfs["ms"], "ms A*": r_ast["ms"], "óptimo": es_optimo})

res_astar = pd.DataFrame(filas)
print(f"  Pares evaluados         : {len(res_astar)}")
print(f"  A* encontró el óptimo   : {optimos}/{len(res_astar)} ({100*optimos/max(len(res_astar),1):.0f}%)")
print(f"  Tiempo medio BFS        : {res_astar['ms BFS'].mean():.2f} ms")
print(f"  Tiempo medio A*         : {res_astar['ms A*'].mean():.2f} ms")
print("\n  Sobre un grafo sin geometría real, A* no compensa: la heurística inventada no")
print("  guía bien la búsqueda y el coste de evaluarla se come la ventaja. A* brilla en")
print("  grafos con geometría genuina (rutas, mapas), no en grafos de transacciones.")

In [ ]:
# =============================================================================
# M5.4 — Diámetro, radio y excentricidad
# =============================================================================

def diametro_doble_barrido(G_und: nx.Graph, repeticiones: int = 12, seed: int = SEED) -> tuple:
    """Cota inferior del diámetro por doble barrido BFS — O(m) por repetición.

    Heurística clásica: desde un nodo al azar se busca el más lejano (a); desde `a`
    se busca el más lejano (b). d(a,b) es una cota inferior del diámetro, y en redes
    reales suele coincidir con el valor exacto. El cálculo exacto necesita un BFS
    desde CADA nodo: O(n·m), inviable sobre componentes grandes.
    """
    rng = random.Random(seed)
    nodos = list(G_und.nodes())
    mejor, par = 0, (None, None)
    for _ in range(repeticiones):
        u = rng.choice(nodos)
        d1 = nx.single_source_shortest_path_length(G_und, u)
        a = max(d1, key=d1.get)
        d2 = nx.single_source_shortest_path_length(G_und, a)
        b = max(d2, key=d2.get)
        if d2[b] > mejor:
            mejor, par = d2[b], (a, b)
    return mejor, par


def metricas_distancia(G_und: nx.Graph, exacto: bool = True, nombre: str = "") -> dict:
    """Diámetro, radio, excentricidad media, centro y periferia de una componente conexa.

    Con `exacto=False` usa el doble barrido para el diámetro y una muestra de nodos
    para la excentricidad media. Sobre componentes de más de unos pocos miles de
    nodos, el cálculo exacto O(n·m) deja de ser práctico en Python.
    """
    if not nx.is_connected(G_und):
        G_und = G_und.subgraph(max(nx.connected_components(G_und), key=len)).copy()

    n, m = G_und.number_of_nodes(), G_und.number_of_edges()
    res = {"grafo": nombre, "nodos": n, "aristas": m, "exacto": exacto}

    if exacto:
        ecc = nx.eccentricity(G_und)
        valores = np.array(list(ecc.values()))
        res.update({
            "diámetro": int(valores.max()),
            "radio": int(valores.min()),
            "excentricidad media": round(float(valores.mean()), 2),
            "nodos en el centro": int(np.sum(valores == valores.min())),
            "nodos en la periferia": int(np.sum(valores == valores.max())),
            "camino medio": round(nx.average_shortest_path_length(G_und), 2),
        })
    else:
        diam, par = diametro_doble_barrido(G_und)
        res.update({"diámetro (cota inferior)": diam, "par más lejano": par})

    return res


titulo("Métricas de distancia")

# Exacto sobre la mayor componente de H (subgrafo de trabajo).
H_lcc = H_und.subgraph(max(nx.connected_components(H_und), key=len)).copy()
m_H, _ = cronometrar(metricas_distancia, H_lcc, etiqueta="exacto sobre H_lcc",
                     exacto=True, nombre="LCC de H")
print()
for k, v in m_H.items():
    if k != "grafo":
        print(f"  {k:24s} {formatear(v):>12s}")

# Aproximado sobre el timestep completo, donde el cálculo exacto ya no compensa.
G_t_und = G_t.to_undirected()
G_t_lcc = G_t_und.subgraph(max(nx.connected_components(G_t_und), key=len)).copy()
print(f"\nMayor componente del timestep {TIMESTEP_FOCO}: {G_t_lcc.number_of_nodes():,} nodos")
m_t, _ = cronometrar(metricas_distancia, G_t_lcc, etiqueta="doble barrido sobre el timestep",
                     exacto=False, nombre=f"LCC t={TIMESTEP_FOCO}")
for k, v in m_t.items():
    if k not in ("grafo", "exacto"):
        print(f"  {k:24s} {v}")

# Distribución de excentricidades: separa el núcleo de la periferia.
ecc_H = nx.eccentricity(H_lcc)
plt.figure(figsize=(9, 3.6))
plt.hist(list(ecc_H.values()), bins=range(min(ecc_H.values()), max(ecc_H.values()) + 2),
         color=COLORES["licito"], edgecolor="white", align="left")
plt.axvline(m_H["radio"], color=COLORES["ok"], ls="--", lw=2, label=f"Radio = {m_H['radio']}")
plt.axvline(m_H["diámetro"], color=COLORES["ilicito"], ls="--", lw=2, label=f"Diámetro = {m_H['diámetro']}")
plt.title("Distribución de excentricidades en la mayor componente de H")
plt.xlabel("Excentricidad")
plt.ylabel("Nº de nodos")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# M5.5 — Aplicación: rastreo del flujo de fondos
# =============================================================================

def rastrear_flujo_fondos(G: nx.DiGraph, origen, profundidad: int = 4,
                          sentido: str = "adelante") -> dict:
    """Sigue el dinero desde (o hacia) una transacción, nivel a nivel.

    `sentido="adelante"` responde a "¿a dónde fue este dinero?" siguiendo aristas
    salientes; `"atras"` responde a "¿de dónde vino?" siguiendo las entrantes. Es la
    operación básica de una investigación: partir de una entidad señalada y expandir.

    Devuelve los nodos alcanzados por nivel, el subgrafo del rastro y —lo importante
    en una investigación real— cuántas entidades ilícitas conocidas aparecen en cada nivel.
    """
    vista = G if sentido == "adelante" else G.reverse(copy=False)
    niveles = nx.single_source_shortest_path_length(vista, origen, cutoff=profundidad)

    por_nivel = defaultdict(list)
    for nodo, nivel in niveles.items():
        por_nivel[nivel].append(nodo)

    rastro = G.subgraph(niveles.keys()).copy()
    resumen = []
    for nivel in sorted(por_nivel):
        nodos = por_nivel[nivel]
        clases = Counter(G.nodes[n].get("clase", "desconocido") for n in nodos)
        resumen.append({
            "nivel": nivel,
            "nodos alcanzados": len(nodos),
            "ilícitos": clases.get("ilicito", 0),
            "lícitos": clases.get("licito", 0),
            "desconocidos": clases.get("desconocido", 0),
        })

    return {"resumen": pd.DataFrame(resumen).set_index("nivel"),
            "por_nivel": dict(por_nivel), "subgrafo": rastro, "total": len(niveles)}


# Partimos de la transacción ilícita con más salidas: el caso típico de dispersión.
ilicitos = df_nodos.loc[df_nodos["clase"] == "ilicito", "txId"]
origen_rastreo = max(ilicitos, key=lambda n: G.out_degree(n))

titulo(f"Rastreo desde la transacción ilícita txId {origen_rastreo}")
print(f"  Out-degree del origen: {G.out_degree(origen_rastreo)} | "
      f"timestep: {G.nodes[origen_rastreo]['timestep']}\n")

for sentido, etiqueta in [("adelante", "¿A DÓNDE FUE EL DINERO?"), ("atras", "¿DE DÓNDE VINO?")]:
    rastro = rastrear_flujo_fondos(G, origen_rastreo, profundidad=4, sentido=sentido)
    print(f"{etiqueta}  (total alcanzado: {rastro['total']:,} transacciones)")
    display(rastro["resumen"])

# Visualización del rastro hacia delante.
rastro_fw = rastrear_flujo_fondos(G, origen_rastreo, profundidad=3, sentido="adelante")
sub_rastro = rastro_fw["subgrafo"]
if sub_rastro.number_of_nodes() <= 900:
    niveles = {n: lv for lv, nodos in rastro_fw["por_nivel"].items() for n in nodos}
    pos = nx.spring_layout(sub_rastro, seed=SEED, k=0.5, iterations=60)
    plt.figure(figsize=(11, 6))
    colores_nodo = [{"ilicito": COLORES["ilicito"], "licito": COLORES["licito"]}.get(
        sub_rastro.nodes[n].get("clase"), COLORES["desconocido"]) for n in sub_rastro.nodes()]
    tam = [220 if n == origen_rastreo else 40 for n in sub_rastro.nodes()]
    nx.draw_networkx_edges(sub_rastro, pos, edge_color="#C0C7D0", width=0.7,
                           arrows=True, arrowsize=7, alpha=0.8)
    nx.draw_networkx_nodes(sub_rastro, pos, node_color=colores_nodo, node_size=tam, linewidths=0)
    plt.title(f"Rastro de fondos a 3 saltos desde txId {origen_rastreo} "
              f"({sub_rastro.number_of_nodes()} transacciones)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

print("La expansión por niveles es exactamente lo que hace un analista de fraude, y también")
print("lo que hace insostenible la investigación manual: a tres saltos ya hay cientos de")
print("entidades. Los módulos de centralidad y comunidades existen para priorizar esa lista.")

---
---

# Módulo 6 — Centralidad

Uno de los capítulos más importantes. "Centralidad" no es una métrica sino **una familia de
respuestas a la pregunta '¿qué nodos importan?'**, y cada miembro de la familia responde a una
pregunta distinta. Elegir la equivocada produce rankings impecables y sin sentido.

| Centralidad | Pregunta que responde | Fórmula | Coste | Señal de fraude |
|---|---|---|---|---|
| **Degree** | ¿Quién tiene más conexiones? | $C_D(v) = \dfrac{k_v}{n-1}$ | $O(m)$ | Concentradores, mulas |
| **In-degree** | ¿Quién recibe de más sitios? | $k_v^{in}/(n-1)$ | $O(m)$ | Recolectores, exchanges |
| **Out-degree** | ¿Quién envía a más sitios? | $k_v^{out}/(n-1)$ | $O(m)$ | Dispersores, mezcladores |
| **Betweenness** | ¿Quién hace de puente? | $\sum_{s\neq v\neq t} \dfrac{\sigma_{st}(v)}{\sigma_{st}}$ | $O(n\,m)$ | Intermediarios de blanqueo |
| **Closeness** | ¿Quién alcanza rápido al resto? | $\dfrac{n-1}{\sum_u d(v,u)}$ | $O(n\,m)$ | Posición operativa central |
| **Eigenvector** | ¿Quién se conecta con importantes? | $\lambda x = A x$ | $O(m\,i)$ | Pertenencia al núcleo |
| **Katz** | Eigenvector con amortiguación | $x = \alpha A^{\!\top} x + \beta$ | $O(m\,i)$ | Influencia con decaimiento |
| **PageRank** | Importancia por vecinos importantes | $x = \alpha P^{\!\top} x + \frac{1-\alpha}{n}$ | $O(m\,i)$ | Acumuladores de flujo |
| **HITS** | Hubs vs. autoridades | $h = A a$, $a = A^{\!\top} h$ | $O(m\,i)$ | Dispersores vs. receptores |

## Las tres familias conceptuales

**1. Centralidad por volumen (degree).** Cuenta vecinos. Barata y sorprendentemente buena como
línea base, pero completamente local: no distingue entre estar conectado a diez nodos
irrelevantes o a diez nodos críticos.

**2. Centralidad por posición (betweenness, closeness).** Se basa en caminos mínimos. Un nodo con
betweenness alta puede tener solo dos aristas y aun así ser crítico: si es el único puente entre
dos comunidades, todo el flujo pasa por él. **En blanqueo de capitales, el intermediario que
importa casi nunca es el que más transacciones tiene, sino el que conecta el mundo ilícito con el
lícito.** Ese nodo tiene betweenness alta y degree mediocre.

**3. Centralidad por recursión (eigenvector, Katz, PageRank, HITS).** "Eres importante si tus
vecinos son importantes." Se resuelven como problemas de punto fijo. Sus diferencias:

- **Eigenvector**: puro autovector principal. Falla en grafos dirigidos (los nodos sin aristas
  entrantes reciben cero y arrastran a todos sus descendientes) y **se concentra por completo en
  una sola componente** si el grafo está desconectado. Lo comprobaremos.
- **Katz**: añade un término constante $\beta$ que da a todo nodo una importancia mínima, y un
  factor $\alpha$ que amortigua las contribuciones lejanas. Resuelve el problema del cero.
  Requiere $\alpha < 1/\lambda_{max}$ para converger.
- **PageRank**: Katz normalizado por el grado de salida, más un factor de teletransporte. Que
  esté normalizado importa: un nodo que enlaza a mil sitios reparte su importancia entre mil, en
  vez de multiplicarla por mil.
- **HITS**: dos puntuaciones acopladas. Un buen **hub** apunta a buenas autoridades; una buena
  **autoridad** es apuntada por buenos hubs. Encaja de forma natural con la dualidad
  dispersor/recolector del fraude.

## Nota sobre Katz en un DAG

En Elliptic la matriz de adyacencia es **nilpotente** ($A^k = 0$ para algún $k$), porque el grafo
es acíclico. Su radio espectral es 0, así que la condición $\alpha < 1/\lambda_{max}$ se cumple
para **cualquier** $\alpha$ y la serie de Neumann $\sum_k \alpha^k (A^{\!\top})^k$ es una suma
finita. Katz converge siempre y en pocas iteraciones. Es una consecuencia bonita del hallazgo
del Módulo 4.

In [ ]:
# =============================================================================
# M6.1 — Cálculo de centralidades con control explícito del coste
# =============================================================================

def katz_sparse(G: nx.Graph, alpha: float = 0.1, beta: float = 1.0,
                max_iter: int = 300, tol: float = 1e-9, nodelist=None) -> dict:
    """Centralidad de Katz por iteración sobre matriz dispersa: x ← α·Aᵀx + β.

    Se implementa a mano en lugar de usar `nx.katz_centrality` por dos razones:
    la versión de NetworkX puede lanzar `PowerIterationFailedConvergence` en grafos
    grandes, y `katz_centrality_numpy` construye una matriz DENSA n×n (332 GB aquí).
    Con `to_scipy_sparse_array` cada iteración es un producto matriz-vector disperso
    de 234.355 elementos no nulos: milisegundos.
    """
    nodelist = list(G.nodes()) if nodelist is None else list(nodelist)
    A = nx.to_scipy_sparse_array(G, nodelist=nodelist, format="csr", dtype=float)
    x = np.full(len(nodelist), beta, dtype=float)
    for i in range(max_iter):
        x_nuevo = alpha * (A.T @ x) + beta
        if np.abs(x_nuevo - x).sum() < tol:
            x = x_nuevo
            break
        x = x_nuevo
    norma = np.linalg.norm(x)
    return dict(zip(nodelist, x / norma if norma else x))


def calcular_centralidades(G: nx.Graph, incluir=None, k_betweenness: int | None = None,
                           seed: int = SEED, verbose: bool = True) -> pd.DataFrame:
    """Calcula varias centralidades sobre el mismo grafo y las devuelve en un DataFrame.

    `k_betweenness` activa la aproximación de Brandes por muestreo: en vez de un BFS
    desde los n nodos, se hace desde k pivotes elegidos al azar, con coste O(k·m) en
    lugar de O(n·m). Con k=None se calcula la betweenness exacta.

    Métricas caras y su alcance recomendado:
      · degree / pagerank / katz / hits → O(m) por iteración: viables sobre 200 mil nodos.
      · betweenness / closeness        → O(n·m): solo sobre subgrafos o con muestreo.
      · eigenvector                    → requiere grafo NO dirigido y, en la práctica,
                                          conexo (ver M6.3).
    """
    disponibles = ["degree", "in_degree", "out_degree", "pagerank", "hits",
                   "katz", "betweenness", "closeness", "eigenvector"]
    incluir = disponibles if incluir is None else incluir
    dirigido = G.is_directed()
    resultados = {}

    def _medir(nombre, fn):
        t0 = time.perf_counter()
        valor = fn()
        if verbose:
            print(f"  · {nombre:14s} {time.perf_counter() - t0:7.2f} s")
        return valor

    if "degree" in incluir:
        resultados["degree"] = _medir("degree", lambda: nx.degree_centrality(G))
    if dirigido and "in_degree" in incluir:
        resultados["in_degree"] = _medir("in_degree", lambda: nx.in_degree_centrality(G))
    if dirigido and "out_degree" in incluir:
        resultados["out_degree"] = _medir("out_degree", lambda: nx.out_degree_centrality(G))
    if "pagerank" in incluir:
        resultados["pagerank"] = _medir("pagerank", lambda: nx.pagerank(G, alpha=0.85))
    if "katz" in incluir:
        resultados["katz"] = _medir("katz", lambda: katz_sparse(G, alpha=0.1))
    if "hits" in incluir:
        hubs, autoridades = _medir("hits", lambda: nx.hits(G, max_iter=200, normalized=True))
        resultados["hits_hub"] = hubs
        resultados["hits_authority"] = autoridades
    if "betweenness" in incluir:
        k = None if k_betweenness is None else min(k_betweenness, G.number_of_nodes())
        etiqueta = "betweenness" if k is None else f"betw (k={k})"
        resultados["betweenness"] = _medir(
            etiqueta, lambda: nx.betweenness_centrality(G, k=k, seed=seed, normalized=True))
    if "closeness" in incluir:
        resultados["closeness"] = _medir("closeness", lambda: nx.closeness_centrality(G))
    if "eigenvector" in incluir:
        # Sobre el grafo DIRIGIDO la iteración de potencia no converge (los nodos sin
        # aristas entrantes reciben cero y lo propagan), así que se usa la proyección
        # no dirigida. Si aun así ARPACK no converge, se informa y se omite la métrica
        # en lugar de abortar el módulo entero.
        base = G.to_undirected() if dirigido else G

        def _eigen():
            try:
                return nx.eigenvector_centrality_numpy(base, max_iter=500)
            except Exception as exc:
                print(f"    eigenvector omitida: {type(exc).__name__}: {exc}")
                return {}

        valores_eigen = _medir("eigenvector", _eigen)
        if valores_eigen:
            resultados["eigenvector"] = valores_eigen

    df = pd.DataFrame(resultados)
    df.index.name = "txId"
    return df.reset_index()


# --- Métricas baratas sobre el grafo COMPLETO ---
titulo("Centralidades O(m) sobre el grafo completo (203.769 nodos)")
cent_global = calcular_centralidades(
    G, incluir=["degree", "in_degree", "out_degree", "pagerank", "katz", "hits"])
cent_global = cent_global.merge(df_nodos, on="txId", how="left")
print(f"\nTabla de centralidades: {cent_global.shape[0]:,} nodos x {cent_global.shape[1]} columnas")

titulo("Top-5 por cada centralidad global", nivel=2)
for metrica in ["pagerank", "katz", "hits_hub", "hits_authority"]:
    top = top_n(cent_global, metrica, 5)[["txId", metrica, "in_degree", "out_degree", "clase"]]
    print(f"\n▸ {metrica}")
    display(top.reset_index(drop=True))

In [ ]:
# =============================================================================
# M6.2 — Métricas O(n·m): dentro de un timestep, que es donde tienen sentido
# =============================================================================

# El Módulo 4 demostró que las componentes son los timesteps. Betweenness y closeness
# miden posición relativa DENTRO de una componente: calcularlas sobre el grafo entero
# mezclaría 49 redes inconexas y produciría un ranking sin significado.
titulo(f"Centralidades de posición sobre el timestep {TIMESTEP_FOCO} "
       f"({G_t.number_of_nodes():,} nodos)")

cent_local = calcular_centralidades(
    G_t, incluir=["degree", "in_degree", "out_degree", "pagerank", "katz",
                  "betweenness", "closeness", "eigenvector"],
    k_betweenness=500)
cent_local = cent_local.merge(df_nodos, on="txId", how="left")

titulo("Top-5 por betweenness: los puentes del timestep", nivel=2)
display(top_n(cent_local, "betweenness", 5)[
    ["txId", "betweenness", "degree", "pagerank", "clase"]].reset_index(drop=True))

titulo("Top-5 por closeness: los mejor posicionados", nivel=2)
display(top_n(cent_local, "closeness", 5)[
    ["txId", "closeness", "betweenness", "degree", "clase"]].reset_index(drop=True))

In [ ]:
# =============================================================================
# M6.3 — Dos patologías que hay que conocer antes de fiarse de un ranking
# =============================================================================

titulo("Patología 1: eigenvector en un grafo desconectado")
try:
    ev_global = nx.eigenvector_centrality_numpy(G_und, max_iter=500)
except Exception as exc:
    ev_global = None
    print(f"  ARPACK no convergió sobre las 49 componentes ({type(exc).__name__}).")
    print("  Ese fallo ES la patología en su forma más cruda: el autovector principal de")
    print("  una matriz diagonal por bloques está mal condicionado cuando varios bloques")
    print("  tienen autovalores parecidos. La conclusión no cambia: no calcules eigenvector")
    print("  sobre un grafo desconectado.")

if ev_global is not None:
    serie_ev = pd.Series(ev_global)
    ev_por_comp = []
    for i, c in enumerate(comp["debiles"][:6]):
        valores = serie_ev.loc[list(c)]
        ev_por_comp.append({
            "componente (timestep)": TS_POR_NODO[next(iter(c))],
            "nodos": len(c),
            "suma |eigenvector|": round(float(np.abs(valores).sum()), 6),
            "máximo": round(float(np.abs(valores).max()), 6),
        })
    display(pd.DataFrame(ev_por_comp))
    print("""La centralidad de eigenvector se concentra casi por completo en UNA componente:
la que tiene el mayor autovalor. Todas las demás reciben valores cercanos a cero, no
porque sus nodos sean irrelevantes, sino porque el autovector principal de una matriz
diagonal por bloques vive en un solo bloque. Comparar el eigenvector de dos nodos de
componentes distintas no significa nada. Por eso en M6.2 se calcula por timestep.""")

titulo("Patología 2: cuánto se pierde al aproximar betweenness por muestreo")
# Sobre H (1.200 nodos) sí se puede calcular la betweenness exacta y comparar.
from scipy.stats import spearmanr

bt_exacta = nx.betweenness_centrality(H, normalized=True)
serie_exacta = pd.Series(bt_exacta)
filas = []
for k in [25, 50, 100, 250, 500]:
    t0 = time.perf_counter()
    aprox = pd.Series(nx.betweenness_centrality(H, k=k, seed=SEED, normalized=True))
    dt = time.perf_counter() - t0
    top50_exacto = set(serie_exacta.nlargest(50).index)
    top50_aprox = set(aprox.nlargest(50).index)
    filas.append({
        "k (pivotes)": k,
        "ρ de Spearman": round(float(spearmanr(serie_exacta, aprox.reindex(serie_exacta.index))[0]), 4),
        "solape del top-50": f"{len(top50_exacto & top50_aprox)}/50",
        "segundos": round(dt, 2),
    })
t0 = time.perf_counter()
nx.betweenness_centrality(H, normalized=True)
filas.append({"k (pivotes)": f"exacta ({H.number_of_nodes()})", "ρ de Spearman": 1.0,
              "solape del top-50": "50/50", "segundos": round(time.perf_counter() - t0, 2)})
display(pd.DataFrame(filas).set_index("k (pivotes)"))
print("Con unos pocos cientos de pivotes el ranking de cabeza ya se recupera casi entero.")
print("Para PRIORIZAR alertas —que es lo que se hace en fraude— la aproximación basta;")
print("para publicar un valor exacto de betweenness, no.")

In [ ]:
# =============================================================================
# M6.4 — ¿Miden lo mismo estas centralidades?
# =============================================================================

def comparar_centralidades(df: pd.DataFrame, columnas=None, metodo: str = "spearman") -> pd.DataFrame:
    """Matriz de correlación entre centralidades y solape de sus top-100.

    Se usa correlación de SPEARMAN y no de Pearson: las centralidades tienen
    distribuciones de cola pesada y lo que importa es si ORDENAN igual, no si sus
    valores son proporcionales.
    """
    columnas = columnas or [c for c in df.columns
                            if c in ("degree", "in_degree", "out_degree", "pagerank", "katz",
                                     "betweenness", "closeness", "eigenvector",
                                     "hits_hub", "hits_authority")]
    return df[columnas].corr(method=metodo)


corr = comparar_centralidades(cent_local)

fig, ejes = plt.subplots(1, 2, figsize=(14, 5.4))
im = ejes[0].imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ejes[0].set_xticks(range(len(corr)))
ejes[0].set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
ejes[0].set_yticks(range(len(corr)))
ejes[0].set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ejes[0].text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                     fontsize=7.5, color="white" if abs(corr.values[i, j]) > 0.55 else "#222")
ejes[0].set_title("Correlación de Spearman entre centralidades")
ejes[0].grid(False)
plt.colorbar(im, ax=ejes[0], fraction=0.046)

# Solape del top-100 entre pares de métricas: más interpretable que la correlación global.
# Se filtra por columnas presentes: eigenvector puede haberse omitido si no convergió.
metricas = [m for m in ["degree", "pagerank", "betweenness", "closeness", "eigenvector", "katz"]
            if m in cent_local.columns]
solape = np.zeros((len(metricas), len(metricas)))
tops = {m: set(cent_local.nlargest(100, m)["txId"]) for m in metricas}
for i, a in enumerate(metricas):
    for j, b in enumerate(metricas):
        solape[i, j] = len(tops[a] & tops[b])

im2 = ejes[1].imshow(solape, cmap="YlOrRd", vmin=0, vmax=100)
ejes[1].set_xticks(range(len(metricas)))
ejes[1].set_xticklabels(metricas, rotation=45, ha="right", fontsize=8)
ejes[1].set_yticks(range(len(metricas)))
ejes[1].set_yticklabels(metricas, fontsize=8)
for i in range(len(metricas)):
    for j in range(len(metricas)):
        ejes[1].text(j, i, int(solape[i, j]), ha="center", va="center", fontsize=8,
                     color="white" if solape[i, j] > 55 else "#222")
ejes[1].set_title("Nodos en común entre los top-100 de cada métrica")
ejes[1].grid(False)
plt.colorbar(im2, ax=ejes[1], fraction=0.046)
plt.tight_layout()
plt.show()

print("""
CÓMO LEER ESTO

· Los pares con correlación alta (degree ↔ pagerank) miden esencialmente lo mismo.
  Meter ambos en un modelo de ML añade multicolinealidad, no información.
· Los pares con correlación baja aportan señales INDEPENDIENTES. Betweenness suele
  ser la que menos se parece al resto, y por eso es la más valiosa como feature:
  identifica nodos que ninguna otra métrica marca.
· El solape de los top-100 es más honesto que la correlación global. Dos métricas
  pueden correlacionar 0,9 sobre todos los nodos y coincidir en solo 30 de los 100
  primeros — y en fraude solo se investigan los primeros.

REGLA PRÁCTICA para elegir centralidad según la pregunta:
    "¿quién mueve más volumen?"        → degree, in/out-degree
    "¿quién es el intermediario?"      → betweenness            ← el más útil en AML
    "¿quién está en el núcleo?"        → eigenvector, katz
    "¿quién acumula flujo?"            → pagerank
    "¿dispersor o recolector?"         → hits_hub vs hits_authority
""")

---
---

# Módulo 7 — Análisis de comunidades

Una **comunidad** es un grupo de nodos con muchas más aristas hacia dentro que hacia fuera.
Formalmente no hay una única definición, y por eso hay tantos algoritmos: cada uno optimiza un
criterio distinto de "densamente conectado".

## Modularidad

Casi todos los métodos clásicos maximizan la **modularidad** $Q$, que compara las aristas
internas observadas con las que habría en un grafo aleatorio con los mismos grados:

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

$Q$ va de $-0{,}5$ a $1$. Por encima de $0{,}3$ se considera estructura comunitaria clara.
Dos límites conocidos que conviene tener presentes:

- **Límite de resolución** (Fortunato & Barthélemy): la modularidad no puede detectar comunidades
  más pequeñas que $\sqrt{2m}$ aristas, por bien definidas que estén. En un grafo de 234.355
  aristas eso son ~680 aristas: un anillo de fraude de 10 cuentas queda **por debajo del umbral
  de detección** y se fusiona con un grupo mayor. El parámetro `resolution` mitiga esto.
- **Degeneración**: existen muchísimas particiones con $Q$ casi máxima y muy distintas entre sí.
  Que un algoritmo devuelva $Q = 0{,}82$ no significa que esa partición sea *la* correcta.

## Los cinco algoritmos

| Algoritmo | Optimiza | Complejidad | Determinista | Punto fuerte / débil |
|---|---|---|---|---|
| **Louvain** | Modularidad (voraz, multinivel) | $O(m \log n)$ | No (depende del orden) | Rapidísimo / puede producir comunidades mal conectadas |
| **Leiden** | Modularidad + garantía de conexión | $O(m \log n)$ | No | Corrige el defecto de Louvain / requiere `igraph` |
| **Girvan-Newman** | Elimina aristas de alta betweenness | $O(m^2 n)$ | Sí | Jerarquía completa e interpretable / inviable > 1.000 nodos |
| **Label Propagation** | Consenso local de etiquetas | $O(m)$ casi lineal | No | El más rápido / muy inestable entre ejecuciones |
| **Infomap** | Longitud de descripción de un random walk | $O(m)$ | No | Base teórica sólida (compresión) / dependencia externa |

**Sobre Leiden**: Louvain puede producir comunidades **internamente desconectadas** —un nodo
puede quedar asignado a una comunidad con la que ya no tiene ninguna arista tras la agregación.
Leiden añade una fase de refinamiento que lo garantiza. En detección de fraude eso importa:
una "comunidad" desconectada no es un grupo, es un artefacto del algoritmo.

**Sobre Infomap**: en lugar de contar aristas, pregunta cuántos bits hacen falta para describir
la trayectoria de un caminante aleatorio. Si el caminante se queda atrapado en un grupo, describir
su trayectoria con un código de dos niveles (comunidad + nodo) es más corto. Es una formulación
distinta y detecta estructuras que la modularidad ignora, sobre todo en grafos dirigidos.

In [ ]:
# =============================================================================
# M7.1 — API unificada de detección de comunidades
# =============================================================================

from sklearn.metrics import normalized_mutual_info_score


def detectar_comunidades(G: nx.Graph, metodo: str = "louvain", seed: int = SEED,
                         resolucion: float = 1.0, n_objetivo: int | None = None) -> dict:
    """Detecta comunidades con el método indicado y devuelve siempre la misma estructura.

    Métodos: "louvain", "leiden", "label_propagation", "greedy", "girvan_newman", "infomap".
    Todos operan sobre la proyección NO DIRIGIDA (la modularidad clásica está definida
    para grafos no dirigidos) y devuelven:

        {"particion": {nodo: id_comunidad}, "comunidades": [set, ...],
         "n_comunidades": int, "modularidad": float, "segundos": float, "metodo": str}

    Si un método no está disponible en el entorno, se informa y se devuelve None en
    lugar de romper la ejecución del notebook.
    """
    Gu = G.to_undirected() if G.is_directed() else G
    Gu = nx.Graph(Gu)   # colapsa multigrafos: los algoritmos de modularidad los rechazan
    t0 = time.perf_counter()

    if metodo == "louvain":
        comunidades = nx.community.louvain_communities(Gu, seed=seed, resolution=resolucion)

    elif metodo == "leiden":
        if not LEIDEN_OK:
            print("  Leiden no disponible (falta `leidenalg`/`igraph`). Se omite.")
            return None
        import igraph as ig
        import leidenalg
        nodos = list(Gu.nodes())
        indice = {n: i for i, n in enumerate(nodos)}
        g_ig = ig.Graph(n=len(nodos), edges=[(indice[u], indice[v]) for u, v in Gu.edges()])
        particion_ig = leidenalg.find_partition(
            g_ig, leidenalg.RBConfigurationVertexPartition,
            resolution_parameter=resolucion, seed=seed)
        comunidades = [{nodos[i] for i in grupo} for grupo in particion_ig]

    elif metodo == "label_propagation":
        comunidades = list(nx.community.asyn_lpa_communities(Gu, seed=seed))

    elif metodo == "greedy":
        comunidades = list(nx.community.greedy_modularity_communities(Gu, resolution=resolucion))

    elif metodo == "girvan_newman":
        objetivo = n_objetivo or 8
        generador = nx.community.girvan_newman(Gu)
        comunidades = next(generador)
        for particion in generador:
            comunidades = particion
            if len(comunidades) >= objetivo:
                break
        comunidades = [set(c) for c in comunidades]

    elif metodo == "infomap":
        if not _disponible("infomap"):
            print("  Infomap no disponible (`pip install infomap`). Se omite.")
            return None
        from infomap import Infomap
        nodos = list(Gu.nodes())
        indice = {n: i for i, n in enumerate(nodos)}
        im = Infomap(silent=True, seed=seed)
        for u, v in Gu.edges():
            im.add_link(indice[u], indice[v])
        im.run()
        agrupado = defaultdict(set)
        for nodo in im.tree:
            if nodo.is_leaf:
                agrupado[nodo.module_id].add(nodos[nodo.node_id])
        comunidades = list(agrupado.values())

    else:
        raise ValueError(f"Método desconocido: {metodo!r}")

    segundos = time.perf_counter() - t0
    particion = {n: i for i, c in enumerate(comunidades) for n in c}
    return {
        "metodo": metodo,
        "particion": particion,
        "comunidades": comunidades,
        "n_comunidades": len(comunidades),
        "modularidad": nx.community.modularity(Gu, comunidades),
        "segundos": segundos,
    }


titulo("Louvain sobre el grafo completo de Elliptic")
louvain_global = detectar_comunidades(G, metodo="louvain")
print(f"  Comunidades detectadas : {louvain_global['n_comunidades']:,}")
print(f"  Modularidad Q          : {louvain_global['modularidad']:.4f}")
print(f"  Tiempo                 : {louvain_global['segundos']:.1f} s")

df_nodos["comunidad"] = df_nodos["txId"].map(louvain_global["particion"])
tam_comunidades = df_nodos["comunidad"].value_counts()
print(f"\n  Tamaño: mediana {tam_comunidades.median():.0f} · máximo {tam_comunidades.max():,} · "
      f"mínimo {tam_comunidades.min():,}")
print(f"  Comunidades de menos de 10 nodos: {(tam_comunidades < 10).sum()}")

# ¿Las comunidades respetan la partición temporal? Tienen que hacerlo: no hay aristas
# entre timesteps, así que ninguna comunidad puede abarcar dos.
comunidades_por_ts = df_nodos.groupby("comunidad")["timestep"].nunique()
print(f"\n  Comunidades que abarcan más de un timestep: {(comunidades_por_ts > 1).sum()}")
print(f"  Comunidades por timestep (media): "
      f"{df_nodos.groupby('timestep')['comunidad'].nunique().mean():.1f}")
print("\n  Louvain subdivide cada timestep en unas pocas comunidades. No podía hacer otra cosa:")
print("  al no existir aristas entre timesteps, fusionarlos daría modularidad negativa.")

In [ ]:
# =============================================================================
# M7.2 — Comparación de algoritmos sobre el mismo grafo
# =============================================================================

def comparar_particiones(G: nx.Graph, resultados: list) -> pd.DataFrame:
    """Compara varias particiones del mismo grafo: calidad, tamaño y acuerdo mutuo.

    El acuerdo se mide con NMI (información mutua normalizada), que va de 0 (sin
    relación) a 1 (particiones idénticas). Es la métrica estándar porque no depende
    de cómo se numeren las comunidades ni de cuántas haya.
    """
    resultados = [r for r in resultados if r is not None]
    nodos = list(G.to_undirected().nodes())

    filas = []
    for r in resultados:
        tams = [len(c) for c in r["comunidades"]]
        filas.append({
            "método": r["metodo"],
            "comunidades": r["n_comunidades"],
            "modularidad Q": round(r["modularidad"], 4),
            "mayor": max(tams),
            "mediana": int(np.median(tams)),
            "singletons": sum(1 for t in tams if t == 1),
            "segundos": round(r["segundos"], 2),
        })
    tabla = pd.DataFrame(filas).set_index("método")

    etiquetas = {r["metodo"]: [r["particion"].get(n, -1) for n in nodos] for r in resultados}
    nombres = list(etiquetas)
    nmi = pd.DataFrame(index=nombres, columns=nombres, dtype=float)
    for a, b in itertools.product(nombres, repeat=2):
        nmi.loc[a, b] = normalized_mutual_info_score(etiquetas[a], etiquetas[b])

    return tabla, nmi.round(3)


# Se compara sobre la mayor componente de un timestep: tamaño realista y grafo conexo.
G_comp = G_t_lcc
titulo(f"Comparación de algoritmos sobre la LCC del timestep {TIMESTEP_FOCO} "
       f"({G_comp.number_of_nodes():,} nodos, {G_comp.number_of_edges():,} aristas)")

resultados = []
for metodo in ["louvain", "leiden", "label_propagation", "greedy", "infomap"]:
    print(f"\n▸ {metodo}")
    r = detectar_comunidades(G_comp, metodo=metodo)
    if r:
        print(f"  {r['n_comunidades']} comunidades · Q = {r['modularidad']:.4f} · {r['segundos']:.2f} s")
        resultados.append(r)

tabla_metodos, matriz_nmi = comparar_particiones(G_comp, resultados)
titulo("Calidad y coste de cada método", nivel=2)
display(tabla_metodos)
titulo("Acuerdo entre métodos (NMI)", nivel=2)
display(matriz_nmi)

In [ ]:
# =============================================================================
# M7.3 — Girvan-Newman: la jerarquía completa, en un grafo donde cabe
# =============================================================================

# Girvan-Newman recalcula la betweenness de TODAS las aristas cada vez que elimina una:
# O(m²n). Sobre 5.000 nodos serían horas; sobre 250, segundos.
sub_gn = muestrear_subgrafo(G, 250, metodo="bfs").to_undirected()
sub_gn = nx.Graph(sub_gn.subgraph(max(nx.connected_components(sub_gn), key=len)))

titulo(f"Girvan-Newman sobre {sub_gn.number_of_nodes()} nodos")
progresion = []
generador = nx.community.girvan_newman(sub_gn)
for paso, particion in enumerate(itertools.islice(generador, 12), start=2):
    comunidades = [set(c) for c in particion]
    progresion.append({
        "nº de comunidades": len(comunidades),
        "modularidad Q": round(nx.community.modularity(sub_gn, comunidades), 4),
        "mayor comunidad": max(len(c) for c in comunidades),
    })
df_gn = pd.DataFrame(progresion)
display(df_gn)

mejor = df_gn.loc[df_gn["modularidad Q"].idxmax()]
plt.figure(figsize=(9, 3.8))
plt.plot(df_gn["nº de comunidades"], df_gn["modularidad Q"], marker="o", color=COLORES["licito"], lw=2)
plt.axvline(mejor["nº de comunidades"], ls="--", color=COLORES["ilicito"],
            label=f"Q máxima = {mejor['modularidad Q']:.3f} con {int(mejor['nº de comunidades'])} comunidades")
plt.xlabel("Nº de comunidades (cortes sucesivos del dendrograma)")
plt.ylabel("Modularidad Q")
plt.title("Girvan-Newman: la modularidad marca dónde cortar la jerarquía")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("La ventaja de Girvan-Newman no es la calidad —Louvain suele igualarla— sino que produce")
print("una JERARQUÍA: se puede elegir el nivel de granularidad a posteriori. En investigación de")
print("fraude eso permite pasar de 'la red criminal' a 'las células dentro de la red'.")

In [ ]:
# =============================================================================
# M7.4 — Visualización de comunidades y su composición
# =============================================================================

def visualizar_comunidades(G: nx.Graph, particion: dict, titulo_fig: str = "",
                           max_comunidades: int = 12, max_nodos: int = 2500,
                           seed: int = SEED) -> None:
    """Dibuja el grafo coloreando cada comunidad, con las menores agrupadas en gris.

    Colorear 300 comunidades es ilegible: ningún ojo distingue 300 tonos. Se colorean
    las `max_comunidades` mayores y el resto se pinta en gris — que es exactamente el
    mensaje correcto, porque la cola de comunidades diminutas rara vez es interpretable.

    Por encima de `max_nodos` se muestrea con BFS: el layout de fuerzas es O(n²) por
    iteración y un dibujo de 50.000 nodos es, además, una mancha sin información.
    """
    Gu = nx.Graph(G.to_undirected() if G.is_directed() else G)
    if Gu.number_of_nodes() > max_nodos:
        Gu = nx.Graph(muestrear_subgrafo(Gu, max_nodos, metodo="bfs"))
        print(f"  (grafo muestreado a {Gu.number_of_nodes()} nodos para la visualización)")
    tam = Counter(particion.get(n) for n in Gu.nodes())
    principales = [c for c, _ in tam.most_common(max_comunidades)]
    paleta = plt.cm.tab20(np.linspace(0, 1, max(len(principales), 2)))
    color_de = {c: paleta[i] for i, c in enumerate(principales)}

    pos = nx.spring_layout(Gu, seed=seed, k=0.28, iterations=55)
    colores = [color_de.get(particion.get(n), (0.78, 0.80, 0.83, 0.85)) for n in Gu.nodes()]

    plt.figure(figsize=(11.5, 7.5))
    nx.draw_networkx_edges(Gu, pos, edge_color="#D5D9DE", width=0.45, alpha=0.75)
    nx.draw_networkx_nodes(Gu, pos, node_color=colores, node_size=22, linewidths=0)
    plt.title(titulo_fig or f"{len(tam)} comunidades · se colorean las {len(principales)} mayores")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


mejor_resultado = max(resultados, key=lambda r: r["modularidad"])
visualizar_comunidades(
    G_comp, mejor_resultado["particion"],
    f"Comunidades por {mejor_resultado['metodo']} · timestep {TIMESTEP_FOCO} · "
    f"Q = {mejor_resultado['modularidad']:.3f}")

# ¿Están el fraude y las comunidades relacionados? Es la pregunta que justifica todo el módulo.
comp_por_comunidad = (
    df_nodos[df_nodos["timestep"] == TIMESTEP_FOCO]
    .assign(com=lambda d: d["txId"].map(mejor_resultado["particion"]))
    .dropna(subset=["com"])
    .groupby("com")
    .agg(nodos=("txId", "size"),
         ilicitos=("clase", lambda s: (s == "ilicito").sum()),
         licitos=("clase", lambda s: (s == "licito").sum()))
)
comp_por_comunidad["etiquetados"] = comp_por_comunidad["ilicitos"] + comp_por_comunidad["licitos"]
comp_por_comunidad["% ilícito"] = (
    100 * comp_por_comunidad["ilicitos"] / comp_por_comunidad["etiquetados"].replace(0, np.nan)
).round(1)

destacadas = comp_por_comunidad[comp_por_comunidad["etiquetados"] >= 20].sort_values(
    "% ilícito", ascending=False)
titulo("Concentración de fraude por comunidad (comunidades con ≥20 nodos etiquetados)", nivel=2)
display(destacadas.head(10))

base = 100 * comp_por_comunidad["ilicitos"].sum() / max(comp_por_comunidad["etiquetados"].sum(), 1)
print(f"\nTasa base de ilícitos en el timestep {TIMESTEP_FOCO}: {base:.1f}%")
if len(destacadas):
    print(f"Comunidad más contaminada: {destacadas['% ilícito'].iloc[0]:.1f}% "
          f"({destacadas['% ilícito'].iloc[0]/max(base,0.01):.1f}x la tasa base)")
print("""
Que el fraude se concentre en unas pocas comunidades y no se reparta uniformemente es
la justificación empírica de todo el enfoque de grafos: significa que la PERTENENCIA A
UNA COMUNIDAD es información predictiva sobre un nodo, más allá de sus propios atributos.
El Módulo 11 convierte esa observación en features y el Módulo 15 la mete en un modelo.""")

---
---

# Módulo 8 — Motifs

Un **motif** es un patrón pequeño de conexión —tres o cuatro nodos— que aparece en la red con
frecuencia significativamente distinta a la esperada por azar. La idea, tomada de la biología de
sistemas (Milo et al., 2002), es que **la función se codifica en la estructura local**: si un
patrón está sobre-representado, algo lo está produciendo.

## Los 16 tríadas dirigidas

Con tres nodos y aristas dirigidas hay exactamente 16 configuraciones posibles. NetworkX las
cuenta todas de golpe con `triadic_census`. Las relevantes para fraude:

| Código | Forma | Nombre | Interpretación en flujo de fondos |
|---|---|---|---|
| `021D` | A ← B → C | **Estrella de salida** (out-star) | **Dispersión**: una cuenta reparte a varias (*smurfing*) |
| `021U` | A → B ← C | **Estrella de entrada** (in-star) | **Agregación**: varias cuentas concentran en una (*mula*) |
| `021C` | A → B → C | **Cadena** | Tránsito: B es intermediario puro |
| `030T` | A→B, B→C, A→C | **Feed-Forward Loop** | Pago directo **y** por intermediario: redundancia o encubrimiento |
| `030C` | A→B→C→A | **Ciclo dirigido** | **Layering**: el dinero vuelve al origen |
| `300` | Todas recíprocas | Triángulo completo | Grupo fuertemente acoplado |

### El Feed-Forward Loop

Es el motif más estudiado. A envía a C directamente **y también** a través de B. En redes
biológicas implementa un retardo y un filtro de ruido. En flujos financieros tiene otra lectura:
el camino directo es la operación real y el indirecto la justifica, o al revés. Un FFL con
importes casi idénticos en ambas ramas es una señal fuerte de operación estructurada.

### Cadenas y ciclos

Una **cadena larga** de cuentas que solo reciben y reenvían casi todo el importe es la firma
canónica del *layering*: cada salto añade una capa de distancia entre el origen del dinero y su
destino. Un **ciclo** cierra la cadena sobre sí misma, y en un grafo de flujo de fondos casi
nunca tiene explicación legítima.

## Significancia: comparar con un modelo nulo

Que un motif aparezca 8.000 veces no dice nada por sí solo. La pregunta es cuántas veces
aparecería en un grafo **aleatorio con la misma secuencia de grados**. El z-score

$$z = \frac{N_{\text{real}} - \langle N_{\text{aleatorio}}\rangle}{\sigma_{\text{aleatorio}}}$$

mide la sobre-representación. Es lo que separa "este patrón existe" de "este patrón es
característico de esta red".

In [ ]:
# =============================================================================
# M8.1 — Censo triádico completo
# =============================================================================

DESCRIPCION_TRIADAS = {
    "003": "vacía (sin aristas)",
    "012": "una sola arista",
    "102": "una arista recíproca",
    "021D": "estrella de SALIDA — dispersión / smurfing",
    "021U": "estrella de ENTRADA — agregación / mula",
    "021C": "cadena A→B→C — tránsito",
    "111D": "recíproca + entrante",
    "111U": "recíproca + saliente",
    "030T": "FEED-FORWARD LOOP",
    "030C": "CICLO dirigido de 3 — layering",
    "201": "dos recíprocas",
    "120D": "estrella de salida + recíproca",
    "120U": "estrella de entrada + recíproca",
    "120C": "cadena + recíproca",
    "210": "casi completa",
    "300": "triángulo completo",
}


def contar_motifs(G: nx.DiGraph, nombre: str = "") -> pd.DataFrame:
    """Censo de las 16 tríadas dirigidas mediante el algoritmo de Batagelj-Mrvar.

    Cuenta cada patrón de 3 nodos en O(m·d̄) en lugar del O(n³) que costaría
    enumerar todos los tríos. Devuelve conteos absolutos y porcentaje sobre las
    tríadas NO vacías (la tríada "003" domina de tal forma en un grafo disperso
    —el 99,99 % de los tríos no tiene ninguna arista— que incluirla aplasta al resto).
    """
    censo = nx.triadic_census(G)
    total_no_vacias = sum(v for k, v in censo.items() if k != "003")

    df = pd.DataFrame({
        "tríada": list(censo.keys()),
        "descripción": [DESCRIPCION_TRIADAS[k] for k in censo],
        "conteo": list(censo.values()),
    })
    df["% de las no vacías"] = (100 * df["conteo"] / max(total_no_vacias, 1)).round(3)
    df.loc[df["tríada"] == "003", "% de las no vacías"] = np.nan
    return df.sort_values("conteo", ascending=False).reset_index(drop=True)


titulo(f"Censo triádico del timestep {TIMESTEP_FOCO}")
censo, _ = cronometrar(contar_motifs, G_t, etiqueta="triadic_census")
display(censo)

interes = censo[censo["tríada"].isin(["021D", "021U", "021C", "030T", "030C", "300"])]
print("\nMotifs de interés para fraude:")
for _, fila in interes.iterrows():
    print(f"  {fila['tríada']:5s} {fila['conteo']:>10,}  {fila['descripción']}")

ciclos_3 = int(censo.loc[censo['tríada'] == '030C', 'conteo'].iloc[0])
ffl = int(censo.loc[censo['tríada'] == '030T', 'conteo'].iloc[0])
print(f"""
Dos lecturas inmediatas:

  · Ciclos dirigidos de 3 (030C): {ciclos_3}. Cero, como exige el Módulo 4: el grafo es
    un DAG. Ninguna búsqueda de layering va a encontrar nada en Elliptic a nivel de
    transacción, por sofisticada que sea. Es una limitación del MODELADO, no del método.
  · Feed-forward loops (030T): {ffl:,}. Existen porque una transacción puede financiar a
    otra directamente y también a través de una tercera, sin violar la aciclicidad.
""")

In [ ]:
# =============================================================================
# M8.2 — Significancia frente a un modelo nulo con los mismos grados
# =============================================================================

def significancia_motifs(G: nx.DiGraph, n_aleatorios: int = 8, seed: int = SEED,
                         triadas=("021D", "021U", "021C", "030T", "030C")) -> pd.DataFrame:
    """Compara el censo real con el de grafos aleatorios de la MISMA secuencia de grados.

    El modelo nulo correcto no es un grafo aleatorio cualquiera: es uno que conserve
    los grados de entrada y salida de cada nodo. Si no, cualquier red con hubs
    parecería llena de estrellas «sobre-representadas», cuando esas estrellas son una
    consecuencia trivial de tener hubs.

    Se generan `n_aleatorios` réplicas con `directed_configuration_model` y se calcula
    el z-score de cada motif.
    """
    censo_real = nx.triadic_census(G)
    gin = [d for _, d in G.in_degree()]
    gout = [d for _, d in G.out_degree()]

    muestras = defaultdict(list)
    for i in range(n_aleatorios):
        R = nx.directed_configuration_model(gin, gout, seed=seed + i)
        R = nx.DiGraph(R)                       # colapsa multi-aristas
        R.remove_edges_from(nx.selfloop_edges(R))
        censo_r = nx.triadic_census(R)
        for t in triadas:
            muestras[t].append(censo_r[t])

    filas = []
    for t in triadas:
        media = float(np.mean(muestras[t]))
        desv = float(np.std(muestras[t]))
        filas.append({
            "tríada": t,
            "descripción": DESCRIPCION_TRIADAS[t],
            "real": censo_real[t],
            "aleatorio (media)": round(media, 1),
            "z-score": round((censo_real[t] - media) / desv, 2) if desv > 0 else np.nan,
            "veces sobre lo esperado": round(censo_real[t] / media, 2) if media > 0 else np.inf,
        })
    return pd.DataFrame(filas).set_index("tríada")


titulo("¿Están sobre-representados estos motifs?")
sub_nulo = muestrear_subgrafo(G, 3000, metodo="bfs")
print(f"Se compara sobre {sub_nulo.number_of_nodes():,} nodos y {sub_nulo.number_of_edges():,} aristas,")
print("contra 8 réplicas aleatorias con idéntica secuencia de grados.\n")
sig, _ = cronometrar(significancia_motifs, sub_nulo, etiqueta="significancia_motifs")
display(sig)
print("\nUn |z| > 3 indica sobre-representación estadísticamente clara: ese patrón no puede")
print("explicarse solo por la distribución de grados y responde a un mecanismo real de la red.")

In [ ]:
# =============================================================================
# M8.3 — Detección directa de patrones: fan, cadenas y ciclos
# =============================================================================

def detectar_patron_fan(G: nx.DiGraph, umbral: int = 20, sentido: str = "salida") -> pd.DataFrame:
    """Localiza nodos con fan-out (dispersión) o fan-in (agregación) por encima de un umbral.

    · Fan-out alto  → un origen reparte hacia muchos destinos: *smurfing*, mezclador.
    · Fan-in alto   → muchos orígenes concentran en un destino: cuenta mula, exchange.

    Se añade el ratio de concentración —qué fracción del vecindario opuesto es de
    grado 1— porque un hub de exchange tiene vecinos con actividad propia, mientras
    que un mezclador tiene vecinos que solo existen para esa operación.
    """
    grados = G.out_degree() if sentido == "salida" else G.in_degree()
    candidatos = [(n, d) for n, d in grados if d >= umbral]

    filas = []
    for nodo, grado in sorted(candidatos, key=lambda x: -x[1]):
        vecinos = list(G.successors(nodo)) if sentido == "salida" else list(G.predecessors(nodo))
        hojas = sum(1 for v in vecinos
                    if (G.out_degree(v) if sentido == "salida" else G.in_degree(v)) == 0)
        filas.append({
            "nodo": nodo,
            f"grado de {sentido}": grado,
            "grado opuesto": G.in_degree(nodo) if sentido == "salida" else G.out_degree(nodo),
            "vecinos terminales": hojas,
            "% terminales": round(100 * hojas / max(len(vecinos), 1), 1),
            "clase": G.nodes[nodo].get("clase", "—"),
        })
    return pd.DataFrame(filas)


def detectar_ciclos(G: nx.DiGraph, longitud_max: int = 6, limite: int = 200) -> list:
    """Enumera ciclos dirigidos simples hasta `longitud_max` aristas.

    En un grafo de flujo de fondos entre ENTIDADES, un ciclo significa que el dinero
    regresa al origen: la firma del layering. La enumeración de ciclos es exponencial
    en el caso general, así que se acota la longitud y el número de resultados.
    """
    try:
        generador = nx.simple_cycles(G, length_bound=longitud_max)
    except TypeError:      # NetworkX < 3.1 no admite length_bound
        generador = (c for c in nx.simple_cycles(G) if len(c) <= longitud_max)
    return list(itertools.islice(generador, limite))


titulo("Patrones de dispersión y agregación en Elliptic")
fan_out = detectar_patron_fan(G, umbral=40, sentido="salida")
fan_in = detectar_patron_fan(G, umbral=100, sentido="entrada")
print(f"Nodos con fan-out ≥ 40  : {len(fan_out)}")
display(fan_out.head(6))
print(f"\nNodos con fan-in ≥ 100  : {len(fan_in)}")
display(fan_in.head(6))

print(f"\nCiclos en Elliptic (longitud ≤ 6): {len(detectar_ciclos(G_t, 6))}")
print("Cero, como estaba previsto. La búsqueda de ciclos necesita el otro grafo.")

In [ ]:
# =============================================================================
# M8.4 — Motifs de fraude donde sí existen: el grafo bancario
# =============================================================================

def proyectar_transferencias(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """Colapsa `cuenta -emite-> Transferencia -recibe-> cuenta` en `cuenta -> cuenta`.

    La reificación es excelente para almacenar y consultar, pero los algoritmos de
    motifs y ciclos necesitan que la arista vaya directamente de entidad a entidad.
    Esta proyección conserva `monto`, `ts` y el identificador de la transferencia
    original, para poder volver del hallazgo al dato de origen.
    """
    P = nx.MultiDiGraph()
    P.add_nodes_from(n for n, d in G.nodes(data=True) if d["tipo"] == "cuenta")
    for trf, datos in G.nodes(data=True):
        if datos["tipo"] != "transferencia":
            continue
        origenes = [u for u in G.predecessors(trf) if G.nodes[u]["tipo"] == "cuenta"]
        destinos = [v for v in G.successors(trf) if G.nodes[v]["tipo"] == "cuenta"]
        for o, d in itertools.product(origenes, destinos):
            P.add_edge(o, d, monto=datos["monto"], ts=datos["ts"], transferencia=trf)
    return P


G_trf = proyectar_transferencias(G_banco)
titulo("Motifs en el grafo de transferencias entre cuentas")
print(f"Proyección: {G_trf.number_of_nodes():,} cuentas, {G_trf.number_of_edges():,} transferencias")
print(f"¿Es un DAG?: {nx.is_directed_acyclic_graph(nx.DiGraph(G_trf))}  ← aquí sí hay ciclos\n")

censo_banco = contar_motifs(nx.DiGraph(G_trf))
display(censo_banco[censo_banco["tríada"].isin(["021D", "021U", "021C", "030T", "030C"])])

# --- Ciclos de layering, contrastados con el ground truth ---
ciclos = detectar_ciclos(nx.DiGraph(G_trf), longitud_max=7, limite=400)
cuentas_layering = {n for n in GT_BANCO.get("aml_layering", set())
                    if G_banco.nodes[n]["tipo"] == "cuenta"}

filas = []
for ciclo in sorted(ciclos, key=len, reverse=True)[:10]:
    en_gt = len(set(ciclo) & cuentas_layering)
    montos = [G_trf[u][v][0]["monto"] for u, v in zip(ciclo, ciclo[1:] + ciclo[:1])
              if G_trf.has_edge(u, v)]
    filas.append({
        "longitud": len(ciclo),
        "cuentas": " → ".join(c.replace("CTA_", "") for c in ciclo[:5]) + ("…" if len(ciclo) > 5 else ""),
        "en ground truth": f"{en_gt}/{len(ciclo)}",
        "monto medio": round(float(np.mean(montos)), 2) if montos else None,
        "dispersión de montos": round(float(np.std(montos) / np.mean(montos)), 3) if montos else None,
    })

print(f"Ciclos dirigidos encontrados (longitud ≤ 7): {len(ciclos)}")
display(pd.DataFrame(filas))

cubiertas = set().union(*[set(c) for c in ciclos]) if ciclos else set()
print(f"\nRecall sobre las cuentas de layering inyectadas: "
      f"{100*len(cubiertas & cuentas_layering)/max(len(cuentas_layering),1):.0f}% "
      f"({len(cubiertas & cuentas_layering)}/{len(cuentas_layering)})")
print("""
La 'dispersión de montos' es la que separa el layering del ruido: en un ciclo de blanqueo
los importes son casi idénticos en cada salto (el dinero es el mismo, menos comisión), así
que su coeficiente de variación es cercano a cero. Un ciclo casual entre cuentas normales
tiene importes sin ninguna relación entre sí. El motif encuentra los candidatos; el atributo
de la arista los filtra. Ninguna de las dos cosas basta por separado.""")

# --- Fan-in / fan-out: money mules ---
titulo("Fan-in y fan-out en el grafo bancario", nivel=2)
fan_in_banco = detectar_patron_fan(nx.DiGraph(G_trf), umbral=8, sentido="entrada")
cuentas_mula = {n for n in GT_BANCO.get("money_mule", set())
                if G_banco.nodes[n]["tipo"] == "cuenta"}
fan_in_banco["¿es mula real?"] = fan_in_banco["nodo"].isin(cuentas_mula)
display(fan_in_banco.head(12))
detectadas = set(fan_in_banco["nodo"]) & cuentas_mula
print(f"Cuentas con fan-in ≥ 8: {len(fan_in_banco)} · de ellas, mulas reales: {len(detectadas)}")
print(f"Precisión: {100*len(detectadas)/max(len(fan_in_banco),1):.0f}% · "
      f"Recall: {100*len(detectadas)/max(len(cuentas_mula),1):.0f}%")

---
---

# Módulo 9 — Clustering

El **coeficiente de clustering** mide cuánto se cierran los triángulos: si dos nodos comparten
un vecino, ¿están también conectados entre sí?

## Local

Para un vértice $v$ con $k_v$ vecinos y $e_v$ aristas entre ellos:

$$C_v = \frac{2 e_v}{k_v(k_v - 1)}$$

Va de 0 (los vecinos no se conocen entre sí: $v$ es el centro de una estrella) a 1 (todos los
vecinos se conocen: $v$ está en un grupo cerrado). Para $k_v < 2$ no está definido y se toma 0.

## Global

Hay dos definiciones que se confunden constantemente y **no dan el mismo número**:

$$\bar{C} = \frac{1}{n}\sum_v C_v \qquad\text{(clustering medio)} \qquad\qquad
T = \frac{3 \times \#\text{triángulos}}{\#\text{tríadas conectadas}} \qquad\text{(transitividad)}$$

- El **clustering medio** promedia coeficientes locales, así que da el mismo peso a un nodo de
  grado 2 que a uno de grado 500. En redes con cola pesada —donde la mayoría son nodos de grado
  bajo— acaba dominado por nodos casi irrelevantes y sale artificialmente alto.
- La **transitividad** es un cociente global, dominado por los nodos de grado alto.

Si $\bar{C} \gg T$, la red tiene muchos nodos periféricos con vecindarios cerrados y hubs con
vecindarios abiertos: la firma de una **estructura jerárquica**.

## Cierre triádico

La hipótesis de **triadic closure** (Rapoport, 1953) dice que si A conoce a B y B conoce a C, la
probabilidad de que A y C acaben conectados es mucho mayor que la de dos nodos al azar. Es el
mecanismo que genera clustering alto en redes sociales, y la base teórica de los algoritmos de
link prediction del Módulo 12 —todos ellos son, en el fondo, formas de contar vecinos comunes.

## Qué esperar en un grafo de transacciones

Poco clustering, y por una razón estructural: **el grafo es un DAG**. Un triángulo no dirigido
$u-v-w$ requiere, en el grafo dirigido, o bien un ciclo (imposible en un DAG) o bien un
feed-forward loop. Solo los FFL contribuyen. Un valor de clustering bajo aquí no indica ausencia
de estructura: indica que la estructura no es de tipo social.

In [ ]:
# =============================================================================
# M9.1 — Coeficientes local y global
# =============================================================================

def metricas_clustering(G: nx.Graph, nombre: str = "", muestra: int | None = None,
                        seed: int = SEED) -> dict:
    """Clustering medio, transitividad y distribución de coeficientes locales.

    Trabaja sobre la proyección no dirigida: el clustering clásico está definido para
    grafos no dirigidos (existe una versión dirigida, `nx.clustering(DiGraph)`, pero
    mezcla cuatro tipos de triángulo y es difícil de interpretar).

    Con `muestra` se calcula el clustering local solo sobre un subconjunto de nodos:
    el coste es O(Σ d²) y en grafos con hubs de grado 500 la cola domina el cálculo.
    """
    Gu = nx.Graph(G.to_undirected() if G.is_directed() else G)
    Gu.remove_edges_from(nx.selfloop_edges(Gu))

    nodos = list(Gu.nodes())
    if muestra and muestra < len(nodos):
        nodos = random.Random(seed).sample(nodos, muestra)

    locales = nx.clustering(Gu, nodes=nodos)
    valores = np.array(list(locales.values()))
    triangulos = nx.triangles(Gu, nodes=nodos)

    return {
        "grafo": nombre,
        "nodos evaluados": len(nodos),
        "clustering medio (C̄)": round(float(valores.mean()), 5),
        "transitividad (T)": round(nx.transitivity(Gu), 5),
        "nodos con C = 0": int(np.sum(valores == 0)),
        "% con C = 0": round(100 * float(np.mean(valores == 0)), 2),
        "nodos con C = 1": int(np.sum(valores == 1)),
        "triángulos totales": sum(triangulos.values()) // 3,
        "_locales": locales,
    }


titulo("Clustering en los grafos de trabajo")
filas = []
for nombre, grafo, muestra in [
    (f"Elliptic · timestep {TIMESTEP_FOCO}", G_t, None),
    ("Elliptic · H (BFS 1.200)", H, None),
    ("Banco · transferencias", G_trf, None),
    ("Banco · clientes por recurso", P_clientes, None),
]:
    r = metricas_clustering(grafo, nombre, muestra=muestra)
    filas.append({k: v for k, v in r.items() if not k.startswith("_")})
display(pd.DataFrame(filas).set_index("grafo"))

clust_t = metricas_clustering(G_t, f"timestep {TIMESTEP_FOCO}")
print(f"""
En el timestep {TIMESTEP_FOCO}: C̄ = {clust_t['clustering medio (C̄)']:.5f} y T = {clust_t['transitividad (T)']:.5f}.
El {clust_t['% con C = 0']:.1f}% de los nodos tiene clustering cero, es decir, sus vecinos no se
conocen entre sí. Coherente con un DAG: los triángulos solo pueden venir de feed-forward
loops, y el censo triádico del Módulo 8 ya mostró que son minoría.

La red de CLIENTES por recurso compartido es el caso opuesto: clustering altísimo. Es lo
esperado, porque compartir un dispositivo es una relación casi de equivalencia — si A y B
comparten un móvil y B y C también, lo normal es que A y C lo compartan. Ese contraste entre
un clustering ~0 y uno ~1 en el mismo dominio muestra que el coeficiente no mide "cantidad de
estructura" sino QUÉ TIPO de estructura hay.""")

In [ ]:
# =============================================================================
# M9.2 — Clustering frente a grado: ¿hay jerarquía?
# =============================================================================

Gt_und = nx.Graph(G_t.to_undirected())
locales_t = nx.clustering(Gt_und)
grados_t = dict(Gt_und.degree())
df_clust = pd.DataFrame({
    "txId": list(locales_t.keys()),
    "clustering": list(locales_t.values()),
    "grado": [grados_t[n] for n in locales_t],
}).merge(df_nodos[["txId", "clase"]], on="txId", how="left")

por_grado = df_clust[df_clust["grado"] >= 2].groupby("grado")["clustering"].agg(["mean", "size"])
por_grado = por_grado[por_grado["size"] >= 5]

fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2))

ejes[0].hist(df_clust["clustering"], bins=40, color=COLORES["licito"], edgecolor="white")
ejes[0].set_yscale("log")
ejes[0].set_title("Distribución del clustering local")
ejes[0].set_xlabel("C(v)")
ejes[0].set_ylabel("Nº de nodos (log)")

if len(por_grado) > 3:
    ejes[1].scatter(por_grado.index, por_grado["mean"], s=26, color=COLORES["acento"], alpha=0.8)
    ejes[1].set_xscale("log")
    ejes[1].set_title("C(k) medio frente al grado k")
    ejes[1].set_xlabel("Grado k")
    ejes[1].set_ylabel("Clustering medio")
    ejes[1].text(0.04, 0.9, "C(k) ~ k⁻¹ indicaría\njerarquía", transform=ejes[1].transAxes, fontsize=8)

por_clase = df_clust[df_clust["clase"] != "desconocido"].groupby("clase")["clustering"].mean()
if len(por_clase):
    ejes[2].bar(por_clase.index, por_clase.values,
                color=[COLORES["ilicito"] if c == "ilicito" else COLORES["licito"] for c in por_clase.index])
    ejes[2].set_title("Clustering medio por clase")
    ejes[2].set_ylabel("C̄")
    for i, v in enumerate(por_clase.values):
        ejes[2].text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

if len(por_clase) == 2:
    print(f"Clustering medio · ilícitos: {por_clase.get('ilicito', 0):.5f} | "
          f"lícitos: {por_clase.get('licito', 0):.5f}")
    print("Si los valores difieren, el clustering local es una feature discriminante y merece")
    print("entrar en el modelo del Módulo 15. Si no, es una métrica descriptiva y nada más.")

In [ ]:
# =============================================================================
# M9.3 — Cierre triádico medido en el tiempo
# =============================================================================

def analizar_cierre_triadico(G_temporal: nx.MultiDiGraph, atributo_ts: str = "ts",
                             percentil_corte: int = 60, max_wedges: int = 60_000,
                             seed: int = SEED) -> dict:
    """Mide empíricamente la hipótesis del cierre triádico usando el tiempo real.

    Procedimiento:
      1. Se parte el grafo por el percentil `percentil_corte` de los timestamps:
         `pasado` (lo ocurrido antes) y `futuro` (lo ocurrido después).
      2. Se enumeran las TRÍADAS ABIERTAS del pasado: pares (u,w) no conectados que
         comparten al menos un vecino.
      3. Se mide qué fracción de esos pares aparece conectada en el futuro.
      4. Se compara con la tasa base: la probabilidad de que un par NO adyacente y
         SIN vecinos comunes se conecte en el futuro.

    El cociente entre ambas tasas es la magnitud del efecto. Si es 1, el cierre
    triádico no existe en esta red y los algoritmos de link prediction basados en
    vecinos comunes (Módulo 12) no tienen fundamento aquí.
    """
    rng = random.Random(seed)
    aristas = [(u, v, d[atributo_ts]) for u, v, d in G_temporal.edges(data=True)
               if atributo_ts in d]
    if not aristas:
        return {"error": f"Ninguna arista tiene el atributo {atributo_ts!r}"}

    corte = float(np.percentile([t for _, _, t in aristas], percentil_corte))
    pasado = nx.Graph()
    pasado.add_nodes_from(G_temporal.nodes())
    pasado.add_edges_from((u, v) for u, v, t in aristas if t <= corte)
    futuro = {frozenset((u, v)) for u, v, t in aristas if t > corte and u != v}

    # Tríadas abiertas del pasado
    wedges = set()
    for v in pasado.nodes():
        vecinos = sorted(pasado.neighbors(v))
        if len(vecinos) < 2:
            continue
        for u, w in itertools.combinations(vecinos, 2):
            if not pasado.has_edge(u, w):
                wedges.add(frozenset((u, w)))
            if len(wedges) >= max_wedges:
                break
        if len(wedges) >= max_wedges:
            break

    cerradas = sum(1 for par in wedges if par in futuro)

    # Tasa base sobre pares aleatorios sin vecinos comunes
    nodos = [n for n in pasado.nodes() if pasado.degree(n) > 0]
    control, intentos = set(), 0
    while len(control) < len(wedges) and intentos < 20 * max(len(wedges), 1):
        intentos += 1
        u, w = rng.sample(nodos, 2)
        if not pasado.has_edge(u, w) and not (set(pasado.neighbors(u)) & set(pasado.neighbors(w))):
            control.add(frozenset((u, w)))
    control_cerradas = sum(1 for par in control if par in futuro)

    tasa_wedge = cerradas / max(len(wedges), 1)
    tasa_base = control_cerradas / max(len(control), 1)

    return {
        "corte temporal (percentil)": percentil_corte,
        "aristas en el pasado": pasado.number_of_edges(),
        "aristas nuevas en el futuro": len(futuro),
        "tríadas abiertas evaluadas": len(wedges),
        "de ellas, cerradas después": cerradas,
        "tasa de cierre triádico": round(tasa_wedge, 5),
        "tasa base (sin vecino común)": round(tasa_base, 5),
        "efecto (veces la tasa base)": round(tasa_wedge / tasa_base, 1) if tasa_base > 0 else np.inf,
    }


titulo("Cierre triádico en el grafo bancario de transferencias")
cierre = analizar_cierre_triadico(G_trf, atributo_ts="ts", percentil_corte=60)
for k, v in cierre.items():
    print(f"  {k:34s} {formatear(v):>12s}")

print("""
Esta medición es la que da (o quita) legitimidad al Módulo 12. Los algoritmos de link
prediction —Jaccard, Adamic-Adar, Resource Allocation— asumen todos que compartir vecinos
predice conexión futura. Si el efecto medido aquí fuera 1, esos algoritmos no serían
mejores que el azar en esta red y habría que usar otra familia (embeddings, atributos).

En Elliptic esta medición no se puede hacer: cada timestep tiene nodos DISTINTOS, así que
no existe ningún par de nodos que esté abierto en t y cerrado en t+1. Es otra consecuencia
del hallazgo del Módulo 4, y condiciona cómo se evalúa el link prediction más adelante.""")

---
---

# Módulo 10 — Redes temporales

Muy importante para fraude, porque **el fraude es un fenómeno temporal**: no es un patrón que
esté ahí, es un patrón que aparece, opera y desaparece. Un grafo estático es la fotografía de
una película.

## Dos formas de modelar el tiempo

| | **Snapshot Graph** | **Dynamic / Continuous-Time Graph** |
|---|---|---|
| Representación | Secuencia $G_1, G_2, \dots, G_T$ de grafos estáticos | Flujo de eventos $(u, v, t)$ |
| Ventaja | Todo el instrumental estático se aplica sin cambios | No pierde resolución temporal |
| Inconveniente | La ventana es arbitraria y difumina lo que pasa dentro | Necesita algoritmos específicos (TGN, TGAT) |
| Cuándo usarlo | Análisis exploratorio, informes periódicos | Detección en tiempo real |

Elliptic viene ya en formato snapshot: los 49 timesteps son ventanas de unas dos semanas.
El grafo bancario lleva `ts` en cada arista, así que admite las dos lecturas y sirve para
demostrar el ciclo de vida de las comunidades.

## Ciclo de vida de una comunidad

Comparando las particiones de dos snapshots consecutivos con el índice de Jaccard

$$J(C_i^t, C_j^{t+1}) = \frac{|C_i^t \cap C_j^{t+1}|}{|C_i^t \cup C_j^{t+1}|}$$

se clasifica cada comunidad en cinco eventos:

- **Aparición** — no se parece a ninguna comunidad anterior. En fraude: una red que se activa.
- **Persistencia** — se corresponde con una sola comunidad anterior. Operación estable.
- **Fusión** — dos o más comunidades anteriores convergen en una.
- **División** — una comunidad anterior se rompe en varias. Típico tras una intervención.
- **Desaparición** — no tiene continuación. En fraude: red desmantelada o abandonada.

## Advertencia sobre Elliptic

El Módulo 4 demostró que **cada timestep tiene nodos completamente distintos**: una transacción
existe en un único timestep. Eso significa que el seguimiento de comunidades por Jaccard sobre
Elliptic daría **cero en todas las comparaciones**, no por ausencia de estructura sino porque
la intersección de conjuntos disjuntos es vacía por definición.

Por tanto: sobre Elliptic se sigue la **evolución de propiedades agregadas**; sobre el grafo
bancario —donde las cuentas persisten— se sigue el **ciclo de vida de las comunidades**. Aplicar
el segundo análisis al primero produciría una tabla llena de ceros presentada como un hallazgo.

In [ ]:
# =============================================================================
# M10.1 — Snapshots y evolución de métricas en Elliptic
# =============================================================================

def construir_snapshots(G: nx.Graph, df_nodos: pd.DataFrame,
                        columna: str = "timestep") -> dict:
    """Parte el grafo en subgrafos, uno por valor de `columna`.

    En Elliptic el resultado coincide exactamente con las componentes débiles, así
    que ningún snapshot pierde aristas: son grafos completos, no cortes truncados.
    """
    snapshots = {}
    for valor, grupo in df_nodos.groupby(columna):
        snapshots[int(valor)] = G.subgraph(grupo["txId"]).copy()
    return dict(sorted(snapshots.items()))


def evolucion_metricas(snapshots: dict, df_nodos: pd.DataFrame,
                       particion: dict | None = None) -> pd.DataFrame:
    """Serie temporal de métricas estructurales, una fila por snapshot.

    Se calculan solo métricas O(n+m) o O(Σd²): a 49 snapshots, cualquier métrica
    cuadrática convertiría esta celda en media hora de cómputo.
    """
    info = df_nodos.set_index("txId")
    filas = []
    for t, g in snapshots.items():
        nodos = list(g.nodes())
        sub = info.loc[nodos]
        etiquetados = sub[sub["clase"] != "desconocido"]
        gu = nx.Graph(g.to_undirected())

        fila = {
            "timestep": t,
            "nodos": g.number_of_nodes(),
            "aristas": g.number_of_edges(),
            "densidad": nx.density(g),
            "grado medio": 2 * g.number_of_edges() / max(g.number_of_nodes(), 1),
            "transitividad": nx.transitivity(gu),
            "componentes": nx.number_connected_components(gu),
            "mayor componente": len(max(nx.connected_components(gu), key=len)) if g.number_of_nodes() else 0,
            "% etiquetado": 100 * len(etiquetados) / max(len(sub), 1),
            "% ilícito": 100 * (etiquetados["clase"] == "ilicito").mean() if len(etiquetados) else np.nan,
            "ilícitos": int((sub["clase"] == "ilicito").sum()),
        }
        if particion is not None:
            fila["comunidades"] = len({particion[n] for n in nodos if n in particion})
        filas.append(fila)

    return pd.DataFrame(filas).set_index("timestep")


titulo("Snapshots de Elliptic")
snapshots = construir_snapshots(G, df_nodos)
print(f"Snapshots construidos: {len(snapshots)}")
evolucion, _ = cronometrar(evolucion_metricas, snapshots, df_nodos,
                           etiqueta="evolucion_metricas", particion=louvain_global["particion"])
display(evolucion.head(8).round(4))
print("…")
display(evolucion.tail(8).round(4))

In [ ]:
# =============================================================================
# M10.2 — El evento del timestep 43
# =============================================================================

fig, ejes = plt.subplots(3, 2, figsize=(14, 10))
paneles = [
    ("nodos", "Volumen de transacciones", COLORES["licito"]),
    ("aristas", "Aristas por snapshot", COLORES["neutro"]),
    ("densidad", "Densidad", COLORES["acento"]),
    ("% ilícito", "% de ilícitos sobre etiquetados", COLORES["ilicito"]),
    ("comunidades", "Comunidades por snapshot", COLORES["ok"]),
    ("transitividad", "Transitividad", "#8E5572"),
]
for eje, (columna, titulo_ax, color) in zip(ejes.ravel(), paneles):
    if columna not in evolucion:
        eje.axis("off")
        continue
    eje.plot(evolucion.index, evolucion[columna], marker="o", ms=3.5, lw=1.6, color=color)
    eje.axvline(43, ls="--", c="#9AA3AD", lw=1.1)
    eje.set_title(titulo_ax, fontsize=10.5)
    eje.set_xlabel("timestep")
fig.suptitle("Evolución de la red a lo largo de los 49 snapshots (línea punteada: t = 43)",
             fontsize=12.5, fontweight="bold")
plt.tight_layout()
plt.show()

antes = evolucion.loc[evolucion.index <= 43, "% ilícito"]
despues = evolucion.loc[evolucion.index > 43, "% ilícito"]
titulo("Antes y después del timestep 43", nivel=2)
print(f"  % ilícito medio hasta t=43 : {antes.mean():.2f}%")
print(f"  % ilícito medio desde t=44 : {despues.mean():.2f}%")
print(f"  Caída relativa             : {100*(1 - despues.mean()/max(antes.mean(),1e-9)):.1f}%")
print(f"\n  Volumen medio hasta t=43   : {evolucion.loc[evolucion.index <= 43, 'nodos'].mean():,.0f} transacciones")
print(f"  Volumen medio desde t=44   : {evolucion.loc[evolucion.index > 43, 'nodos'].mean():,.0f} transacciones")
print("""
Es el hecho documentado en el paper original de Elliptic (Weber et al., 2019): en el
timestep 43 se produjo el cierre de un mercado negro, y la actividad ilícita etiquetada
se desploma a partir de ahí.

Este evento es la razón por la que el Módulo 15 usa un SPLIT TEMPORAL y no uno aleatorio.
Con un split aleatorio, el modelo vería ejemplos de después del cierre durante el
entrenamiento y las métricas saldrían infladas: estaría prediciendo un mundo que en el
momento de la decisión aún no existía. Es fuga de información temporal, y es el error
más común al aplicar ML a datos de fraude.""")

In [ ]:
# =============================================================================
# M10.3 — Ciclo de vida de comunidades (grafo bancario)
# =============================================================================

def snapshots_por_tiempo(G: nx.MultiDiGraph, atributo_ts: str = "ts",
                         n_ventanas: int = 8) -> dict:
    """Corta un grafo con timestamps en `n_ventanas` snapshots de igual duración.

    Cada snapshot contiene las aristas de su ventana y los nodos que participan en
    ellas. A diferencia de Elliptic, aquí las ENTIDADES persisten entre ventanas, que
    es la condición necesaria para poder seguir comunidades en el tiempo.
    """
    aristas = [(u, v, d[atributo_ts]) for u, v, d in G.edges(data=True) if atributo_ts in d]
    if not aristas:
        return {}
    t_min = min(t for _, _, t in aristas)
    t_max = max(t for _, _, t in aristas)
    bordes = np.linspace(t_min, t_max + 1, n_ventanas + 1)

    snapshots = {}
    for i in range(n_ventanas):
        seleccion = [(u, v) for u, v, t in aristas if bordes[i] <= t < bordes[i + 1]]
        g = nx.Graph()
        g.add_edges_from(seleccion)
        snapshots[i] = g
    return snapshots


def seguir_comunidades(snapshots: dict, umbral_jaccard: float = 0.3,
                       metodo: str = "louvain", tam_min: int = 3) -> dict:
    """Sigue comunidades entre snapshots consecutivos y clasifica su ciclo de vida.

    Para cada comunidad de t+1 se calcula el Jaccard con todas las de t:
      · sin ninguna por encima del umbral      → APARICIÓN
      · exactamente una                        → PERSISTENCIA
      · dos o más                              → FUSIÓN
    Y simétricamente, una comunidad de t con dos o más sucesores es una DIVISIÓN y
    una sin ninguno, una DESAPARICIÓN.

    Devuelve la tabla de eventos y la distribución de longevidad (en cuántos
    snapshots consecutivos sobrevive cada comunidad).
    """
    particiones = {}
    for t, g in snapshots.items():
        if g.number_of_nodes() < 4:
            particiones[t] = []
            continue
        r = detectar_comunidades(g, metodo=metodo)
        particiones[t] = [c for c in r["comunidades"] if len(c) >= tam_min]

    def jaccard(a, b):
        return len(a & b) / len(a | b) if (a | b) else 0.0

    eventos = []
    tiempos = sorted(snapshots)
    for t_prev, t_act in zip(tiempos[:-1], tiempos[1:]):
        prev, act = particiones[t_prev], particiones[t_act]
        sucesores = defaultdict(list)

        for j, c_act in enumerate(act):
            emparejados = [i for i, c_prev in enumerate(prev) if jaccard(c_prev, c_act) >= umbral_jaccard]
            for i in emparejados:
                sucesores[i].append(j)
            if not emparejados:
                tipo = "aparición"
            elif len(emparejados) == 1:
                tipo = "persistencia"
            else:
                tipo = "fusión"
            eventos.append({"de": t_prev, "a": t_act, "comunidad": j,
                            "tamaño": len(c_act), "evento": tipo,
                            "predecesores": len(emparejados)})

        for i, c_prev in enumerate(prev):
            if i not in sucesores:
                eventos.append({"de": t_prev, "a": t_act, "comunidad": i,
                                "tamaño": len(c_prev), "evento": "desaparición",
                                "predecesores": 0})
            elif len(sucesores[i]) > 1:
                eventos.append({"de": t_prev, "a": t_act, "comunidad": i,
                                "tamaño": len(c_prev), "evento": "división",
                                "predecesores": len(sucesores[i])})

    return {
        "eventos": pd.DataFrame(eventos),
        "particiones": particiones,
        "comunidades_por_snapshot": {t: len(p) for t, p in particiones.items()},
    }


titulo("Ciclo de vida de comunidades en el grafo bancario")
snaps_banco = snapshots_por_tiempo(G_trf, "ts", n_ventanas=8)
for t, g in snaps_banco.items():
    print(f"  ventana {t}: {g.number_of_nodes():>5,} cuentas activas, {g.number_of_edges():>5,} transferencias")

ciclo = seguir_comunidades(snaps_banco, umbral_jaccard=0.3)
eventos = ciclo["eventos"]

if len(eventos):
    resumen_eventos = eventos.groupby("evento").agg(
        veces=("evento", "size"), tamaño_medio=("tamaño", "mean")).round(1)
    titulo("Eventos detectados", nivel=2)
    display(resumen_eventos)

    tabla = eventos.pivot_table(index="a", columns="evento", values="comunidad",
                                aggfunc="size", fill_value=0)
    fig, ejes = plt.subplots(1, 2, figsize=(14, 4.2))
    tabla.plot(kind="bar", stacked=True, ax=ejes[0], width=0.8, colormap="Set2")
    ejes[0].set_title("Eventos del ciclo de vida por ventana temporal")
    ejes[0].set_xlabel("Ventana de destino")
    ejes[0].set_ylabel("Nº de eventos")
    ejes[0].legend(fontsize=8, ncol=2)

    serie = pd.Series(ciclo["comunidades_por_snapshot"])
    ejes[1].plot(serie.index, serie.values, marker="o", color=COLORES["licito"], lw=2)
    ejes[1].set_title("Comunidades activas (≥3 cuentas) por ventana")
    ejes[1].set_xlabel("Ventana")
    ejes[1].set_ylabel("Nº de comunidades")
    plt.tight_layout()
    plt.show()

print("""
La persistencia es la señal más útil de las cinco. Una comunidad que aparece, opera durante
dos o tres ventanas y desaparece por completo tiene un perfil operativo muy distinto al de
la actividad bancaria normal, que persiste. En un sistema real esa métrica —cuántas ventanas
sobrevive un grupo— entra directamente como feature de riesgo.""")

---
---

# Módulo 11 — Ingeniería de features

Aquí empieza el machine learning. El objetivo es convertir la **posición de un nodo en el grafo**
en columnas numéricas que un modelo tabular pueda consumir. Es el punto donde el análisis de
grafos deja de ser descriptivo y pasa a ser predictivo.

## Las tres familias de features

| Familia | Ejemplos | ¿Necesita etiquetas? | Riesgo de fuga |
|---|---|---|---|
| **Topológicas** | grado, PageRank, betweenness, clustering | No | Ninguno |
| **De comunidad** | id de comunidad, tamaño, ratio de frontera | No | Bajo |
| **De vecindad etiquetada** | nº y ratio de vecinos fraudulentos, distancia al fraude | **Sí** | **Alto** |

Las dos primeras son seguras: solo dependen de la estructura. La tercera es la más predictiva y
también la más peligrosa, y merece una sección propia.

## El problema de la fuga de información

Una feature como *"cuántos vecinos de este nodo son fraudulentos"* es enormemente predictiva —el
fraude se agrupa— pero solo es **legítima** si, en el momento de la predicción, esas etiquetas de
los vecinos realmente se conocen. Tres reglas:

1. **Solo se pueden usar etiquetas del conjunto de entrenamiento.** Usar la etiqueta de un nodo
   de test para construir la feature de otro nodo de test es fuga pura.
2. **Solo se pueden usar etiquetas anteriores al momento de predicción.** Si se predice el
   timestep 40, las etiquetas del 41 no existen todavía.
3. **La feature debe poder calcularse en producción.** Si en el sistema real las etiquetas llegan
   con tres meses de retraso, entrenar con etiquetas instantáneas produce un modelo que en
   producción no funciona.

### Y en Elliptic hay un agravante

El Módulo 4 demostró que **ninguna arista cruza timesteps**. Con el split temporal estándar
(entrenar con $t \leq 34$, probar con $t \geq 35$), **todos** los vecinos de un nodo de test están
también en test. Por tanto:

- Calcular *"vecinos fraudulentos"* con todas las etiquetas → **fuga del 100 %**.
- Calcularlo solo con etiquetas de entrenamiento → **siempre cero** en test: la feature no aporta nada.

No es un matiz teórico: es la diferencia entre reportar un F1 de 0,95 y uno de 0,75, y solo el
segundo es real. La celda M11.3 lo demuestra numéricamente en lugar de afirmarlo.

In [ ]:
# =============================================================================
# M11.1 — Features topológicas y de comunidad (seguras)
# =============================================================================

def features_comunidad(G: nx.Graph, particion: dict) -> pd.DataFrame:
    """Features derivadas de la partición en comunidades.

    · `comunidad`            — identificador (categórico; útil como agregación, no como número)
    · `tamano_comunidad`     — cuántos nodos la componen
    · `comunidades_vecinas`  — a cuántas comunidades distintas apunta el nodo
    · `ratio_frontera`       — fracción de aristas del nodo que salen de su comunidad.
                               Un valor alto marca a un nodo puente: la firma del intermediario.
    """
    tam = Counter(particion.values())
    filas = []
    for n in G.nodes():
        propia = particion.get(n)
        vecinos = set(G.successors(n)) | set(G.predecessors(n)) if G.is_directed() else set(G.neighbors(n))
        comunidades_vecinas = {particion.get(v) for v in vecinos if v in particion}
        externas = sum(1 for v in vecinos if particion.get(v) != propia)
        filas.append({
            "txId": n,
            "comunidad": propia,
            "tamano_comunidad": tam.get(propia, 0),
            "comunidades_vecinas": len(comunidades_vecinas),
            "ratio_frontera": externas / max(len(vecinos), 1),
        })
    return pd.DataFrame(filas)


def construir_features_grafo(G: nx.Graph, df_nodos: pd.DataFrame, particion: dict,
                             centralidades: pd.DataFrame | None = None) -> pd.DataFrame:
    """Ensambla la tabla de features topológicas y de comunidad de todos los nodos.

    Reutiliza lo ya calculado en módulos anteriores (`calcular_centralidades`,
    `detectar_comunidades`) en lugar de recalcularlo: sobre 200 mil nodos, repetir
    PageRank cuesta minutos y no aporta nada nuevo.
    """
    Gu = nx.Graph(G.to_undirected() if G.is_directed() else G)

    # Solo se arrastra `timestep` de la tabla maestra: `clase`, `y` y `comunidad` se
    # incorporan más tarde y traerlas aquí provocaría columnas duplicadas en los merges.
    df = tabla_grados(G, df_nodos[["txId", "timestep"]])
    df["ratio_in_out"] = df["in_degree"] / (df["out_degree"] + 1) if "in_degree" in df else np.nan
    df["es_fuente"] = (df["in_degree"] == 0).astype(int) if "in_degree" in df else 0
    df["es_sumidero"] = (df["out_degree"] == 0).astype(int) if "out_degree" in df else 0

    if centralidades is not None:
        columnas = [c for c in ("pagerank", "katz", "hits_hub", "hits_authority")
                    if c in centralidades.columns]
        df = df.merge(centralidades[["txId"] + columnas], on="txId", how="left")

    df = df.merge(features_comunidad(G, particion), on="txId", how="left")
    df["clustering"] = df["txId"].map(nx.clustering(Gu))

    # Estadísticos del vecindario: describen el entorno sin usar ninguna etiqueta.
    grado_de = dict(Gu.degree())
    medias, maximos = [], []
    for n in df["txId"]:
        vecinos = list(Gu.neighbors(n))
        grados_vec = [grado_de[v] for v in vecinos] if vecinos else [0]
        medias.append(float(np.mean(grados_vec)))
        maximos.append(int(np.max(grados_vec)))
    df["grado_medio_vecinos"] = medias
    df["grado_max_vecinos"] = maximos

    return df


titulo("Construcción de features topológicas")
features_grafo, _ = cronometrar(
    construir_features_grafo, G, df_nodos, louvain_global["particion"],
    etiqueta="construir_features_grafo", centralidades=cent_global)
print(f"\nTabla de features: {features_grafo.shape[0]:,} nodos x {features_grafo.shape[1]} columnas")
print(f"Columnas: {[c for c in features_grafo.columns]}")
display(features_grafo.head())

In [ ]:
# =============================================================================
# M11.2 — Features de vecindad etiquetada, con control explícito de qué se conoce
# =============================================================================

def features_de_vecindad(G: nx.Graph, etiquetas_visibles: dict,
                         prefijo: str = "") -> pd.DataFrame:
    """Features basadas en las etiquetas de los vecinos.

    `etiquetas_visibles` es un dict {nodo: 0|1} que contiene EXCLUSIVAMENTE las
    etiquetas que se pueden usar. Ese argumento es el mecanismo de control: pasar
    todas las etiquetas produce fuga, pasar solo las de entrenamiento produce la
    versión honesta. La función no decide por ti; te obliga a decidir.
    """
    Gu = G.to_undirected(as_view=True) if G.is_directed() else G
    filas = []
    for n in G.nodes():
        vecinos = list(Gu.neighbors(n))
        conocidos = [etiquetas_visibles[v] for v in vecinos if v in etiquetas_visibles]
        n_fraude = int(sum(conocidos))
        filas.append({
            "txId": n,
            f"{prefijo}vecinos_etiquetados": len(conocidos),
            f"{prefijo}vecinos_fraude": n_fraude,
            f"{prefijo}ratio_vecinos_fraude": n_fraude / len(conocidos) if conocidos else 0.0,
        })
    return pd.DataFrame(filas)


def distancia_a_fraude(G: nx.Graph, fuentes, corte: int = 6) -> dict:
    """Distancia en saltos de cada nodo al nodo fraudulento conocido más cercano.

    Se resuelve con un BFS multi-fuente (un solo recorrido desde TODAS las fuentes a
    la vez, O(m log n)), no con un BFS por cada nodo fraudulento. Los nodos a más de
    `corte` saltos no aparecen en el resultado; quien llama decide con qué valor
    rellenarlos —usar `corte + 1` es la convención habitual.
    """
    Gu = G.to_undirected(as_view=True) if G.is_directed() else G
    fuentes = {f for f in fuentes if f in Gu}
    if not fuentes:
        return {}
    return nx.multi_source_dijkstra_path_length(Gu, fuentes, cutoff=corte)


# Definición del split temporal, el mismo que usarán los Módulos 15 y 16.
CORTE_TEMPORAL = 34
mask_train = (df_nodos["timestep"] <= CORTE_TEMPORAL) & (df_nodos["y"] >= 0)
mask_test = (df_nodos["timestep"] > CORTE_TEMPORAL) & (df_nodos["y"] >= 0)

etiquetas_todas = dict(zip(df_nodos.loc[df_nodos["y"] >= 0, "txId"],
                           df_nodos.loc[df_nodos["y"] >= 0, "y"]))
etiquetas_train = dict(zip(df_nodos.loc[mask_train, "txId"], df_nodos.loc[mask_train, "y"]))

titulo("Features de vecindad en dos versiones")
print(f"  Etiquetas totales disponibles       : {len(etiquetas_todas):,}")
print(f"  Etiquetas visibles en entrenamiento : {len(etiquetas_train):,} (t ≤ {CORTE_TEMPORAL})")
print(f"  Nodos de entrenamiento              : {int(mask_train.sum()):,}")
print(f"  Nodos de test                       : {int(mask_test.sum()):,}\n")

vec_fuga, _ = cronometrar(features_de_vecindad, G, etiquetas_todas,
                          etiqueta="versión con fuga (todas las etiquetas)", prefijo="fuga_")
vec_honesta, _ = cronometrar(features_de_vecindad, G, etiquetas_train,
                             etiqueta="versión honesta (solo train)", prefijo="")

dist_fuga = distancia_a_fraude(G, [n for n, y in etiquetas_todas.items() if y == 1])
dist_honesta = distancia_a_fraude(G, [n for n, y in etiquetas_train.items() if y == 1])

features_grafo = features_grafo.merge(vec_honesta, on="txId", how="left")
features_grafo = features_grafo.merge(vec_fuga, on="txId", how="left")
features_grafo["dist_a_fraude"] = features_grafo["txId"].map(dist_honesta).fillna(7)
features_grafo["fuga_dist_a_fraude"] = features_grafo["txId"].map(dist_fuga).fillna(7)

In [ ]:
# =============================================================================
# M11.3 — Demostración numérica de la fuga
# =============================================================================

titulo("¿Qué ven realmente estas features en el conjunto de test?")

fg = features_grafo.merge(df_nodos[["txId", "y"]], on="txId", how="left")
test = fg[(fg["timestep"] > CORTE_TEMPORAL) & (fg["y"] >= 0)]
train = fg[(fg["timestep"] <= CORTE_TEMPORAL) & (fg["y"] >= 0)]

comparacion = pd.DataFrame({
    "versión honesta (solo etiquetas de train)": [
        train["vecinos_etiquetados"].mean(),
        test["vecinos_etiquetados"].mean(),
        test["ratio_vecinos_fraude"].mean(),
        (test["dist_a_fraude"] < 7).mean() * 100,
    ],
    "versión con fuga (todas las etiquetas)": [
        train["fuga_vecinos_etiquetados"].mean(),
        test["fuga_vecinos_etiquetados"].mean(),
        test["fuga_ratio_vecinos_fraude"].mean(),
        (test["fuga_dist_a_fraude"] < 7).mean() * 100,
    ],
}, index=["vecinos etiquetados por nodo (TRAIN)",
          "vecinos etiquetados por nodo (TEST)",
          "ratio medio de vecinos fraudulentos (TEST)",
          "% de nodos de TEST a <7 saltos de un fraude conocido"]).round(4)
display(comparacion)

print(f"""
La segunda fila es la prueba. Con la versión honesta, un nodo de test tiene
{test['vecinos_etiquetados'].mean():.4f} vecinos etiquetados de media: prácticamente cero. Con la
versión con fuga, tiene {test['fuga_vecinos_etiquetados'].mean():.2f}.

La razón es exactamente la del Módulo 4: como ninguna arista cruza timesteps, TODOS los
vecinos de un nodo de test (t > {CORTE_TEMPORAL}) están también en test. Sus etiquetas son
precisamente lo que el modelo debe predecir.

CONSECUENCIA PRÁCTICA
  · La versión con fuga se queda FUERA del modelo del Módulo 15. Está calculada aquí solo
    para poder medir cuánto infla las métricas, no para usarla.
  · La versión honesta es inútil en este dataset concreto (siempre cero en test), pero es
    la implementación CORRECTA y sí funciona en grafos donde las relaciones cruzan el
    tiempo — que es el caso de cualquier grafo bancario real, donde un cliente de hoy
    transfiere a una cuenta señalada hace seis meses.
  · Si en tu proyecto la feature 'vecinos fraudulentos' aparece como la más importante del
    modelo, comprueba esto ANTES de celebrarlo.
""")

# Poder discriminante de las features seguras, medido con información mutua.
from sklearn.feature_selection import mutual_info_classif

SEGURAS = ["in_degree", "out_degree", "degree", "ratio_in_out", "es_fuente", "es_sumidero",
           "pagerank", "katz", "hits_hub", "hits_authority", "tamano_comunidad",
           "comunidades_vecinas", "ratio_frontera", "clustering",
           "grado_medio_vecinos", "grado_max_vecinos"]
SEGURAS = [c for c in SEGURAS if c in fg.columns]

etiquetados = fg[fg["y"] >= 0]
mi = mutual_info_classif(etiquetados[SEGURAS].fillna(0), etiquetados["y"], random_state=SEED)
imp = pd.Series(mi, index=SEGURAS).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(imp.index[::-1], imp.values[::-1], color=COLORES["licito"])
plt.title("Información mutua con la etiqueta — solo features SIN fuga")
plt.xlabel("Información mutua (bits)")
plt.tight_layout()
plt.show()

print("Estas son las features que sí se pueden usar. Ninguna es espectacular por sí sola —la")
print("información mutua es baja en todas—, y esa es justamente la situación normal: la señal")
print("está en la COMBINACIÓN, que es lo que el modelo del Módulo 15 tiene que encontrar.")

In [ ]:
# =============================================================================
# M11.4 — Unión con las 165 features originales de Elliptic
# =============================================================================

def unir_con_features_originales(df_grafo: pd.DataFrame, df_features: pd.DataFrame,
                                 incluir_agregadas: bool = True) -> pd.DataFrame:
    """Une las features topológicas con las 165 features numéricas del dataset.

    El notebook original anunciaba esta unión en el markdown del Módulo 11 pero nunca
    la ejecutaba: `ml_df` solo contenía grado, PageRank, comunidad y clase, y las 165
    columnas de `df_features` se quedaban sin usar.

    Sobre las agregadas (`agg_1..72`): son estadísticos de las features locales de los
    VECINOS a un salto. Es decir, Elliptic ya trae una capa de feature engineering de
    grafo hecha. Conviene saberlo antes de sorprenderse de que aporten tanto.
    """
    columnas = ["txId"] + [c for c in df_features.columns if c.startswith("local_")]
    if incluir_agregadas:
        columnas += [c for c in df_features.columns if c.startswith("agg_")]
    # `timestep` se excluye a propósito: df_grafo ya lo trae, y duplicarlo produciría
    # las columnas timestep_x / timestep_y que romperían el split temporal de M15.
    return df_grafo.merge(df_features[columnas], on="txId", how="left")


titulo("Dataset final para machine learning")
DATOS_ML = unir_con_features_originales(
    features_grafo.merge(df_nodos[["txId", "y", "clase"]], on="txId", how="left"),
    df_features)

COLS_LOCALES = [c for c in DATOS_ML.columns if c.startswith("local_")]
COLS_AGG = [c for c in DATOS_ML.columns if c.startswith("agg_")]
COLS_GRAFO = [c for c in SEGURAS if c in DATOS_ML.columns]

print(f"  Filas (nodos)                  : {len(DATOS_ML):,}")
print(f"  Features topológicas (seguras) : {len(COLS_GRAFO)}")
print(f"  Features locales de Elliptic   : {len(COLS_LOCALES)}")
print(f"  Features agregadas de Elliptic : {len(COLS_AGG)}")
print(f"  TOTAL de features usables      : {len(COLS_GRAFO) + len(COLS_LOCALES) + len(COLS_AGG)}")
print(f"  Nodos etiquetados              : {(DATOS_ML['y'] >= 0).sum():,}")
print(f"    · entrenamiento (t ≤ {CORTE_TEMPORAL})     : {((DATOS_ML['y'] >= 0) & (DATOS_ML['timestep'] <= CORTE_TEMPORAL)).sum():,}")
print(f"    · test (t > {CORTE_TEMPORAL})              : {((DATOS_ML['y'] >= 0) & (DATOS_ML['timestep'] > CORTE_TEMPORAL)).sum():,}")
display(DATOS_ML[["txId", "timestep", "clase", "degree", "pagerank", "tamano_comunidad",
                  "clustering", "local_1", "agg_1"]].head())

---
---

# Módulo 12 — Link Prediction

Predecir qué aristas **aparecerán** (o cuáles faltan en un grafo incompleto). En fraude tiene tres
usos concretos: anticipar la siguiente cuenta de una red de mulas, completar relaciones que el
dato no registra (dos clientes que operan juntos sin vínculo declarado) y detectar aristas
**anómalas** — una conexión que el modelo consideraba improbable y que sin embargo ocurrió.

## Los cuatro índices clásicos

Sea $\Gamma(u)$ el conjunto de vecinos de $u$. Todos parten de la misma intuición —el cierre
triádico del Módulo 9— y se diferencian en cómo pesan a los vecinos comunes:

| Índice | Fórmula | Idea | Sesgo |
|---|---|---|---|
| **Vecinos comunes** | $\|\Gamma(u) \cap \Gamma(v)\|$ | Cuantos más amigos comunes, más probable | Favorece nodos de grado alto |
| **Jaccard** | $\dfrac{\|\Gamma(u) \cap \Gamma(v)\|}{\|\Gamma(u) \cup \Gamma(v)\|}$ | Normaliza por el tamaño del vecindario | Penaliza a los hubs |
| **Adamic-Adar** | $\displaystyle\sum_{w \in \Gamma(u)\cap\Gamma(v)} \frac{1}{\log \|\Gamma(w)\|}$ | Un vecino común **raro** vale más que uno popular | Equilibrado |
| **Resource Allocation** | $\displaystyle\sum_{w \in \Gamma(u)\cap\Gamma(v)} \frac{1}{\|\Gamma(w)\|}$ | Como AA pero penaliza más fuerte | El mejor en redes dispersas |
| **Preferential Attachment** | $\|\Gamma(u)\| \cdot \|\Gamma(v)\|$ | Los ricos se hacen más ricos | **No usa vecinos comunes** |

Preferential Attachment es el raro del grupo: no mira la vecindad compartida en absoluto. Predice
que dos nodos de grado alto se conectarán simplemente porque son de grado alto. Es la línea base
que hay que batir — si un índice sofisticado no supera a PA, no está aportando nada.

## El problema del muestreo de negativos

Aquí es donde se falsean —normalmente sin querer— casi todas las evaluaciones de link prediction.

El conjunto de test necesita ejemplos positivos (aristas que sí existen) y negativos (pares que
no). Si los negativos se eligen **al azar** entre todos los pares posibles, en un grafo con
densidad $5{,}6\times10^{-6}$ casi todos serán pares de nodos lejanísimos, sin ningún vecino
común y con score exactamente 0. Cualquier método los separa sin esfuerzo y el **AUC sale ~0,99**.

Un AUC de 0,99 en link prediction con negativos aleatorios no significa que el modelo sea bueno:
significa que el problema planteado era trivial. La evaluación honesta usa **negativos difíciles**:
pares que están a distancia 2 (comparten al menos un vecino) pero no están conectados. Ahí es
donde el modelo tiene que decidir de verdad. La celda M12.2 mide las dos y compara.

In [ ]:
# =============================================================================
# M12.1 — Los cinco índices
# =============================================================================

def predecir_enlaces(G_und: nx.Graph, pares: list, metodo: str = "adamic_adar") -> list:
    """Puntúa una lista de pares (u, v) con el índice indicado.

    Todos los índices exigen un grafo NO dirigido y simple: NetworkX lanza excepción
    con DiGraph o MultiGraph. La proyección se hace fuera para no ocultar el coste.
    Devuelve una lista de tuplas (u, v, score).
    """
    if metodo == "common_neighbors":
        return [(u, v, len(list(nx.common_neighbors(G_und, u, v)))) for u, v in pares]

    funciones = {
        "jaccard": nx.jaccard_coefficient,
        "adamic_adar": nx.adamic_adar_index,
        "resource_allocation": nx.resource_allocation_index,
        "preferential_attachment": nx.preferential_attachment,
    }
    if metodo not in funciones:
        raise ValueError(f"Método desconocido: {metodo!r}")
    return list(funciones[metodo](G_und, pares))


METODOS_LP = ["common_neighbors", "jaccard", "adamic_adar",
              "resource_allocation", "preferential_attachment"]

# Ejemplo ilustrativo sobre un grafo diminuto donde se puede comprobar a mano.
demo = nx.Graph([("A", "C"), ("B", "C"), ("A", "D"), ("B", "D"), ("A", "E"),
                 ("C", "F"), ("D", "F"), ("E", "F"), ("F", "G"), ("F", "H")])
titulo("Los cinco índices sobre el par (A, B) — comparten C y D")
print(f"  Γ(A) = {sorted(demo.neighbors('A'))}   Γ(B) = {sorted(demo.neighbors('B'))}")
print(f"  Vecinos comunes: {sorted(nx.common_neighbors(demo, 'A', 'B'))}")
print(f"  Grados: C={demo.degree('C')}, D={demo.degree('D')}, F={demo.degree('F')}\n")
for metodo in METODOS_LP:
    score = predecir_enlaces(demo, [("A", "B")], metodo)[0][2]
    print(f"  {metodo:26s} {score:.4f}")
print("\n  Nota: F es el nodo más conectado (grado 5). Adamic-Adar y Resource Allocation")
print("  descontarían su aportación si fuera vecino común, porque un vecino popular es")
print("  poca evidencia de relación: está conectado con todo el mundo.")

In [ ]:
# =============================================================================
# M12.2 — Evaluación honesta: negativos aleatorios vs. negativos difíciles
# =============================================================================

from sklearn.metrics import roc_auc_score, average_precision_score


def muestrear_negativos(G_und: nx.Graph, n: int, dificultad: str = "dificil",
                        seed: int = SEED) -> list:
    """Genera pares NO conectados para usar como ejemplos negativos.

    · "aleatorio" — dos nodos al azar sin arista entre ellos. En un grafo disperso
                    casi nunca comparten vecinos, así que el problema se vuelve trivial.
    · "dificil"   — pares a distancia exactamente 2: comparten al menos un vecino pero
                    no están conectados. Es el negativo que de verdad discrimina.
    """
    rng = random.Random(seed)
    nodos = [n_ for n_ in G_und.nodes() if G_und.degree(n_) > 0]
    negativos, vistos = [], set()

    if dificultad == "aleatorio":
        intentos = 0
        while len(negativos) < n and intentos < 60 * n:
            intentos += 1
            u, v = rng.sample(nodos, 2)
            par = frozenset((u, v))
            if par not in vistos and not G_und.has_edge(u, v):
                vistos.add(par)
                negativos.append((u, v))
        return negativos

    # Difíciles: se recorren vecindarios buscando pares a distancia 2.
    rng.shuffle(nodos)
    for w in nodos:
        vecinos = list(G_und.neighbors(w))
        if len(vecinos) < 2:
            continue
        for u, v in itertools.combinations(rng.sample(vecinos, min(len(vecinos), 6)), 2):
            par = frozenset((u, v))
            if par not in vistos and not G_und.has_edge(u, v):
                vistos.add(par)
                negativos.append((u, v))
                if len(negativos) >= n:
                    return negativos
    return negativos


def evaluar_link_prediction(G_und: nx.Graph, frac_test: float = 0.10,
                            dificultad: str = "dificil", seed: int = SEED,
                            metodos=None, k_precision: int = 100) -> pd.DataFrame:
    """Oculta una fracción de aristas y mide si los índices las recuperan.

    Protocolo:
      1. Se retira al azar `frac_test` de las aristas → son los POSITIVOS de test.
      2. Se muestrea el mismo número de NEGATIVOS con la dificultad indicada.
      3. Cada índice puntúa ambos conjuntos sobre el grafo SIN las aristas ocultas.
      4. Se reportan ROC-AUC, average precision y precision@K.

    Precision@K es la métrica que importa en producción: de las K aristas que el
    sistema señala como más probables, ¿cuántas eran reales? Un analista revisa 100
    alertas, no 200.000 pares.
    """
    metodos = metodos or METODOS_LP
    rng = random.Random(seed)

    aristas = list(G_und.edges())
    n_test = max(int(len(aristas) * frac_test), 1)
    positivos = rng.sample(aristas, n_test)

    G_train = G_und.copy()
    G_train.remove_edges_from(positivos)
    negativos = muestrear_negativos(G_train, n_test, dificultad=dificultad, seed=seed)

    pares = positivos + negativos
    y = np.array([1] * len(positivos) + [0] * len(negativos))

    filas = []
    for metodo in metodos:
        scores = np.array([s for _, _, s in predecir_enlaces(G_train, pares, metodo)], dtype=float)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0)
        orden = np.argsort(-scores)[:k_precision]
        filas.append({
            "método": metodo,
            "ROC-AUC": round(roc_auc_score(y, scores), 4),
            "Average Precision": round(average_precision_score(y, scores), 4),
            f"Precision@{k_precision}": round(float(y[orden].mean()), 4),
            "% con score 0": round(100 * float(np.mean(scores == 0)), 1),
        })
    return pd.DataFrame(filas).set_index("método")


# El grafo de un timestep: dentro de él sí hay estructura local que explotar.
lp_grafo = nx.Graph(G_t.to_undirected())
titulo(f"Link prediction sobre el timestep {TIMESTEP_FOCO} "
       f"({lp_grafo.number_of_nodes():,} nodos, {lp_grafo.number_of_edges():,} aristas)")

for dificultad, etiqueta in [("aleatorio", "NEGATIVOS ALEATORIOS (evaluación optimista)"),
                             ("dificil", "NEGATIVOS DIFÍCILES (evaluación honesta)")]:
    print(f"\n▸ {etiqueta}")
    resultado = evaluar_link_prediction(lp_grafo, frac_test=0.10, dificultad=dificultad)
    display(resultado)

print("""
La diferencia entre las dos tablas es el mensaje del módulo. Con negativos aleatorios todos
los índices rozan el AUC perfecto; con negativos difíciles el rendimiento cae, y solo
entonces se distinguen entre sí.

Fíjate también en '% con score 0': en un DAG disperso, la mayoría de pares NO comparte
ningún vecino común, así que los índices basados en vecindad devuelven cero y no pueden
ordenar. Preferential Attachment nunca da cero —siempre hay grados que multiplicar— y por
eso a veces gana en AUC sin ser mejor: simplemente rompe empates donde los demás no pueden.""")

In [ ]:
# =============================================================================
# M12.3 — Evaluación temporal sobre el grafo bancario
# =============================================================================

def evaluar_lp_temporal(G_temporal: nx.MultiDiGraph, atributo_ts: str = "ts",
                        percentil_corte: int = 70, dificultad: str = "dificil",
                        seed: int = SEED, k_precision: int = 50) -> pd.DataFrame:
    """Link prediction con un split TEMPORAL real, que es como se evalúa en producción.

    Se entrena con las aristas anteriores al corte y se predicen las que aparecen
    después. A diferencia del protocolo de ocultar aristas al azar, aquí no hay
    ninguna posibilidad de fuga: la información futura simplemente no está en el grafo
    de entrenamiento.
    """
    rng = random.Random(seed)
    aristas = [(u, v, d[atributo_ts]) for u, v, d in G_temporal.edges(data=True)
               if atributo_ts in d and u != v]
    corte = float(np.percentile([t for _, _, t in aristas], percentil_corte))

    G_pasado = nx.Graph()
    G_pasado.add_nodes_from(G_temporal.nodes())
    G_pasado.add_edges_from((u, v) for u, v, t in aristas if t <= corte)

    futuras = {frozenset((u, v)) for u, v, t in aristas
               if t > corte and not G_pasado.has_edge(u, v)}
    positivos = [tuple(p) for p in futuras if len(p) == 2]
    if not positivos:
        return pd.DataFrame()

    negativos = muestrear_negativos(G_pasado, len(positivos), dificultad=dificultad, seed=seed)
    pares = positivos + negativos
    y = np.array([1] * len(positivos) + [0] * len(negativos))

    filas = []
    for metodo in METODOS_LP:
        scores = np.array([s for _, _, s in predecir_enlaces(G_pasado, pares, metodo)], dtype=float)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0)
        orden = np.argsort(-scores)[:k_precision]
        filas.append({
            "método": metodo,
            "ROC-AUC": round(roc_auc_score(y, scores), 4) if len(set(y)) > 1 else np.nan,
            f"Precision@{k_precision}": round(float(y[orden].mean()), 4),
        })
    resumen = pd.DataFrame(filas).set_index("método")
    resumen.attrs["positivos"] = len(positivos)
    resumen.attrs["negativos"] = len(negativos)
    return resumen


titulo("Link prediction temporal sobre el grafo bancario")
lp_temporal = evaluar_lp_temporal(G_trf, percentil_corte=70)
if len(lp_temporal):
    print(f"Aristas futuras a predecir: {lp_temporal.attrs['positivos']:,} · "
          f"negativos: {lp_temporal.attrs['negativos']:,}\n")
    display(lp_temporal)
    print("""
El rendimiento aquí es más modesto, y es el número creíble. En un grafo de transferencias
las conexiones futuras no dependen solo de la topología: dependen de decisiones económicas
que el grafo no observa. Los índices de vecindad capturan una parte —la tendencia a operar
dentro del propio círculo— y nada más.

Esto marca el límite de los métodos topológicos puros y motiva los dos módulos siguientes:
los EMBEDDINGS (M13) aprenden una representación en lugar de contar vecinos, y las GNN (M16)
combinan topología con atributos de nodo.""")
else:
    print("No hay aristas nuevas suficientes tras el corte para evaluar.")

---
---

# Módulo 13 — Embeddings

Representar cada nodo como un vector denso $\mathbf{z}_v \in \mathbb{R}^d$ de forma que **la
proximidad en el grafo se traduzca en proximidad en el espacio vectorial**. Es el puente entre
el grafo y cualquier algoritmo que espere una matriz de números: clasificadores, clustering,
detección de anomalías.

## Las dos grandes familias

| Familia | Método | Idea | Captura |
|---|---|---|---|
| **Factorización matricial** | SVD, HOPE, LINE (2º orden) | Descomponer una matriz de proximidad | Estructura **global** |
| **Basados en recorridos** | DeepWalk, Node2Vec | Word2Vec sobre caminatas aleatorias | Estructura **local** y de rol |

### SVD de la adyacencia

La más simple: $A \approx U \Sigma V^{\!\top}$ y se toma $Z = U_k \Sigma_k$. Directa, determinista
y sorprendentemente competitiva. Su límite es que solo ve conexiones de primer orden: dos nodos
sin ningún vecino común quedan lejos aunque estén en la misma región de la red.

### HOPE

Factoriza una matriz de **proximidad de alto orden** en vez de la adyacencia. Con la proximidad
de Katz $S = \beta A + \beta^2 A^2 + \dots$, dos nodos son cercanos si hay muchos caminos entre
ellos, no solo aristas directas. Preserva además la **asimetría**, lo que importa en un grafo
dirigido: que A financie a B no es lo mismo que lo contrario.

### DeepWalk y Node2Vec

Se generan caminatas aleatorias y se tratan como frases: cada nodo es una palabra y Word2Vec
aprende su vector. La diferencia entre ambos está en cómo se camina:

- **DeepWalk**: caminata uniforme. Cada vecino es igual de probable.
- **Node2Vec**: caminata sesgada por dos parámetros:
  - $p$ (*return*) — alto desalienta volver al nodo anterior.
  - $q$ (*in-out*) — $q < 1$ favorece **alejarse** (exploración tipo DFS, captura comunidades);
    $q > 1$ favorece **quedarse cerca** (exploración tipo BFS, captura **roles estructurales**).

Esa distinción es la que importa en fraude. Con $q > 1$, dos cuentas mula situadas en redes
distintas y sin ninguna conexión entre sí acaban con vectores parecidos porque **desempeñan el
mismo papel**. Con $q < 1$, cada una se parece a su propio vecindario. Son objetivos opuestos.

### LINE

Optimiza explícitamente dos proximidades: la de **primer orden** (nodos conectados deben tener
vectores similares) y la de **segundo orden** (nodos con vecindarios similares deben tener
vectores similares). Se entrena con muestreo negativo, igual que Word2Vec.

## Nota sobre el notebook original

Las celdas de embeddings del notebook original fallaban con
`AttributeError: 'csr_array' object has no attribute 'asfptype'`. La causa: NetworkX 3.x devuelve
un `csr_array` (API nueva de SciPy) en vez del antiguo `csr_matrix`, y `.asfptype()` solo existía
en la clase vieja. La corrección es pedir el tipo directamente al construir la matriz, que además
evita una copia.

In [ ]:
# =============================================================================
# M13.1 — Embeddings por factorización: SVD y HOPE
# =============================================================================

from scipy.sparse.linalg import svds


def embeddings_svd(G: nx.Graph, dim: int = 32, nodelist=None, escalar: bool = True) -> tuple:
    """Embeddings por SVD truncada de la matriz de adyacencia.

    Se construye la matriz ya en punto flotante con `to_scipy_sparse_array(dtype=float)`:
    es la corrección del `AttributeError: 'csr_array' has no attribute 'asfptype'` del
    notebook original, y ahorra la copia que provocaba `.astype(float)`.

    Devuelve `(Z, nodelist)` con Z de forma (n, dim). El orden de `nodelist` es el
    contrato con el resto del notebook: nunca se debe reordenar por separado.
    """
    nodelist = list(G.nodes()) if nodelist is None else list(nodelist)
    A = nx.to_scipy_sparse_array(G, nodelist=nodelist, dtype=float, format="csr")
    u, s, _ = svds(A, k=min(dim, min(A.shape) - 1))
    orden = np.argsort(-s)
    Z = u[:, orden] * (s[orden] if escalar else 1.0)
    return Z, nodelist


def embeddings_hope(G: nx.Graph, dim: int = 32, beta: float = 0.05,
                    orden_max: int = 3, nodelist=None) -> tuple:
    """HOPE con proximidad de Katz truncada: S ≈ βA + β²A² + … + β^K A^K.

    La formulación exacta usa (I − βA)⁻¹βA, que es una matriz DENSA n×n y por tanto
    imposible a esta escala. Se trunca la serie al orden `orden_max`, lo que mantiene
    la matriz dispersa. En un DAG la aproximación es especialmente buena: la serie es
    finita porque A es nilpotente.
    """
    nodelist = list(G.nodes()) if nodelist is None else list(nodelist)
    A = nx.to_scipy_sparse_array(G, nodelist=nodelist, dtype=float, format="csr")
    S = beta * A
    potencia = A.copy()
    for k in range(2, orden_max + 1):
        potencia = potencia @ A
        S = S + (beta ** k) * potencia
    u, s, vt = svds(S, k=min(dim // 2, min(S.shape) - 1))
    orden = np.argsort(-s)
    fuente = u[:, orden] * np.sqrt(s[orden])      # rol como ORIGEN
    destino = vt[orden].T * np.sqrt(s[orden])     # rol como DESTINO
    return np.hstack([fuente, destino]), nodelist


titulo("Embeddings por factorización sobre el grafo completo")
(Z_svd_global, nodos_svd), _ = cronometrar(
    embeddings_svd, G, etiqueta="SVD sobre el grafo completo", dim=32)
print(f"  Matriz de embeddings: {Z_svd_global.shape} ({Z_svd_global.nbytes/1024**2:.1f} MB)")
print("  Sobre el grafo completo la SVD es viable porque opera sobre la matriz DISPERSA:")
print("  234.355 elementos no nulos, no 203.769² = 41.500 millones.")

# El resto del módulo trabaja sobre un timestep: los métodos por caminatas
# necesitan generar millones de tokens y sobre 200 mil nodos no terminarían en clase.
G_emb = nx.Graph(G_t.to_undirected())
G_emb.remove_edges_from(nx.selfloop_edges(G_emb))
NODOS_EMB = list(G_emb.nodes())
etiquetas_emb = df_nodos.set_index("txId").reindex(NODOS_EMB)["y"].fillna(-1).astype(int).values
print(f"\nGrafo de trabajo del módulo: timestep {TIMESTEP_FOCO} — "
      f"{G_emb.number_of_nodes():,} nodos, {G_emb.number_of_edges():,} aristas")
print(f"  Etiquetados: {(etiquetas_emb >= 0).sum():,} "
      f"(ilícitos: {(etiquetas_emb == 1).sum():,})")

In [ ]:
# =============================================================================
# M13.2 — Caminatas aleatorias: DeepWalk y Node2Vec
# =============================================================================

def random_walks(G: nx.Graph, num_walks: int = 10, walk_length: int = 40,
                 p: float = 1.0, q: float = 1.0, seed: int = SEED) -> list:
    """Genera caminatas aleatorias sesgadas al estilo Node2Vec.

    Con p = q = 1 el sesgo desaparece y son las caminatas uniformes de DeepWalk.
    En otro caso, la probabilidad de ir de `actual` a `siguiente` viniendo de
    `anterior` se pondera con:

        1/p   si siguiente == anterior           (volver atrás)
        1     si siguiente es vecino de anterior (quedarse en el vecindario)
        1/q   en cualquier otro caso             (alejarse)

    Se implementa sin precalcular las tablas de alias: con grados medios bajos el
    muestreo directo es más simple y no domina el tiempo total (lo domina Word2Vec).
    """
    rng = random.Random(seed)
    nodos = list(G.nodes())
    vecinos = {n: list(G.neighbors(n)) for n in nodos}
    vecinos_set = {n: set(v) for n, v in vecinos.items()}
    caminatas = []

    for _ in range(num_walks):
        rng.shuffle(nodos)
        for inicio in nodos:
            caminata = [inicio]
            while len(caminata) < walk_length:
                actual = caminata[-1]
                candidatos = vecinos[actual]
                if not candidatos:
                    break
                if len(caminata) == 1 or (p == 1.0 and q == 1.0):
                    caminata.append(rng.choice(candidatos))
                    continue
                anterior = caminata[-2]
                pesos = [
                    1.0 / p if c == anterior else (1.0 if c in vecinos_set[anterior] else 1.0 / q)
                    for c in candidatos
                ]
                caminata.append(rng.choices(candidatos, weights=pesos, k=1)[0])
            if len(caminata) > 1:
                caminatas.append([str(n) for n in caminata])
    return caminatas


def entrenar_word2vec(caminatas: list, nodelist: list, dim: int = 32,
                      ventana: int = 5, epocas: int = 5, seed: int = SEED) -> np.ndarray:
    """Aprende un vector por nodo a partir de las caminatas.

    Con gensim usa skip-gram con muestreo negativo. Sin gensim, cae a una
    factorización PPMI + SVD, que NO es un apaño: Levy & Goldberg (2014) demostraron
    que skip-gram con muestreo negativo factoriza implícitamente la matriz PPMI de
    co-ocurrencias. El resultado es comparable y no necesita ninguna dependencia extra.
    """
    if GENSIM_OK:
        from gensim.models import Word2Vec
        modelo = Word2Vec(caminatas, vector_size=dim, window=ventana, min_count=0,
                          sg=1, negative=5, workers=4, epochs=epocas, seed=seed)
        return np.array([modelo.wv[str(n)] if str(n) in modelo.wv else np.zeros(dim)
                         for n in nodelist])

    print("  gensim no disponible → factorización PPMI + SVD (equivalente implícito)")
    indice = {str(n): i for i, n in enumerate(nodelist)}
    n = len(nodelist)
    filas, columnas = [], []
    for caminata in caminatas:
        for i, centro in enumerate(caminata):
            for j in range(max(0, i - ventana), min(len(caminata), i + ventana + 1)):
                if i != j and centro in indice and caminata[j] in indice:
                    filas.append(indice[centro])
                    columnas.append(indice[caminata[j]])
    C = sp.coo_matrix((np.ones(len(filas)), (filas, columnas)), shape=(n, n)).tocsr()

    total = C.sum()
    marginal_f = np.asarray(C.sum(axis=1)).ravel() + 1e-12
    marginal_c = np.asarray(C.sum(axis=0)).ravel() + 1e-12
    C = C.tocoo()
    pmi = np.log((C.data * total) / (marginal_f[C.row] * marginal_c[C.col]))
    positivos = pmi > 0
    PPMI = sp.coo_matrix((pmi[positivos], (C.row[positivos], C.col[positivos])),
                         shape=(n, n)).tocsr()
    u, s, _ = svds(PPMI, k=min(dim, n - 1))
    orden = np.argsort(-s)
    return u[:, orden] * np.sqrt(s[orden])


titulo("Generación de caminatas")
config = {
    "DeepWalk (p=1, q=1)": dict(p=1.0, q=1.0),
    "Node2Vec DFS (q=0.5) — comunidades": dict(p=1.0, q=0.5),
    "Node2Vec BFS (q=2.0) — roles": dict(p=1.0, q=2.0),
}
EMBEDDINGS = {}
for nombre, kwargs in config.items():
    t0 = time.perf_counter()
    caminatas = random_walks(G_emb, num_walks=10, walk_length=40, seed=SEED, **kwargs)
    Z = entrenar_word2vec(caminatas, NODOS_EMB, dim=32)
    EMBEDDINGS[nombre] = Z
    print(f"  {nombre:38s} {len(caminatas):>7,} caminatas · {Z.shape} · "
          f"{time.perf_counter()-t0:5.1f} s")

In [ ]:
# =============================================================================
# M13.3 — LINE de primer orden, implementado con muestreo negativo
# =============================================================================

def embeddings_line(G: nx.Graph, dim: int = 32, epocas: int = 30, lr: float = 0.05,
                    n_negativos: int = 5, lote: int = 2048, seed: int = SEED) -> np.ndarray:
    """LINE de primer orden: nodos conectados deben tener vectores similares.

    Maximiza  Σ_(u,v)∈E [ log σ(zᵤ·z_v) + Σ_neg log σ(−zᵤ·z_n) ]  por descenso de
    gradiente estocástico con muestreo negativo. Los negativos se sortean con la
    distribución de ruido de Word2Vec, proporcional a grado^0.75: sin ese exponente
    los hubs acaparan todos los negativos y los nodos raros nunca se separan.

    Todo vectorizado en NumPy. `np.add.at` es imprescindible aquí: en un lote el mismo
    nodo aparece varias veces y una asignación normal sobrescribiría los gradientes
    en vez de acumularlos.
    """
    rng = np.random.default_rng(seed)
    nodos = list(G.nodes())
    indice = {n: i for i, n in enumerate(nodos)}
    n = len(nodos)

    E = np.array([[indice[u], indice[v]] for u, v in G.edges()], dtype=np.int64)
    if len(E) == 0:
        return np.zeros((n, dim))

    grados = np.array([max(G.degree(x), 1) for x in nodos], dtype=float)
    p_ruido = grados ** 0.75
    p_ruido /= p_ruido.sum()

    Z = rng.normal(0, 0.1, size=(n, dim))
    sigmoide = lambda x: 1.0 / (1.0 + np.exp(-np.clip(x, -12, 12)))

    for epoca in range(epocas):
        orden = rng.permutation(len(E))
        for inicio in range(0, len(E), lote):
            b = E[orden[inicio:inicio + lote]]
            u, v = b[:, 0], b[:, 1]
            neg = rng.choice(n, size=(len(b), n_negativos), p=p_ruido)

            # Par positivo
            g_pos = (1.0 - sigmoide(np.sum(Z[u] * Z[v], axis=1)))[:, None]
            grad_u = g_pos * Z[v]
            grad_v = g_pos * Z[u]

            # Pares negativos
            puntuacion_neg = np.einsum("ij,ikj->ik", Z[u], Z[neg])
            g_neg = -sigmoide(puntuacion_neg)
            grad_u += np.einsum("ik,ikj->ij", g_neg, Z[neg])
            grad_neg = g_neg[:, :, None] * Z[u][:, None, :]

            np.add.at(Z, u, lr * grad_u)
            np.add.at(Z, v, lr * grad_v)
            np.add.at(Z, neg.ravel(), lr * grad_neg.reshape(-1, dim))

    normas = np.linalg.norm(Z, axis=1, keepdims=True)
    return Z / np.where(normas == 0, 1.0, normas)


titulo("Todos los métodos de embedding sobre el mismo grafo")
t0 = time.perf_counter()
EMBEDDINGS["SVD (adyacencia)"] = embeddings_svd(G_emb, dim=32, nodelist=NODOS_EMB)[0]
print(f"  SVD              {time.perf_counter()-t0:5.1f} s")
t0 = time.perf_counter()
# HOPE se calcula sobre el grafo DIRIGIDO: su gracia es preservar la asimetría
# (vector como origen ≠ vector como destino), que se perdería en la proyección.
EMBEDDINGS["HOPE (Katz orden 3)"] = embeddings_hope(G_t, dim=32, nodelist=NODOS_EMB)[0]
print(f"  HOPE             {time.perf_counter()-t0:5.1f} s")
t0 = time.perf_counter()
EMBEDDINGS["LINE (1er orden)"] = embeddings_line(G_emb, dim=32, epocas=30)
print(f"  LINE             {time.perf_counter()-t0:5.1f} s")

for nombre, Z in EMBEDDINGS.items():
    print(f"  {nombre:38s} {Z.shape}")

In [ ]:
# =============================================================================
# M13.4 — ¿Sirven estos vectores? Evaluación cuantitativa
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler


def evaluar_embeddings(embeddings: dict, etiquetas: np.ndarray, cv: int = 4,
                       seed: int = SEED) -> pd.DataFrame:
    """Mide la calidad de cada embedding con una tarea posterior común.

    Se entrena una regresión logística sobre los vectores para predecir ilícito/lícito
    y se reporta el ROC-AUC por validación cruzada. Es la evaluación estándar de
    embeddings no supervisados: el vector no ha visto ninguna etiqueta, así que lo que
    se mide es cuánta información sobre el fraude captura la ESTRUCTURA por sí sola.
    """
    mask = etiquetas >= 0
    y = etiquetas[mask]
    if len(np.unique(y)) < 2:
        return pd.DataFrame()

    filas = []
    for nombre, Z in embeddings.items():
        X = StandardScaler().fit_transform(Z[mask])
        modelo = LogisticRegression(max_iter=1500, class_weight="balanced", random_state=seed)
        auc = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")
        filas.append({
            "embedding": nombre,
            "dimensión": Z.shape[1],
            "ROC-AUC medio": round(float(auc.mean()), 4),
            "desviación": round(float(auc.std()), 4),
        })
    return pd.DataFrame(filas).sort_values("ROC-AUC medio", ascending=False).set_index("embedding")


titulo("Poder predictivo de cada embedding (solo estructura, sin atributos)")
evaluacion_emb = evaluar_embeddings(EMBEDDINGS, etiquetas_emb)
display(evaluacion_emb)
print("""
Un AUC claramente por encima de 0,5 significa que la POSICIÓN en el grafo, por sí sola y
sin mirar ningún atributo de la transacción, ya contiene información sobre si es ilícita.
Ese es el argumento central de todo el notebook, ahora medido en lugar de afirmado.

Comparación útil: el Módulo 15 entrenará con las 165 features originales de Elliptic. Si los
embeddings estructurales aportan un AUC comparable partiendo de CERO información de atributos,
la combinación de ambos debería superar a cualquiera por separado.""")

In [ ]:
# =============================================================================
# M13.5 — Visualización: PCA, t-SNE y una nota sobre UMAP
# =============================================================================

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


def visualizar_embeddings(embeddings: dict, etiquetas: np.ndarray, metodo: str = "tsne",
                          max_puntos: int = 4000, seed: int = SEED) -> None:
    """Proyecta los embeddings a 2D y los colorea por clase.

    · PCA    — lineal, determinista, rápido. Conserva la varianza GLOBAL. Si dos
               grupos aparecen separados en PCA, la separación es real y grande.
    · t-SNE  — no lineal. Conserva la vecindad LOCAL a costa de deformar las
               distancias globales: en un gráfico t-SNE, el TAMAÑO de los grupos y la
               DISTANCIA entre ellos no significan nada. Solo la pertenencia.
    · UMAP   — similar a t-SNE, mejor preservación global y mucho más rápido, pero
               requiere `umap-learn`. Se documenta como alternativa, no se instala.
    """
    mask = etiquetas >= 0
    indices = np.where(mask)[0]
    if len(indices) > max_puntos:
        indices = np.random.default_rng(seed).choice(indices, max_puntos, replace=False)
    y = etiquetas[indices]

    n = len(embeddings)
    fig, ejes = plt.subplots(1, n, figsize=(4.4 * n, 4.3))
    ejes = np.atleast_1d(ejes)

    for eje, (nombre, Z) in zip(ejes, embeddings.items()):
        X = StandardScaler().fit_transform(Z[indices])
        if metodo == "pca":
            proy = PCA(n_components=2, random_state=seed).fit_transform(X)
        else:
            perplejidad = min(30, max(5, len(X) // 100))
            proy = TSNE(n_components=2, random_state=seed, init="pca",
                        perplexity=perplejidad, learning_rate="auto").fit_transform(X)

        for valor, color, etiqueta in [(0, COLORES["licito"], "Lícito"),
                                       (1, COLORES["ilicito"], "Ilícito")]:
            sel = y == valor
            eje.scatter(proy[sel, 0], proy[sel, 1], s=7, alpha=0.55, c=color,
                        label=f"{etiqueta} ({sel.sum()})")
        eje.set_title(f"{nombre}\n({metodo.upper()})", fontsize=9.5)
        eje.set_xticks([])
        eje.set_yticks([])
        eje.legend(fontsize=7.5, markerscale=1.8)

    plt.tight_layout()
    plt.show()


subconjunto = {k: EMBEDDINGS[k] for k in list(EMBEDDINGS)[:4]}
visualizar_embeddings(subconjunto, etiquetas_emb, metodo="pca")
visualizar_embeddings(subconjunto, etiquetas_emb, metodo="tsne")

print("""
CÓMO NO ENGAÑARSE CON ESTOS GRÁFICOS

· En t-SNE, la distancia entre dos grupos NO es interpretable. Dos nubes muy separadas
  pueden estar pegadas en el espacio original. Lo único fiable es qué puntos caen juntos.
· La perplejidad cambia radicalmente el dibujo. Un t-SNE con una sola configuración no es
  evidencia de nada; hay que probar varias antes de concluir.
· Que los ilícitos NO formen una nube limpia y separada es lo normal y no invalida el
  embedding: la evaluación cuantitativa de M13.4 ya mostró que hay señal. Una proyección a
  2 dimensiones desde 32 pierde casi toda la información — el clasificador trabaja en 32D.
· UMAP (`pip install umap-learn`) conserva mejor la estructura global y escala mucho mejor:
      import umap; proy = umap.UMAP(n_neighbors=15, min_dist=0.1).fit_transform(X)
""")

---
---

# Módulo 14 — Detección de anomalías

Muy útil para fraude, y por una razón práctica: **el fraude nuevo no tiene etiquetas**. Un
clasificador supervisado solo detecta lo que ya se ha visto; la detección de anomalías busca lo
que no encaja, sin necesidad de saber de antemano qué se busca.

## Las cuatro familias

| Familia | Método | Qué considera anómalo | Necesita etiquetas |
|---|---|---|---|
| **Estructural** | OddBall | Egonets que rompen las leyes de potencias de la red | No |
| **Basada en distancia** | Isolation Forest sobre embeddings | Puntos aislados en el espacio vectorial | No |
| **Por reconstrucción** | Autoencoder, DOMINANT, ONE | Nodos que el modelo no consigue reconstruir | No |
| **Supervisada** | Módulo 15 | Lo que se parece a fraude conocido | Sí |

## OddBall

Akoglu, McGlohon & Faloutsos (2010). La idea es elegante: en redes reales, las propiedades del
**egonet** de un nodo (el nodo, sus vecinos y las aristas entre ellos) siguen leyes de potencias.
Concretamente, entre el número de nodos $N_i$ y el de aristas $E_i$ del egonet:

$$E_i \propto N_i^{\theta}, \qquad \theta \in [1, 2]$$

$\theta = 1$ significa que todos los egonets son estrellas; $\theta = 2$, que son cliques. Las
redes reales caen en medio. **La anomalía es la desviación de esa recta**:

- **Por encima** (más aristas de las esperadas) → *near-clique*: un grupo anormalmente cerrado.
  En fraude: un anillo cuyos miembros solo interactúan entre sí.
- **Por debajo** (menos aristas) → *near-star*: un nodo cuyos vecinos no se conocen entre sí.
  En fraude: un dispersor, un mezclador, una cuenta mula.

El score combina la desviación relativa y la absoluta:

$$\text{score}(i) = \frac{\max(E_i, \hat{E_i})}{\min(E_i, \hat{E_i})} \cdot \log(|E_i - \hat{E_i}| + 1)$$

Lo notable de OddBall es que **no tiene hiperparámetros** y no necesita entrenamiento: la propia
red define qué es normal.

## DOMINANT y ONE

- **DOMINANT** (Ding et al., 2019): un autoencoder de grafo. Un codificador GCN produce $Z$, y
  dos decodificadores reconstruyen la **estructura** ($\hat{A} = \sigma(ZZ^{\!\top})$) y los
  **atributos** ($\hat{X} = \text{GCN}(Z)$). El score es la combinación de ambos errores:
  $s_i = \alpha\|A_i - \hat{A_i}\| + (1-\alpha)\|X_i - \hat{X_i}\|$. Detecta tanto nodos con
  conexiones raras como nodos cuyos atributos no cuadran con los de su vecindario.
- **ONE** (Bandyopadhyay et al., 2019): factoriza conjuntamente estructura y atributos con pesos
  por nodo que se aprenden, de forma que los nodos anómalos —los que estropean la reconstrucción—
  reciben peso bajo y dejan de contaminar el embedding. Es más robusto que DOMINANT ante una
  proporción alta de anomalías.

## La advertencia importante

**Anomalía no es fraude.** Un nodo raro puede ser un exchange legítimo, un error de datos o un
cliente atípico. La detección de anomalías genera *candidatos a investigar*, no veredictos. La
métrica que importa es **precision@K** —de las K alertas que genera el sistema, cuántas valen la
pena— porque un analista revisa 50 casos al día, no 200.000.

In [ ]:
# =============================================================================
# M14.1 — OddBall: anomalías estructurales sin entrenamiento
# =============================================================================

def oddball(G: nx.Graph, muestra: int | None = 20_000, seed: int = SEED) -> pd.DataFrame:
    """Detección de anomalías estructurales por desviación de la ley de potencias del egonet.

    Para cada nodo se calculan las dos propiedades de su egonet (Nᵢ nodos, Eᵢ aristas),
    se ajusta log Eᵢ = θ·log Nᵢ + b por mínimos cuadrados y se puntúa la distancia a
    esa recta. Sin hiperparámetros: la red define su propia normalidad.

    El muestreo es por coste: extraer el egonet de cada uno de 200 mil nodos es caro,
    y el ajuste de la recta converge con unas pocas decenas de miles.
    """
    Gu = nx.Graph(G.to_undirected() if G.is_directed() else G)
    Gu.remove_edges_from(nx.selfloop_edges(Gu))
    nodos = list(Gu.nodes())
    if muestra and muestra < len(nodos):
        nodos = random.Random(seed).sample(nodos, muestra)

    registros = []
    for n in nodos:
        ego = nx.ego_graph(Gu, n, radius=1)
        Ni, Ei = ego.number_of_nodes(), ego.number_of_edges()
        if Ni >= 2 and Ei >= 1:
            registros.append((n, Ni, Ei))

    df = pd.DataFrame(registros, columns=["txId", "N_ego", "E_ego"])
    if df.empty:
        return df

    log_n = np.log10(df["N_ego"].values)
    log_e = np.log10(df["E_ego"].values)
    theta, b = np.polyfit(log_n, log_e, 1)
    esperado = 10 ** (theta * log_n + b)

    maximo = np.maximum(df["E_ego"].values, esperado)
    minimo = np.minimum(df["E_ego"].values, esperado)
    df["E_esperado"] = esperado.round(2)
    df["score_oddball"] = (maximo / minimo) * np.log(np.abs(df["E_ego"].values - esperado) + 1)
    df["tipo"] = np.where(df["E_ego"].values > esperado, "near-clique", "near-star")
    df.attrs["theta"] = theta
    df.attrs["b"] = b
    return df.sort_values("score_oddball", ascending=False).reset_index(drop=True)


titulo("OddBall sobre Elliptic")
odd, _ = cronometrar(oddball, G, etiqueta="oddball (muestra de 20.000 egonets)", muestra=20_000)
# `attrs` no sobrevive de forma fiable a un merge: se capturan antes de enriquecer la tabla.
THETA_ODD, B_ODD = odd.attrs["theta"], odd.attrs["b"]
odd = odd.merge(df_nodos[["txId", "clase", "timestep"]], on="txId", how="left")

print(f"\n  Exponente ajustado θ = {THETA_ODD:.3f}")
print("  θ ≈ 1 → egonets con forma de estrella | θ ≈ 2 → egonets con forma de clique")
print(f"  Egonets evaluados: {len(odd):,}")
print(f"  Reparto: {dict(odd['tipo'].value_counts())}\n")
display(odd.head(10)[["txId", "N_ego", "E_ego", "E_esperado", "score_oddball", "tipo", "clase"]])

# Gráfico canónico de OddBall: la nube de egonets y la recta de normalidad.
plt.figure(figsize=(8.5, 5.5))
normales = odd.iloc[200:]
anomalos = odd.iloc[:200]
plt.scatter(normales["N_ego"], normales["E_ego"], s=6, alpha=0.28,
            color=COLORES["neutro"], label="Egonets")
plt.scatter(anomalos["N_ego"], anomalos["E_ego"], s=26, alpha=0.85,
            color=COLORES["ilicito"], label="Top-200 anómalos")
rango = np.logspace(np.log10(odd["N_ego"].min()), np.log10(odd["N_ego"].max()), 60)
plt.plot(rango, 10 ** (THETA_ODD * np.log10(rango) + B_ODD),
         "--", color=COLORES["acento"], lw=2, label=f"Ajuste: E ∝ N^{THETA_ODD:.2f}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Nodos del egonet (N)")
plt.ylabel("Aristas del egonet (E)")
plt.title("OddBall — la anomalía es la distancia a la recta, no la posición")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# M14.2 — Isolation Forest sobre embeddings y features
# =============================================================================

from sklearn.ensemble import IsolationForest


def isolation_forest_embeddings(Z: np.ndarray, contaminacion: float = 0.05,
                                seed: int = SEED) -> np.ndarray:
    """Puntuación de anomalía por Isolation Forest sobre una matriz de vectores.

    Isolation Forest aísla puntos con particiones aleatorias: un punto anómalo se
    aísla en pocos cortes porque está en una región poco poblada. No asume ninguna
    distribución y escala linealmente, lo que lo hace la opción por defecto para
    puntuar embeddings.

    Devuelve un score donde MAYOR = más anómalo (se invierte el signo de
    `score_samples`, que en sklearn devuelve valores más negativos para lo anómalo).
    """
    modelo = IsolationForest(n_estimators=250, contamination=contaminacion,
                             random_state=seed, n_jobs=-1)
    modelo.fit(Z)
    return -modelo.score_samples(Z)


titulo("Isolation Forest sobre los embeddings del Módulo 13")
scores_if = {}
for nombre, Z in EMBEDDINGS.items():
    scores_if[nombre] = isolation_forest_embeddings(StandardScaler().fit_transform(Z))
    print(f"  {nombre:38s} score medio {scores_if[nombre].mean():.4f}")

In [ ]:
# =============================================================================
# M14.3 — Reconstrucción con redes neuronales: autoencoder y DOMINANT
# =============================================================================

def normalizar_adyacencia(A: sp.spmatrix, anadir_self_loops: bool = True) -> sp.coo_matrix:
    """Normalización simétrica de Kipf & Welling: Â = D^(-1/2) (A + I) D^(-1/2).

    Los self-loops hacen que cada nodo conserve su propia señal al agregar; sin ellos,
    una capa GCN sustituye la representación de un nodo por la media de sus vecinos y
    la información propia se pierde. La normalización simétrica evita que los nodos de
    grado alto dominen la agregación.

    El Módulo 16 reutiliza esta función para todos los modelos de GNN.
    """
    A = sp.csr_matrix(A)
    if anadir_self_loops:
        A = A + sp.eye(A.shape[0], format="csr")
    grados = np.asarray(A.sum(axis=1)).ravel()
    inv_sqrt = np.zeros_like(grados, dtype=float)
    np.divide(1.0, np.sqrt(grados), out=inv_sqrt, where=grados > 0)
    D = sp.diags(inv_sqrt)
    return (D @ A @ D).tocoo()


def sparse_a_torch(M):
    """Convierte una matriz dispersa de SciPy en un tensor disperso de PyTorch."""
    import torch
    M = M.tocoo()
    indices = torch.from_numpy(np.vstack([M.row, M.col]).astype(np.int64))
    valores = torch.from_numpy(M.data.astype(np.float32))
    return torch.sparse_coo_tensor(indices, valores, M.shape).coalesce()


def autoencoder_anomalias(Z: np.ndarray, dim_oculta: int = 16, epocas: int = 120,
                          lr: float = 1e-2, seed: int = SEED) -> np.ndarray:
    """Autoencoder denso: el error de reconstrucción es el score de anomalía.

    La hipótesis es que el autoencoder aprende a reconstruir bien lo frecuente y mal
    lo raro, porque el cuello de botella le obliga a quedarse con los patrones
    dominantes. Los nodos con residuo alto son los que no encajan en ningún patrón.
    """
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    X = torch.tensor(StandardScaler().fit_transform(Z), dtype=torch.float32)
    modelo = nn.Sequential(
        nn.Linear(X.shape[1], 32), nn.ReLU(),
        nn.Linear(32, dim_oculta), nn.ReLU(),
        nn.Linear(dim_oculta, 32), nn.ReLU(),
        nn.Linear(32, X.shape[1]),
    )
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    for _ in range(epocas):
        opt.zero_grad()
        perdida = nn.functional.mse_loss(modelo(X), X)
        perdida.backward()
        opt.step()
    with torch.no_grad():
        return ((modelo(X) - X) ** 2).mean(dim=1).numpy()


def dominant_simplificado(G: nx.Graph, X: np.ndarray, nodelist: list, dim_oculta: int = 32,
                          epocas: int = 100, alpha: float = 0.6, lr: float = 5e-3,
                          seed: int = SEED) -> np.ndarray:
    """Versión compacta de DOMINANT: codificador GCN + doble decodificador.

    Arquitectura:
        Z  = GCN(X, Â)                        codificador de dos capas
        Â' = σ(Z Zᵀ)                          decodificador de ESTRUCTURA
        X' = GCN(Z, Â)                        decodificador de ATRIBUTOS
        sᵢ = α·‖Aᵢ − Â'ᵢ‖ + (1−α)·‖Xᵢ − X'ᵢ‖  score de anomalía

    Un nodo puntúa alto si sus conexiones son raras, si sus atributos no encajan con
    los de su vecindario, o ambas cosas.

    ⏱️ COSTE: el decodificador de estructura reconstruye la matriz de adyacencia COMPLETA
    en denso (n² entradas por época). Con ~4.000 nodos son unos 16 millones de entradas
    y el entrenamiento tarda alrededor de un minuto en CPU; con 20.000 nodos ya no cabe
    en memoria. Las implementaciones de producción muestrean pares en vez de reconstruir
    A entera, a costa de un gradiente más ruidoso.
    """
    if len(nodelist) > 8000:
        raise MemoryError(
            f"{len(nodelist):,} nodos harían una matriz densa de "
            f"{len(nodelist)**2*4/1024**3:.1f} GB. Usa un subgrafo más pequeño.")
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    A = nx.to_scipy_sparse_array(G, nodelist=nodelist, dtype=float, format="csr")
    A_norm = sparse_a_torch(normalizar_adyacencia(A))
    A_densa = torch.tensor(A.toarray(), dtype=torch.float32)
    X_t = torch.tensor(StandardScaler().fit_transform(X), dtype=torch.float32)

    class GCN(nn.Module):
        def __init__(self, entrada, oculta, salida):
            super().__init__()
            self.l1 = nn.Linear(entrada, oculta)
            self.l2 = nn.Linear(oculta, salida)

        def forward(self, x, a):
            h = torch.relu(torch.sparse.mm(a, self.l1(x)))
            return torch.sparse.mm(a, self.l2(h))

    codificador = GCN(X_t.shape[1], 64, dim_oculta)
    decodificador_attr = GCN(dim_oculta, 64, X_t.shape[1])
    opt = torch.optim.Adam(
        list(codificador.parameters()) + list(decodificador_attr.parameters()), lr=lr)

    for _ in range(epocas):
        opt.zero_grad()
        Z = codificador(X_t, A_norm)
        A_rec = torch.sigmoid(Z @ Z.T)
        X_rec = decodificador_attr(Z, A_norm)
        perdida = alpha * nn.functional.mse_loss(A_rec, A_densa) + \
                  (1 - alpha) * nn.functional.mse_loss(X_rec, X_t)
        perdida.backward()
        opt.step()

    with torch.no_grad():
        Z = codificador(X_t, A_norm)
        err_estructura = ((torch.sigmoid(Z @ Z.T) - A_densa) ** 2).mean(dim=1)
        err_atributos = ((decodificador_attr(Z, A_norm) - X_t) ** 2).mean(dim=1)
        score = alpha * err_estructura + (1 - alpha) * err_atributos
    return score.numpy()


titulo("Métodos por reconstrucción")
scores_nn = {}
if TORCH_OK:
    Z_base = EMBEDDINGS["SVD (adyacencia)"]
    t0 = time.perf_counter()
    scores_nn["Autoencoder (sobre SVD)"] = autoencoder_anomalias(Z_base)
    print(f"  Autoencoder          {time.perf_counter()-t0:5.1f} s")

    # DOMINANT necesita atributos de nodo: se usan las features originales de Elliptic.
    X_attr = (df_features.set_index("txId")
              .reindex(NODOS_EMB)[[f"local_{i}" for i in range(1, 31)]]
              .fillna(0).values)
    t0 = time.perf_counter()
    try:
        scores_nn["DOMINANT (simplificado)"] = dominant_simplificado(G_emb, X_attr, NODOS_EMB)
        print(f"  DOMINANT             {time.perf_counter()-t0:5.1f} s")
    except MemoryError as exc:
        print(f"  DOMINANT omitido: {exc}")
else:
    print("  PyTorch no disponible: se omiten autoencoder y DOMINANT.")
    print("  Instálalo con `pip install torch` (en Colab ya viene preinstalado).")

print("""
ONE (Outlier aware Network Embedding) no se implementa aquí porque su valor añadido —pesos
por nodo aprendidos que descuentan a los outliers durante el propio entrenamiento— solo se
aprecia con una proporción alta de anomalías, y requiere una optimización alternada de tres
bloques que ocuparía más espacio del que aporta. La referencia es Bandyopadhyay, Lokesh &
Murty (AAAI 2019).""")

In [ ]:
# =============================================================================
# M14.4 — ¿Qué método encuentra fraude de verdad? Precision@K
# =============================================================================

def ranking_anomalias(scores: dict, etiquetas: np.ndarray, ks=(50, 100, 250, 500)) -> pd.DataFrame:
    """Evalúa detectores de anomalías con precision@K contra las etiquetas conocidas.

    Precision@K = de los K nodos con score más alto, qué fracción es realmente ilícita.
    Se contrasta con la TASA BASE (proporción de ilícitos entre los etiquetados): un
    detector solo aporta valor si su precision@K supera esa tasa base. La columna de
    'lift' hace explícita esa comparación, que es la que decide si el sistema se
    despliega o no.
    """
    mask = etiquetas >= 0
    y = etiquetas.copy()
    tasa_base = float((y[mask] == 1).mean())

    filas = []
    for nombre, s in scores.items():
        fila = {"detector": nombre}
        for k in ks:
            orden = np.argsort(-s)
            top = [i for i in orden if mask[i]][:k]
            precision = float(np.mean(y[top] == 1)) if top else 0.0
            fila[f"P@{k}"] = round(precision, 4)
            fila[f"lift@{k}"] = round(precision / tasa_base, 2) if tasa_base > 0 else np.nan
        filas.append(fila)

    tabla = pd.DataFrame(filas).set_index("detector")
    tabla.attrs["tasa_base"] = tasa_base
    return tabla


# Se alinean todos los scores sobre el mismo conjunto de nodos (el timestep de trabajo).
scores_todos = dict(scores_if)
scores_todos.update(scores_nn)

odd_por_nodo = odd.set_index("txId")["score_oddball"]
scores_todos["OddBall"] = np.nan_to_num(
    pd.Series(NODOS_EMB).map(odd_por_nodo).values.astype(float), nan=0.0)
scores_todos["Grado (línea base)"] = np.array([G_emb.degree(n) for n in NODOS_EMB], dtype=float)
scores_todos["Aleatorio (control)"] = np.random.default_rng(SEED).random(len(NODOS_EMB))

titulo(f"Precision@K de cada detector — timestep {TIMESTEP_FOCO}")
tabla_anomalias = ranking_anomalias(scores_todos, etiquetas_emb)
print(f"Tasa base de ilícitos entre etiquetados: {100*tabla_anomalias.attrs['tasa_base']:.2f}%\n")
display(tabla_anomalias.sort_values("P@100", ascending=False))

fig, eje = plt.subplots(figsize=(10, 4.5))
orden_detectores = tabla_anomalias.sort_values("P@100", ascending=False).index
ks = [50, 100, 250, 500]
ancho = 0.8 / len(orden_detectores)
for i, det in enumerate(orden_detectores):
    valores = [tabla_anomalias.loc[det, f"P@{k}"] * 100 for k in ks]
    eje.bar(np.arange(len(ks)) + i * ancho, valores, ancho, label=det)
eje.axhline(100 * tabla_anomalias.attrs["tasa_base"], ls="--", color=COLORES["ilicito"],
            lw=1.8, label="Tasa base (azar)")
eje.set_xticks(np.arange(len(ks)) + 0.4 - ancho / 2)
eje.set_xticklabels([f"K={k}" for k in ks])
eje.set_ylabel("Precision@K (%)")
eje.set_title("Precisión de cada detector no supervisado frente al azar")
eje.legend(fontsize=7.5, ncol=2)
plt.tight_layout()
plt.show()

print("""
CÓMO INTERPRETAR ESTA TABLA

· 'Aleatorio (control)' debe salir clavado en la tasa base. Si no, hay un error en la
  evaluación. Es la comprobación de cordura del experimento, no un método.
· 'Grado (línea base)' es la línea base honesta: ordenar por grado no cuesta nada. Todo
  método que no la supere no justifica su complejidad.
· Un lift de 2 significa que revisar las alertas del sistema es el doble de productivo
  que revisar casos al azar. En un equipo con capacidad para 100 revisiones diarias, eso
  se traduce directamente en el doble de fraude detectado con el mismo personal.
· Estos métodos NO han visto ni una etiqueta. Lo que miden es cuánta señal de fraude hay
  en la pura irregularidad estructural. El Módulo 15 pondrá el listón usando las etiquetas.""")

---
---

# Módulo 15 — Graph Machine Learning

Cuatro tareas, según qué se predice:

| Tarea | Se predice | Ejemplo en fraude | En este módulo |
|---|---|---|---|
| **Node classification** | Etiqueta de un nodo | ¿Es esta transacción ilícita? | M15.2 |
| **Edge classification** | Etiqueta de una arista | ¿Es esta transferencia fraudulenta? | M15.4 |
| **Graph classification** | Etiqueta de un grafo entero | ¿Contiene fraude este vecindario? | M15.5 |
| **Link prediction** | Existencia de una arista | ¿Aparecerá esta relación? | Módulo 12 |

## El protocolo de evaluación, que es lo que hay que acertar

Sobre Elliptic hay un protocolo estándar (Weber et al., 2019) y conviene respetarlo tanto para
poder comparar con la literatura como porque es el correcto:

- **Split temporal**: entrenar con $t \leq 34$, probar con $t \geq 35$. Nunca aleatorio.
- **Métricas sobre la clase ilícita**: precision, recall y F1 de la clase 1. La *accuracy* está
  prohibida: con un 9,8 % de ilícitos, predecir "todo lícito" da un 90 % de acierto.
- **Sin las features de vecindad etiquetada** del Módulo 11: ya se demostró que son fuga pura
  bajo este split.

### Por qué el split aleatorio miente

Con un split aleatorio, el modelo entrena con transacciones del timestep 40 y prueba con otras
del mismo timestep 40 — vecinas suyas en el grafo, generadas por el mismo proceso, a veces del
mismo actor. El resultado es una estimación optimista de algo que en producción nunca ocurre:
en producción siempre se predice el futuro con datos del pasado.

Además, sobre Elliptic hay un agravante concreto: el **cierre del mercado negro en $t=43$**
(Módulo 10). Un split aleatorio reparte ese evento entre entrenamiento y test; el temporal lo
deja íntegramente en test, que es donde debe estar. La celda M15.3 mide cuánto se degrada el
modelo tras ese evento — y ese número es el que de verdad describe cómo envejece un modelo de
fraude en producción.

In [ ]:
# =============================================================================
# M15.1 — Utilidades de evaluación y preparación de los datos
# =============================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluar_modelo(y_true, y_pred, y_prob=None, nombre: str = "") -> dict:
    """Métricas centradas en la clase minoritaria (ilícito = 1).

    Se reporta también la accuracy, pero acompañada de la 'accuracy trivial' —la que
    obtendría un modelo que predijera siempre la clase mayoritaria— para dejar claro
    de un vistazo que no significa nada por sí sola.
    """
    resultado = {
        "modelo": nombre,
        "precision (ilícito)": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "recall (ilícito)": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "F1 (ilícito)": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "F1 micro": round(f1_score(y_true, y_pred, average="micro"), 4),
        "accuracy": round(float(np.mean(y_true == y_pred)), 4),
        "accuracy trivial": round(float(max(np.mean(y_true == 0), np.mean(y_true == 1))), 4),
    }
    if y_prob is not None and len(np.unique(y_true)) > 1:
        resultado["ROC-AUC"] = round(roc_auc_score(y_true, y_prob), 4)
        resultado["PR-AUC"] = round(average_precision_score(y_true, y_prob), 4)
    return resultado


def split_temporal(df: pd.DataFrame, columnas: list, corte: int = CORTE_TEMPORAL) -> tuple:
    """Divide el dataset por timestep, tal como exige el protocolo de Elliptic."""
    etiquetados = df[df["y"] >= 0].copy()
    train = etiquetados[etiquetados["timestep"] <= corte]
    test = etiquetados[etiquetados["timestep"] > corte]
    return (train[columnas].fillna(0).values, train["y"].values,
            test[columnas].fillna(0).values, test["y"].values, test)


CONJUNTOS_FEATURES = {
    "Solo grafo (topológicas)": COLS_GRAFO,
    "Solo locales de Elliptic": COLS_LOCALES,
    "Locales + agregadas (AF)": COLS_LOCALES + COLS_AGG,
    "AF + features de grafo": COLS_LOCALES + COLS_AGG + COLS_GRAFO,
}

titulo("Configuración del experimento")
X_tr, y_tr, X_te, y_te, test_df = split_temporal(DATOS_ML, COLS_GRAFO)
print(f"  Corte temporal      : t ≤ {CORTE_TEMPORAL} entrena · t > {CORTE_TEMPORAL} prueba")
print(f"  Entrenamiento       : {len(y_tr):,} nodos ({y_tr.sum():,} ilícitos, {100*y_tr.mean():.1f}%)")
print(f"  Test                : {len(y_te):,} nodos ({y_te.sum():,} ilícitos, {100*y_te.mean():.1f}%)")
print("\n  Conjuntos de features a comparar:")
for nombre, cols in CONJUNTOS_FEATURES.items():
    print(f"    {nombre:32s} {len(cols):>4} columnas")

In [ ]:
# =============================================================================
# M15.2 — Node classification: ¿aportan algo las features de grafo?
# =============================================================================

titulo("Clasificación de nodos con split temporal")
resultados_ml, modelos, importancias = [], {}, {}

for nombre, columnas in CONJUNTOS_FEATURES.items():
    X_tr, y_tr, X_te, y_te, _ = split_temporal(DATOS_ML, columnas)

    modelo = RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
    t0 = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    pred = modelo.predict(X_te)
    prob = modelo.predict_proba(X_te)[:, 1]

    fila = evaluar_modelo(y_te, pred, prob, nombre=f"RF · {nombre}")
    fila["segundos"] = round(time.perf_counter() - t0, 1)
    resultados_ml.append(fila)
    modelos[nombre] = modelo
    importancias[nombre] = pd.Series(modelo.feature_importances_, index=columnas)
    print(f"  {nombre:32s} F1(ilícito) = {fila['F1 (ilícito)']:.4f}   ({fila['segundos']} s)")

# Línea base sin ninguna capacidad: predecir siempre lícito.
X_tr, y_tr, X_te, y_te, test_df = split_temporal(DATOS_ML, COLS_LOCALES + COLS_AGG + COLS_GRAFO)
resultados_ml.append(evaluar_modelo(y_te, np.zeros_like(y_te), nombre="Línea base: todo lícito"))

tabla_ml = pd.DataFrame(resultados_ml).set_index("modelo")
titulo("Resultados", nivel=2)
display(tabla_ml)

print("""
LECTURA

· La línea base 'todo lícito' consigue una accuracy alta y un F1 de la clase ilícita de
  cero. Es la demostración de por qué la accuracy no se reporta sola en problemas
  desbalanceados.
· 'Solo grafo' usa exclusivamente posición topológica: ni un solo atributo de la
  transacción. Que consiga un F1 muy por encima de cero es el resultado que justifica
  todo el notebook.
· Las features agregadas (agg_1..72) aportan mucho porque YA SON features de grafo:
  estadísticos de los vecinos a un salto, calculados por Elliptic. Comparar 'solo locales'
  con 'locales + agregadas' es, en el fondo, medir el valor de la información de vecindad.
· Los valores obtenidos son del orden de los publicados por Weber et al. (2019) con este
  mismo protocolo (F1 de la clase ilícita en torno a 0,7–0,8 con Random Forest). Si te
  sale muy por encima de 0,9, casi seguro se ha colado una feature con fuga.
""")

# Importancia de variables y matriz de confusión del mejor modelo.
mejor_nombre = tabla_ml.drop(index="Línea base: todo lícito")["F1 (ilícito)"].idxmax().replace("RF · ", "")
mejor_modelo = modelos[mejor_nombre]
X_tr, y_tr, X_te, y_te, test_df = split_temporal(DATOS_ML, CONJUNTOS_FEATURES[mejor_nombre])
pred_mejor = mejor_modelo.predict(X_te)

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.6))
top_imp = importancias[mejor_nombre].nlargest(18)
colores_imp = [COLORES["acento"] if c in COLS_GRAFO else COLORES["licito"] for c in top_imp.index]
ejes[0].barh(top_imp.index[::-1], top_imp.values[::-1], color=colores_imp[::-1])
ejes[0].set_title(f"Top-18 variables · {mejor_nombre}\n(naranja = feature de grafo)")
ejes[0].set_xlabel("Importancia (Gini)")

cm = confusion_matrix(y_te, pred_mejor)
ejes[1].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ejes[1].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=13,
                     color="white" if cm[i, j] > cm.max() / 2 else "#222")
ejes[1].set_xticks([0, 1]); ejes[1].set_xticklabels(["Pred. lícito", "Pred. ilícito"])
ejes[1].set_yticks([0, 1]); ejes[1].set_yticklabels(["Real lícito", "Real ilícito"])
ejes[1].set_title("Matriz de confusión sobre el test temporal")
ejes[1].grid(False)
plt.tight_layout()
plt.show()

n_grafo_top = sum(1 for c in top_imp.index if c in COLS_GRAFO)
print(f"Features de grafo entre las 18 más importantes: {n_grafo_top}")
print("\nInforme completo del mejor modelo:")
print(classification_report(y_te, pred_mejor, target_names=["Lícito", "Ilícito"], digits=4))

In [ ]:
# =============================================================================
# M15.3 — Cómo envejece el modelo: rendimiento timestep a timestep
# =============================================================================

test_df = test_df.copy()
test_df["pred"] = pred_mejor
test_df["prob"] = mejor_modelo.predict_proba(X_te)[:, 1]

por_timestep = []
for t, grupo in test_df.groupby("timestep"):
    if grupo["y"].sum() == 0:
        continue
    por_timestep.append({
        "timestep": int(t),
        "nodos": len(grupo),
        "ilícitos reales": int(grupo["y"].sum()),
        "precision": precision_score(grupo["y"], grupo["pred"], pos_label=1, zero_division=0),
        "recall": recall_score(grupo["y"], grupo["pred"], pos_label=1, zero_division=0),
        "F1": f1_score(grupo["y"], grupo["pred"], pos_label=1, zero_division=0),
    })
df_tiempo = pd.DataFrame(por_timestep).set_index("timestep")

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.2))
for metrica, color in [("precision", COLORES["licito"]), ("recall", COLORES["ilicito"]),
                       ("F1", COLORES["acento"])]:
    ejes[0].plot(df_tiempo.index, df_tiempo[metrica], marker="o", ms=4, label=metrica, color=color)
ejes[0].axvline(43, ls="--", color="#9AA3AD", lw=1.3)
ejes[0].annotate("cierre del\nmercado negro", xy=(43, 0.5), xytext=(44.5, 0.7), fontsize=8.5,
                 arrowprops=dict(arrowstyle="->", color="#9AA3AD"))
ejes[0].set_title("Rendimiento sobre la clase ilícita, timestep a timestep")
ejes[0].set_xlabel("timestep")
ejes[0].set_ylim(-0.03, 1.03)
ejes[0].legend()

ejes[1].bar(df_tiempo.index, df_tiempo["ilícitos reales"], color=COLORES["ilicito"], width=0.7)
ejes[1].axvline(43, ls="--", color="#9AA3AD", lw=1.3)
ejes[1].set_title("Transacciones ilícitas reales por timestep en el test")
ejes[1].set_xlabel("timestep")
plt.tight_layout()
plt.show()

antes_43 = df_tiempo[df_tiempo.index <= 43]["F1"]
despues_43 = df_tiempo[df_tiempo.index > 43]["F1"]
print(f"  F1 medio en t ∈ [35, 43] : {antes_43.mean():.4f}")
print(f"  F1 medio en t > 43       : {despues_43.mean():.4f}" if len(despues_43) else
      "  Sin timesteps con ilícitos después de t=43")
print("""
Esta caída no es un fallo del modelo: es la realidad del fraude. Cuando el entorno cambia
—se cierra un mercado, los defraudadores cambian de técnica—, el modelo entrenado con el
pasado deja de servir. En la literatura esto se llama *concept drift* y en fraude es la
norma, no la excepción.

Implicación operativa: un sistema de fraude no se entrena una vez. Necesita reentrenamiento
periódico, monitorización de la deriva de las distribuciones de entrada, y métricas que
detecten la degradación ANTES de que el recall se desplome. Un F1 agregado sobre todo el
test habría ocultado por completo este comportamiento.""")

In [ ]:
# =============================================================================
# M15.4 — Edge classification
# =============================================================================

def features_arista(G: nx.Graph, aristas: list, tabla_nodos: pd.DataFrame,
                    columnas: list) -> np.ndarray:
    """Construye features de arista combinando las de sus dos extremos.

    Tres operadores binarios habituales, cada uno con una intuición distinta:
      · concatenación [z_u, z_v] — conserva la dirección (u es origen, v destino)
      · producto Hadamard z_u ⊙ z_v — simétrico, captura coincidencia de magnitudes
      · diferencia absoluta |z_u − z_v| — captura DISPARIDAD entre los extremos

    Se usan los tres a la vez: una transferencia de una cuenta enorme a una diminuta es
    sospechosa precisamente por la disparidad, que la concatenación por sí sola no expresa.
    """
    idx = tabla_nodos.set_index("txId")[columnas].fillna(0)
    origen = idx.reindex([u for u, _ in aristas]).values
    destino = idx.reindex([v for _, v in aristas]).values
    return np.hstack([origen, destino, origen * destino, np.abs(origen - destino)])


titulo("Clasificación de aristas")
# Una arista es "de riesgo" si alguno de sus extremos es ilícito. Se restringe a aristas
# con AMBOS extremos etiquetados: en el resto la etiqueta de arista no está definida.
etiqueta_nodo = dict(zip(DATOS_ML["txId"], DATOS_ML["y"]))
ts_nodo = dict(zip(DATOS_ML["txId"], DATOS_ML["timestep"]))

aristas_etiquetadas, y_aristas, ts_aristas = [], [], []
for u, v in G.edges():
    yu, yv = etiqueta_nodo.get(u, -1), etiqueta_nodo.get(v, -1)
    if yu >= 0 and yv >= 0:
        aristas_etiquetadas.append((u, v))
        y_aristas.append(int(yu == 1 or yv == 1))
        ts_aristas.append(ts_nodo.get(u, 0))

y_aristas = np.array(y_aristas)
ts_aristas = np.array(ts_aristas)
print(f"  Aristas con ambos extremos etiquetados: {len(aristas_etiquetadas):,}")
print(f"  De riesgo (algún extremo ilícito)     : {y_aristas.sum():,} ({100*y_aristas.mean():.1f}%)")

if len(aristas_etiquetadas) > 500 and len(np.unique(y_aristas)) > 1:
    COLS_ARISTA = COLS_GRAFO + COLS_LOCALES[:30]
    X_aristas = features_arista(G, aristas_etiquetadas, DATOS_ML, COLS_ARISTA)
    tr = ts_aristas <= CORTE_TEMPORAL
    te = ~tr
    if te.sum() > 50 and len(np.unique(y_aristas[te])) > 1:
        modelo_arista = RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                               n_jobs=-1, random_state=SEED)
        modelo_arista.fit(X_aristas[tr], y_aristas[tr])
        pred_a = modelo_arista.predict(X_aristas[te])
        prob_a = modelo_arista.predict_proba(X_aristas[te])[:, 1]
        display(pd.DataFrame([evaluar_modelo(y_aristas[te], pred_a, prob_a,
                                             nombre="RF sobre aristas")]).set_index("modelo"))
        print(f"  Dimensión de las features de arista: {X_aristas.shape[1]} "
              f"(= 4 operadores x {len(COLS_ARISTA)} columnas)")
    else:
        print("  Test insuficiente para evaluar la clasificación de aristas.")

In [ ]:
# =============================================================================
# M15.5 — Graph classification sobre redes ego
# =============================================================================

def descriptores_grafo(g: nx.Graph) -> dict:
    """Vector de características de un GRAFO completo (no de sus nodos).

    Son los descriptores clásicos previos a las GNN: baratos, interpretables y
    sorprendentemente competitivos en grafos pequeños. Un Graph Neural Network
    aprendería estas features en vez de definirlas a mano — esa es exactamente la
    diferencia que el Módulo 16 pone a prueba.
    """
    gu = nx.Graph(g.to_undirected() if g.is_directed() else g)
    gu.remove_edges_from(nx.selfloop_edges(gu))
    n, m = gu.number_of_nodes(), gu.number_of_edges()
    grados = [d for _, d in gu.degree()] or [0]
    return {
        "n_nodos": n,
        "n_aristas": m,
        "densidad": nx.density(gu) if n > 1 else 0.0,
        "grado_medio": float(np.mean(grados)),
        "grado_max": int(np.max(grados)),
        "grado_std": float(np.std(grados)),
        "transitividad": nx.transitivity(gu) if n > 2 else 0.0,
        "clustering_medio": nx.average_clustering(gu) if n > 2 else 0.0,
        "n_triangulos": sum(nx.triangles(gu).values()) // 3 if n > 2 else 0,
        "componentes": nx.number_connected_components(gu) if n else 0,
    }


titulo("Clasificación de grafos: ¿contiene fraude esta red ego?")
rng = np.random.default_rng(SEED)
centros = DATOS_ML[(DATOS_ML["y"] >= 0) & (DATOS_ML["degree"] >= 3)]
if len(centros) > 3000:
    centros = centros.sample(3000, random_state=SEED)

filas_grafos, y_grafos, ts_grafos = [], [], []
for _, fila in centros.iterrows():
    ego = nx.ego_graph(G_und, fila["txId"], radius=1)
    if ego.number_of_nodes() < 3:
        continue
    filas_grafos.append(descriptores_grafo(ego))
    contiene = any(etiqueta_nodo.get(n, -1) == 1 for n in ego.nodes())
    y_grafos.append(int(contiene))
    ts_grafos.append(fila["timestep"])

if filas_grafos:
    X_g = pd.DataFrame(filas_grafos).fillna(0)
    y_g = np.array(y_grafos)
    ts_g = np.array(ts_grafos)
    tr, te = ts_g <= CORTE_TEMPORAL, ts_g > CORTE_TEMPORAL
    print(f"  Redes ego construidas : {len(X_g):,}")
    print(f"  Contienen fraude      : {y_g.sum():,} ({100*y_g.mean():.1f}%)")
    print(f"  Descriptores por grafo: {X_g.shape[1]}\n")

    if tr.sum() > 100 and te.sum() > 50 and len(np.unique(y_g[te])) > 1:
        modelo_g = RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                          n_jobs=-1, random_state=SEED)
        modelo_g.fit(X_g[tr], y_g[tr])
        pred_g = modelo_g.predict(X_g[te])
        prob_g = modelo_g.predict_proba(X_g[te])[:, 1]
        display(pd.DataFrame([evaluar_modelo(y_g[te], pred_g, prob_g,
                                             nombre="RF sobre redes ego")]).set_index("modelo"))
        imp_g = pd.Series(modelo_g.feature_importances_, index=X_g.columns).sort_values()
        plt.figure(figsize=(8, 3.8))
        plt.barh(imp_g.index, imp_g.values, color=COLORES["ok"])
        plt.title("Qué descriptor del vecindario delata la presencia de fraude")
        plt.xlabel("Importancia")
        plt.tight_layout()
        plt.show()
    else:
        print("  Muestras insuficientes en el split temporal para esta tarea.")

---
---

# Módulo 16 — Graph Neural Networks

Estado del arte. La idea común a todas las arquitecturas es el **paso de mensajes**: cada nodo
actualiza su representación agregando la de sus vecinos, y al apilar $L$ capas cada nodo acaba
viendo su vecindario a $L$ saltos.

$$h_v^{(l+1)} = \text{ACTUALIZAR}\Big(h_v^{(l)},\; \text{AGREGAR}\big(\{h_u^{(l)} : u \in \mathcal{N}(v)\}\big)\Big)$$

Todas las arquitecturas de este módulo son la misma ecuación con distinta función de agregación:

| Arquitectura | Agregación | Aportación | Coste |
|---|---|---|---|
| **GCN** | Media normalizada $D^{-1/2}\hat{A}D^{-1/2}$ | La formulación base, simple y eficaz | $O(m \cdot d)$ |
| **GraphSAGE** | Media de vecinos **+ transformación propia separada** | Distingue "yo" de "mis vecinos"; admite muestreo e inducción | $O(m \cdot d)$ |
| **GAT** | Media **ponderada por atención aprendida** | Aprende qué vecinos importan | $O(m \cdot d)$ |
| **GIN** | **Suma** + MLP | Máximo poder discriminativo (equivale al test de Weisfeiler-Lehman) | $O(m \cdot d)$ |
| **RGCN** | Una matriz de pesos **por tipo de relación** | Grafos heterogéneos | $O(m \cdot d \cdot R)$ |

## Por qué la suma y no la media (GIN)

Xu et al. (2019) demostraron que la media y el máximo **no distinguen** ciertos vecindarios: la
media de $\{1, 1, 1\}$ y la de $\{1\}$ son idénticas. Una cuenta con 3 vecinos idénticos y otra
con 1 producen la misma representación. La **suma** sí los distingue, y por eso GIN alcanza el
poder expresivo máximo posible para una GNN de paso de mensajes. En fraude importa: el volumen
de vecinos *es* la señal en un patrón de smurfing.

## Por qué GraphSAGE es el que se usa en producción

GCN es **transductivo**: la normalización depende de la matriz de adyacencia completa, así que
un nodo nuevo obliga a recalcular. GraphSAGE es **inductivo**: aprende una función de agregación
que se aplica a cualquier vecindario, incluido el de una cuenta abierta hace cinco minutos. En un
sistema de fraude real, donde llegan entidades nuevas continuamente, esa diferencia lo decide todo.

## Sobre la implementación de este módulo

Todo está escrito en **PyTorch puro sobre matrices dispersas**, sin `torch-geometric` ni `DGL`.
No es por purismo: son ~120 líneas, se ejecutan en cualquier entorno con torch instalado —el de
Colab lo trae— y hacen visible que una GNN no es magia, sino una multiplicación dispersa seguida
de una capa lineal. Para producción, usa `torch-geometric`: tiene kernels optimizados, muestreo
de vecindarios y decenas de arquitecturas.

**Graph Transformer**: sustituye la agregación local por atención global —cada nodo atiende a
todos los demás, más una codificación estructural que reinyecta la topología. Escala $O(n^2)$, así
que en grafos grandes se aplica sobre subgrafos. Se comenta en el Módulo 23.

In [ ]:
# =============================================================================
# M16.1 — Infraestructura común
# =============================================================================

if TORCH_OK:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch {torch.__version__} · dispositivo: {DISPOSITIVO}")
else:
    print("PyTorch no disponible. Este módulo se salta por completo.")
    print("Instálalo con `pip install torch` (en Colab viene preinstalado).")


def preparar_tensores_elliptic(G: nx.Graph, datos: pd.DataFrame, columnas: list,
                               corte: int = CORTE_TEMPORAL) -> dict:
    """Construye los tensores que consumen todas las GNN del módulo.

    El punto crítico es el ORDEN: la fila i de X debe corresponder al nodo i de la
    matriz de adyacencia. `DATOS_ML` se construyó a partir de `list(G.nodes())` y todos
    los merges posteriores fueron `how="left"`, así que el orden se conserva — pero se
    verifica explícitamente, porque un desalineamiento aquí no lanza ninguna excepción:
    simplemente produce un modelo que entrena, converge y no significa nada.
    """
    nodelist = list(G.nodes())
    assert np.array_equal(datos["txId"].values, np.array(nodelist)), \
        "DATOS_ML no está alineado con G.nodes(): las features quedarían asignadas a otro nodo"

    # csr_matrix (API antigua) y no csr_array: se mezcla con sp.eye y sp.diags, que
    # devuelven spmatrix, y combinar los dos APIs de SciPy produce resultados inconsistentes.
    A = sp.csr_matrix(nx.to_scipy_sparse_array(G, nodelist=nodelist, dtype=float, format="csr"))
    A_und = A + A.T                                   # el paso de mensajes usa ambos sentidos
    A_und.data[:] = 1.0

    X = StandardScaler().fit_transform(datos[columnas].fillna(0).values).astype(np.float32)
    y = datos["y"].values
    ts = datos["timestep"].values

    grados = np.asarray(A_und.sum(axis=1)).ravel()
    inv = np.zeros_like(grados, dtype=float)
    np.divide(1.0, grados, out=inv, where=grados > 0)

    aristas = sp.coo_matrix(A_und + sp.eye(A_und.shape[0], format="csr"))

    return {
        "nodelist": nodelist,
        "X": torch.tensor(X, device=DISPOSITIVO),
        "y": torch.tensor((y == 1).astype(np.int64), device=DISPOSITIVO),
        "A_gcn": sparse_a_torch(normalizar_adyacencia(A_und)).to(DISPOSITIVO),
        "A_media": sparse_a_torch(sp.diags(inv) @ A_und).to(DISPOSITIVO),
        "A_suma": sparse_a_torch(sp.coo_matrix(A_und)).to(DISPOSITIVO),
        "edge_index": torch.tensor(np.vstack([aristas.row, aristas.col]).astype(np.int64),
                                   device=DISPOSITIVO),
        "mask_train": torch.tensor((ts <= corte) & (y >= 0), device=DISPOSITIVO),
        "mask_test": torch.tensor((ts > corte) & (y >= 0), device=DISPOSITIVO),
    }


if TORCH_OK:
    COLS_GNN = COLS_LOCALES + COLS_AGG
    DATOS_GNN, _ = cronometrar(preparar_tensores_elliptic, G, DATOS_ML, COLS_GNN,
                               etiqueta="preparar_tensores_elliptic")
    print(f"  X            : {tuple(DATOS_GNN['X'].shape)}  "
          f"({DATOS_GNN['X'].element_size()*DATOS_GNN['X'].nelement()/1024**2:.0f} MB)")
    print(f"  Aristas      : {DATOS_GNN['edge_index'].shape[1]:,} (con self-loops, ambos sentidos)")
    print(f"  Entrenamiento: {int(DATOS_GNN['mask_train'].sum()):,} nodos")
    print(f"  Test         : {int(DATOS_GNN['mask_test'].sum()):,} nodos")

In [ ]:
# =============================================================================
# M16.2 — Las cinco arquitecturas
# =============================================================================

if TORCH_OK:

    class CapaGCN(nn.Module):
        """h' = Â · (h W).  Kipf & Welling (2017)."""
        def __init__(self, entrada, salida):
            super().__init__()
            self.lin = nn.Linear(entrada, salida)

        def forward(self, X, A):
            return torch.sparse.mm(A, self.lin(X))

    class CapaSAGE(nn.Module):
        """h' = W_self·h + W_vec·media(h_vecinos).  Hamilton et al. (2017).

        La clave está en tener DOS matrices de pesos: la representación propia y la
        agregada de los vecinos se transforman por separado, así que la red puede
        aprender a darles distinta importancia. GCN las mezcla y no puede.
        """
        def __init__(self, entrada, salida):
            super().__init__()
            self.lin_propio = nn.Linear(entrada, salida)
            self.lin_vecinos = nn.Linear(entrada, salida, bias=False)

        def forward(self, X, A_media):
            return self.lin_propio(X) + self.lin_vecinos(torch.sparse.mm(A_media, X))

    class CapaGAT(nn.Module):
        """Atención sobre aristas: α_uv = softmax_v(LeakyReLU(aᵀ[Wh_u ‖ Wh_v])).

        La softmax se calcula por nodo DESTINO. Sin `torch-geometric` no hay un
        scatter_softmax listo, así que se implementa con `index_add_`: se acumulan los
        exponentes por destino y se divide. Es exactamente lo mismo, en cuatro líneas.
        """
        def __init__(self, entrada, salida, pendiente: float = 0.2):
            super().__init__()
            self.lin = nn.Linear(entrada, salida, bias=False)
            self.att_origen = nn.Parameter(torch.empty(salida))
            self.att_destino = nn.Parameter(torch.empty(salida))
            nn.init.normal_(self.att_origen, std=0.1)
            nn.init.normal_(self.att_destino, std=0.1)
            self.pendiente = pendiente

        def forward(self, X, edge_index):
            H = self.lin(X)
            origen, destino = edge_index[0], edge_index[1]
            e = F.leaky_relu((H * self.att_origen).sum(-1)[origen] +
                             (H * self.att_destino).sum(-1)[destino], self.pendiente)
            e = e - e.max()
            exp = e.exp()
            denom = torch.zeros(X.size(0), device=X.device).index_add_(0, destino, exp) + 1e-16
            alpha = (exp / denom[destino]).unsqueeze(-1)
            salida = torch.zeros_like(H).index_add_(0, destino, H[origen] * alpha)
            return salida

    class CapaGIN(nn.Module):
        """h' = MLP((1+ε)·h + Σ h_vecinos).  Xu et al. (2019).

        Agregación por SUMA, no media: es lo que le da poder discriminativo máximo.
        ε es un parámetro aprendible que regula cuánto pesa la representación propia.
        """
        def __init__(self, entrada, salida):
            super().__init__()
            self.mlp = nn.Sequential(nn.Linear(entrada, salida), nn.ReLU(),
                                     nn.Linear(salida, salida))
            self.eps = nn.Parameter(torch.zeros(1))

        def forward(self, X, A_suma):
            return self.mlp((1 + self.eps) * X + torch.sparse.mm(A_suma, X))

    class CapaRGCN(nn.Module):
        """Una matriz de pesos por tipo de relación.  Schlichtkrull et al. (2018).

        h'_v = W_0·h_v + Σ_r Σ_{u∈N_r(v)} (1/c_{v,r})·W_r·h_u

        Es la generalización natural de GCN a grafos heterogéneos: 'transfiere a' y
        'comparte dispositivo con' son relaciones semánticamente distintas y deben
        transformarse con pesos distintos. El coste es lineal en el nº de relaciones,
        que es lo que motiva la descomposición en bases del paper original.
        """
        def __init__(self, entrada, salida, n_relaciones):
            super().__init__()
            self.propio = nn.Linear(entrada, salida)
            self.por_relacion = nn.ModuleList(
                [nn.Linear(entrada, salida, bias=False) for _ in range(n_relaciones)])

        def forward(self, X, matrices):
            salida = self.propio(X)
            for lin, A_r in zip(self.por_relacion, matrices):
                salida = salida + torch.sparse.mm(A_r, lin(X))
            return salida

    class RedGNN(nn.Module):
        """Envoltorio de dos capas + dropout, común a las cuatro arquitecturas homogéneas."""
        def __init__(self, tipo, entrada, oculta, salida=2, dropout=0.4):
            super().__init__()
            capas = {"GCN": CapaGCN, "GraphSAGE": CapaSAGE, "GAT": CapaGAT, "GIN": CapaGIN}
            Capa = capas[tipo]
            self.tipo = tipo
            self.capa1 = Capa(entrada, oculta)
            self.capa2 = Capa(oculta, salida)
            self.dropout = dropout

        def forward(self, X, estructura):
            h = F.relu(self.capa1(X, estructura))
            h = F.dropout(h, p=self.dropout, training=self.training)
            return self.capa2(h, estructura)

    print("Arquitecturas definidas: GCN, GraphSAGE, GAT, GIN, RGCN")

In [ ]:
# =============================================================================
# M16.3 — Entrenamiento y comparación con el baseline tabular
# =============================================================================

if TORCH_OK:

    def entrenar_gnn(modelo, datos: dict, estructura_clave: str, epocas: int = 120,
                     lr: float = 0.01, peso_decay: float = 5e-4, seed: int = SEED,
                     verbose: bool = False) -> dict:
        """Entrena una GNN a grafo completo y evalúa sobre el split temporal.

        Detalles que importan:
          · `class_weight` inverso a la frecuencia. Sin él, con un 10 % de ilícitos la
            red converge a predecir todo lícito y la pérdida baja de maravilla.
          · La pérdida se calcula SOLO sobre los nodos de entrenamiento, pero el paso de
            mensajes usa el grafo COMPLETO. Eso es correcto y no es fuga: se propagan
            features, no etiquetas. Un nodo de test contribuye con sus atributos, que
            en producción también están disponibles.
        """
        torch.manual_seed(seed)
        modelo = modelo.to(DISPOSITIVO)
        opt = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=peso_decay)

        y = datos["y"]
        m_tr, m_te = datos["mask_train"], datos["mask_test"]
        estructura = datos[estructura_clave]

        n_pos = float(y[m_tr].sum())
        n_neg = float((~y[m_tr].bool()).sum())
        pesos = torch.tensor([1.0, n_neg / max(n_pos, 1.0)], dtype=torch.float32,
                             device=DISPOSITIVO)

        t0 = time.perf_counter()
        historial = []
        for epoca in range(epocas):
            modelo.train()
            opt.zero_grad()
            salida = modelo(datos["X"], estructura)
            perdida = F.cross_entropy(salida[m_tr], y[m_tr], weight=pesos)
            perdida.backward()
            opt.step()
            if (epoca + 1) % 20 == 0:
                modelo.eval()
                with torch.no_grad():
                    pred = modelo(datos["X"], estructura).argmax(1)
                f1 = f1_score(y[m_te].cpu().numpy(), pred[m_te].cpu().numpy(),
                              pos_label=1, zero_division=0)
                historial.append({"época": epoca + 1, "pérdida": float(perdida), "F1 test": f1})
                if verbose:
                    print(f"    época {epoca+1:>3}  pérdida {float(perdida):.4f}  F1 {f1:.4f}")

        modelo.eval()
        with torch.no_grad():
            logits = modelo(datos["X"], estructura)
            prob = F.softmax(logits, dim=1)[:, 1]
            pred = logits.argmax(1)

        y_te = y[m_te].cpu().numpy()
        resultado = evaluar_modelo(y_te, pred[m_te].cpu().numpy(), prob[m_te].cpu().numpy(),
                                   nombre=modelo.tipo if hasattr(modelo, "tipo") else "GNN")
        resultado["segundos"] = round(time.perf_counter() - t0, 1)
        resultado["parámetros"] = sum(p.numel() for p in modelo.parameters())
        resultado["_historial"] = historial
        return resultado

    titulo("Entrenamiento de las cuatro GNN homogéneas ⏱️")
    ESTRUCTURA = {"GCN": "A_gcn", "GraphSAGE": "A_media", "GAT": "edge_index", "GIN": "A_suma"}
    resultados_gnn, historiales = [], {}

    for tipo, clave in ESTRUCTURA.items():
        print(f"\n▸ {tipo}")
        modelo = RedGNN(tipo, DATOS_GNN["X"].shape[1], oculta=64)
        r = entrenar_gnn(modelo, DATOS_GNN, clave, epocas=120)
        historiales[tipo] = r.pop("_historial")
        resultados_gnn.append(r)
        print(f"  F1(ilícito) = {r['F1 (ilícito)']:.4f} · precision = {r['precision (ilícito)']:.4f} · "
              f"recall = {r['recall (ilícito)']:.4f} · {r['segundos']} s · {r['parámetros']:,} parámetros")

    # Comparación con el Random Forest del Módulo 15, sobre EXACTAMENTE el mismo split.
    tabla_gnn = pd.DataFrame(resultados_gnn).set_index("modelo")
    fila_rf = tabla_ml.loc[[i for i in tabla_ml.index if "AF + features de grafo" in i]]
    comparativa_final = pd.concat([
        tabla_gnn[["precision (ilícito)", "recall (ilícito)", "F1 (ilícito)", "ROC-AUC", "PR-AUC"]],
        fila_rf[["precision (ilícito)", "recall (ilícito)", "F1 (ilícito)", "ROC-AUC", "PR-AUC"]],
    ])
    titulo("GNN frente a Random Forest — mismo split temporal, mismas features", nivel=2)
    display(comparativa_final)

    fig, ejes = plt.subplots(1, 2, figsize=(14, 4.3))
    for tipo, hist in historiales.items():
        if hist:
            ejes[0].plot([h["época"] for h in hist], [h["F1 test"] for h in hist],
                         marker="o", ms=3.5, label=tipo)
    ejes[0].set_title("Convergencia: F1 de la clase ilícita en test")
    ejes[0].set_xlabel("Época")
    ejes[0].set_ylabel("F1")
    ejes[0].legend(fontsize=8.5)

    orden = comparativa_final.sort_values("F1 (ilícito)")
    ejes[1].barh(range(len(orden)), orden["F1 (ilícito)"],
                 color=[COLORES["acento"] if "RF" in str(i) else COLORES["licito"] for i in orden.index])
    ejes[1].set_yticks(range(len(orden)))
    ejes[1].set_yticklabels([str(i)[:34] for i in orden.index], fontsize=8.5)
    ejes[1].set_xlabel("F1 (clase ilícita)")
    ejes[1].set_title("Comparativa final (naranja = baseline tabular)")
    plt.tight_layout()
    plt.show()

    print("""
UN RESULTADO QUE CONVIENE NO MAQUILLAR

En Elliptic, Random Forest sobre las features tabulares suele igualar o superar a las GNN.
Está documentado en la literatura y no es un fallo de esta implementación. Las razones:

  · Las 72 features 'agregadas' de Elliptic YA son el resultado de agregar el vecindario a
    un salto. El dataset trae media capa de GNN incorporada, así que el margen de mejora
    del paso de mensajes es mucho menor de lo habitual.
  · El grafo son 49 componentes disjuntas de diámetro pequeño. Hay poca estructura global
    que propagar: a dos saltos, una GNN ya ha visto casi todo lo que hay.
  · 46.564 nodos etiquetados con un 10 % de positivos es poco para una red neuronal, y
    mucho para un ensamble de árboles.

Dónde ganan las GNN de verdad: grafos con relaciones que cruzan el tiempo, atributos de nodo
pobres (donde la estructura es la única señal), y escenarios inductivos con entidades nuevas
constantemente — es decir, un grafo bancario real, no este dataset.""")

In [ ]:
# =============================================================================
# M16.4 — RGCN sobre el grafo heterogéneo bancario
# =============================================================================

if TORCH_OK:

    def preparar_rgcn_banco(G: nx.MultiDiGraph, relaciones=None) -> dict:
        """Prepara tensores para RGCN: una matriz de adyacencia por tipo de relación.

        Las features de nodo son one-hot del TIPO de entidad más dos señales
        estructurales (grados). Es deliberado: se quiere comprobar cuánto puede
        aprender el modelo a partir casi solo del cableado del grafo.
        """
        nodos = list(G.nodes())
        indice = {n: i for i, n in enumerate(nodos)}
        tipos = sorted({d["tipo"] for _, d in G.nodes(data=True)})
        relaciones = relaciones or sorted({d["rel"] for _, _, d in G.edges(data=True)})

        matrices = []
        for rel in relaciones:
            filas, columnas = [], []
            for u, v, d in G.edges(data=True):
                if d["rel"] == rel:
                    filas += [indice[u], indice[v]]        # simétrica: los mensajes van y vienen
                    columnas += [indice[v], indice[u]]
            A_r = sp.coo_matrix((np.ones(len(filas)), (filas, columnas)),
                                shape=(len(nodos), len(nodos))).tocsr()
            A_r.data[:] = 1.0
            grados = np.asarray(A_r.sum(axis=1)).ravel()
            inv = np.zeros_like(grados, dtype=float)
            np.divide(1.0, grados, out=inv, where=grados > 0)
            matrices.append(sparse_a_torch(sp.diags(inv) @ A_r).to(DISPOSITIVO))

        X = np.zeros((len(nodos), len(tipos) + 2), dtype=np.float32)
        for i, n in enumerate(nodos):
            X[i, tipos.index(G.nodes[n]["tipo"])] = 1.0
            X[i, -2] = math.log1p(G.in_degree(n))
            X[i, -1] = math.log1p(G.out_degree(n))

        y = np.array([1 if G.nodes[n]["patron_fraude"] else 0 for n in nodos], dtype=np.int64)
        rng = np.random.default_rng(SEED)
        train = rng.random(len(nodos)) < 0.6

        return {
            "nodos": nodos, "tipos": tipos, "relaciones": relaciones,
            "X": torch.tensor(X, device=DISPOSITIVO),
            "y": torch.tensor(y, device=DISPOSITIVO),
            "matrices": matrices,
            "mask_train": torch.tensor(train, device=DISPOSITIVO),
            "mask_test": torch.tensor(~train, device=DISPOSITIVO),
        }

    titulo("RGCN sobre el grafo bancario heterogéneo")
    D_rgcn = preparar_rgcn_banco(G_banco)
    print(f"  Nodos      : {len(D_rgcn['nodos']):,}")
    print(f"  Tipos      : {D_rgcn['tipos']}")
    print(f"  Relaciones : {D_rgcn['relaciones']}")
    print(f"  Positivos  : {int(D_rgcn['y'].sum()):,} nodos implicados en algún patrón\n")

    class RedRGCN(nn.Module):
        def __init__(self, entrada, oculta, n_rel, salida=2):
            super().__init__()
            self.tipo = "RGCN"
            self.c1 = CapaRGCN(entrada, oculta, n_rel)
            self.c2 = CapaRGCN(oculta, salida, n_rel)

        def forward(self, X, matrices):
            h = F.relu(self.c1(X, matrices))
            h = F.dropout(h, p=0.3, training=self.training)
            return self.c2(h, matrices)

    torch.manual_seed(SEED)
    modelo_r = RedRGCN(D_rgcn["X"].shape[1], 32, len(D_rgcn["relaciones"])).to(DISPOSITIVO)
    opt = torch.optim.Adam(modelo_r.parameters(), lr=0.01, weight_decay=5e-4)
    n_pos = float(D_rgcn["y"][D_rgcn["mask_train"]].sum())
    n_neg = float((D_rgcn["y"][D_rgcn["mask_train"]] == 0).sum())
    pesos = torch.tensor([1.0, n_neg / max(n_pos, 1.0)], dtype=torch.float32, device=DISPOSITIVO)

    for epoca in range(200):
        modelo_r.train()
        opt.zero_grad()
        salida = modelo_r(D_rgcn["X"], D_rgcn["matrices"])
        perdida = F.cross_entropy(salida[D_rgcn["mask_train"]],
                                  D_rgcn["y"][D_rgcn["mask_train"]], weight=pesos)
        perdida.backward()
        opt.step()

    modelo_r.eval()
    with torch.no_grad():
        logits = modelo_r(D_rgcn["X"], D_rgcn["matrices"])
        pred = logits.argmax(1)
        prob = F.softmax(logits, 1)[:, 1]
    m = D_rgcn["mask_test"]
    display(pd.DataFrame([evaluar_modelo(
        D_rgcn["y"][m].cpu().numpy(), pred[m].cpu().numpy(), prob[m].cpu().numpy(),
        nombre="RGCN · grafo bancario")]).set_index("modelo"))

    print("""
El RGCN parte de features casi vacías —el tipo de entidad y dos grados— y aun así separa
los nodos implicados en fraude. Toda la señal que usa viene del CABLEADO: qué tipos de
relación conectan a cada nodo y con qué clase de entidades. Es la demostración más directa
de por qué modelar el dominio como grafo heterogéneo tiene valor.""")

---
---

# Módulo 17 — Visualización

## Herramientas

| Herramienta | Tipo | Escala útil | Cuándo |
|---|---|---|---|
| **NetworkX + Matplotlib** | Estático | < 5.000 nodos | Informes, documentación, exploración rápida |
| **Plotly** | Interactivo (notebook/web) | < 20.000 nodos | Dashboards, exploración con zoom y tooltip |
| **PyVis** | Interactivo (HTML autónomo) | < 10.000 nodos | Compartir un caso concreto con un investigador |
| **Gephi** | Escritorio | < 1.000.000 nodos | Análisis visual serio, publicaciones |
| **Cytoscape** | Escritorio | < 500.000 nodos | Bioinformática, redes con muchos atributos |

Para Gephi y Cytoscape lo relevante no es la herramienta sino **exportar bien**: este módulo
incluye las funciones de exportación a GEXF, GraphML y JSON de Cytoscape.

## Layouts

| Layout | Algoritmo | Bueno para | Coste |
|---|---|---|---|
| **Spring / Force-Directed** (Fruchterman-Reingold) | Muelles atractivos + repulsión eléctrica | Comunidades, estructura general | $O(n^2)$ por iteración |
| **Kamada-Kawai** | Minimiza el error frente a distancias de grafo | Grafos pequeños, resultado muy legible | $O(n^3)$ |
| **Circular** | Nodos en un círculo | Comparar densidad de conexiones | $O(n)$ |
| **Shell** | Círculos concéntricos por capas | Jerarquías, distancia a un nodo foco | $O(n)$ |
| **Spectral** | Autovectores del laplaciano | Separación de componentes | $O(n \cdot m)$ |
| **Multipartite** | Columnas por atributo | Grafos bipartitos y heterogéneos | $O(n)$ |

## La regla que casi nadie sigue

**Por encima de unos pocos miles de nodos, dibujar el grafo entero no comunica nada.** El
resultado es una mancha —lo que en la jerga se llama *hairball*— de la que no se extrae ninguna
conclusión. Las cuatro alternativas que sí funcionan:

1. **Agregar**: dibujar el grafo de comunidades, no el de nodos. Un nodo por comunidad, tamaño
   proporcional a sus miembros, aristas con grosor según el flujo entre ellas.
2. **Filtrar**: quedarse con el subgrafo relevante (la red ego del caso, la componente sospechosa).
3. **Muestrear**: BFS desde un punto de interés, como en el Módulo 2.
4. **No dibujar**: a menudo una tabla ordenada por centralidad es más informativa que cualquier
   gráfico. La visualización de red es una herramienta de exploración y de comunicación, no de
   análisis.

In [ ]:
# =============================================================================
# M17.1 — Los seis layouts sobre el mismo grafo
# =============================================================================

def calcular_layout(G: nx.Graph, tipo: str = "spring", seed: int = SEED, **kwargs) -> dict:
    """Devuelve las posiciones de los nodos según el layout indicado.

    Unifica las diferencias de firma de NetworkX: unos aceptan `seed`, otros no, y
    `multipartite_layout` necesita un atributo de nodo que sirva de capa.
    """
    layouts = {
        "spring": lambda: nx.spring_layout(G, seed=seed, k=kwargs.get("k"),
                                           iterations=kwargs.get("iterations", 60)),
        "kamada_kawai": lambda: nx.kamada_kawai_layout(G),
        "circular": lambda: nx.circular_layout(G),
        "shell": lambda: nx.shell_layout(G, nlist=kwargs.get("nlist")),
        "spectral": lambda: nx.spectral_layout(G),
        "random": lambda: nx.random_layout(G, seed=seed),
    }
    if tipo == "multipartite":
        return nx.multipartite_layout(G, subset_key=kwargs.get("subset_key", "capa"))
    if tipo not in layouts:
        raise ValueError(f"Layout desconocido: {tipo!r}")
    return layouts[tipo]()


def dibujar_grafo(G: nx.Graph, pos: dict | None = None, layout: str = "spring",
                  color_por: str | None = None, tam_por: str | None = None,
                  titulo_fig: str = "", eje=None, con_etiquetas: bool = False,
                  paleta: dict | None = None, seed: int = SEED) -> dict:
    """Dibuja un grafo con codificación visual por atributos de nodo.

    `color_por` y `tam_por` nombran atributos de nodo. El color es categórico (se
    asigna un tono por valor distinto) y el tamaño es continuo (se escala al rango).
    Devuelve las posiciones para poder reutilizarlas: recalcular el layout entre dos
    figuras del mismo grafo hace imposible compararlas.
    """
    if pos is None:
        pos = calcular_layout(G, layout, seed=seed)
    if eje is None:
        _, eje = plt.subplots(figsize=(9, 6.5))

    if color_por:
        valores = [G.nodes[n].get(color_por) for n in G.nodes()]
        distintos = sorted({v for v in valores if v is not None}, key=str)
        if paleta is None:
            tab = plt.cm.tab20(np.linspace(0, 1, max(len(distintos), 2)))
            paleta = {v: tab[i] for i, v in enumerate(distintos)}
        colores = [paleta.get(v, (0.8, 0.8, 0.8, 1.0)) for v in valores]
    else:
        colores = COLORES["licito"]

    if tam_por:
        crudos = np.array([float(G.nodes[n].get(tam_por, 0) or 0) for n in G.nodes()])
        rango = crudos.max() - crudos.min()
        tam = 30 + 380 * ((crudos - crudos.min()) / rango if rango > 0 else np.zeros_like(crudos))
    else:
        tam = 60

    nx.draw_networkx_edges(G, pos, ax=eje, edge_color="#CDD3DA", width=0.7,
                           arrows=G.is_directed(), arrowsize=8, alpha=0.75)
    nx.draw_networkx_nodes(G, pos, ax=eje, node_color=colores, node_size=tam,
                           linewidths=0.4, edgecolors="white")
    if con_etiquetas:
        nx.draw_networkx_labels(G, pos, ax=eje, font_size=6.5)
    eje.set_title(titulo_fig or f"{G.number_of_nodes()} nodos · {G.number_of_edges()} aristas",
                  fontsize=10.5)
    eje.axis("off")
    return pos


# Grafo pequeño y conexo, para que las diferencias entre layouts se aprecien.
G_vis = muestrear_subgrafo(G, 160, metodo="bfs").to_undirected()
G_vis = nx.Graph(G_vis.subgraph(max(nx.connected_components(G_vis), key=len)))

titulo(f"Seis layouts sobre el mismo grafo ({G_vis.number_of_nodes()} nodos)")
fig, ejes = plt.subplots(2, 3, figsize=(15, 9))
for eje, tipo in zip(ejes.ravel(), ["spring", "kamada_kawai", "circular",
                                    "shell", "spectral", "random"]):
    t0 = time.perf_counter()
    pos = calcular_layout(G_vis, tipo)
    dt = time.perf_counter() - t0
    dibujar_grafo(G_vis, pos=pos, color_por="clase", eje=eje,
                  titulo_fig=f"{tipo}  ({dt*1000:.0f} ms)",
                  paleta={"ilicito": COLORES["ilicito"], "licito": COLORES["licito"],
                          "desconocido": COLORES["desconocido"]})
plt.tight_layout()
plt.show()

print("""
· spring y kamada_kawai son los únicos que revelan ESTRUCTURA: agrupan lo que está conectado.
· circular y shell no la revelan, pero permiten comparar densidades de forma sistemática.
· spectral separa componentes con nitidez y colapsa el interior de cada una.
· random es el control: si tu gráfico se parece a este, el layout no está aportando nada.
""")

In [ ]:
# =============================================================================
# M17.2 — El grafo bancario heterogéneo, con codificación por tipo
# =============================================================================

# Vista de un cliente con fraude inyectado: su red ego a dos saltos.
cliente_fraude = next(
    (n for n, d in G_banco.nodes(data=True)
     if d["tipo"] == "cliente" and d.get("patron_fraude") == "device_sharing"),
    next(n for n, d in G_banco.nodes(data=True) if d["tipo"] == "cliente"))

ego_banco = nx.ego_graph(nx.Graph(G_banco), cliente_fraude, radius=2)
if ego_banco.number_of_nodes() > 220:
    ego_banco = nx.ego_graph(nx.Graph(G_banco), cliente_fraude, radius=1)

PALETA_TIPOS = {
    "cliente": "#2E86AB", "cuenta": "#2A9D8F", "tarjeta": "#F18F01",
    "comercio": "#8E5572", "ip": "#D7263D", "dispositivo": "#6A4C93",
    "telefono": "#457B9D", "direccion": "#A8DADC", "atm": "#E9C46A",
    "agencia": "#264653", "transferencia": "#B0B7BE", "compra": "#CBD5E0",
}

titulo(f"Red ego de {cliente_fraude} (patrón: {G_banco.nodes[cliente_fraude]['patron_fraude']})")
fig, ejes = plt.subplots(1, 2, figsize=(15, 6.5))

pos_banco = dibujar_grafo(ego_banco, layout="spring", color_por="tipo", eje=ejes[0],
                          paleta=PALETA_TIPOS,
                          titulo_fig=f"Coloreado por TIPO de entidad ({ego_banco.number_of_nodes()} nodos)")

# El mismo grafo, coloreado por patrón de fraude en vez de por tipo.
paleta_fraude = {"money_mule": "#D7263D", "device_sharing": "#F18F01",
                 "card_testing": "#8E5572", "account_takeover": "#6A4C93",
                 "identidad_sintetica": "#2A9D8F", "aml_layering": "#E76F51", None: "#DDE3EA"}
dibujar_grafo(ego_banco, pos=pos_banco, color_por="patron_fraude", eje=ejes[1],
              paleta=paleta_fraude, titulo_fig="El mismo grafo, coloreado por PATRÓN de fraude")

manijas = [plt.Line2D([], [], marker="o", ls="", color=c, label=t, markersize=7)
           for t, c in PALETA_TIPOS.items()
           if t in {G_banco.nodes[n]["tipo"] for n in ego_banco.nodes()}]
ejes[0].legend(handles=manijas, fontsize=7.5, loc="upper left", ncol=2, frameon=True)
plt.tight_layout()
plt.show()

# Vista bipartita: clientes contra los recursos que comparten.
recursos = [n for n, d in G_banco.nodes(data=True)
            if d["tipo"] in ("dispositivo", "ip") and
            sum(1 for u in G_banco.predecessors(n) if G_banco.nodes[u]["tipo"] == "cliente") >= 4]
if recursos:
    clientes_vinculados = {u for r in recursos for u in G_banco.predecessors(r)
                           if G_banco.nodes[u]["tipo"] == "cliente"}
    B = nx.Graph()
    for r in recursos:
        for u in G_banco.predecessors(r):
            if G_banco.nodes[u]["tipo"] == "cliente":
                B.add_node(u, capa=0, tipo="cliente")
                B.add_node(r, capa=1, tipo=G_banco.nodes[r]["tipo"])
                B.add_edge(u, r)

    plt.figure(figsize=(12, 6))
    pos_b = nx.multipartite_layout(B, subset_key="capa")
    colores_b = [PALETA_TIPOS[B.nodes[n]["tipo"]] for n in B.nodes()]
    nx.draw_networkx_edges(B, pos_b, edge_color="#C7CDD4", width=0.8)
    nx.draw_networkx_nodes(B, pos_b, node_color=colores_b, node_size=110, edgecolors="white")
    nx.draw_networkx_labels(B, pos_b, font_size=5.5,
                            labels={n: n for n in B.nodes() if B.degree(n) >= 4})
    plt.title(f"Vista bipartita: {len(clientes_vinculados)} clientes compartiendo "
              f"{len(recursos)} dispositivos/IPs — los anillos saltan a la vista")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# M17.3 — Visualización interactiva: Plotly y PyVis
# =============================================================================

def grafo_plotly(G: nx.Graph, pos: dict | None = None, color_por: str = "tipo",
                 titulo_fig: str = "", paleta: dict | None = None, seed: int = SEED):
    """Grafo interactivo con Plotly: zoom, desplazamiento y tooltip por nodo.

    Las aristas se dibujan como UNA sola traza con None entre segmentos. Es el truco
    clave: una traza por arista haría que un grafo de mil aristas tardara segundos en
    renderizarse en el navegador.
    """
    if not PLOTLY_OK:
        print("Plotly no disponible (`pip install plotly`). Se omite la vista interactiva.")
        return None
    import plotly.graph_objects as go

    pos = pos or nx.spring_layout(G, seed=seed)
    x_aristas, y_aristas = [], []
    for u, v in G.edges():
        x_aristas += [pos[u][0], pos[v][0], None]
        y_aristas += [pos[u][1], pos[v][1], None]

    valores = [str(G.nodes[n].get(color_por, "—")) for n in G.nodes()]
    distintos = sorted(set(valores))
    paleta = paleta or {v: c for v, c in zip(
        distintos, ["#2E86AB", "#D7263D", "#F18F01", "#2A9D8F", "#6A4C93",
                    "#8E5572", "#457B9D", "#E9C46A", "#264653", "#B0B7BE",
                    "#A8DADC", "#CBD5E0"] * 3)}

    texto = [f"<b>{n}</b><br>{color_por}: {G.nodes[n].get(color_por, '—')}"
             f"<br>grado: {G.degree(n)}" for n in G.nodes()]

    figura = go.Figure()
    figura.add_trace(go.Scatter(x=x_aristas, y=y_aristas, mode="lines",
                                line=dict(width=0.6, color="#CDD3DA"),
                                hoverinfo="none", showlegend=False))
    figura.add_trace(go.Scatter(
        x=[pos[n][0] for n in G.nodes()], y=[pos[n][1] for n in G.nodes()],
        mode="markers", text=texto, hoverinfo="text",
        marker=dict(size=9, color=[paleta.get(v, "#999") for v in valores],
                    line=dict(width=1, color="white")), showlegend=False))
    figura.update_layout(title=titulo_fig, height=560, hovermode="closest",
                         xaxis=dict(visible=False), yaxis=dict(visible=False),
                         plot_bgcolor="rgba(0,0,0,0)", margin=dict(l=10, r=10, t=45, b=10))
    return figura


def grafo_pyvis(G: nx.Graph, ruta: str = "grafo_interactivo.html",
                color_por: str = "tipo", paleta: dict | None = None):
    """Genera un HTML autónomo con física interactiva (PyVis / vis.js).

    Es el formato más práctico para pasarle un caso a un investigador que no programa:
    un único fichero que se abre en cualquier navegador, con nodos arrastrables.
    """
    if not PYVIS_OK:
        print("PyVis no disponible (`pip install pyvis`). Se omite.")
        return None
    from pyvis.network import Network

    red = Network(height="600px", width="100%", directed=G.is_directed(),
                  bgcolor="#ffffff", font_color="#222")
    red.barnes_hut(gravity=-8000, spring_length=120)
    paleta = paleta or PALETA_TIPOS
    for n, d in G.nodes(data=True):
        red.add_node(str(n), label=str(n), color=paleta.get(d.get(color_por), "#B0B7BE"),
                     title=f"{color_por}: {d.get(color_por)}<br>grado: {G.degree(n)}")
    for u, v, d in G.edges(data=True):
        red.add_edge(str(u), str(v), title=str(d.get("rel", "")))
    red.save_graph(ruta)
    print(f"  Grafo interactivo guardado en: {ruta}")
    return ruta


titulo("Vistas interactivas")
figura = grafo_plotly(ego_banco, pos=pos_banco, color_por="tipo",
                      titulo_fig="Red ego bancaria — pasa el ratón sobre un nodo",
                      paleta=PALETA_TIPOS)
if figura is not None:
    figura.show()

grafo_pyvis(ego_banco, ruta="grafo_banco_interactivo.html")

In [ ]:
# =============================================================================
# M17.4 — Exportación a Gephi, Cytoscape y otras herramientas
# =============================================================================

def limpiar_para_exportar(G: nx.Graph) -> nx.Graph:
    """Prepara un grafo para exportarlo: GEXF y GraphML no admiten valores None.

    Es el error más habitual al exportar desde NetworkX: `None` en un atributo hace
    fallar la serialización con un mensaje poco claro. Aquí se convierten a cadena
    vacía y los tipos de NumPy a tipos nativos de Python, que tampoco se serializan.
    """
    H = G.copy()
    for _, datos in H.nodes(data=True):
        for clave, valor in list(datos.items()):
            if valor is None:
                datos[clave] = ""
            elif isinstance(valor, (np.integer,)):
                datos[clave] = int(valor)
            elif isinstance(valor, (np.floating,)):
                datos[clave] = float(valor)
    for _, _, datos in H.edges(data=True):
        for clave, valor in list(datos.items()):
            if valor is None:
                datos[clave] = ""
            elif isinstance(valor, (np.integer,)):
                datos[clave] = int(valor)
            elif isinstance(valor, (np.floating,)):
                datos[clave] = float(valor)
    return H


def exportar_grafo(G: nx.Graph, base: str = "grafo", formatos=("gexf", "graphml", "cytoscape")) -> dict:
    """Exporta un grafo a los formatos que consumen las herramientas de escritorio.

    · GEXF      → Gephi. Conserva atributos y admite dinámica temporal.
    · GraphML   → estándar amplio: Cytoscape, yEd, igraph.
    · Cytoscape → JSON nativo de Cytoscape.js, también apto para la web.
    """
    H = limpiar_para_exportar(G)
    salidas = {}
    if "gexf" in formatos:
        nx.write_gexf(H, f"{base}.gexf")
        salidas["gexf"] = f"{base}.gexf"
    if "graphml" in formatos:
        nx.write_graphml(H, f"{base}.graphml")
        salidas["graphml"] = f"{base}.graphml"
    if "cytoscape" in formatos:
        with open(f"{base}_cytoscape.json", "w", encoding="utf-8") as f:
            json.dump(nx.cytoscape_data(H), f, ensure_ascii=False, indent=1)
        salidas["cytoscape"] = f"{base}_cytoscape.json"
    return salidas


titulo("Exportación para análisis visual externo")
archivos = exportar_grafo(ego_banco, base="ego_banco")
for formato, ruta in archivos.items():
    tam = os.path.getsize(ruta) / 1024
    print(f"  {formato:10s} → {ruta:32s} ({tam:.1f} KB)")

# Grafo AGREGADO de comunidades: la alternativa correcta al hairball.
def grafo_de_comunidades(G: nx.Graph, particion: dict, min_tam: int = 5) -> nx.Graph:
    """Colapsa cada comunidad en un solo nodo. Un grafo de 200 mil nodos cabe en 300.

    Tamaño del nodo = número de miembros; grosor de la arista = nº de conexiones entre
    las dos comunidades. Es la única forma de ver una red grande ENTERA sin mentir.
    """
    C = nx.Graph()
    tam = Counter(particion.values())
    for com, n in tam.items():
        if n >= min_tam:
            C.add_node(com, miembros=n, capa=0)
    entre = Counter()
    for u, v in G.edges():
        cu, cv = particion.get(u), particion.get(v)
        if cu is not None and cv is not None and cu != cv and cu in C and cv in C:
            entre[tuple(sorted((cu, cv)))] += 1
    for (a, b), peso in entre.items():
        C.add_edge(a, b, peso=peso)
    return C


C_com = grafo_de_comunidades(G_und, louvain_global["particion"], min_tam=50)
print(f"\nGrafo agregado de comunidades: {C_com.number_of_nodes()} nodos "
      f"(desde {G.number_of_nodes():,}), {C_com.number_of_edges()} aristas")

if C_com.number_of_nodes():
    plt.figure(figsize=(10, 6.5))
    pos_c = nx.spring_layout(C_com, seed=SEED, k=0.6)
    tam_c = [12 + C_com.nodes[n]["miembros"] / 6 for n in C_com.nodes()]
    nx.draw_networkx_edges(C_com, pos_c, edge_color="#C7CDD4",
                           width=[0.4 + C_com[u][v]["peso"] / 40 for u, v in C_com.edges()])
    nx.draw_networkx_nodes(C_com, pos_c, node_size=tam_c, node_color=COLORES["licito"],
                           edgecolors="white", linewidths=0.8)
    plt.title("Grafo agregado de comunidades — el tamaño es el nº de transacciones")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

print("""
Este grafo tiene la misma información topológica de alto nivel que los 203.769 nodos
originales y sí se puede leer. Al no haber aristas entre timesteps, las comunidades quedan
agrupadas por snapshot: se ven 49 islas, cada una con sus subcomunidades. La estructura del
dataset queda visible de un vistazo, cosa que el hairball completo jamás habría mostrado.""")

---
---

# Módulo 18 — Grafos para fraude bancario

Modelado de entidades. Este módulo formaliza el esquema del grafo generado en el Módulo 1 y
muestra las operaciones específicas de un grafo **heterogéneo**: estadísticas por tipo,
proyecciones bipartitas y conteo de meta-caminos.

## Catálogo de nodos

| Entidad | Qué representa | Por qué es un nodo y no un atributo |
|---|---|---|
| **Cliente** | La persona o empresa titular | Es el sujeto de casi toda investigación |
| **Cuenta** | Un producto de depósito | Un cliente tiene varias; el dinero fluye entre cuentas, no entre clientes |
| **Tarjeta** | Un instrumento de pago | Se compromete de forma independiente de la cuenta |
| **Comercio** | El punto de venta | Concentra fraude por sector y por comercio concreto |
| **IP** | Origen de la conexión | **Conecta clientes que se declaran independientes** |
| **Dispositivo** | Móvil, navegador, huella digital | Idem, y es más estable que la IP |
| **Teléfono** | Número de contacto | Compartirlo entre titulares distintos es señal fuerte |
| **Dirección** | Domicilio declarado | Idem; base de la identidad sintética |
| **ATM** | Cajero automático | Sitúa la operación geográficamente |
| **Agencia** | Oficina | Permite agregar riesgo por sucursal |
| **Transferencia** | Un envío de dinero | **Reificada**: lleva importe, canal y timestamp propios |
| **Compra** | Una operación con tarjeta | Reificada por las mismas razones |

La columna de la derecha es la que importa. Las cuatro entidades de identificador compartido
—IP, dispositivo, teléfono, dirección— son las que **crean caminos entre clientes sin relación
declarada**. Si se modelan como columnas de una tabla de clientes, esos caminos no existen y
ningún algoritmo los va a encontrar. Modelar es decidir qué preguntas serán respondibles.

## Catálogo de relaciones

| Relación | Origen → Destino | Dirigida | Múltiple |
|---|---|---|---|
| `posee` | Cliente → Cuenta / Tarjeta | Sí | No |
| `emite` | Cuenta → Transferencia | Sí | Sí |
| `recibe` | Transferencia → Cuenta | Sí | Sí |
| `realiza` | Tarjeta → Compra | Sí | Sí |
| `en` | Compra → Comercio | Sí | Sí |
| `utiliza` | Cliente → Dispositivo / IP / Teléfono | Sí | Sí (con timestamp) |
| `reside_en` | Cliente → Dirección | Sí | No |
| `opera_en` | Cuenta → ATM | Sí | Sí |
| `pertenece_a` | ATM → Agencia | Sí | No |

## Meta-caminos

En un grafo heterogéneo, un **meta-camino** es una secuencia de tipos de nodo y relación. Es la
unidad semántica de consulta:

- `Cliente -posee-> Tarjeta -realiza-> Compra -en-> Comercio` → *"dónde compra un cliente"*
- `Cliente -utiliza-> Dispositivo <-utiliza- Cliente` → *"clientes que comparten dispositivo"*
- `Cuenta -emite-> Transferencia -recibe-> Cuenta` → *"flujo de dinero"*

Contar instancias de un meta-camino entre dos entidades es la forma natural de medir su
proximidad en un grafo heterogéneo, y es también lo que aprenden implícitamente los modelos de
Heterogeneous GNN (HAN, HGT) del Módulo 23.

In [ ]:
# =============================================================================
# M18.1 — Anatomía del grafo heterogéneo
# =============================================================================

def estadisticas_por_tipo(G: nx.MultiDiGraph) -> tuple:
    """Descompone las métricas del grafo por tipo de nodo y por tipo de relación.

    En un grafo heterogéneo, el "grado medio" global no significa nada: mezcla el grado
    de un cliente (que tiene decenas de vínculos) con el de una compra (que siempre
    tiene exactamente dos). Hay que desagregar por tipo o no medir.
    """
    filas_nodo = []
    for tipo in sorted({d["tipo"] for _, d in G.nodes(data=True)}):
        nodos = [n for n, d in G.nodes(data=True) if d["tipo"] == tipo]
        gin = [G.in_degree(n) for n in nodos]
        gout = [G.out_degree(n) for n in nodos]
        filas_nodo.append({
            "tipo de nodo": tipo,
            "cantidad": len(nodos),
            "in-degree medio": round(float(np.mean(gin)), 2),
            "out-degree medio": round(float(np.mean(gout)), 2),
            "grado máximo": int(max(max(gin, default=0), max(gout, default=0))),
            "implicados en fraude": sum(1 for n in nodos if G.nodes[n].get("patron_fraude")),
        })

    filas_rel = []
    for rel in sorted({d["rel"] for _, _, d in G.edges(data=True)}):
        aristas = [(u, v) for u, v, d in G.edges(data=True) if d["rel"] == rel]
        pares_tipo = Counter((G.nodes[u]["tipo"], G.nodes[v]["tipo"]) for u, v in aristas)
        origen, destino = pares_tipo.most_common(1)[0][0]
        filas_rel.append({
            "relación": rel,
            "cantidad": len(aristas),
            "origen → destino": f"{origen} → {destino}",
            "tipos distintos que conecta": len(pares_tipo),
        })

    return (pd.DataFrame(filas_nodo).set_index("tipo de nodo"),
            pd.DataFrame(filas_rel).set_index("relación"))


titulo("Anatomía del grafo bancario")
tipos_nodo, tipos_rel = estadisticas_por_tipo(G_banco)
display(tipos_nodo)
display(tipos_rel)

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.2))
tipos_nodo["cantidad"].sort_values().plot(
    kind="barh", ax=ejes[0], color=[PALETA_TIPOS.get(t, "#999") for t in
                                    tipos_nodo["cantidad"].sort_values().index])
ejes[0].set_title("Nodos por tipo de entidad")
ejes[0].set_xlabel("Cantidad")
tipos_rel["cantidad"].sort_values().plot(kind="barh", ax=ejes[1], color=COLORES["neutro"])
ejes[1].set_title("Aristas por tipo de relación")
ejes[1].set_xlabel("Cantidad")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# M18.2 — Meta-caminos
# =============================================================================

def contar_metapath(G: nx.MultiDiGraph, metapath: list, limite: int = 200_000,
                    devolver_caminos: bool = False):
    """Enumera instancias de un meta-camino en un grafo heterogéneo.

    `metapath` alterna tipo y relación:
        ["cliente", "posee", "tarjeta", "realiza", "compra", "en", "comercio"]

    Se recorre nivel a nivel, expandiendo solo las aristas cuyo `rel` y cuyo tipo de
    destino coinciden con el patrón. El número de instancias crece de forma
    multiplicativa con la longitud, así que `limite` corta la expansión: sin él, un
    meta-camino de cuatro saltos sobre un grafo mediano agota la memoria.
    """
    tipos, relaciones = metapath[::2], metapath[1::2]
    caminos = [[n] for n, d in G.nodes(data=True) if d["tipo"] == tipos[0]]

    for i, rel in enumerate(relaciones):
        siguientes = []
        for camino in caminos:
            for _, v, datos in G.out_edges(camino[-1], data=True):
                if datos.get("rel") == rel and G.nodes[v]["tipo"] == tipos[i + 1]:
                    siguientes.append(camino + [v])
                    if len(siguientes) >= limite:
                        break
            if len(siguientes) >= limite:
                break
        caminos = siguientes
        if not caminos:
            break

    return caminos if devolver_caminos else len(caminos)


def proyeccion_por_metapath(G: nx.MultiDiGraph, metapath: list, limite: int = 200_000) -> nx.Graph:
    """Construye un grafo bipartito entre los extremos de un meta-camino.

    El peso de la arista es el número de instancias del meta-camino que conectan ese
    par. Es la forma correcta de proyectar un grafo heterogéneo: la relación resultante
    tiene un significado explícito, definido por el meta-camino que la generó.
    """
    caminos = contar_metapath(G, metapath, limite=limite, devolver_caminos=True)
    P = nx.Graph()
    for camino in caminos:
        origen, destino = camino[0], camino[-1]
        if P.has_edge(origen, destino):
            P[origen][destino]["peso"] += 1
        else:
            P.add_edge(origen, destino, peso=1)
            P.nodes[origen]["capa"] = 0
            P.nodes[destino]["capa"] = 1
    return P


titulo("Meta-caminos en el grafo bancario")
METAPATHS = {
    "Cliente → Cuenta": ["cliente", "posee", "cuenta"],
    "Cliente → Tarjeta": ["cliente", "posee", "tarjeta"],
    "Cliente → Dispositivo": ["cliente", "utiliza", "dispositivo"],
    "Cuenta → Transferencia → Cuenta": ["cuenta", "emite", "transferencia", "recibe", "cuenta"],
    "Tarjeta → Compra → Comercio": ["tarjeta", "realiza", "compra", "en", "comercio"],
    "Cliente → Tarjeta → Compra → Comercio":
        ["cliente", "posee", "tarjeta", "realiza", "compra", "en", "comercio"],
}
filas = []
for nombre, mp in METAPATHS.items():
    t0 = time.perf_counter()
    n = contar_metapath(G_banco, mp)
    filas.append({"meta-camino": nombre, "longitud (saltos)": len(mp) // 2,
                  "instancias": n, "ms": round((time.perf_counter() - t0) * 1000, 1)})
display(pd.DataFrame(filas).set_index("meta-camino"))

In [ ]:
# =============================================================================
# M18.3 — Proyecciones bipartitas
# =============================================================================

titulo("Proyección Cliente ↔ Comercio a través de tarjeta y compra")
P_cc = proyeccion_por_metapath(
    G_banco, ["cliente", "posee", "tarjeta", "realiza", "compra", "en", "comercio"])
clientes_p = [n for n, d in P_cc.nodes(data=True) if d.get("capa") == 0]
comercios_p = [n for n, d in P_cc.nodes(data=True) if d.get("capa") == 1]
print(f"  Clientes  : {len(clientes_p):,}")
print(f"  Comercios : {len(comercios_p):,}")
print(f"  Aristas   : {P_cc.number_of_edges():,}")
print(f"  ¿Bipartito?: {nx.is_bipartite(P_cc)}")

if P_cc.number_of_edges():
    pesos = [d["peso"] for _, _, d in P_cc.edges(data=True)]
    print(f"  Compras por par cliente-comercio: media {np.mean(pesos):.2f}, máximo {max(pesos)}")

    # Comercios donde se concentran los clientes con fraude inyectado.
    riesgo = []
    for com in comercios_p:
        vecinos = list(P_cc.neighbors(com))
        sospechosos = sum(1 for c in vecinos if G_banco.nodes[c].get("patron_fraude"))
        if len(vecinos) >= 5:
            riesgo.append({
                "comercio": com,
                "categoría": G_banco.nodes[com].get("categoria"),
                "riesgo declarado": G_banco.nodes[com].get("riesgo"),
                "clientes": len(vecinos),
                "clientes marcados": sospechosos,
                "% marcados": round(100 * sospechosos / len(vecinos), 1),
            })
    df_riesgo = pd.DataFrame(riesgo).sort_values("% marcados", ascending=False)
    titulo("Comercios ordenados por concentración de clientes marcados", nivel=2)
    display(df_riesgo.head(10).set_index("comercio"))

# Proyección cliente-cliente por recursos compartidos (ya construida en el Módulo 4),
# ahora medida como red en sí misma.
titulo("Red Cliente ↔ Cliente por recurso compartido", nivel=2)
est_p = estadisticas_basicas(P_clientes, "Clientes por recurso compartido")
for k, v in est_p.items():
    if k != "grafo" and v is not None:
        print(f"  {k:26s} {formatear(v):>12s}")

grados_p = pd.Series(dict(P_clientes.degree()))
marcados = {n for n in P_clientes.nodes() if G_banco.nodes[n].get("patron_fraude")}
print(f"\n  Grado medio · clientes marcados : {grados_p[list(marcados)].mean():.2f}"
      if marcados else "")
print(f"  Grado medio · resto             : "
      f"{grados_p[[n for n in P_clientes.nodes() if n not in marcados]].mean():.2f}")
print("\n  Si los clientes marcados tienen un grado claramente superior en esta proyección,")
print("  compartir recursos es por sí solo una señal de riesgo — y basta un conteo de grado")
print("  para explotarla. El Módulo 19 convierte esa observación en detectores concretos.")

---
---

# Módulo 19 — Casos de uso

Seis tipologías de fraude, cada una con su **firma topológica** y su detector. Todas se evalúan
contra el ground truth inyectado en el Módulo 1, así que los números de precisión y recall son
reales, no estimaciones.

| Tipología | Qué es | Firma en el grafo | Detector |
|---|---|---|---|
| **Money mule** | Cuentas que reciben fondos de terceros y los reenvían | Fan-in de muchos orígenes + fan-out inmediato de casi todo el importe | M19.1 |
| **AML / Layering** | Capas de transferencias para romper la trazabilidad | Ciclo dirigido con importes casi constantes | M19.2 |
| **Device sharing** | Varias identidades operadas desde el mismo sitio | Muchos clientes colgando de un dispositivo o IP | M19.3 |
| **Identidad sintética** | Identidades fabricadas con datos reales mezclados | Clientes distintos que comparten teléfono y dirección | M19.4 |
| **Account takeover** | Toma de control de una cuenta legítima | Dispositivo e IP nuevos + transferencia grande inmediata | M19.5 |
| **Card testing** | Validar tarjetas robadas con micro-compras | Una tarjeta, decenas de compras diminutas, muchos comercios, pocas horas | M19.6 |

## Chargeback y friendly fraud: por qué no hay detector aquí

**Friendly fraud** (el titular legítimo niega una compra que sí hizo) y **chargeback fraud**
(abuso sistemático del proceso de devolución) no tienen firma topológica en el grafo de
transacciones. Su señal está en el **historial de disputas**, que aquí no se modela.

Modelarlas requiere añadir un nodo `Disputa` conectado a `Compra` y a `Cliente`, y entonces sí
aparece la firma: un cliente con muchas disputas sobre comercios sin ninguna otra queja, o un
patrón temporal de compra→disputa demasiado regular. **Lo que no está en el modelo no se puede
detectar** — y esa es la lección más importante del Módulo 1 aplicada aquí.

## Una advertencia sobre estos números

El ground truth es sintético, así que la precisión y el recall que se miden abajo son
**optimistas frente a un despliegue real**: los patrones inyectados son más limpios que los
reales. Su valor está en la comparación relativa —qué detector recupera más, cuál genera más
falsos positivos— y en verificar que cada detector encuentra lo que dice buscar.

In [ ]:
# =============================================================================
# M19.1 — Money mule
# =============================================================================

DIA = 86400


def detectar_money_mule(G_trf: nx.MultiDiGraph, min_origenes: int = 7,
                        ventana_dias: int = 7, ratio_reenvio: float = 0.65,
                        dias_reenvio: int = 10) -> pd.DataFrame:
    """Cuentas que concentran fondos de muchos orígenes y los reenvían casi íntegros.

    La firma tiene tres componentes y las tres son necesarias:
      1. FAN-IN: ≥ `min_origenes` cuentas distintas ingresan en una ventana de
         `ventana_dias` días. Solo con esto, cualquier nómina sería una mula.
      2. REENVÍO: sale al menos `ratio_reenvio` del importe recibido. Es lo que
         distingue a una cuenta que ACUMULA de una que hace de TRÁNSITO.
      3. INMEDIATEZ: el reenvío ocurre en pocos días. Una mula no invierte el dinero.

    La ventana se busca deslizando sobre los ingresos ordenados por timestamp, no
    partiendo el tiempo en bloques fijos: un patrón a caballo entre dos bloques fijos
    pasaría desapercibido.
    """
    ventana = ventana_dias * DIA
    filas = []

    for cuenta in G_trf.nodes():
        entrantes = sorted(
            [(u, d["monto"], d["ts"]) for u, _, d in G_trf.in_edges(cuenta, data=True)],
            key=lambda x: x[2])
        if len(entrantes) < min_origenes:
            continue

        salientes = [(v, d["monto"], d["ts"]) for _, v, d in G_trf.out_edges(cuenta, data=True)]

        mejor = None
        inicio = 0
        for fin in range(len(entrantes)):
            while entrantes[fin][2] - entrantes[inicio][2] > ventana:
                inicio += 1
            bloque = entrantes[inicio:fin + 1]
            origenes = {u for u, _, _ in bloque}
            if len(origenes) >= min_origenes:
                total = sum(m for _, m, _ in bloque)
                if mejor is None or total > mejor["recibido"]:
                    mejor = {"recibido": total, "origenes": len(origenes),
                             "t_ini": bloque[0][2], "t_fin": bloque[-1][2]}

        if mejor is None:
            continue

        reenviado = sum(m for _, m, t in salientes
                        if mejor["t_ini"] <= t <= mejor["t_fin"] + dias_reenvio * DIA)
        ratio = reenviado / mejor["recibido"] if mejor["recibido"] else 0.0
        if ratio >= ratio_reenvio:
            destinos = {v for v, _, t in salientes
                        if mejor["t_ini"] <= t <= mejor["t_fin"] + dias_reenvio * DIA}
            filas.append({
                "cuenta": cuenta,
                "orígenes distintos": mejor["origenes"],
                "recibido": round(mejor["recibido"], 2),
                "reenviado": round(reenviado, 2),
                "ratio de reenvío": round(ratio, 3),
                "destinos": len(destinos),
                "días de la ventana": round((mejor["t_fin"] - mejor["t_ini"]) / DIA, 1),
            })

    return pd.DataFrame(filas).sort_values("orígenes distintos", ascending=False) \
        if filas else pd.DataFrame()


titulo("Detector de money mule")
mulas = detectar_money_mule(G_trf)
gt_mulas = {n for n in GT_BANCO.get("money_mule", set()) if G_banco.nodes[n]["tipo"] == "cuenta"}
if len(mulas):
    mulas["¿en ground truth?"] = mulas["cuenta"].isin(gt_mulas)
    display(mulas.head(12).set_index("cuenta"))
    aciertos = set(mulas["cuenta"]) & gt_mulas
    print(f"  Detectadas: {len(mulas)} · aciertos: {len(aciertos)} · "
          f"cuentas mula reales: {len(gt_mulas)}")
else:
    print("  Ninguna cuenta cumple las tres condiciones. Prueba a relajar `min_origenes`.")

In [ ]:
# =============================================================================
# M19.2 — AML / Layering
# =============================================================================

def detectar_layering_aml(G_trf: nx.MultiDiGraph, longitud_max: int = 7,
                          cv_max: float = 0.25, limite_ciclos: int = 600) -> pd.DataFrame:
    """Ciclos de transferencias con importes casi constantes: la firma del layering.

    Un ciclo dirigido por sí solo no basta —entre cuentas activas aparecen por azar—.
    Lo que lo delata es que **el importe apenas varía en cada salto**: es el mismo
    dinero dando vueltas, menos la comisión. Se mide con el coeficiente de variación
    (desviación / media) de los importes del ciclo; por debajo de `cv_max` el ciclo es
    sospechoso.

    Se comprueba además la MONOTONÍA temporal: en un layering real los saltos ocurren
    en orden, no de forma desordenada.
    """
    simple = nx.DiGraph(G_trf)
    ciclos = detectar_ciclos(simple, longitud_max=longitud_max, limite=limite_ciclos)

    filas = []
    for ciclo in ciclos:
        pares = list(zip(ciclo, ciclo[1:] + ciclo[:1]))
        montos, tiempos = [], []
        for u, v in pares:
            if G_trf.has_edge(u, v):
                datos = list(G_trf[u][v].values())
                mejor = max(datos, key=lambda d: d.get("monto", 0))
                montos.append(mejor.get("monto", 0.0))
                tiempos.append(mejor.get("ts", 0))
        if len(montos) < 3 or np.mean(montos) == 0:
            continue
        cv = float(np.std(montos) / np.mean(montos))
        if cv <= cv_max:
            filas.append({
                "longitud": len(ciclo),
                "cuentas": ciclo,
                "monto medio": round(float(np.mean(montos)), 2),
                "CV de importes": round(cv, 4),
                "ordenado en el tiempo": bool(np.all(np.diff(tiempos) > 0)) if len(tiempos) > 1 else False,
                "días que dura": round((max(tiempos) - min(tiempos)) / DIA, 1) if tiempos else 0,
            })
    return pd.DataFrame(filas).sort_values("CV de importes") if filas else pd.DataFrame()


titulo("Detector de layering (AML)")
layering = detectar_layering_aml(G_trf)
gt_layering = {n for n in GT_BANCO.get("aml_layering", set()) if G_banco.nodes[n]["tipo"] == "cuenta"}
if len(layering):
    layering["cuentas en GT"] = layering["cuentas"].apply(lambda c: len(set(c) & gt_layering))
    vista = layering.copy()
    vista["cuentas"] = vista["cuentas"].apply(
        lambda c: " → ".join(x.replace("CTA_", "") for x in c[:5]) + ("…" if len(c) > 5 else ""))
    display(vista.head(10).reset_index(drop=True))
    detectadas_lay = set().union(*[set(c) for c in layering["cuentas"]])
    print(f"  Ciclos con CV ≤ 0,25: {len(layering)} · cuentas implicadas: {len(detectadas_lay)}")
else:
    detectadas_lay = set()
    print("  Ningún ciclo con importes suficientemente constantes.")

In [ ]:
# =============================================================================
# M19.3 y M19.4 — Device sharing e identidad sintética
# =============================================================================

def detectar_device_sharing(G: nx.MultiDiGraph, umbral: int = 4,
                            tipos=("dispositivo", "ip")) -> pd.DataFrame:
    """Recursos técnicos usados por un número anómalo de clientes distintos.

    Compartir un dispositivo entre dos o tres titulares es normal (familia, empresa).
    Por encima de `umbral` deja de serlo. El detector devuelve el recurso, sus usuarios
    y si esos usuarios tienen alguna otra relación entre ellos: un anillo de fraude
    comparte el dispositivo y NADA más, mientras que una familia real comparte también
    dirección o transferencias.
    """
    filas = []
    for recurso, datos in G.nodes(data=True):
        if datos["tipo"] not in tipos:
            continue
        clientes = sorted({u for u in G.predecessors(recurso)
                           if G.nodes[u]["tipo"] == "cliente"})
        if len(clientes) < umbral:
            continue
        # ¿Comparten los usuarios alguna OTRA cosa además de este recurso?
        otros = set()
        for c in clientes:
            for _, v, d in G.out_edges(c, data=True):
                if v != recurso and G.nodes[v]["tipo"] in ("direccion", "telefono"):
                    otros.add(v)
        filas.append({
            "recurso": recurso,
            "tipo": datos["tipo"],
            "clientes": len(clientes),
            "otros vínculos compartidos": len(otros) < len(clientes),
            "país/SO": datos.get("pais") or datos.get("so"),
            "_clientes": clientes,
        })
    return pd.DataFrame(filas).sort_values("clientes", ascending=False) if filas else pd.DataFrame()


def detectar_identidad_sintetica(G: nx.MultiDiGraph, min_clientes: int = 2) -> pd.DataFrame:
    """Clientes distintos que comparten identificadores que deberían ser únicos.

    Un teléfono o una dirección compartidos por varios titulares es el indicio
    canónico de identidad sintética: los defraudadores reutilizan datos reales
    (dirección de un inmueble, número de prepago) para dar de alta identidades
    fabricadas. A diferencia del dispositivo, aquí el umbral es bajo — dos titulares
    con el mismo móvil declarado ya merece revisión.
    """
    filas = []
    for recurso, datos in G.nodes(data=True):
        if datos["tipo"] not in ("telefono", "direccion"):
            continue
        clientes = sorted({u for u in G.predecessors(recurso)
                           if G.nodes[u]["tipo"] == "cliente"})
        if len(clientes) >= min_clientes:
            antiguedades = [G.nodes[c]["antiguedad_dias"] for c in clientes]
            filas.append({
                "identificador": recurso,
                "tipo": datos["tipo"],
                "titulares": len(clientes),
                "antigüedad media (días)": int(np.mean(antiguedades)),
                "antigüedad mínima": int(min(antiguedades)),
                "_clientes": clientes,
            })
    return pd.DataFrame(filas).sort_values("titulares", ascending=False) if filas else pd.DataFrame()


titulo("Detector de device sharing")
sharing = detectar_device_sharing(G_banco, umbral=4)
gt_sharing = {n for n in GT_BANCO.get("device_sharing", set())
              if G_banco.nodes[n]["tipo"] == "cliente"}
if len(sharing):
    display(sharing.drop(columns=["_clientes"]).head(8).set_index("recurso"))
    clientes_sharing = set().union(*sharing["_clientes"])
    print(f"  Recursos señalados: {len(sharing)} · clientes implicados: {len(clientes_sharing)}")
else:
    clientes_sharing = set()
    print("  Ningún recurso supera el umbral.")

titulo("Detector de identidad sintética")
sinteticas = detectar_identidad_sintetica(G_banco)
gt_sinteticas = {n for n in GT_BANCO.get("identidad_sintetica", set())
                 if G_banco.nodes[n]["tipo"] == "cliente"}
if len(sinteticas):
    display(sinteticas.drop(columns=["_clientes"]).head(8).set_index("identificador"))
    clientes_sinteticos = set().union(*sinteticas["_clientes"])
    print(f"  Identificadores compartidos: {len(sinteticas)} · "
          f"clientes implicados: {len(clientes_sinteticos)}")
else:
    clientes_sinteticos = set()
    print("  Ningún teléfono ni dirección compartido.")

In [ ]:
# =============================================================================
# M19.5 y M19.6 — Account takeover y card testing
# =============================================================================

def detectar_account_takeover(G: nx.MultiDiGraph, percentil_reciente: int = 70,
                              horas_max: int = 48, percentil_monto: int = 95) -> pd.DataFrame:
    """Dispositivo/IP nuevos en una cuenta antigua, seguidos de una salida grande.

    La firma del ATO es una RUPTURA con el comportamiento previo, no un valor extremo
    aislado. Se busca la conjunción de tres cosas en una ventana corta:
      · un dispositivo o IP visto por primera vez (y solo por este cliente),
      · una transferencia de salida por encima del percentil `percentil_monto`,
      · menos de `horas_max` entre lo uno y lo otro.

    Ninguna de las tres es sospechosa por separado. Las tres juntas, sí. Los detectores
    de fraude útiles casi siempre tienen esta forma conjuntiva.
    """
    ts_todos = [d["ts"] for _, _, d in G.edges(data=True) if "ts" in d]
    montos = [d["monto"] for _, d in G.nodes(data=True)
              if d["tipo"] == "transferencia" and "monto" in d]
    if not ts_todos or not montos:
        return pd.DataFrame()
    umbral_t = float(np.percentile(ts_todos, percentil_reciente))
    umbral_m = float(np.percentile(montos, percentil_monto))

    filas = []
    for cliente, datos in G.nodes(data=True):
        if datos["tipo"] != "cliente":
            continue
        nuevos = [(v, d["ts"]) for _, v, d in G.out_edges(cliente, data=True)
                  if d.get("rel") == "utiliza" and "ts" in d and d["ts"] >= umbral_t
                  and G.nodes[v]["tipo"] in ("dispositivo", "ip")
                  and sum(1 for u in G.predecessors(v) if G.nodes[u]["tipo"] == "cliente") == 1]
        if not nuevos:
            continue
        t_nuevo = min(t for _, t in nuevos)

        cuentas = [v for _, v, d in G.out_edges(cliente, data=True)
                   if d.get("rel") == "posee" and G.nodes[v]["tipo"] == "cuenta"]
        for cta in cuentas:
            for _, trf, d in G.out_edges(cta, data=True):
                if d.get("rel") != "emite" or G.nodes[trf]["tipo"] != "transferencia":
                    continue
                monto, t = G.nodes[trf]["monto"], G.nodes[trf]["ts"]
                if monto >= umbral_m and 0 <= t - t_nuevo <= horas_max * 3600:
                    filas.append({
                        "cliente": cliente,
                        "antigüedad (días)": datos["antiguedad_dias"],
                        "recursos nuevos": len(nuevos),
                        "transferencia": trf,
                        "monto": round(monto, 2),
                        "horas tras el alta del recurso": round((t - t_nuevo) / 3600, 1),
                    })
    return pd.DataFrame(filas).sort_values("monto", ascending=False) if filas else pd.DataFrame()


def detectar_card_testing(G: nx.MultiDiGraph, min_compras: int = 12, horas_max: int = 24,
                          monto_max: float = 8.0, min_comercios: int = 6) -> pd.DataFrame:
    """Tarjetas con una ráfaga de micro-compras en muchos comercios distintos.

    Es el patrón de validación de tarjetas robadas: se prueban importes minúsculos
    —que suelen pasar sin autenticación reforzada— en muchos comercios para ver cuáles
    siguen activas. La diversidad de comercios es esencial: 30 micro-compras en el mismo
    sitio son una máquina de vending, no un ataque.
    """
    filas = []
    for tarjeta, datos in G.nodes(data=True):
        if datos["tipo"] != "tarjeta":
            continue
        compras = []
        for _, cmp_, d in G.out_edges(tarjeta, data=True):
            if d.get("rel") == "realiza" and G.nodes[cmp_]["tipo"] == "compra":
                comercios = [v for v in G.successors(cmp_) if G.nodes[v]["tipo"] == "comercio"]
                compras.append((G.nodes[cmp_]["ts"], G.nodes[cmp_]["monto"],
                                comercios[0] if comercios else None))
        if len(compras) < min_compras:
            continue
        compras.sort()
        ventana = horas_max * 3600
        inicio = 0
        for fin in range(len(compras)):
            while compras[fin][0] - compras[inicio][0] > ventana:
                inicio += 1
            bloque = compras[inicio:fin + 1]
            micro = [c for c in bloque if c[1] <= monto_max]
            comercios = {c[2] for c in micro if c[2]}
            if len(micro) >= min_compras and len(comercios) >= min_comercios:
                filas.append({
                    "tarjeta": tarjeta,
                    "micro-compras": len(micro),
                    "comercios distintos": len(comercios),
                    "monto medio": round(float(np.mean([c[1] for c in micro])), 2),
                    "horas": round((bloque[-1][0] - bloque[0][0]) / 3600, 1),
                    "límite de la tarjeta": datos.get("limite"),
                })
                break
    return pd.DataFrame(filas).sort_values("micro-compras", ascending=False) if filas else pd.DataFrame()


titulo("Detector de account takeover")
ato = detectar_account_takeover(G_banco)
gt_ato = {n for n in GT_BANCO.get("account_takeover", set()) if G_banco.nodes[n]["tipo"] == "cliente"}
if len(ato):
    ato["¿en ground truth?"] = ato["cliente"].isin(gt_ato)
    display(ato.head(10).set_index("cliente"))
    clientes_ato = set(ato["cliente"])
else:
    clientes_ato = set()
    print("  Ningún caso cumple las tres condiciones simultáneamente.")

titulo("Detector de card testing")
testing = detectar_card_testing(G_banco)
gt_testing = {n for n in GT_BANCO.get("card_testing", set()) if G_banco.nodes[n]["tipo"] == "tarjeta"}
if len(testing):
    testing["¿en ground truth?"] = testing["tarjeta"].isin(gt_testing)
    display(testing.head(10).set_index("tarjeta"))
    tarjetas_testing = set(testing["tarjeta"])
else:
    tarjetas_testing = set()
    print("  Ninguna tarjeta con ráfaga de micro-compras.")

In [ ]:
# =============================================================================
# M19.7 — Evaluación conjunta contra el ground truth
# =============================================================================

def evaluar_detectores(detecciones: dict, ground_truth: dict, G: nx.MultiDiGraph,
                       tipos_objetivo: dict) -> pd.DataFrame:
    """Precisión, recall y F1 de cada detector, restringidos a su tipo de entidad.

    Cada detector señala entidades de un tipo concreto (el de mulas señala cuentas, el
    de card testing señala tarjetas), así que el ground truth se filtra a ese mismo
    tipo antes de comparar. Sin ese filtro, el recall saldría artificialmente bajo
    porque el ground truth incluye también las transferencias y los recursos implicados.
    """
    filas = []
    for patron, detectados in detecciones.items():
        tipos = tipos_objetivo.get(patron, ())
        reales = {n for n in ground_truth.get(patron, set())
                  if G.nodes[n]["tipo"] in tipos}
        aciertos = detectados & reales
        precision = len(aciertos) / len(detectados) if detectados else 0.0
        recall = len(aciertos) / len(reales) if reales else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        filas.append({
            "patrón": patron,
            "tipo señalado": "/".join(tipos),
            "detectados": len(detectados),
            "reales": len(reales),
            "aciertos": len(aciertos),
            "precisión": round(precision, 3),
            "recall": round(recall, 3),
            "F1": round(f1, 3),
        })
    return pd.DataFrame(filas).set_index("patrón")


DETECCIONES = {
    "money_mule": set(mulas["cuenta"]) if len(mulas) else set(),
    "aml_layering": detectadas_lay,
    "device_sharing": clientes_sharing,
    "identidad_sintetica": clientes_sinteticos,
    "account_takeover": clientes_ato,
    "card_testing": tarjetas_testing,
}
TIPOS_OBJETIVO = {
    "money_mule": ("cuenta",),
    "aml_layering": ("cuenta",),
    "device_sharing": ("cliente",),
    "identidad_sintetica": ("cliente",),
    "account_takeover": ("cliente",),
    "card_testing": ("tarjeta",),
}

titulo("Evaluación de los seis detectores contra el ground truth")
evaluacion = evaluar_detectores(DETECCIONES, GT_BANCO, G_banco, TIPOS_OBJETIVO)
display(evaluacion)

fig, eje = plt.subplots(figsize=(10, 4.3))
x = np.arange(len(evaluacion))
eje.bar(x - 0.22, evaluacion["precisión"], 0.44, label="Precisión", color=COLORES["licito"])
eje.bar(x + 0.22, evaluacion["recall"], 0.44, label="Recall", color=COLORES["ilicito"])
eje.set_xticks(x)
eje.set_xticklabels(evaluacion.index, rotation=18, ha="right", fontsize=9)
eje.set_ylim(0, 1.08)
eje.set_ylabel("Valor")
eje.set_title("Rendimiento de los detectores basados en reglas de grafo")
eje.legend()
for i, (p, r) in enumerate(zip(evaluacion["precisión"], evaluacion["recall"])):
    eje.text(i - 0.22, p + 0.02, f"{p:.2f}", ha="center", fontsize=8)
    eje.text(i + 0.22, r + 0.02, f"{r:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

print(f"""
LECTURA

· Recall alto y precisión alta a la vez indican una firma topológica NÍTIDA: el patrón se
  distingue del comportamiento normal con una regla simple. Es el caso del device sharing
  y de la identidad sintética, donde compartir un identificador único es de por sí anómalo.
· Recall alto con precisión baja significa que la regla captura el patrón pero también
  mucho comportamiento legítimo. La solución no es endurecer el umbral —eso hunde el
  recall— sino AÑADIR CONDICIONES, como hace el detector de ATO con sus tres requisitos
  simultáneos.
· Estos detectores no usan machine learning. Son reglas sobre estructura, y en fraude
  siguen siendo la primera línea porque son EXPLICABLES: un investigador entiende
  "esta cuenta recibió de 14 orígenes y reenvió el 93 % en dos días" mucho mejor que
  "la red neuronal le dio 0,87".
· La arquitectura habitual en producción combina ambos: reglas de grafo para las
  tipologías conocidas, y modelos de los Módulos 14–16 para lo que las reglas no cubren.

Recall medio de los seis detectores: {evaluacion['recall'].mean():.1%}
Precisión media                   : {evaluacion['precisión'].mean():.1%}""")

---
---

# Módulo 20 — Bases de datos de grafos

NetworkX es una librería de **análisis en memoria**: perfecta para investigar, inútil para
servir consultas concurrentes sobre datos que cambian. Una base de datos de grafos aporta
persistencia, transacciones, índices, control de acceso y un lenguaje de consulta.

| Motor | Modelo | Lenguaje | Licencia | Punto fuerte | Punto débil |
|---|---|---|---|---|---|
| **Neo4j** | Property Graph | Cypher | GPL / comercial | Ecosistema y madurez; el estándar de facto en fraude | Escalado horizontal solo en la edición comercial |
| **TigerGraph** | Property Graph | GSQL | Comercial | Consultas profundas paralelizadas; muy rápido a gran escala | Comercial; curva de aprendizaje de GSQL |
| **AWS Neptune** | Property Graph **y** RDF | Gremlin, openCypher, SPARQL | Servicio gestionado | Soporta los dos modelos; operación gestionada | Dependencia de AWS; menos analítica integrada |
| **ArangoDB** | Multi-modelo (documentos + grafo) | AQL | Apache 2.0 | Un solo motor para documentos y grafo | Menos especializado en grafos puros |
| **JanusGraph** | Property Graph | Gremlin | Apache 2.0 | Escala sobre Cassandra/HBase; sin límite de tamaño | Complejidad operativa alta |
| **Memgraph** | Property Graph | Cypher | BSL / comercial | En memoria, compatible con Cypher, orientado a *streaming* | Coste de RAM; ecosistema más joven |

## Cómo elegir

- **Fraude bancario, equipo pequeño, tiempo real moderado** → Neo4j. El ecosistema (Bloom para
  investigadores, GDS para algoritmos, Cypher) está construido justo para este caso.
- **Detección en streaming con latencia de milisegundos** → Memgraph. Está pensado para ingerir
  de Kafka y responder mientras la transacción está en curso.
- **Cientos de miles de millones de aristas** → TigerGraph o JanusGraph.
- **Ya estás en AWS y necesitas RDF** → Neptune.
- **El grafo es solo una parte de un sistema documental** → ArangoDB.

## Lo que casi siempre se decide mal

La arquitectura habitual **no** es "todo en la base de datos de grafos". Es híbrida:

```
   Core bancario (relacional)          Data lake (Parquet/Delta)
             │                                    │
             └──────────► ETL ◄───────────────────┘
                           │
        ┌──────────────────┴──────────────────┐
        ▼                                     ▼
  BD de grafos                        Spark GraphFrames
  (consultas online, investigación)   (analítica batch: PageRank,
   sub-segundo, subgrafos pequeños)    comunidades sobre el grafo entero)
```

La base de datos de grafos responde preguntas **locales** (el vecindario de una alerta) en
milisegundos. La analítica **global** —PageRank sobre 500 millones de nodos, detección de
comunidades— se hace en batch con Spark y sus resultados se escriben de vuelta como propiedades
de los nodos. Intentar hacer PageRank global dentro de la base transaccional es el error clásico.

In [ ]:
# =============================================================================
# M20.1 — Exportación a Neo4j
# =============================================================================

def exportar_neo4j_csv(G: nx.MultiDiGraph, directorio: str = "neo4j_import",
                       atributos_nodo=None) -> dict:
    """Genera los CSV de nodos y relaciones con las cabeceras que espera Neo4j.

    Neo4j importa a partir de dos ficheros con una convención de cabecera concreta:
      · nodos     : `id:ID`, `:LABEL`, y una columna por propiedad con su tipo
      · relaciones: `:START_ID`, `:END_ID`, `:TYPE`, más propiedades

    Se generan un CSV de nodos por ETIQUETA (Neo4j lo prefiere así y el import es más
    rápido) y uno único de relaciones. Se devuelve también el comando de importación
    y las consultas Cypher de creación de índices, que es lo que en la práctica marca
    la diferencia entre una consulta de 5 ms y una de 5 segundos.
    """
    os.makedirs(directorio, exist_ok=True)
    salidas = {}

    tipos = sorted({d["tipo"] for _, d in G.nodes(data=True)})
    for tipo in tipos:
        nodos = [(n, d) for n, d in G.nodes(data=True) if d["tipo"] == tipo]
        propiedades = sorted({k for _, d in nodos for k, v in d.items()
                              if k != "tipo" and v is not None})
        if atributos_nodo:
            propiedades = [p for p in propiedades if p in atributos_nodo]

        filas = []
        for n, d in nodos:
            fila = {"id:ID": n, ":LABEL": tipo.capitalize()}
            for p in propiedades:
                valor = d.get(p)
                fila[p] = "" if valor is None else valor
            filas.append(fila)

        ruta = os.path.join(directorio, f"nodos_{tipo}.csv")
        pd.DataFrame(filas).to_csv(ruta, index=False)
        salidas[f"nodos_{tipo}"] = ruta

    filas_rel = []
    for u, v, d in G.edges(data=True):
        fila = {":START_ID": u, ":END_ID": v, ":TYPE": d["rel"].upper()}
        for k, valor in d.items():
            if k != "rel" and valor is not None:
                fila[k] = valor
        filas_rel.append(fila)
    ruta_rel = os.path.join(directorio, "relaciones.csv")
    pd.DataFrame(filas_rel).to_csv(ruta_rel, index=False)
    salidas["relaciones"] = ruta_rel

    ficheros_nodo = " ".join(
        f"--nodes={os.path.basename(r)}" for k, r in salidas.items() if k.startswith("nodos_"))
    salidas["_comando"] = (
        f"neo4j-admin database import full \\\n"
        f"  {ficheros_nodo} \\\n"
        f"  --relationships=relaciones.csv \\\n"
        f"  --overwrite-destination=true fraude")
    salidas["_indices"] = "\n".join(
        f"CREATE INDEX idx_{t} IF NOT EXISTS FOR (n:{t.capitalize()}) ON (n.id);" for t in tipos)

    return salidas


titulo("Exportación del grafo bancario a Neo4j")
export_neo4j = exportar_neo4j_csv(G_banco)
for clave, ruta in export_neo4j.items():
    if not clave.startswith("_"):
        print(f"  {clave:24s} → {os.path.basename(ruta):28s} "
              f"({os.path.getsize(ruta)/1024:>7.1f} KB)")

print("\nComando de importación masiva (offline, el más rápido):\n")
print(export_neo4j["_comando"])
print("\nÍndices a crear después de importar (sin ellos, cada MATCH hace un escaneo completo):\n")
print(export_neo4j["_indices"])

print("""
Nota sobre el rendimiento de la importación:

  · `neo4j-admin database import` es para la carga inicial: escribe los ficheros del
    almacén directamente, con la base parada. Millones de nodos por minuto.
  · `LOAD CSV` funciona con la base en marcha y es lo adecuado para cargas incrementales,
    pero es órdenes de magnitud más lento. Úsalo siempre con `CALL { … } IN TRANSACTIONS
    OF 10000 ROWS`: sin eso, una carga grande construye una sola transacción gigante y
    agota la memoria.
  · Crea los índices DESPUÉS de cargar, nunca antes: mantenerlos actualizados durante la
    inserción multiplica el tiempo de carga.""")

---
---

# Módulo 21 — Lenguajes de consulta

Tres lenguajes, tres filosofías. La forma más rápida de entender la diferencia es escribir **la
misma consulta** en los tres. Usamos la del Módulo 19: *"encuentra cuentas que reciban de al
menos 7 orígenes distintos y reenvíen casi todo poco después"*.

## Cypher — declarativo, orientado a patrones

Neo4j y Memgraph. Se **dibuja** el patrón con arte ASCII y el motor lo encuentra. Es el más
legible de los tres y el que menor curva de aprendizaje tiene para quien viene de SQL.

## Gremlin — imperativo, orientado a recorridos

JanusGraph, Neptune, TinkerPop. Se **describe el recorrido** paso a paso: empieza aquí, sigue
estas aristas, filtra, agrupa. Más verboso, pero da control explícito sobre el orden de
exploración — lo que importa cuando el plan que elige el optimizador no es el bueno.

## SPARQL — declarativo, orientado a tripletas

RDF y knowledge graphs. Todo son patrones de tripletas `(sujeto, predicado, objeto)`. Su fuerza
es la **federación**: una consulta puede atacar varios endpoints remotos a la vez, algo que ni
Cypher ni Gremlin hacen de forma nativa. Su debilidad, la ergonomía: los atributos de arista
obligan a reificar y el patrón se hincha.

In [ ]:
# =============================================================================
# M21.1 — La misma consulta en los tres lenguajes
# =============================================================================

CYPHER_MULAS = """
// ── CYPHER (Neo4j / Memgraph) ─────────────────────────────────────────────
// Cuentas mula: ≥7 orígenes distintos en 7 días y reenvío de ≥65% del importe

MATCH (origen:Cuenta)-[:EMITE]->(entrada:Transferencia)-[:RECIBE]->(mula:Cuenta)
WITH  mula,
      collect(DISTINCT origen)      AS origenes,
      sum(entrada.monto)            AS recibido,
      min(entrada.ts)               AS t_ini,
      max(entrada.ts)               AS t_fin
WHERE size(origenes) >= 7
  AND (t_fin - t_ini) <= 7 * 86400

MATCH (mula)-[:EMITE]->(salida:Transferencia)-[:RECIBE]->(destino:Cuenta)
WHERE salida.ts >= t_ini AND salida.ts <= t_fin + 10 * 86400
WITH  mula, origenes, recibido,
      sum(salida.monto)             AS reenviado,
      collect(DISTINCT destino)     AS destinos
WHERE reenviado >= 0.65 * recibido

RETURN mula.id                      AS cuenta,
       size(origenes)               AS n_origenes,
       recibido, reenviado,
       round(100.0 * reenviado / recibido) AS pct_reenvio,
       size(destinos)               AS n_destinos
ORDER BY n_origenes DESC
LIMIT 20;
"""

GREMLIN_MULAS = """
// ── GREMLIN (JanusGraph / Neptune / TinkerPop) ────────────────────────────
// El mismo patrón, expresado como recorrido explícito

g.V().hasLabel('Cuenta').as('mula')
  .where(
    __.in('RECIBE').hasLabel('Transferencia')
      .in('EMITE').hasLabel('Cuenta')
      .dedup().count().is(gte(7))
  )
  .project('cuenta', 'n_origenes', 'recibido', 'reenviado')
    .by(values('id'))
    .by(__.in('RECIBE').in('EMITE').dedup().count())
    .by(__.in('RECIBE').values('monto').sum())
    .by(__.out('EMITE').values('monto').sum())
  .where(
    __.select('reenviado').math('_ * 1.0')
      .as('r').select('recibido').math('_ * 0.65')
      .where(lte('r'))
  )
  .order().by(select('n_origenes'), desc)
  .limit(20)
"""

SPARQL_MULAS = """
# ── SPARQL (RDF / Neptune / GraphDB) ──────────────────────────────────────
# En RDF la transferencia YA es un recurso, así que la reificación no cuesta nada extra

PREFIX b:    <http://banca.ejemplo/>
PREFIX rel:  <http://banca.ejemplo/rel/>
PREFIX prop: <http://banca.ejemplo/prop/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

SELECT ?mula (COUNT(DISTINCT ?origen) AS ?n_origenes)
             (SUM(?monto_in) AS ?recibido)
WHERE {
  ?origen   a          b:clase/Cuenta ;
            rel:emite  ?entrada .
  ?entrada  a          b:clase/Transferencia ;
            rel:recibe ?mula ;
            prop:monto ?monto_in ;
            prop:ts    ?ts_in .
  ?mula     a          b:clase/Cuenta .

  FILTER(?origen != ?mula)
}
GROUP BY ?mula
HAVING (COUNT(DISTINCT ?origen) >= 7)
ORDER BY DESC(?n_origenes)
LIMIT 20
"""

titulo("La misma pregunta en tres lenguajes")
for nombre, consulta in [("CYPHER", CYPHER_MULAS), ("GREMLIN", GREMLIN_MULAS),
                         ("SPARQL", SPARQL_MULAS)]:
    print(consulta)

print("""
COMPARACIÓN

  Legibilidad      Cypher > SPARQL > Gremlin
  Control del plan Gremlin > Cypher > SPARQL
  Agregaciones     Cypher > SPARQL > Gremlin
  Federación       SPARQL >> los otros dos (es su razón de ser)
  Portabilidad     Gremlin (TinkerPop) y SPARQL (W3C) son estándares reales;
                   Cypher lo está siendo vía openCypher / GQL (ISO/IEC 39075:2024)

GQL, el estándar ISO publicado en 2024, es esencialmente Cypher elevado a norma
internacional. Es la apuesta segura a medio plazo: si empiezas hoy, empieza por Cypher.
""")

In [ ]:
# =============================================================================
# M21.2 — Generación de consultas ejecutables desde el grafo
# =============================================================================

def generar_cypher(G: nx.MultiDiGraph, limite_nodos: int = 60,
                   limite_aristas: int = 120) -> str:
    """Emite sentencias CREATE de Cypher listas para pegar en el navegador de Neo4j.

    Para una muestra pequeña —justo lo que se quiere para explorar visualmente un caso—
    es más cómodo que montar la importación por CSV. Los valores se escapan y se
    tipifican: las cadenas van entre comillas, los números no.
    """
    def valor_cypher(v):
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, (int, float, np.integer, np.floating)):
            return str(v)
        return '"' + str(v).replace('\\', '\\\\').replace('"', '\\"') + '"'

    nodos = list(G.nodes())[:limite_nodos]
    conjunto = set(nodos)
    lineas = ["// Grafo bancario — muestra generada automáticamente",
              "// Pega esto en el navegador de Neo4j (http://localhost:7474)", ""]

    for n in nodos:
        d = G.nodes[n]
        props = {"id": n}
        props.update({k: v for k, v in d.items() if k != "tipo" and v is not None})
        cuerpo = ", ".join(f"{k}: {valor_cypher(v)}" for k, v in props.items())
        lineas.append(f"CREATE (:{d['tipo'].capitalize()} {{{cuerpo}}})")

    lineas.append("")
    contador = 0
    for u, v, d in G.edges(data=True):
        if u in conjunto and v in conjunto and contador < limite_aristas:
            props = {k: x for k, x in d.items() if k != "rel" and x is not None}
            cuerpo = (" {" + ", ".join(f"{k}: {valor_cypher(x)}" for k, x in props.items()) + "}") if props else ""
            lineas.append(
                f"MATCH (a {{id: {valor_cypher(u)}}}), (b {{id: {valor_cypher(v)}}}) "
                f"CREATE (a)-[:{d['rel'].upper()}{cuerpo}]->(b);")
            contador += 1

    lineas += ["", "// Consulta de comprobación:",
               "MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 100;"]
    return "\n".join(lineas)


def exportar_rdf_turtle(G: nx.MultiDiGraph, nodos=None,
                        base: str = "http://banca.ejemplo/") -> str:
    """Serializa un fragmento del grafo a Turtle, el formato RDF legible por humanos.

    Turtle es N-Triples con prefijos y agrupación por sujeto: mismo contenido, una
    fracción del tamaño y muchísimo más legible. Es el formato que consume cualquier
    endpoint SPARQL.
    """
    nodos = list(G.nodes())[:40] if nodos is None else list(nodos)
    conjunto = set(nodos)
    lineas = [f"@prefix b:    <{base}> .",
              f"@prefix clase: <{base}clase/> .",
              f"@prefix prop: <{base}prop/> .",
              f"@prefix rel:  <{base}rel/> .",
              "@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .", ""]

    for n in nodos:
        d = G.nodes[n]
        partes = [f"    rdf:type clase:{d['tipo'].capitalize()}"]
        for k, v in d.items():
            if k in ("tipo", "patron_fraude") or v is None:
                continue
            literal = f'"{v}"' if isinstance(v, str) else f'"{v}"^^<http://www.w3.org/2001/XMLSchema#decimal>'
            partes.append(f"    prop:{k} {literal}")
        for _, destino, datos in G.out_edges(n, data=True):
            if destino in conjunto:
                partes.append(f"    rel:{datos['rel']} b:{destino}")
        lineas.append(f"b:{n}\n" + " ;\n".join(partes) + " .\n")

    return "\n".join(lineas)


titulo("Cypher generado desde el grafo")
muestra = nx.Graph(nx.ego_graph(nx.Graph(G_banco), cliente_fraude, radius=1))
sub_banco = G_banco.subgraph(list(muestra.nodes())[:40])
cypher = generar_cypher(sub_banco, limite_nodos=40, limite_aristas=80)
print("\n".join(cypher.split("\n")[:18]))
print(f"…\n[{len(cypher.splitlines())} líneas en total]")

with open("muestra_neo4j.cypher", "w", encoding="utf-8") as f:
    f.write(cypher)
print("\nGuardado en: muestra_neo4j.cypher")

titulo("RDF / Turtle generado desde el mismo fragmento", nivel=2)
turtle = exportar_rdf_turtle(sub_banco, nodos=list(sub_banco.nodes())[:6])
print(turtle[:1400])

with open("muestra_grafo.ttl", "w", encoding="utf-8") as f:
    f.write(exportar_rdf_turtle(sub_banco, nodos=list(sub_banco.nodes())))
print("Guardado en: muestra_grafo.ttl")

---
---

# Módulo 22 — Escalabilidad

NetworkX es el estándar para aprender y explorar, y no es el adecuado para producción a gran
escala. Conviene saber exactamente dónde está su techo y qué hay al otro lado.

| Herramienta | Lenguaje | Techo práctico | Paralelo | Cuándo |
|---|---|---|---|---|
| **NetworkX** | Python puro | ~10⁶ nodos | No | Prototipado, docencia, análisis exploratorio |
| **igraph** | C con bindings | ~10⁷ nodos | Parcial | Misma API mental, 10–100× más rápido |
| **graph-tool** | C++ con OpenMP | ~10⁸ nodos | Sí | Analítica estadística intensiva; instalación difícil |
| **Spark GraphFrames** | JVM + Python | 10⁹⁺ aristas | Sí (clúster) | Batch distribuido sobre un data lake |
| **GraphX** | Scala/Spark | 10⁹⁺ aristas | Sí (clúster) | Igual, API de Scala, más control |
| **cuGraph (RAPIDS)** | CUDA | ~10⁸ nodos | Sí (GPU) | Cuando hay GPU y el grafo cabe en su memoria |

## Por qué NetworkX es lento (y cuándo da igual)

Un grafo de NetworkX es un **diccionario de diccionarios**. Cada acceso a un vecino es una
búsqueda en un dict de Python con toda su sobrecarga de objetos. Eso da flexibilidad total —los
nodos pueden ser cualquier objeto hashable, los atributos cualquier cosa— y cuesta entre uno y
dos órdenes de magnitud frente a un array de C.

La consecuencia práctica es menos dramática de lo que parece: **la mayoría de los algoritmos se
pueden reescribir como operaciones sobre matrices dispersas de SciPy**, que sí están en C. La
celda M22.2 reimplementa PageRank así y mide la diferencia. La regla: si un algoritmo se expresa
como productos matriz-vector, sácalo de NetworkX; si es un recorrido irregular (BFS, ciclos,
motifs), NetworkX está bien.

In [ ]:
# =============================================================================
# M22.1 — Benchmark: NetworkX frente a igraph
# =============================================================================

def benchmark_libreria(G: nx.Graph, repeticiones: int = 1) -> pd.DataFrame:
    """Cronometra las mismas operaciones en NetworkX y, si está disponible, en igraph.

    Se miden tres operaciones de perfil distinto:
      · PageRank            — iterativo, dominado por productos matriz-vector
      · Componentes conexas — recorrido puro
      · Grados              — acceso masivo a la estructura

    La conversión a igraph se cronometra aparte: si solo se va a ejecutar un algoritmo,
    ese coste de conversión puede comerse toda la ventaja.
    """
    filas = []

    def medir(etiqueta, biblioteca, fn):
        tiempos = []
        for _ in range(repeticiones):
            t0 = time.perf_counter()
            fn()
            tiempos.append(time.perf_counter() - t0)
        filas.append({"operación": etiqueta, "biblioteca": biblioteca,
                      "segundos": round(float(np.mean(tiempos)), 3)})

    medir("PageRank", "NetworkX", lambda: nx.pagerank(G, alpha=0.85))
    medir("Componentes débiles", "NetworkX",
          lambda: list(nx.weakly_connected_components(G) if G.is_directed()
                       else nx.connected_components(G)))
    medir("Grados", "NetworkX", lambda: dict(G.degree()))

    if IGRAPH_OK:
        import igraph as ig
        nodos = list(G.nodes())
        indice = {n: i for i, n in enumerate(nodos)}

        t0 = time.perf_counter()
        g_ig = ig.Graph(n=len(nodos),
                        edges=[(indice[u], indice[v]) for u, v in G.edges()],
                        directed=G.is_directed())
        conversion = time.perf_counter() - t0
        filas.append({"operación": "Conversión nx→igraph", "biblioteca": "igraph",
                      "segundos": round(conversion, 3)})

        medir("PageRank", "igraph", lambda: g_ig.pagerank(damping=0.85))
        medir("Componentes débiles", "igraph", lambda: g_ig.connected_components(mode="weak"))
        medir("Grados", "igraph", lambda: g_ig.degree())
    else:
        print("igraph no disponible (`pip install python-igraph`): solo se mide NetworkX.\n")

    tabla = pd.DataFrame(filas).pivot_table(index="operación", columns="biblioteca",
                                            values="segundos")
    if "igraph" in tabla.columns:
        tabla["aceleración"] = (tabla["NetworkX"] / tabla["igraph"]).round(1)
    return tabla


titulo(f"Benchmark sobre Elliptic ({G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas)")
bench = benchmark_libreria(G)
display(bench)

In [ ]:
# =============================================================================
# M22.2 — El patrón que de verdad importa: sacar el álgebra de NetworkX
# =============================================================================

def pagerank_disperso(G: nx.Graph, alpha: float = 0.85, max_iter: int = 100,
                      tol: float = 1e-9, nodelist=None) -> dict:
    """PageRank por iteración de potencia sobre matrices dispersas de SciPy.

    Es el MISMO algoritmo que `nx.pagerank`, pero cada iteración es un producto
    matriz-vector disperso ejecutado en C en vez de un bucle de Python sobre
    diccionarios. La diferencia no está en la teoría, sino en dónde se ejecuta el bucle.

    Detalle que suele olvidarse: los **nodos colgantes** (sin aristas de salida) se
    tragan la probabilidad. Su masa hay que redistribuirla entre todos los nodos en
    cada iteración, o el vector deja de sumar 1 y el resultado se degrada en silencio.
    """
    nodelist = list(G.nodes()) if nodelist is None else list(nodelist)
    n = len(nodelist)
    A = sp.csr_matrix(nx.to_scipy_sparse_array(G, nodelist=nodelist, dtype=float, format="csr"))

    grados = np.asarray(A.sum(axis=1)).ravel()
    colgantes = grados == 0
    inv = np.zeros(n)
    np.divide(1.0, grados, out=inv, where=grados > 0)
    M = (sp.diags(inv) @ A).T.tocsr()          # matriz de transición, estocástica por columnas

    x = np.full(n, 1.0 / n)
    for _ in range(max_iter):
        masa_colgante = alpha * x[colgantes].sum() / n
        x_nuevo = alpha * (M @ x) + masa_colgante + (1 - alpha) / n
        if np.abs(x_nuevo - x).sum() < tol:
            x = x_nuevo
            break
        x = x_nuevo
    return dict(zip(nodelist, x / x.sum()))


titulo("Mismo algoritmo, distinto motor")
pr_nx, t_nx = cronometrar(nx.pagerank, G, etiqueta="nx.pagerank (bucles Python)", alpha=0.85)
pr_sp, t_sp = cronometrar(pagerank_disperso, G, etiqueta="pagerank_disperso (SciPy)", alpha=0.85)

nodos_comunes = list(G.nodes())
v_nx = np.array([pr_nx[n] for n in nodos_comunes])
v_sp = np.array([pr_sp[n] for n in nodos_comunes])
print(f"\n  Aceleración            : {t_nx / max(t_sp, 1e-9):.1f}x")
print(f"  Diferencia máxima      : {np.abs(v_nx - v_sp).max():.2e}")
print(f"  Correlación de rangos  : {float(spearmanr(v_nx, v_sp)[0]):.6f}")
print("\n  El resultado es el mismo hasta la precisión numérica. Lo único que cambió es")
print("  dónde se ejecuta el bucle interno. Este patrón —expresar el algoritmo como")
print("  álgebra dispersa— es lo que más rendimiento da antes de cambiar de herramienta.")

titulo("Escalado del coste con el tamaño del grafo", nivel=2)
filas = []
for tam in [1_000, 5_000, 20_000, 60_000, G.number_of_nodes()]:
    sub = G if tam >= G.number_of_nodes() else muestrear_subgrafo(G, tam, metodo="bfs")
    _, t1 = cronometrar(nx.pagerank, sub, etiqueta=f"nx.pagerank n={sub.number_of_nodes():,}")
    _, t2 = cronometrar(pagerank_disperso, sub, etiqueta=f"disperso   n={sub.number_of_nodes():,}")
    filas.append({"nodos": sub.number_of_nodes(), "aristas": sub.number_of_edges(),
                  "NetworkX (s)": round(t1, 3), "SciPy disperso (s)": round(t2, 4),
                  "aceleración": round(t1 / max(t2, 1e-9), 1)})
escalado = pd.DataFrame(filas).set_index("nodos")
display(escalado)

plt.figure(figsize=(9, 4))
plt.plot(escalado.index, escalado["NetworkX (s)"], marker="o", lw=2,
         color=COLORES["ilicito"], label="nx.pagerank")
plt.plot(escalado.index, escalado["SciPy disperso (s)"], marker="s", lw=2,
         color=COLORES["ok"], label="pagerank_disperso")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Nodos (log)")
plt.ylabel("Segundos (log)")
plt.title("Ambos son O(m·i); la constante multiplicativa es lo que separa a Python de C")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# M22.3 — Más allá de una sola máquina: Spark GraphFrames y GraphX
# =============================================================================

CODIGO_GRAPHFRAMES = '''
# ── Spark GraphFrames (PySpark) ─────────────────────────────────────────────
# pyspark --packages graphframes:graphframes:0.8.3-spark3.5-s_2.12

from pyspark.sql import SparkSession
from graphframes import GraphFrame

spark = (SparkSession.builder
         .appName("grafo-fraude")
         .config("spark.sql.shuffle.partitions", "400")
         .getOrCreate())
spark.sparkContext.setCheckpointDir("/tmp/checkpoints")   # obligatorio para connectedComponents

# Los vértices necesitan una columna "id"; las aristas, "src" y "dst".
vertices = spark.read.parquet("s3://banco/grafo/cuentas/")
aristas  = spark.read.parquet("s3://banco/grafo/transferencias/")
g = GraphFrame(vertices, aristas)

# --- Métricas globales, distribuidas ---
pr    = g.pageRank(resetProbability=0.15, maxIter=20)
comp  = g.connectedComponents()
tri   = g.triangleCount()
grados = g.inDegrees.join(g.outDegrees, "id", "outer").fillna(0)

# --- Búsqueda de motifs: la joya de GraphFrames ---
# Un ciclo de tres saltos con importes casi constantes (layering)
ciclos = (g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")
           .filter("e1.monto > 10000")
           .filter("abs(e1.monto - e2.monto) / e1.monto < 0.1")
           .filter("abs(e2.monto - e3.monto) / e2.monto < 0.1")
           .filter("e1.ts < e2.ts AND e2.ts < e3.ts"))

# Fan-in: cuentas que reciben de muchos orígenes distintos en poco tiempo
from pyspark.sql import functions as F
fan_in = (aristas.groupBy("dst")
          .agg(F.countDistinct("src").alias("origenes"),
               F.sum("monto").alias("recibido"),
               (F.max("ts") - F.min("ts")).alias("span"))
          .filter("origenes >= 7 AND span <= 604800"))

# Los resultados se escriben de vuelta como propiedades de los nodos, para que la base
# de datos de grafos los sirva en consultas online sin recalcular nada.
pr.vertices.write.mode("overwrite").parquet("s3://banco/grafo/pagerank/")
'''

CODIGO_GRAPHX = '''
// ── Spark GraphX (Scala) ────────────────────────────────────────────────────
import org.apache.spark.graphx._

val aristas: RDD[Edge[Double]] =
  sc.textFile("hdfs:///banco/transferencias.csv").map { linea =>
    val c = linea.split(","); Edge(c(0).toLong, c(1).toLong, c(2).toDouble)
  }
val grafo = Graph.fromEdges(aristas, defaultValue = 0.0)

val ranks  = grafo.pageRank(0.0001).vertices
val comps  = grafo.connectedComponents().vertices

// Pregel: propagación de riesgo desde cuentas señaladas.
// Cada iteración difunde el riesgo a los vecinos con un factor de amortiguación.
val riesgoInicial = grafo.mapVertices((id, _) => if (senaladas.contains(id)) 1.0 else 0.0)
val propagado = riesgoInicial.pregel(0.0, maxIterations = 5)(
  (id, actual, entrante) => math.max(actual, entrante),
  triplet => if (triplet.srcAttr > 0.01)
               Iterator((triplet.dstId, triplet.srcAttr * 0.6))
             else Iterator.empty,
  (a, b) => math.max(a, b)
)
'''

titulo("Código de referencia para escala distribuida")
print(CODIGO_GRAPHFRAMES)
print(CODIGO_GRAPHX)

print("""
CUÁNDO DAR EL SALTO A SPARK

No por el número de nodos, sino por tres señales concretas:

  1. El grafo no cabe en la RAM de una máquina. Regla aproximada: NetworkX consume
     entre 200 y 400 bytes por arista, así que 100 millones de aristas son ~30 GB.
  2. Los datos ya viven en un data lake y moverlos cuesta más que procesarlos donde están.
  3. Hay que recalcular métricas globales de forma periódica sobre datos que cambian.

Y cuándo NO: si la pregunta es local ("el vecindario de esta alerta"), Spark es la
herramienta equivocada por mucho que el grafo sea enorme. Esa consulta la responde una
base de datos de grafos en milisegundos, mientras que un trabajo de Spark tarda minutos
solo en arrancar. Es la división de trabajo del Módulo 20.

ESCALERA DE DECISIÓN
    < 10⁵ nodos   → NetworkX. No optimices lo que no hace falta.
    10⁵ – 10⁶     → NetworkX + SciPy disperso para el álgebra (M22.2).
    10⁶ – 10⁷     → igraph o graph-tool.
    10⁷ – 10⁸     → graph-tool, o cuGraph si hay GPU.
    > 10⁸         → Spark GraphFrames / GraphX, o un motor distribuido como TigerGraph.
""")

---
---

# Módulo 23 — Investigación (estado del arte)

Las siete líneas activas más relevantes para grafos aplicados a fraude, con lo que aportan y
dónde están sus límites reales.

## 1. Temporal Graph Neural Networks

Las GNN del Módulo 16 tratan el grafo como estático. Las temporales incorporan **cuándo** ocurrió
cada arista:

- **TGAT** (Xu et al., 2020) — codificación temporal funcional: el tiempo entra como una
  característica continua mediante funciones de Bochner, en vez de discretizarlo en snapshots.
- **TGN** (Rossi et al., 2020) — cada nodo mantiene un vector de **memoria** que se actualiza con
  cada interacción. Es lo más parecido a un sistema de fraude real: el estado de una cuenta
  evoluciona con cada operación y la predicción se hace sobre ese estado.
- **JODIE** (Kumar et al., 2019) — dos RNN acopladas que proyectan la trayectoria futura del
  embedding entre interacciones.

Por qué importa aquí: el orden de las operaciones **es** la señal. Diez transferencias entrantes
seguidas de una saliente es una mula; las mismas once operaciones en otro orden, no es nada.
Un modelo estático no puede distinguirlas.

## 2. Dynamic Graph Embeddings

Reentrenar embeddings desde cero cada vez que llega una arista es inviable. Las técnicas
incrementales —**DynamicTriad**, **EvolveGCN** (que hace evolucionar los *pesos* de la GCN con una
RNN), **tNodeEmbed**— actualizan solo la región afectada. Es un requisito operativo, no una mejora.

## 3. Graph Transformers

Sustituyen la agregación local por atención global. Resuelven dos limitaciones de las GNN:

- **Over-smoothing**: al apilar capas, todos los embeddings convergen al mismo valor.
- **Alcance limitado**: una GNN de $L$ capas solo ve $L$ saltos.

**Graphormer** (Ying et al., 2021) introduce tres codificaciones estructurales —de grado, de
centralidad y de distancia entre pares— para reinyectar la topología que la atención global
ignora. **SAN** y **GraphGPS** combinan atención local y global. Su límite es el $O(n^2)$ de la
atención: en grafos grandes se aplican sobre subgrafos.

## 4. Heterogeneous GNN

El RGCN del Módulo 16 es el punto de partida. Más allá:

- **HAN** (Wang et al., 2019) — atención a dos niveles: dentro de cada meta-camino y entre
  meta-caminos. El modelo aprende *qué meta-camino importa* para cada tarea.
- **HGT** (Hu et al., 2020) — atención con parámetros por tipo de nodo y de arista, sin necesidad
  de definir meta-caminos a mano. Es el estado del arte práctico para grafos bancarios.

## 5. Knowledge Graph Embeddings

Aprenden vectores para entidades **y relaciones**, de modo que las tripletas verdaderas cumplan
una ecuación geométrica:

| Modelo | Ecuación | Modela |
|---|---|---|
| **TransE** | $h + r \approx t$ | Relaciones 1-a-1; falla en 1-a-N |
| **DistMult** | $\langle h, r, t\rangle$ | Simétricas; no distingue $(h,r,t)$ de $(t,r,h)$ |
| **ComplEx** | Producto en $\mathbb{C}$ | Asimétricas |
| **RotatE** | $t \approx h \circ r$ (rotación en $\mathbb{C}$) | Simetría, antisimetría, inversión y composición |

Sirven para **completar el grafo**: inferir relaciones que los datos no registran pero que la
estructura implica. En banca: deducir que dos clientes pertenecen a la misma organización sin que
ningún campo lo declare. La celda M23.1 implementa TransE sobre el grafo bancario.

## 6. Explainable GNN

Un modelo que marca una cuenta como fraudulenta sin justificación **no se puede desplegar** en
banca: hay obligación regulatoria de explicar las decisiones adversas.

- **GNNExplainer** (Ying et al., 2019) — busca el subgrafo mínimo y el subconjunto de features que
  preservan la predicción. La salida es directamente accionable: "estas 4 aristas la explican".
- **PGExplainer** — aprende un explicador global, reutilizable entre instancias.
- **SubgraphX** — usa valores de Shapley sobre subgrafos; más riguroso y más caro.

## 7. Graph Foundation Models

La línea más reciente: modelos preentrenados sobre muchos grafos y transferibles a grafos nuevos
sin reentrenar, al estilo de los modelos de lenguaje. El obstáculo de fondo es que **no existe un
vocabulario común entre grafos**: los nodos de una red social y los de una red de transacciones no
comparten espacio de features. Las propuestas actuales (GraphGPT, OFA, GFM) lo abordan alineando
descripciones textuales de los nodos con un LLM. Prometedor y todavía no operativo en producción.

In [ ]:
# =============================================================================
# M23.1 — TransE sobre el grafo bancario como Knowledge Graph
# =============================================================================

def extraer_tripletas(G: nx.MultiDiGraph, excluir_tipos=("compra",)) -> list:
    """Convierte el grafo de propiedades en tripletas (cabeza, relación, cola)."""
    return [(u, d["rel"], v) for u, v, d in G.edges(data=True)
            if G.nodes[u]["tipo"] not in excluir_tipos
            and G.nodes[v]["tipo"] not in excluir_tipos]


def transe(tripletas: list, dim: int = 48, epocas: int = 220, lr: float = 0.02,
           margen: float = 1.0, lote: int = 512, seed: int = SEED) -> dict:
    """TransE: aprende h + r ≈ t con pérdida de ranking por margen.

    Para cada tripleta verdadera se genera una CORRUPTA sustituyendo la cabeza o la
    cola por una entidad al azar, y se optimiza:

        L = Σ max(0, margen + d(h+r, t) − d(h'+r, t'))

    es decir, "la tripleta verdadera debe estar al menos `margen` más cerca de cumplir
    la ecuación que la falsa". Los gradientes se derivan a mano —la derivada de la norma
    L2 respecto a su argumento es v/‖v‖— y se aplican con `np.add.at` para acumular
    correctamente cuando una entidad aparece varias veces en el mismo lote.

    La normalización de las entidades a norma 1 tras cada época es parte del algoritmo,
    no un detalle: sin ella el modelo minimiza la pérdida agrandando los vectores en vez
    de aprender estructura.
    """
    rng = np.random.default_rng(seed)
    entidades = sorted({h for h, _, _ in tripletas} | {t for _, _, t in tripletas})
    relaciones = sorted({r for _, r, _ in tripletas})
    idx_e = {e: i for i, e in enumerate(entidades)}
    idx_r = {r: i for i, r in enumerate(relaciones)}

    T = np.array([[idx_e[h], idx_r[r], idx_e[t]] for h, r, t in tripletas], dtype=np.int64)
    n_e, n_r = len(entidades), len(relaciones)

    limite = 6.0 / math.sqrt(dim)
    E = rng.uniform(-limite, limite, (n_e, dim))
    R = rng.uniform(-limite, limite, (n_r, dim))
    R /= np.linalg.norm(R, axis=1, keepdims=True)

    historial = []
    for epoca in range(epocas):
        E /= np.linalg.norm(E, axis=1, keepdims=True)
        orden = rng.permutation(len(T))
        perdida_total = 0.0

        for inicio in range(0, len(T), lote):
            b = T[orden[inicio:inicio + lote]]
            h, r, t = b[:, 0], b[:, 1], b[:, 2]

            # Corrupción: mitad de cabezas, mitad de colas
            h_c, t_c = h.copy(), t.copy()
            cambiar_cabeza = rng.random(len(b)) < 0.5
            h_c[cambiar_cabeza] = rng.integers(0, n_e, cambiar_cabeza.sum())
            t_c[~cambiar_cabeza] = rng.integers(0, n_e, (~cambiar_cabeza).sum())

            v_pos = E[h] + R[r] - E[t]
            v_neg = E[h_c] + R[r] - E[t_c]
            d_pos = np.linalg.norm(v_pos, axis=1) + 1e-12
            d_neg = np.linalg.norm(v_neg, axis=1) + 1e-12

            perdida = np.maximum(0.0, margen + d_pos - d_neg)
            activos = perdida > 0
            perdida_total += float(perdida.sum())
            if not activos.any():
                continue

            gp = (v_pos[activos] / d_pos[activos, None])
            gn = (v_neg[activos] / d_neg[activos, None])

            np.add.at(E, h[activos], -lr * gp)
            np.add.at(E, t[activos], +lr * gp)
            np.add.at(R, r[activos], -lr * (gp - gn))
            np.add.at(E, h_c[activos], +lr * gn)
            np.add.at(E, t_c[activos], -lr * gn)

        if (epoca + 1) % 40 == 0:
            historial.append({"época": epoca + 1, "pérdida": round(perdida_total / len(T), 4)})

    return {"E": E / np.linalg.norm(E, axis=1, keepdims=True), "R": R,
            "entidades": entidades, "relaciones": relaciones,
            "idx_e": idx_e, "idx_r": idx_r, "historial": historial}


def evaluar_transe(modelo: dict, tripletas_test: list, muestra: int = 300,
                   seed: int = SEED) -> dict:
    """Mean rank y Hits@K: para cada tripleta, ¿en qué puesto queda la cola verdadera?

    Se puntúan todas las entidades como posible cola y se mira dónde cae la correcta.
    Es la evaluación estándar en knowledge graph completion, y su lectura es directa:
    Hits@10 = fracción de veces que la respuesta correcta está entre las 10 primeras.
    """
    rng = random.Random(seed)
    E, R, idx_e, idx_r = modelo["E"], modelo["R"], modelo["idx_e"], modelo["idx_r"]
    muestra_test = rng.sample(tripletas_test, min(muestra, len(tripletas_test)))

    rangos = []
    for h, r, t in muestra_test:
        if h not in idx_e or t not in idx_e or r not in idx_r:
            continue
        objetivo = E[idx_e[h]] + R[idx_r[r]]
        distancias = np.linalg.norm(E - objetivo, axis=1)
        rangos.append(int((distancias < distancias[idx_e[t]]).sum()) + 1)

    rangos = np.array(rangos)
    if len(rangos) == 0:
        return {}
    return {
        "tripletas evaluadas": len(rangos),
        "entidades candidatas": len(E),
        "rango medio": round(float(rangos.mean()), 1),
        "rango mediano": int(np.median(rangos)),
        "MRR": round(float((1.0 / rangos).mean()), 4),
        "Hits@1": round(float((rangos <= 1).mean()), 4),
        "Hits@10": round(float((rangos <= 10).mean()), 4),
        "Hits@100": round(float((rangos <= 100).mean()), 4),
    }


titulo("TransE sobre el grafo bancario")
tripletas = extraer_tripletas(G_banco)
rng_split = random.Random(SEED)
rng_split.shuffle(tripletas)
corte = int(len(tripletas) * 0.9)
tripletas_train, tripletas_test = tripletas[:corte], tripletas[corte:]

print(f"  Tripletas totales : {len(tripletas):,}")
print(f"  Entrenamiento     : {len(tripletas_train):,}")
print(f"  Test              : {len(tripletas_test):,}")
print(f"  Relaciones        : {sorted({r for _, r, _ in tripletas})}\n")

modelo_kg, _ = cronometrar(transe, tripletas_train, etiqueta="entrenamiento de TransE",
                           dim=48, epocas=220)
print("\n  Evolución de la pérdida:")
for h in modelo_kg["historial"]:
    print(f"    época {h['época']:>3}  pérdida media {h['pérdida']:.4f}")

titulo("Evaluación (link prediction sobre el knowledge graph)", nivel=2)
metricas_kg = evaluar_transe(modelo_kg, tripletas_test)
for k, v in metricas_kg.items():
    print(f"  {k:24s} {formatear(v):>10s}")

print(f"""
Un Hits@10 de {metricas_kg.get('Hits@10', 0):.1%} significa que, para esa fracción de las tripletas
retiradas, el modelo coloca la entidad correcta entre sus 10 primeras candidatas de entre
{metricas_kg.get('entidades candidatas', 0):,}. El azar daría {10/max(metricas_kg.get('entidades candidatas',1),1):.2%}.

Ahí está el uso práctico: el modelo propone relaciones que NO figuran en los datos. Una
tripleta que el modelo puntúa muy alto y que no existe es una hipótesis a investigar
—"este cliente probablemente usa este dispositivo"— y una que existe pero el modelo puntúa
muy bajo es una anomalía: una relación que no encaja con la estructura del resto del grafo.""")

In [ ]:
# =============================================================================
# M23.2 — Visualización del espacio de entidades aprendido
# =============================================================================

E_kg = modelo_kg["E"]
tipos_kg = [G_banco.nodes[e]["tipo"] for e in modelo_kg["entidades"]]
fraude_kg = [G_banco.nodes[e].get("patron_fraude") for e in modelo_kg["entidades"]]

indices = np.random.default_rng(SEED).choice(len(E_kg), min(2500, len(E_kg)), replace=False)
proy = PCA(n_components=2, random_state=SEED).fit_transform(E_kg[indices])

fig, ejes = plt.subplots(1, 2, figsize=(14, 5.4))
for tipo in sorted(set(tipos_kg)):
    sel = [i for i, j in enumerate(indices) if tipos_kg[j] == tipo]
    if sel:
        ejes[0].scatter(proy[sel, 0], proy[sel, 1], s=9, alpha=0.6,
                        c=PALETA_TIPOS.get(tipo, "#999"), label=tipo)
ejes[0].set_title("Espacio de entidades de TransE, coloreado por TIPO")
ejes[0].legend(fontsize=7, ncol=2, markerscale=1.6)
ejes[0].set_xticks([]); ejes[0].set_yticks([])

marcados = [i for i, j in enumerate(indices) if fraude_kg[j]]
limpios = [i for i, j in enumerate(indices) if not fraude_kg[j]]
ejes[1].scatter(proy[limpios, 0], proy[limpios, 1], s=8, alpha=0.35,
                c=COLORES["desconocido"], label="Sin marcar")
ejes[1].scatter(proy[marcados, 0], proy[marcados, 1], s=22, alpha=0.9,
                c=COLORES["ilicito"], label="En algún patrón de fraude")
ejes[1].set_title("El mismo espacio, marcando las entidades implicadas en fraude")
ejes[1].legend(fontsize=8, markerscale=1.6)
ejes[1].set_xticks([]); ejes[1].set_yticks([])
plt.tight_layout()
plt.show()

print("""
TransE agrupa las entidades por tipo sin que nadie se lo haya dicho: el tipo nunca entró en
el modelo, solo las tripletas. La estructura de relaciones basta para inducirlo, que es
exactamente lo que promete un knowledge graph embedding.

Y algo más útil todavía: si las entidades marcadas como fraude ocupan regiones concretas del
espacio en lugar de repartirse al azar, ese embedding sirve como feature para un clasificador
— cerrando el círculo con los Módulos 13 y 15.""")

---

## Cierre

### Lo que se ha hecho

Veinticuatro módulos, dos grafos y unas setenta funciones reutilizables. El recorrido completo:
del dato tabular al grafo (M1–M2), de la estructura a la métrica (M3–M9), del instante a la
serie temporal (M10), de la métrica a la feature (M11–M13), y de la feature al modelo (M14–M16).
Los módulos 17 a 23 cubren cómo se comunica, se almacena, se consulta, se escala y hacia dónde va.

### Los cinco hallazgos que dejó el análisis

1. **Las 49 componentes débiles de Elliptic son exactamente los 49 timesteps.** Ninguna arista
   cruza el tiempo. No es un grafo temporal: son 49 grafos independientes, y eso condiciona qué
   análisis tienen sentido (M4).
2. **El grafo es un DAG.** Cero ciclos, cero componentes fuertemente conexas no triviales, cero
   reciprocidad. La detección de layering por ciclos es inaplicable a nivel de transacción, y
   Katz converge para cualquier α como efecto secundario (M4, M6, M8).
3. **Las features de "vecinos fraudulentos" son fuga pura bajo el split temporal.** Como los
   vecinos de un nodo de test están siempre en test, usar sus etiquetas es usar la respuesta.
   Se midió en lugar de suponerlo (M11).
4. **El fraude se concentra en comunidades concretas**, muy por encima de la tasa base. Es la
   justificación empírica de usar features de comunidad en el modelo (M7, M15).
5. **El evento del timestep 43 degrada el modelo de forma visible.** Un F1 agregado lo habría
   ocultado; el desglose por timestep lo hace evidente y explica por qué un sistema de fraude
   necesita reentrenamiento continuo (M10, M15).

### Correcciones sobre el notebook original

| Defecto | Corrección |
|---|---|
| Bloque Módulo 9/11/12 repetido cuatro veces | Estructura lineal sin duplicados |
| `AttributeError: 'csr_array' has no attribute 'asfptype'` | `to_scipy_sparse_array(dtype=float)` (M13) |
| `df_clasess` (typo) | `df_classes` |
| Rama de error que asignaba `df = None` | `FileNotFoundError` con instrucciones accionables (M2) |
| 165 features cargadas y nunca usadas | `unir_con_features_originales` las integra (M11) |
| `H = list(G.nodes())[:80]`, nunca conexo | `muestrear_subgrafo` con cuatro estrategias documentadas (M2) |
| Betweenness anunciada y no calculada | Calculada, y medido el error de la aproximación (M6) |
| Ejecución fuera de orden, sin `pip install` | Notebook lineal y portable, con dependencias declaradas |

### Qué haría falta para llevar esto a producción

- **Un grafo cuyas relaciones crucen el tiempo.** Es la limitación de fondo de Elliptic y la
  razón por la que las GNN no lucen aquí. Un grafo bancario real la resuelve por construcción.
- **Etiquetas con su fecha de disponibilidad.** No basta con saber que algo fue fraude: hay que
  saber *cuándo se supo*, o toda la ingeniería de features vuelve a tener fuga.
- **Reentrenamiento y monitorización de deriva.** M15 mostró que el modelo se degrada cuando el
  entorno cambia.
- **Explicabilidad.** Un modelo que no justifica su decisión no se despliega en banca (M23).
- **La arquitectura híbrida del M20**: base de datos de grafos para lo local y online, Spark para
  la analítica global en batch.

### Referencias

- Weber et al. (2019). *Anti-Money Laundering in Bitcoin: Experimenting with Graph Convolutional
  Networks for Financial Forensics*. — El paper del dataset Elliptic.
- Akoglu, Tong & Koutra (2015). *Graph-based Anomaly Detection and Description: A Survey*.
- Hamilton, Ying & Leskovec (2017). *Inductive Representation Learning on Large Graphs* (GraphSAGE).
- Kipf & Welling (2017). *Semi-Supervised Classification with Graph Convolutional Networks*.
- Veličković et al. (2018). *Graph Attention Networks*.
- Xu et al. (2019). *How Powerful are Graph Neural Networks?* (GIN).
- Schlichtkrull et al. (2018). *Modeling Relational Data with Graph Convolutional Networks* (RGCN).
- Rossi et al. (2020). *Temporal Graph Networks for Deep Learning on Dynamic Graphs*.
- Ying et al. (2019). *GNNExplainer: Generating Explanations for Graph Neural Networks*.
- Blondel et al. (2008). *Fast unfolding of communities in large networks* (Louvain).
- Traag, Waltman & van Eck (2019). *From Louvain to Leiden: guaranteeing well-connected communities*.
- Clauset, Shalizi & Newman (2009). *Power-law distributions in empirical data*.
- Akoglu, McGlohon & Faloutsos (2010). *OddBall: Spotting Anomalies in Weighted Graphs*.
- Ding et al. (2019). *Deep Anomaly Detection on Attributed Networks* (DOMINANT).
- Grover & Leskovec (2016). *node2vec: Scalable Feature Learning for Networks*.
- Ou et al. (2016). *Asymmetric Transitivity Preserving Graph Embedding* (HOPE).
- Tang et al. (2015). *LINE: Large-scale Information Network Embedding*.
- Bordes et al. (2013). *Translating Embeddings for Modeling Multi-relational Data* (TransE).
- Milo et al. (2002). *Network Motifs: Simple Building Blocks of Complex Networks*.